<a href="https://colab.research.google.com/github/nehin17/network-attack-forecasting/blob/model_branch/dataset_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# =============================================================================
# CREATE NETWORK FORECASTING PROJECT STRUCTURE
# =============================================================================

import os

PROJECT_ROOT = "/content/drive/MyDrive/Network-Attack-Forecasting"

folders = [
    "notebooks",
    "src",
    "data/raw",
    "data/processed",
    "data/forecasting",
    "models",
    "results",
    "results/metrics",
    "results/figures",
]

for folder in folders:
    os.makedirs(
        os.path.join(PROJECT_ROOT, folder),
        exist_ok=True
    )

print("Project created at:")
print(PROJECT_ROOT)

print("\nStructure:")

for root, dirs, files in os.walk(PROJECT_ROOT):
    level = root.replace(PROJECT_ROOT, "").count(os.sep)
    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")

Project created at:
/content/drive/MyDrive/Network-Attack-Forecasting

Structure:
Network-Attack-Forecasting/
    results/
        metrics/
        figures/
    models/
    notebooks/
    data/
        forecasting/
        raw/
        processed/
    src/


In [5]:
# =============================================================================
# NETWORK ATTACK FORECASTING — DATASET PREPARATION
# CICIDS2017 Generated Labelled Flows / TrafficLabelling
# =============================================================================
#
# PURPOSE
# -------
# This notebook builds the canonical dataset that will later be used for:
#
#   1. Future attack prediction
#      -> Will an attack occur in the next forecast window?
#
#   2. Future attack-type prediction
#      -> What type of attack is expected?
#
#   3. Lead-time prediction
#      -> How long until the attack begins?
#
#   4. Temporal graph modelling
#      -> Which source/destination nodes or edges are likely to be involved?
#
#
# RAW DATA
# --------
# We use the 8 CICIDS2017 TrafficLabelling / GeneratedLabelledFlows CSV files.
#
# These files preserve:
#   - Timestamp
#   - Source IP
#   - Destination IP
#   - Source/Destination ports
#   - Protocol
#   - CICFlowMeter traffic features
#   - Original multiclass attack Label
#
#
# IMPORTANT
# ---------
# We will NOT immediately discard features.
# First we construct a clean canonical flow-level dataset.
#
# Forecasting windows, engineered temporal features, model-specific feature
# selection, scaling, and train/validation/test splitting will be performed
# AFTER the canonical dataset has been validated.
#
# Known raw-data issue:
# The Thursday WebAttacks CSV contains completely empty appended rows.
# These will be removed only after explicitly verifying that they are empty.
#
# =============================================================================


# =============================================================================
# CELL 1 — IMPORT LIBRARIES
# =============================================================================
#
# What we are doing:
# Importing the libraries required for dataset construction and validation.
# No dataset is loaded or modified in this cell.
#

import os
import gc
import glob
import numpy as np
import pandas as pd

from collections import Counter

print("=" * 90)
print("NETWORK FORECASTING — DATASET PREPARATION")
print("=" * 90)

print("\nPandas version :", pd.__version__)
print("NumPy version  :", np.__version__)

print("\n✓ Libraries imported successfully.")

NETWORK FORECASTING — DATASET PREPARATION

Pandas version : 2.2.3
NumPy version  : 2.1.3

✓ Libraries imported successfully.


In [6]:
# ==============================================================================================================
# RUNTIME RECOVERY — REMOUNT GOOGLE DRIVE
# ==============================================================================================================

import os

print("=" * 110)
print("CICIDS2017 — RUNTIME RECOVERY")
print("=" * 110)

# Remove stale Colab Drive mount if one exists
try:
    from google.colab import drive
    drive.flush_and_unmount()
except Exception:
    pass

# If /content/drive exists as a normal directory with stale files,
# rename it rather than deleting anything
if os.path.exists("/content/drive") and not os.path.ismount("/content/drive"):
    if os.listdir("/content/drive"):
        stale_path = "/content/drive_stale"

        if os.path.exists(stale_path):
            import shutil
            shutil.rmtree(stale_path)

        os.rename("/content/drive", stale_path)
        print("✓ Stale mountpoint moved to:", stale_path)

# Mount Drive
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

print("\n✓ Google Drive mounted.")

CICIDS2017 — RUNTIME RECOVERY
Drive not mounted, so nothing to flush and unmount.
✓ Stale mountpoint moved to: /content/drive_stale
Mounted at /content/drive

✓ Google Drive mounted.


In [7]:
# ==============================================================================================================
# VERIFY SAVED CICIDS2017 ARTIFACTS
# ==============================================================================================================

import os
import glob

PROCESSED_DIR = "/content/drive/MyDrive/Network-Attack-Forecasting/data/processed"
TEMPORAL_DIR  = "/content/drive/MyDrive/Network-Attack-Forecasting/data/temporal"

processed_files = sorted(
    glob.glob(os.path.join(PROCESSED_DIR, "*_cleaned.parquet"))
)

print("=" * 110)
print("SAVED CICIDS2017 ARTIFACT CHECK")
print("=" * 110)

print("\nProcessed Parquet files:", len(processed_files))

for i, path in enumerate(processed_files, 1):
    print(f"{i}. {os.path.basename(path)}")

print("\nTemporal artifacts:")

for filename in [
    "segment_metadata.parquet",
    "large_gap_audit.parquet",
    "flow_timeline.parquet"
]:
    path = os.path.join(TEMPORAL_DIR, filename)
    print(f"{'✓' if os.path.exists(path) else '✗'} {filename}")

print("\n" + "=" * 110)

if len(processed_files) == 8:
    print("✓ CLEANED CICIDS2017 DATA RECOVERED.")
    print("✓ NO NEED TO RE-RUN RAW CLEANING.")
else:
    print("❌ Expected 8 cleaned Parquet files but found:", len(processed_files))

print("=" * 110)

SAVED CICIDS2017 ARTIFACT CHECK

Processed Parquet files: 8
1. Friday-WorkingHours-Afternoon-DDos.pcap_ISCX_cleaned.parquet
2. Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX_cleaned.parquet
3. Friday-WorkingHours-Morning.pcap_ISCX_cleaned.parquet
4. Monday-WorkingHours.pcap_ISCX_cleaned.parquet
5. Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX_cleaned.parquet
6. Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX_cleaned.parquet
7. Tuesday-WorkingHours.pcap_ISCX_cleaned.parquet
8. Wednesday-workingHours.pcap_ISCX_cleaned.parquet

Temporal artifacts:
✓ segment_metadata.parquet
✓ large_gap_audit.parquet
✓ flow_timeline.parquet

✓ CLEANED CICIDS2017 DATA RECOVERED.
✓ NO NEED TO RE-RUN RAW CLEANING.


In [8]:
# =============================================================================
# CELL 3 — LOCATE RAW CICIDS2017 TRAFFICLABELLING FILES
# =============================================================================
#
# What we are doing:
# Finding the 8 original CICIDS2017 TrafficLabelling / GeneratedLabelledFlows
# CSV files stored in Google Drive.
#
# WHY:
# These files will be our canonical RAW source because they preserve:
#
#   - timestamps
#   - source/destination IP identities
#   - ports
#   - protocol
#   - original multiclass attack labels
#   - full CICFlowMeter feature information
#
# At this stage we are ONLY locating the files.
# We are NOT combining, cleaning, sorting, or modifying anything.
# =============================================================================


RAW_DIR = "/content/drive/MyDrive/TrafficLabelling "

print("=" * 100)
print("LOCATING CICIDS2017 TRAFFICLABELLING FILES")
print("=" * 100)

print("\nRaw data directory:")
print(repr(RAW_DIR))


# -----------------------------------------------------------------------------
# Check folder exists
# -----------------------------------------------------------------------------

assert os.path.isdir(RAW_DIR), (
    f"\nFolder not found:\n{RAW_DIR}\n\n"
    "Check the exact Google Drive folder name."
)

print("\n✓ Raw data directory exists.")


# -----------------------------------------------------------------------------
# Find CSV files
# -----------------------------------------------------------------------------

raw_files = sorted(
    glob.glob(os.path.join(RAW_DIR, "*.csv"))
)

print(f"\nCSV files found: {len(raw_files)}")


# -----------------------------------------------------------------------------
# Display files
# -----------------------------------------------------------------------------

for i, path in enumerate(raw_files, 1):

    size_mb = os.path.getsize(path) / (1024 ** 2)

    print(
        f"\n{i}. {os.path.basename(path)}"
        f"\n   Size : {size_mb:.2f} MB"
        f"\n   Path : {path}"
    )


# -----------------------------------------------------------------------------
# We expect exactly 8 CICIDS2017 TrafficLabelling files
# -----------------------------------------------------------------------------

print("\n" + "=" * 100)
print("FILE COUNT CHECK")
print("=" * 100)

if len(raw_files) == 8:

    print("\n✓ Exactly 8 CSV files found.")

else:

    print(
        f"\n❌ Expected 8 CSV files but found {len(raw_files)}."
    )

    print(
        "\nDo NOT continue until we understand why."
    )


# -----------------------------------------------------------------------------
# Save aliases used later in the notebook
# -----------------------------------------------------------------------------

glf_files = raw_files

print("\n✓ File paths stored in variable: raw_files")
print("✓ Compatibility alias created: glf_files")

LOCATING CICIDS2017 TRAFFICLABELLING FILES

Raw data directory:
'/content/drive/MyDrive/TrafficLabelling '

✓ Raw data directory exists.

CSV files found: 8

1. Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
   Size : 91.65 MB
   Path : /content/drive/MyDrive/TrafficLabelling /Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv

2. Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
   Size : 97.16 MB
   Path : /content/drive/MyDrive/TrafficLabelling /Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv

3. Friday-WorkingHours-Morning.pcap_ISCX.csv
   Size : 71.89 MB
   Path : /content/drive/MyDrive/TrafficLabelling /Friday-WorkingHours-Morning.pcap_ISCX.csv

4. Monday-WorkingHours.pcap_ISCX.csv
   Size : 256.20 MB
   Path : /content/drive/MyDrive/TrafficLabelling /Monday-WorkingHours.pcap_ISCX.csv

5. Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
   Size : 103.69 MB
   Path : /content/drive/MyDrive/TrafficLabelling /Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX

In [9]:
# =============================================================================
# CELL 4 — VALIDATE RAW DATASET SCHEMA
# =============================================================================
#
# What we are doing:
# Inspecting ONLY the column headers of all 8 CICIDS2017 TrafficLabelling files.
#
# WHY:
# Before combining millions of rows, we need to confirm that:
#
#   1. All 8 files have the same number of columns.
#   2. All 8 files use the same schema.
#   3. The columns required for our forecasting pipeline are present.
#   4. The columns required for future graph modelling are present.
#
# IMPORTANT:
# We are NOT loading the full datasets in this cell.
# nrows=0 reads only the CSV headers, so this is fast and memory-safe.
#
# We also use latin1 because the WebAttacks file contains a known
# non-UTF-8 character in some attack labels.
# =============================================================================


print("=" * 110)
print("CICIDS2017 — RAW SCHEMA VALIDATION")
print("=" * 110)


# -----------------------------------------------------------------------------
# Columns that are absolutely necessary for our final system
# -----------------------------------------------------------------------------

critical_columns = [
    "Flow ID",
    "Source IP",
    "Source Port",
    "Destination IP",
    "Destination Port",
    "Protocol",
    "Timestamp",
    "Label"
]


# -----------------------------------------------------------------------------
# Store schema information for every file
# -----------------------------------------------------------------------------

schemas = {}

for i, path in enumerate(raw_files, 1):

    filename = os.path.basename(path)

    print("\n" + "#" * 110)
    print(f"FILE {i}/8 — {filename}")
    print("#" * 110)

    # Read header only
    header = pd.read_csv(
        path,
        nrows=0,
        encoding="latin1"
    )

    # CICIDS2017 contains leading/trailing spaces in some column names.
    # We normalize ONLY the in-memory header names here.
    cleaned_columns = [
        str(col).strip()
        for col in header.columns
    ]

    schemas[filename] = cleaned_columns

    print(f"\nNumber of columns: {len(cleaned_columns)}")

    print("\nCRITICAL COLUMN CHECK:")

    for col in critical_columns:

        exists = col in cleaned_columns

        print(
            f"{'✓' if exists else '❌'} "
            f"{col:<22} : {exists}"
        )


# =============================================================================
# CHECK WHETHER ALL FILES HAVE THE SAME SCHEMA
# =============================================================================

print("\n\n" + "=" * 110)
print("CROSS-FILE SCHEMA CONSISTENCY")
print("=" * 110)

reference_filename = list(schemas.keys())[0]
reference_schema = schemas[reference_filename]

all_same_schema = all(
    schema == reference_schema
    for schema in schemas.values()
)

print("\nReference file:")
print(reference_filename)

print("\nAll files have identical schema:", all_same_schema)


# =============================================================================
# CHECK COLUMN COUNTS
# =============================================================================

column_counts = {
    filename: len(schema)
    for filename, schema in schemas.items()
}

print("\nColumn counts:")

for filename, count in column_counts.items():

    print(
        f"{filename:<65} {count}"
    )

all_85_columns = all(
    count == 85
    for count in column_counts.values()
)

print("\nAll files contain exactly 85 columns:", all_85_columns)


# =============================================================================
# CHECK CRITICAL COLUMNS ACROSS EVERY FILE
# =============================================================================

all_critical_present = all(

    all(
        col in schema
        for col in critical_columns
    )

    for schema in schemas.values()
)

print(
    "All files contain every critical column:",
    all_critical_present
)


# =============================================================================
# DISPLAY COMPLETE COLUMN LIST ONCE
# =============================================================================

print("\n" + "=" * 110)
print("COMPLETE CICIDS2017 COLUMN LIST")
print("=" * 110)

for i, col in enumerate(reference_schema, 1):

    print(
        f"{i:>2}. {col}"
    )


# =============================================================================
# FINAL RESULT
# =============================================================================

print("\n" + "=" * 110)
print("SCHEMA VALIDATION RESULT")
print("=" * 110)

checks = {

    "8 schemas inspected":
        len(schemas) == 8,

    "Identical schema across all files":
        all_same_schema,

    "Exactly 85 columns per file":
        all_85_columns,

    "All critical columns present":
        all_critical_present
}

for name, passed in checks.items():

    print(
        f"{'✓' if passed else '❌'} "
        f"{name}: {passed}"
    )


if all(checks.values()):

    print(
        "\n✓ RAW SCHEMA VALIDATION PASSED."
    )

    print(
        "The 8 files are structurally compatible for later combination."
    )

else:

    print(
        "\n❌ RAW SCHEMA VALIDATION FAILED."
    )

    print(
        "Do NOT continue to dataset construction until the issue is inspected."
    )

CICIDS2017 — RAW SCHEMA VALIDATION

##############################################################################################################
FILE 1/8 — Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
##############################################################################################################

Number of columns: 85

CRITICAL COLUMN CHECK:
✓ Flow ID                : True
✓ Source IP              : True
✓ Source Port            : True
✓ Destination IP         : True
✓ Destination Port       : True
✓ Protocol               : True
✓ Timestamp              : True
✓ Label                  : True

##############################################################################################################
FILE 2/8 — Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
##############################################################################################################

Number of columns: 85

CRITICAL COLUMN CHECK:
✓ Flow ID                : True
✓ Source IP  

In [10]:
# =============================================================================
# CELL 5 — RAW DATA INTEGRITY AND ATTACK-LABEL AUDIT
# =============================================================================
#
# What we are doing:
# Inspecting the CONTENT of all 8 raw CICIDS2017 TrafficLabelling files before
# performing any cleaning.
#
# We want to determine:
#
#   1. How many raw rows each file contains.
#   2. How many rows are completely empty.
#   3. Whether any NON-EMPTY row has a missing attack label.
#   4. The attack-label distribution in each file.
#   5. The global attack-label distribution across CICIDS2017.
#   6. Whether the known Thursday WebAttacks blank-row issue is exactly what
#      we previously observed.
#
# IMPORTANT:
#   - Nothing is modified.
#   - Nothing is saved.
#   - Empty rows are counted, NOT removed yet.
#   - Labels are inspected in their raw form.
#
# We use encoding="latin1" because the WebAttacks file contains a non-UTF-8
# dash character in its raw attack labels.
# =============================================================================


from collections import Counter

print("=" * 110)
print("CICIDS2017 — RAW DATA INTEGRITY & LABEL AUDIT")
print("=" * 110)


# =============================================================================
# STORAGE FOR RESULTS
# =============================================================================

file_audit_results = []

global_raw_labels = Counter()

global_raw_rows = 0
global_empty_rows = 0
global_real_rows = 0
global_nonempty_missing_labels = 0


# =============================================================================
# AUDIT EACH RAW FILE
# =============================================================================

for i, path in enumerate(raw_files, 1):

    filename = os.path.basename(path)

    print("\n" + "#" * 110)
    print(f"FILE {i}/8 — {filename}")
    print("#" * 110)

    # -------------------------------------------------------------------------
    # Load one raw file
    # -------------------------------------------------------------------------

    df_temp = pd.read_csv(
        path,
        encoding="latin1",
        low_memory=False
    )

    # Normalize column NAMES in memory only.
    # This does NOT modify the original CSV.
    df_temp.columns = (
        df_temp.columns
        .astype(str)
        .str.strip()
    )

    raw_rows = len(df_temp)

    # -------------------------------------------------------------------------
    # Identify completely empty rows
    # -------------------------------------------------------------------------

    empty_mask = df_temp.isna().all(axis=1)

    empty_rows = int(empty_mask.sum())

    nonempty_mask = ~empty_mask

    real_rows = int(nonempty_mask.sum())

    # -------------------------------------------------------------------------
    # Check labels ONLY among non-empty rows
    # -------------------------------------------------------------------------

    missing_label_nonempty = int(
        df_temp.loc[nonempty_mask, "Label"]
        .isna()
        .sum()
    )

    # -------------------------------------------------------------------------
    # Raw label distribution
    #
    # IMPORTANT:
    # We exclude completely empty rows from this distribution.
    # We do NOT normalize the attack names yet.
    # -------------------------------------------------------------------------

    raw_labels = (
        df_temp.loc[
            nonempty_mask & df_temp["Label"].notna(),
            "Label"
        ]
        .astype(str)
        .str.strip()
    )

    label_counts = Counter(
        raw_labels.value_counts().to_dict()
    )

    # -------------------------------------------------------------------------
    # Update global counters
    # -------------------------------------------------------------------------

    global_raw_rows += raw_rows
    global_empty_rows += empty_rows
    global_real_rows += real_rows
    global_nonempty_missing_labels += missing_label_nonempty

    global_raw_labels.update(label_counts)

    # -------------------------------------------------------------------------
    # Print file-level results
    # -------------------------------------------------------------------------

    print(f"\nRaw rows                     : {raw_rows:,}")
    print(f"Completely empty rows        : {empty_rows:,}")
    print(f"Non-empty rows               : {real_rows:,}")
    print(f"Non-empty rows missing Label : {missing_label_nonempty:,}")

    print(f"\nUnique raw labels            : {len(label_counts)}")

    print("\nRAW LABEL DISTRIBUTION:")

    for label, count in label_counts.most_common():

        pct = (
            count / real_rows * 100
            if real_rows > 0
            else 0
        )

        print(
            f"{repr(label):<48}"
            f"{count:>12,}"
            f" ({pct:7.3f}%)"
        )

    # -------------------------------------------------------------------------
    # Store summary
    # -------------------------------------------------------------------------

    file_audit_results.append({

        "File": filename,

        "Raw_Rows": raw_rows,

        "Empty_Rows": empty_rows,

        "Nonempty_Rows": real_rows,

        "Nonempty_Missing_Label":
            missing_label_nonempty,

        "Unique_Raw_Labels":
            len(label_counts)
    })

    # Free memory before loading next large CSV
    del df_temp
    gc.collect()


# =============================================================================
# GLOBAL DATASET SUMMARY
# =============================================================================

print("\n\n" + "=" * 110)
print("GLOBAL RAW DATASET SUMMARY")
print("=" * 110)

print(f"\nFiles                         : {len(raw_files)}")
print(f"Raw rows across files         : {global_raw_rows:,}")
print(f"Completely empty rows         : {global_empty_rows:,}")
print(f"Real / non-empty rows         : {global_real_rows:,}")
print(
    f"Non-empty rows missing Label  : "
    f"{global_nonempty_missing_labels:,}"
)


# =============================================================================
# GLOBAL RAW LABEL DISTRIBUTION
# =============================================================================

print("\n" + "=" * 110)
print("GLOBAL RAW LABEL DISTRIBUTION")
print("=" * 110)

print(
    f"\nUnique raw labels: "
    f"{len(global_raw_labels)}\n"
)

for label, count in global_raw_labels.most_common():

    pct = count / global_real_rows * 100

    print(
        f"{repr(label):<48}"
        f"{count:>12,}"
        f" ({pct:7.3f}%)"
    )


# =============================================================================
# FILE SUMMARY TABLE
# =============================================================================

file_audit_df = pd.DataFrame(
    file_audit_results
)

print("\n" + "=" * 110)
print("FILE-LEVEL AUDIT SUMMARY")
print("=" * 110)

display(file_audit_df)


# =============================================================================
# EMPTY-ROW LOCATION CHECK
# =============================================================================
#
# We expect the known malformed/blank-row issue to occur only in the
# Thursday WebAttacks file.
# =============================================================================

files_with_empty_rows = (
    file_audit_df.loc[
        file_audit_df["Empty_Rows"] > 0,
        ["File", "Empty_Rows"]
    ]
)

print("\n" + "=" * 110)
print("FILES CONTAINING COMPLETELY EMPTY ROWS")
print("=" * 110)

if len(files_with_empty_rows) == 0:

    print("\nNo completely empty rows found.")

else:

    display(files_with_empty_rows)


# =============================================================================
# IMPORTANT EXPECTED COUNTS
# =============================================================================

print("\n" + "=" * 110)
print("EXPECTED DATASET CHECKS")
print("=" * 110)

checks = {

    "Exactly 8 raw files":
        len(raw_files) == 8,

    "Exactly 2,830,743 real flows":
        global_real_rows == 2_830_743,

    "No non-empty row has missing Label":
        global_nonempty_missing_labels == 0,

    "Exactly 15 raw attack/benign labels":
        len(global_raw_labels) == 15,

    "BENIGN label exists":
        "BENIGN" in global_raw_labels,

    "DDoS label exists":
        "DDoS" in global_raw_labels,

    "PortScan label exists":
        "PortScan" in global_raw_labels,

    "Heartbleed label exists":
        "Heartbleed" in global_raw_labels
}


for name, passed in checks.items():

    print(
        f"{'✓' if passed else '❌'} "
        f"{name}: {passed}"
    )


# =============================================================================
# KNOWN WEBATTACKS EMPTY-ROW CHECK
# =============================================================================

webattack_rows = file_audit_df[
    file_audit_df["File"]
    .str.contains(
        "WebAttacks",
        case=False,
        na=False
    )
]

print("\n" + "=" * 110)
print("WEBATTACKS RAW-FILE CHECK")
print("=" * 110)

if len(webattack_rows) == 1:

    web_empty = int(
        webattack_rows.iloc[0]["Empty_Rows"]
    )

    web_real = int(
        webattack_rows.iloc[0]["Nonempty_Rows"]
    )

    print(
        f"\nWebAttacks completely empty rows : "
        f"{web_empty:,}"
    )

    print(
        f"WebAttacks real rows             : "
        f"{web_real:,}"
    )

    print(
        "\nExpected from our previous investigation:"
    )

    print(
        "  Empty rows : 288,602"
    )

    print(
        "  Real rows  : 170,366"
    )

    print(
        "\nMatches previous empty-row finding:",
        web_empty == 288_602
    )

    print(
        "Matches expected real-row count:",
        web_real == 170_366
    )

else:

    print(
        "\n❌ Could not uniquely identify "
        "the WebAttacks file."
    )


# =============================================================================
# FINAL STATUS
# =============================================================================

print("\n" + "=" * 110)
print("RAW INTEGRITY AUDIT STATUS")
print("=" * 110)

if all(checks.values()):

    print(
        "\n✓ CORE RAW-DATA INTEGRITY CHECKS PASSED."
    )

    print(
        "The dataset is ready for controlled cleaning."
    )

else:

    print(
        "\n❌ One or more raw-data checks failed."
    )

    print(
        "Do NOT clean or combine the files yet."
    )

CICIDS2017 — RAW DATA INTEGRITY & LABEL AUDIT

##############################################################################################################
FILE 1/8 — Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
##############################################################################################################

Raw rows                     : 225,745
Completely empty rows        : 0
Non-empty rows               : 225,745
Non-empty rows missing Label : 0

Unique raw labels            : 2

RAW LABEL DISTRIBUTION:
'DDoS'                                               128,027 ( 56.713%)
'BENIGN'                                              97,718 ( 43.287%)

##############################################################################################################
FILE 2/8 — Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
##############################################################################################################

Raw rows                     : 286,

,File,Raw_Rows,Empty_Rows,Nonempty_Rows,Nonempty_Missing_Label,Unique_Raw_Labels
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,225745,0,225745,0,2
1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,286467,0,286467,0,2
2,Friday-WorkingHours-Morning.pcap_ISCX.csv,191033,0,191033,0,2
3,Monday-WorkingHours.pcap_ISCX.csv,529918,0,529918,0,1
4,Thursday-WorkingHours-Afternoon-Infilteration....,288602,0,288602,0,2
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,458968,288602,170366,0,4
6,Tuesday-WorkingHours.pcap_ISCX.csv,445909,0,445909,0,3
7,Wednesday-workingHours.pcap_ISCX.csv,692703,0,692703,0,6



FILES CONTAINING COMPLETELY EMPTY ROWS


,File,Empty_Rows
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,288602



EXPECTED DATASET CHECKS
✓ Exactly 8 raw files: True
✓ Exactly 2,830,743 real flows: True
✓ No non-empty row has missing Label: True
✓ Exactly 15 raw attack/benign labels: True
✓ BENIGN label exists: True
✓ DDoS label exists: True
✓ PortScan label exists: True
✓ Heartbleed label exists: True

WEBATTACKS RAW-FILE CHECK

WebAttacks completely empty rows : 288,602
WebAttacks real rows             : 170,366

Expected from our previous investigation:
  Empty rows : 288,602
  Real rows  : 170,366

Matches previous empty-row finding: True
Matches expected real-row count: True

RAW INTEGRITY AUDIT STATUS

✓ CORE RAW-DATA INTEGRITY CHECKS PASSED.
The dataset is ready for controlled cleaning.


In [11]:
# =============================================================================
# CELL 6 — CLEAN CANONICAL FLOW-LEVEL FILES
# =============================================================================
#
# For EACH of the 8 raw CSV files independently, we:
#
#   1. Load ONE raw CSV file.
#   2. Strip whitespace from column names.
#   3. Remove ONLY rows that are completely empty.
#   4. Normalize malformed Web Attack label names.
#   5. Preserve the multiclass attack label as "Attack_Type".
#   6. Create a binary attack indicator:
#
#          Attack_Label = 0  -> BENIGN
#          Attack_Label = 1  -> any attack
#
#   7. Preserve the source filename for traceability.
#   8. Preserve the original timestamp string.
#   9. Parse timestamps using the mixed CICIDS2017 timestamp formats.
#  10. Verify that every timestamp parsed successfully.
#  11. Save the cleaned file to Google Drive.
#  12. Delete the dataframe from RAM before processing the next file.
#
#
# IMPORTANT DIFFERENCE FROM OLD CELL 6:
#
#
# We DO NOT create canonical_df in this cell.
#
# All original CICFlowMeter features are preserved.
# No model features are selected yet.
# No chronological sorting is performed yet.
# No temporal segments are created yet.
# No forecasting samples are created yet.
#
# =============================================================================


import os
import gc
import pandas as pd
import numpy as np


# =============================================================================
# 1. VERIFY INPUT FILES
# =============================================================================

assert "raw_files" in globals(), (
    "raw_files does not exist. Run the earlier file-location cell first."
)

assert len(raw_files) == 8, (
    f"Expected 8 raw CICIDS2017 files, found {len(raw_files)}."
)


# =============================================================================
# 2. CREATE OUTPUT DIRECTORY
# =============================================================================

PROJECT_ROOT = "/content/drive/MyDrive/Network-Attack-Forecasting"

PROCESSED_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed"
)

os.makedirs(
    PROCESSED_DIR,
    exist_ok=True
)


print("=" * 110)
print("CICIDS2017 — RAM-SAFE CANONICAL FLOW CONSTRUCTION")
print("=" * 110)

print("\nRaw files:")
print(len(raw_files))

print("\nProcessed files will be saved to:")
print(PROCESSED_DIR)


# =============================================================================
# 3. LABEL NORMALIZATION FUNCTION
# =============================================================================
#
# CICIDS2017 contains malformed/non-standard text in some Web Attack labels.
#
# Instead of relying on the exact corrupted characters, we normalize labels
# using their meaningful attack-name components.
#
# Examples ultimately become:
#
#   Web Attack - Brute Force
#   Web Attack - XSS
#   Web Attack - Sql Injection
#
# All other labels are preserved after whitespace cleaning.
#
# =============================================================================

def normalize_attack_label(value):

    if pd.isna(value):
        return value

    label = str(value).strip()

    lower = label.lower()

    # ---------------------------------------------------------
    # Web Attack labels
    # ---------------------------------------------------------

    if "web attack" in lower:

        if "brute force" in lower:
            return "Web Attack - Brute Force"

        if "xss" in lower:
            return "Web Attack - XSS"

        if "sql injection" in lower:
            return "Web Attack - Sql Injection"

    # ---------------------------------------------------------
    # Everything else
    # ---------------------------------------------------------

    return label


# =============================================================================
# 4. ROBUST CSV READER
# =============================================================================
#
# One CICIDS2017 file previously caused UTF-8 decoding problems.
#
# We therefore try UTF-8 first and fall back to Windows-1252.
#
# =============================================================================

def read_cicids_file(path):

    try:

        df = pd.read_csv(
            path,
            encoding="utf-8",
            low_memory=False
        )

        encoding_used = "utf-8"

    except UnicodeDecodeError:

        df = pd.read_csv(
            path,
            encoding="cp1252",
            low_memory=False
        )

        encoding_used = "cp1252"

    return df, encoding_used


# =============================================================================
# 5. STORAGE FOR SMALL AUDIT INFORMATION ONLY
# =============================================================================
#
# IMPORTANT:
#
# We store only tiny summary dictionaries here.
# We NEVER store the full dataframes.
#
# =============================================================================

processing_summary = []

processed_files = []


# =============================================================================
# 6. PROCESS ONE FILE AT A TIME
# =============================================================================

for file_number, path in enumerate(raw_files, start=1):

    filename = os.path.basename(path)

    print("\n\n")
    print("#" * 110)
    print(
        f"PROCESSING FILE {file_number}/{len(raw_files)}"
    )
    print(filename)
    print("#" * 110)


    # =========================================================================
    # 6A. LOAD ONLY THIS FILE
    # =========================================================================

    df, encoding_used = read_cicids_file(path)

    raw_rows = len(df)
    raw_columns = len(df.columns)

    print(f"\nEncoding used : {encoding_used}")
    print(f"Raw rows      : {raw_rows:,}")
    print(f"Raw columns   : {raw_columns}")


    # =========================================================================
    # 6B. CLEAN COLUMN NAMES
    # =========================================================================

    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
    )


    # =========================================================================
    # 6C. VERIFY REQUIRED COLUMNS
    # =========================================================================

    required_columns = [
        "Flow ID",
        "Source IP",
        "Source Port",
        "Destination IP",
        "Destination Port",
        "Protocol",
        "Timestamp",
        "Label"
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in df.columns
    ]

    assert len(missing_columns) == 0, (
        f"{filename} is missing required columns: "
        f"{missing_columns}"
    )


    # =========================================================================
    # 6D. REMOVE ONLY COMPLETELY EMPTY ROWS
    # =========================================================================
    #
    # We DO NOT drop rows merely because one feature is missing.
    #
    # Only rows where EVERY column is NaN are removed.
    #
    # =========================================================================

    rows_before_empty_removal = len(df)

    df = (
        df
        .dropna(
            how="all"
        )
        .reset_index(drop=True)
    )

    empty_rows_removed = (
        rows_before_empty_removal
        - len(df)
    )

    print(
        f"Completely empty rows removed : "
        f"{empty_rows_removed:,}"
    )


    # =========================================================================
    # 6E. NORMALIZE ATTACK LABEL
    # =========================================================================

    df["Attack_Type"] = (
        df["Label"]
        .apply(normalize_attack_label)
    )


    # =========================================================================
    # 6F. CREATE BINARY ATTACK LABEL
    # =========================================================================
    #
    # BENIGN -> 0
    # ANY ATTACK -> 1
    #
    # =========================================================================

    df["Attack_Label"] = (
        df["Attack_Type"]
        .str.upper()
        .ne("BENIGN")
        .astype("int8")
    )


    # =========================================================================
    # 6G. PRESERVE SOURCE FILE
    # =========================================================================

    df["Source_File"] = filename


    # =========================================================================
    # 6H. PRESERVE ORIGINAL TIMESTAMP
    # =========================================================================
    #
    # This lets us audit or reconstruct timestamp parsing later if necessary.
    #
    # =========================================================================

    df["Original_Timestamp"] = (
        df["Timestamp"]
        .astype("string")
        .str.strip()
    )


    # =========================================================================
    # 6I. ROBUST TIMESTAMP PARSING
    # =========================================================================
    #
    # CICIDS2017 contains mixed timestamp representations.
    #
    # Examples include:
    #
    #       7/7/2017 3:30
    #
    # and:
    #
    #       03/07/2017 11:04:03
    #
    # format="mixed" allows pandas to parse both.
    #
    # dayfirst=True is required for this dataset.
    #
    # =========================================================================

    df["Timestamp"] = pd.to_datetime(
        df["Original_Timestamp"],
        format="mixed",
        dayfirst=True,
        errors="coerce"
    )


    # =========================================================================
    # 6J. TIMESTAMP VALIDATION
    # =========================================================================

    failed_timestamps = (
        df["Timestamp"]
        .isna()
        .sum()
    )

    valid_timestamps = (
        df["Timestamp"]
        .notna()
        .sum()
    )

    print(
        f"Valid timestamps              : "
        f"{valid_timestamps:,}"
    )

    print(
        f"Failed timestamps             : "
        f"{failed_timestamps:,}"
    )


    assert failed_timestamps == 0, (
        f"Timestamp parsing failed for "
        f"{failed_timestamps:,} rows in {filename}."
    )


    # =========================================================================
    # 6K. CREATE DATE
    # =========================================================================

    df["Date"] = (
        df["Timestamp"]
        .dt.date
    )


    # =========================================================================
    # 6L. FILE-LEVEL LABEL AUDIT
    # =========================================================================

    attack_count = int(
        df["Attack_Label"]
        .sum()
    )

    benign_count = (
        len(df)
        - attack_count
    )

    unique_attack_types = (
        df["Attack_Type"]
        .nunique()
    )

    start_time = (
        df["Timestamp"]
        .min()
    )

    end_time = (
        df["Timestamp"]
        .max()
    )


    print(f"\nClean rows      : {len(df):,}")
    print(f"Benign flows    : {benign_count:,}")
    print(f"Attack flows    : {attack_count:,}")
    print(f"Unique labels   : {unique_attack_types}")

    print("\nTemporal range:")
    print("Start :", start_time)
    print("End   :", end_time)


    print("\nLabel distribution:")

    print(
        df["Attack_Type"]
        .value_counts()
    )


    # =========================================================================
    # 6M. SAVE CLEANED FILE
    # =========================================================================
    #
    # Parquet is preferable here because:
    #
    #   - it is substantially smaller than CSV
    #   - datetime types are preserved
    #   - numeric dtypes are preserved
    #   - later we can load only selected columns
    #
    # =========================================================================

    safe_filename = (
        os.path.splitext(filename)[0]
        + "_cleaned.parquet"
    )

    output_path = os.path.join(
        PROCESSED_DIR,
        safe_filename
    )


    df.to_parquet(
        output_path,
        index=False,
        compression="snappy"
    )


    processed_files.append(
        output_path
    )


    output_size_mb = (
        os.path.getsize(output_path)
        / 1024**2
    )


    print("\n✓ Saved:")
    print(output_path)

    print(
        f"Saved file size : "
        f"{output_size_mb:.2f} MB"
    )


    # =========================================================================
    # 6N. STORE ONLY SMALL SUMMARY INFORMATION
    # =========================================================================

    processing_summary.append({

        "Source_File":
            filename,

        "Encoding":
            encoding_used,

        "Raw_Rows":
            raw_rows,

        "Clean_Rows":
            len(df),

        "Empty_Rows_Removed":
            empty_rows_removed,

        "Benign_Flows":
            benign_count,

        "Attack_Flows":
            attack_count,

        "Unique_Attack_Types":
            unique_attack_types,

        "Failed_Timestamps":
            failed_timestamps,

        "Start_Time":
            start_time,

        "End_Time":
            end_time,

        "Saved_Size_MB":
            round(
                output_size_mb,
                2
            )
    })


    # =========================================================================
    # 6O. DELETE DATAFRAME FROM RAM
    # =========================================================================
    #
    # THIS IS THE CRITICAL RAM-SAVING STEP.
    #
    # Once the cleaned file has been written to Drive, we do not need the
    # dataframe anymore.
    #
    # =========================================================================

    del df

    gc.collect()

    print(
        "\n✓ Dataframe deleted from RAM "
        "before processing next file."
    )


# =============================================================================
# 7. CREATE SMALL PROCESSING SUMMARY
# =============================================================================

processing_summary_df = pd.DataFrame(
    processing_summary
)


print("\n\n")
print("=" * 110)
print("RAM-SAFE PROCESSING SUMMARY")
print("=" * 110)

display(
    processing_summary_df
)


# =============================================================================
# 8. GLOBAL COUNTS FROM SUMMARIES
# =============================================================================

total_raw_rows = (
    processing_summary_df["Raw_Rows"]
    .sum()
)

total_clean_rows = (
    processing_summary_df["Clean_Rows"]
    .sum()
)

total_empty_removed = (
    processing_summary_df["Empty_Rows_Removed"]
    .sum()
)

total_benign = (
    processing_summary_df["Benign_Flows"]
    .sum()
)

total_attacks = (
    processing_summary_df["Attack_Flows"]
    .sum()
)

total_failed_timestamps = (
    processing_summary_df["Failed_Timestamps"]
    .sum()
)


print("\n" + "=" * 110)
print("GLOBAL CLEANING TOTALS")
print("=" * 110)

print(
    f"\nRaw rows             : "
    f"{total_raw_rows:,}"
)

print(
    f"Clean rows           : "
    f"{total_clean_rows:,}"
)

print(
    f"Empty rows removed   : "
    f"{total_empty_removed:,}"
)

print(
    f"Benign flows         : "
    f"{total_benign:,}"
)

print(
    f"Attack flows         : "
    f"{total_attacks:,}"
)

print(
    f"Failed timestamps    : "
    f"{total_failed_timestamps:,}"
)

print(
    f"Processed files saved: "
    f"{len(processed_files)}"
)


# =============================================================================
# 9. FINAL VALIDATION
# =============================================================================
#
# These values were established by our previous complete audit.
#
# =============================================================================

checks = {

    "Exactly 8 processed files":
        len(processed_files) == 8,

    "Expected total flow count":
        total_clean_rows == 2_830_743,

    "Expected attack flow count":
        total_attacks == 557_646,

    "Expected benign flow count":
        total_benign == 2_273_097,

    "All timestamps parsed":
        total_failed_timestamps == 0,

    "Every processed file exists":
        all(
            os.path.exists(path)
            for path in processed_files
        )
}


print("\n" + "=" * 110)
print("FINAL CELL 6 VALIDATION")
print("=" * 110)

for name, result in checks.items():

    print(
        f"{'✓' if result else '❌'} "
        f"{name}: {result}"
    )


print("\n" + "=" * 110)

if all(checks.values()):

    print("✓ CELL 6 COMPLETED SUCCESSFULLY.")
    print()
    print(
        "All 8 CICIDS2017 files were cleaned "
        "and saved independently."
    )

    print(
        "The full 2.83-million-row dataset was "
        "NEVER held in RAM at once."
    )

    print()
    print(
        "IMPORTANT: canonical_df has intentionally "
        "NOT been created."
    )

    print()
    print(
        "NEXT: Cell 7 will validate the processed "
        "files directly from disk."
    )

else:

    print("❌ CELL 6 VALIDATION FAILED.")
    print("Do NOT continue to Cell 7.")

print("=" * 110)
# We are NOT yet:
#   - selecting model features
#   - scaling features
#   - creating minute-level aggregation
#   - creating forecasting windows
#   - creating lead-time targets
#   - splitting train/validation/test
#
# Those steps come AFTER the canonical flow dataset is validated.
#
# Expected final number of rows:
#
#       2,830,743
#
# =============================================================================


print("=" * 110)
print("BUILDING CLEAN CANONICAL CICIDS2017 FLOW DATASET")
print("=" * 110)


# =============================================================================
# STORAGE FOR CLEANED FILES
# =============================================================================

cleaned_parts = []


# =============================================================================
# PROCESS EACH RAW FILE
# =============================================================================

for i, path in enumerate(raw_files, 1):

    filename = os.path.basename(path)

    print("\n" + "#" * 110)
    print(f"PROCESSING FILE {i}/8")
    print(filename)
    print("#" * 110)


    # -------------------------------------------------------------------------
    # Load raw file
    # -------------------------------------------------------------------------

    df_part = pd.read_csv(
        path,
        encoding="latin1",
        low_memory=False
    )

    rows_before = len(df_part)


    # -------------------------------------------------------------------------
    # Normalize COLUMN NAMES
    # -------------------------------------------------------------------------

    df_part.columns = (
        df_part.columns
        .astype(str)
        .str.strip()
    )


    # -------------------------------------------------------------------------
    # Remove completely empty rows
    #
    # IMPORTANT:
    # We already proved in Cell 5 that the 288,602 problematic WebAttack rows
    # are completely empty.
    #
    # We are NOT deleting rows merely because one feature is missing.
    # -------------------------------------------------------------------------

    empty_mask = df_part.isna().all(axis=1)

    empty_removed = int(empty_mask.sum())

    df_part = (
        df_part.loc[~empty_mask]
        .copy()
    )

    rows_after = len(df_part)


    # -------------------------------------------------------------------------
    # Clean raw attack label strings
    # -------------------------------------------------------------------------

    df_part["Label"] = (
        df_part["Label"]
        .astype(str)
        .str.strip()
    )


    # -------------------------------------------------------------------------
    # Normalize malformed Web Attack labels
    #
    # Raw labels contain the Windows-1252/Latin-1 character \x96.
    #
    # Example:
    #
    #   Web Attack \x96 Brute Force
    #
    # becomes:
    #
    #   Web Attack - Brute Force
    # -------------------------------------------------------------------------

    df_part["Label"] = (
        df_part["Label"]
        .str.replace(
            "\x96",
            "-",
            regex=False
        )
        .str.replace(
            "  ",
            " ",
            regex=False
        )
        .str.strip()
    )


    # -------------------------------------------------------------------------
    # Rename multiclass target
    #
    # Attack_Type will contain:
    #
    # BENIGN
    # DDoS
    # PortScan
    # DoS Hulk
    # Bot
    # etc.
    # -------------------------------------------------------------------------

    df_part = df_part.rename(
        columns={
            "Label": "Attack_Type"
        }
    )


    # -------------------------------------------------------------------------
    # Create binary attack target
    #
    # BENIGN -> 0
    # ATTACK -> 1
    # -------------------------------------------------------------------------

    df_part["Attack_Label"] = (
        df_part["Attack_Type"]
        .ne("BENIGN")
        .astype(np.int8)
    )


    # -------------------------------------------------------------------------
    # Preserve source file identity
    #
    # This is useful for:
    #   - debugging
    #   - auditing
    #   - tracing flows back to original CICIDS2017 file
    # -------------------------------------------------------------------------

    df_part["Source_File"] = filename


    # -------------------------------------------------------------------------
    # File-level validation
    # -------------------------------------------------------------------------

    assert df_part["Attack_Type"].notna().all(), (
        f"Missing Attack_Type found in {filename}"
    )

    assert df_part["Attack_Label"].isin([0, 1]).all(), (
        f"Invalid Attack_Label found in {filename}"
    )


    print(f"\nRows before cleaning : {rows_before:,}")
    print(f"Empty rows removed   : {empty_removed:,}")
    print(f"Rows after cleaning  : {rows_after:,}")

    print(
        f"Unique attack types  : "
        f"{df_part['Attack_Type'].nunique()}"
    )

    print(
        f"Attack flows         : "
        f"{int(df_part['Attack_Label'].sum()):,}"
    )

    print(
        f"Benign flows         : "
        f"{int((df_part['Attack_Label'] == 0).sum()):,}"
    )


    # -------------------------------------------------------------------------
    # Store cleaned part
    # -------------------------------------------------------------------------

    cleaned_parts.append(df_part)


# =============================================================================
# COMBINE ALL 8 CLEANED FILES
# =============================================================================

print("\n\n" + "=" * 110)
print("COMBINING CLEANED FILES")
print("=" * 110)

canonical_df = pd.concat(
    cleaned_parts,
    ignore_index=True
)

print(
    f"\nCombined dataframe shape: "
    f"{canonical_df.shape}"
)


# Free list references after concatenation
del cleaned_parts
gc.collect()


# =============================================================================
# BASIC VALIDATION
# =============================================================================

print("\n" + "=" * 110)
print("CANONICAL DATASET — BASIC VALIDATION")
print("=" * 110)

print(
    f"\nTotal flows       : "
    f"{len(canonical_df):,}"
)

print(
    f"Total columns     : "
    f"{canonical_df.shape[1]}"
)

print(
    f"Unique attack types: "
    f"{canonical_df['Attack_Type'].nunique()}"
)

print(
    f"Attack flows      : "
    f"{int(canonical_df['Attack_Label'].sum()):,}"
)

print(
    f"Benign flows      : "
    f"{int((canonical_df['Attack_Label'] == 0).sum()):,}"
)


# =============================================================================
# ATTACK TYPE DISTRIBUTION
# =============================================================================

print("\n" + "=" * 110)
print("NORMALIZED ATTACK TYPE DISTRIBUTION")
print("=" * 110)

attack_distribution = (
    canonical_df["Attack_Type"]
    .value_counts()
)

for label, count in attack_distribution.items():

    pct = (
        count /
        len(canonical_df) *
        100
    )

    print(
        f"{label:<40}"
        f"{count:>12,}"
        f" ({pct:7.3f}%)"
    )


# =============================================================================
# BINARY TARGET DISTRIBUTION
# =============================================================================

print("\n" + "=" * 110)
print("BINARY ATTACK LABEL DISTRIBUTION")
print("=" * 110)

binary_distribution = (
    canonical_df["Attack_Label"]
    .value_counts()
    .sort_index()
)

for label, count in binary_distribution.items():

    name = (
        "BENIGN"
        if label == 0
        else "ATTACK"
    )

    pct = (
        count /
        len(canonical_df) *
        100
    )

    print(
        f"{label} ({name:<6}) "
        f"{count:>12,}"
        f" ({pct:7.3f}%)"
    )


# =============================================================================
# FINAL SANITY CHECKS
# =============================================================================

print("\n" + "=" * 110)
print("CANONICAL DATASET SANITY CHECKS")
print("=" * 110)

checks = {

    "Exactly 2,830,743 flows":
        len(canonical_df) == 2_830_743,

    "Exactly 15 attack categories":
        canonical_df["Attack_Type"].nunique() == 15,

    "No missing Attack_Type":
        canonical_df["Attack_Type"].isna().sum() == 0,

    "Binary target contains only 0/1":
        canonical_df["Attack_Label"]
        .isin([0, 1])
        .all(),

    "BENIGN maps only to 0":
        (
            canonical_df.loc[
                canonical_df["Attack_Type"] == "BENIGN",
                "Attack_Label"
            ] == 0
        ).all(),

    "Every attack maps to 1":
        (
            canonical_df.loc[
                canonical_df["Attack_Type"] != "BENIGN",
                "Attack_Label"
            ] == 1
        ).all(),

    "All 8 source files represented":
        canonical_df["Source_File"].nunique() == 8
}


for name, passed in checks.items():

    print(
        f"{'✓' if passed else '❌'} "
        f"{name}: {passed}"
    )


# =============================================================================
# PREVIEW IMPORTANT COLUMNS
# =============================================================================

print("\n" + "=" * 110)
print("CANONICAL DATASET PREVIEW")
print("=" * 110)

preview_columns = [
    "Timestamp",
    "Source IP",
    "Source Port",
    "Destination IP",
    "Destination Port",
    "Protocol",
    "Attack_Type",
    "Attack_Label",
    "Source_File"
]

display(
    canonical_df[
        preview_columns
    ].head(10)
)


# =============================================================================
# FINAL STATUS
# =============================================================================

print("\n" + "=" * 110)

if all(checks.values()):

    print(
        "✓ CANONICAL FLOW DATASET SUCCESSFULLY CONSTRUCTED"
    )

    print(
        "✓ Raw multiclass attack information preserved."
    )

    print(
        "✓ Binary attack indicator created."
    )

    print(
        "✓ Full CICFlowMeter feature space preserved."
    )

    print(
        "✓ Source/destination identity preserved for future graph modelling."
    )

else:

    print(
        "❌ CANONICAL DATASET VALIDATION FAILED"
    )

    print(
        "Do NOT continue until the failed checks are investigated."
    )

print("=" * 110)

CICIDS2017 — RAM-SAFE CANONICAL FLOW CONSTRUCTION

Raw files:
8

Processed files will be saved to:
/content/drive/MyDrive/Network-Attack-Forecasting/data/processed



##############################################################################################################
PROCESSING FILE 1/8
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
##############################################################################################################

Encoding used : utf-8
Raw rows      : 225,745
Raw columns   : 85
Completely empty rows removed : 0
Valid timestamps              : 225,745
Failed timestamps             : 0

Clean rows      : 225,745
Benign flows    : 97,718
Attack flows    : 128,027
Unique labels   : 2

Temporal range:
Start : 2017-07-07 03:30:00
End   : 2017-07-07 05:02:00

Label distribution:
Attack_Type
DDoS      128027
BENIGN     97718
Name: count, dtype: int64

✓ Saved:
/content/drive/MyDrive/Network-Attack-Forecasting/data/processed/Friday-WorkingHours-Afternoon

,Source_File,Encoding,Raw_Rows,Clean_Rows,Empty_Rows_Removed,Benign_Flows,Attack_Flows,Unique_Attack_Types,Failed_Timestamps,Start_Time,End_Time,Saved_Size_MB
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,utf-8,225745,225745,0,97718,128027,2,0,2017-07-07 03:30:00,2017-07-07 05:02:00,30.24
1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,utf-8,286467,286467,0,127537,158930,2,0,2017-07-07 01:00:00,2017-07-07 03:29:00,21.86
2,Friday-WorkingHours-Morning.pcap_ISCX.csv,utf-8,191033,191033,0,189067,1966,2,0,2017-07-07 08:59:00,2017-07-07 12:59:00,25.59
3,Monday-WorkingHours.pcap_ISCX.csv,utf-8,529918,529918,0,529918,0,1,0,2017-07-03 01:00:01,2017-07-03 12:59:58,80.33
4,Thursday-WorkingHours-Afternoon-Infilteration....,utf-8,288602,288602,0,288566,36,2,0,2017-07-06 01:00:00,2017-07-06 05:04:00,32.12
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,cp1252,458968,170366,288602,168186,2180,4,0,2017-07-06 08:59:00,2017-07-06 12:59:00,23.00
6,Tuesday-WorkingHours.pcap_ISCX.csv,utf-8,445909,445909,0,432074,13835,3,0,2017-07-04 01:00:00,2017-07-04 12:59:00,63.39
7,Wednesday-workingHours.pcap_ISCX.csv,utf-8,692703,692703,0,440031,252672,6,0,2017-07-05 01:00:00,2017-07-05 12:59:00,90.07



GLOBAL CLEANING TOTALS

Raw rows             : 3,119,345
Clean rows           : 2,830,743
Empty rows removed   : 288,602
Benign flows         : 2,273,097
Attack flows         : 557,646
Failed timestamps    : 0
Processed files saved: 8

FINAL CELL 6 VALIDATION
✓ Exactly 8 processed files: True
✓ Expected total flow count: True
✓ Expected attack flow count: True
✓ Expected benign flow count: True
✓ All timestamps parsed: True
✓ Every processed file exists: True

✓ CELL 6 COMPLETED SUCCESSFULLY.

All 8 CICIDS2017 files were cleaned and saved independently.
The full 2.83-million-row dataset was NEVER held in RAM at once.

IMPORTANT: canonical_df has intentionally NOT been created.

NEXT: Cell 7 will validate the processed files directly from disk.
BUILDING CLEAN CANONICAL CICIDS2017 FLOW DATASET

##############################################################################################################
PROCESSING FILE 1/8
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
###############

,Timestamp,Source IP,Source Port,Destination IP,Destination Port,Protocol,Attack_Type,Attack_Label,Source_File
0,7/7/2017 3:30,104.16.207.165,443.0,192.168.10.5,54865.0,6.0,BENIGN,0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
1,7/7/2017 3:30,104.16.28.216,80.0,192.168.10.5,55054.0,6.0,BENIGN,0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
2,7/7/2017 3:30,104.16.28.216,80.0,192.168.10.5,55055.0,6.0,BENIGN,0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
3,7/7/2017 3:30,104.17.241.25,443.0,192.168.10.16,46236.0,6.0,BENIGN,0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
4,7/7/2017 3:30,104.19.196.102,443.0,192.168.10.5,54863.0,6.0,BENIGN,0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
5,7/7/2017 3:30,104.20.10.120,443.0,192.168.10.5,54871.0,6.0,BENIGN,0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
6,7/7/2017 3:30,104.20.10.120,443.0,192.168.10.5,54925.0,6.0,BENIGN,0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
7,7/7/2017 3:30,104.20.10.120,443.0,192.168.10.5,54925.0,6.0,BENIGN,0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
8,7/7/2017 3:30,104.28.13.116,443.0,192.168.10.8,9282.0,6.0,BENIGN,0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
9,7/7/2017 3:30,104.97.123.193,443.0,192.168.10.5,55153.0,6.0,BENIGN,0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv



✓ CANONICAL FLOW DATASET SUCCESSFULLY CONSTRUCTED
✓ Raw multiclass attack information preserved.
✓ Binary attack indicator created.
✓ Full CICFlowMeter feature space preserved.
✓ Source/destination identity preserved for future graph modelling.


In [12]:
# =============================================================================
# CELL 7 — VALIDATE RAM-SAFE PROCESSED FLOW FILES
# =============================================================================
#
# What we are doing:
#
# Cell 6 cleaned the 8 raw CICIDS2017 TrafficLabelling CSV files and saved
# them independently as Parquet files.
#
# We intentionally did NOT create one giant canonical_df because doing so
# caused Google Colab to run out of RAM.
#
# This cell verifies that the files written by Cell 6 are correct BEFORE
# we begin temporal segmentation and forecasting-dataset construction.
#
#
# THIS CELL WILL:
#
#   1. Locate the 8 cleaned Parquet files created by Cell 6.
#
#   2. Read ONLY lightweight columns needed for validation:
#
#          Timestamp
#          Attack_Type
#          Attack_Label
#          Source_File
#
#      We DO NOT load all ~90 columns into memory.
#
#   3. Validate:
#
#          - total number of flows
#          - attack/benign counts
#          - multiclass attack labels
#          - timestamp validity
#          - temporal range of each file
#          - source-file traceability
#
#   4. Build a small global attack-label distribution.
#
#   5. Verify the expected CICIDS2017 totals established by our previous
#      audit.
#
#
# IMPORTANT:
#
# This is still FLOW-LEVEL validation.
#
# We are NOT yet:
#
#   - creating temporal segments
#   - aggregating flows by minute
#   - creating forecasting windows
#   - creating lead-time targets
#   - training any model
#
#
# After this cell passes:
#
#                 CLEAN FLOW FILES
#                         ↓
#                 TEMPORAL SEGMENTATION
#                         ↓
#                 MINUTE-LEVEL FEATURES
#                         ↓
#                 FORECASTING WINDOWS
#                         ↓
#                 TARGET CONSTRUCTION
#
# =============================================================================


import os
import glob
import gc

import pandas as pd
import numpy as np


# =============================================================================
# 1. DEFINE PROJECT DIRECTORIES
# =============================================================================

PROJECT_ROOT = (
    "/content/drive/MyDrive/"
    "Network-Attack-Forecasting"
)

PROCESSED_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed"
)


# =============================================================================
# 2. LOCATE PROCESSED FILES
# =============================================================================

processed_files = sorted(
    glob.glob(
        os.path.join(
            PROCESSED_DIR,
            "*_cleaned.parquet"
        )
    )
)


print("=" * 110)
print("CELL 7 — PROCESSED FLOW DATASET VALIDATION")
print("=" * 110)

print("\nProcessed directory:")
print(PROCESSED_DIR)

print(
    f"\nProcessed Parquet files found: "
    f"{len(processed_files)}"
)


for i, path in enumerate(
    processed_files,
    start=1
):

    size_mb = (
        os.path.getsize(path)
        / 1024**2
    )

    print(
        f"\n{i}. {os.path.basename(path)}"
    )

    print(
        f"   Size: {size_mb:.2f} MB"
    )


assert len(processed_files) == 8, (
    f"Expected 8 processed Parquet files, "
    f"but found {len(processed_files)}."
)


# =============================================================================
# 3. COLUMNS REQUIRED FOR THIS VALIDATION
# =============================================================================
#
# We deliberately load ONLY these columns.
#
# There is no reason to load the full CICFlowMeter feature space simply to
# validate chronology and labels.
#
# =============================================================================

validation_columns = [

    "Timestamp",
    "Attack_Type",
    "Attack_Label",
    "Source_File"

]


# =============================================================================
# 4. SMALL STORAGE OBJECTS
# =============================================================================
#
# These contain only summaries, not millions of flow rows.
#
# =============================================================================

file_summaries = []

global_label_counts = {}

global_binary_counts = {
    0: 0,
    1: 0
}


# =============================================================================
# 5. VALIDATE EACH PROCESSED FILE INDEPENDENTLY
# =============================================================================

for file_number, path in enumerate(
    processed_files,
    start=1
):

    filename = os.path.basename(path)

    print("\n\n")
    print("#" * 110)

    print(
        f"VALIDATING FILE "
        f"{file_number}/{len(processed_files)}"
    )

    print(filename)

    print("#" * 110)


    # =========================================================================
    # 5A. READ ONLY VALIDATION COLUMNS
    # =========================================================================

    df = pd.read_parquet(
        path,
        columns=validation_columns
    )


    # =========================================================================
    # 5B. BASIC COUNTS
    # =========================================================================

    total_rows = len(df)

    benign_flows = int(
        (df["Attack_Label"] == 0)
        .sum()
    )

    attack_flows = int(
        (df["Attack_Label"] == 1)
        .sum()
    )


    # =========================================================================
    # 5C. TIMESTAMP VALIDATION
    # =========================================================================

    missing_timestamps = int(
        df["Timestamp"]
        .isna()
        .sum()
    )

    start_time = (
        df["Timestamp"]
        .min()
    )

    end_time = (
        df["Timestamp"]
        .max()
    )


    # =========================================================================
    # 5D. ATTACK-TYPE VALIDATION
    # =========================================================================

    missing_attack_types = int(
        df["Attack_Type"]
        .isna()
        .sum()
    )

    label_counts = (
        df["Attack_Type"]
        .value_counts()
    )


    # =========================================================================
    # 5E. BINARY LABEL VALIDATION
    # =========================================================================

    unique_binary_labels = set(
        df["Attack_Label"]
        .dropna()
        .unique()
        .tolist()
    )

    binary_valid = (
        unique_binary_labels
        .issubset({0, 1})
    )


    # =========================================================================
    # 5F. VERIFY BENIGN ↔ BINARY MAPPING
    # =========================================================================

    benign_mapping_valid = bool(

        (
            df.loc[
                df["Attack_Type"]
                .str.upper()
                .eq("BENIGN"),
                "Attack_Label"
            ]
            == 0
        )
        .all()

    )


    attack_mapping_valid = bool(

        (
            df.loc[
                ~df["Attack_Type"]
                .str.upper()
                .eq("BENIGN"),
                "Attack_Label"
            ]
            == 1
        )
        .all()

    )


    # =========================================================================
    # 5G. SOURCE FILE TRACEABILITY
    # =========================================================================

    source_file_count = (
        df["Source_File"]
        .nunique()
    )

    source_file_valid = (
        source_file_count == 1
    )


    # =========================================================================
    # 5H. ADD LABEL COUNTS TO GLOBAL TOTAL
    # =========================================================================

    for label, count in label_counts.items():

        global_label_counts[label] = (
            global_label_counts.get(
                label,
                0
            )
            + int(count)
        )


    global_binary_counts[0] += (
        benign_flows
    )

    global_binary_counts[1] += (
        attack_flows
    )


    # =========================================================================
    # 5I. STORE SMALL FILE SUMMARY
    # =========================================================================

    file_summaries.append({

        "Processed_File":
            filename,

        "Rows":
            total_rows,

        "Benign_Flows":
            benign_flows,

        "Attack_Flows":
            attack_flows,

        "Unique_Attack_Types":
            df["Attack_Type"].nunique(),

        "Missing_Timestamps":
            missing_timestamps,

        "Missing_Attack_Types":
            missing_attack_types,

        "Binary_Label_Valid":
            binary_valid,

        "Benign_Mapping_Valid":
            benign_mapping_valid,

        "Attack_Mapping_Valid":
            attack_mapping_valid,

        "Source_File_Valid":
            source_file_valid,

        "Start_Time":
            start_time,

        "End_Time":
            end_time

    })


    # =========================================================================
    # 5J. DISPLAY FILE RESULTS
    # =========================================================================

    print(
        f"\nRows               : "
        f"{total_rows:,}"
    )

    print(
        f"Benign flows       : "
        f"{benign_flows:,}"
    )

    print(
        f"Attack flows       : "
        f"{attack_flows:,}"
    )

    print(
        f"Unique attack types: "
        f"{df['Attack_Type'].nunique()}"
    )

    print(
        f"Missing timestamps : "
        f"{missing_timestamps:,}"
    )

    print(
        f"Missing attack type: "
        f"{missing_attack_types:,}"
    )

    print(
        f"Temporal start     : "
        f"{start_time}"
    )

    print(
        f"Temporal end       : "
        f"{end_time}"
    )


    print("\nLabels:")

    for label, count in label_counts.items():

        print(
            f"  {label:<35} "
            f"{int(count):>10,}"
        )


    print("\nChecks:")

    print(
        "✓" if binary_valid else "❌",
        "Binary label contains only 0/1"
    )

    print(
        "✓" if benign_mapping_valid else "❌",
        "BENIGN maps to 0"
    )

    print(
        "✓" if attack_mapping_valid else "❌",
        "Attacks map to 1"
    )

    print(
        "✓" if source_file_valid else "❌",
        "Source-file traceability preserved"
    )


    # =========================================================================
    # 5K. DELETE THIS FILE FROM RAM
    # =========================================================================

    del df

    gc.collect()

    print(
        "\n✓ Validation dataframe "
        "deleted from RAM."
    )


# =============================================================================
# 6. BUILD FILE-LEVEL SUMMARY
# =============================================================================

validation_summary_df = pd.DataFrame(
    file_summaries
)


print("\n\n")
print("=" * 110)
print("FILE-LEVEL VALIDATION SUMMARY")
print("=" * 110)

display(
    validation_summary_df
)


# =============================================================================
# 7. GLOBAL DATASET COUNTS
# =============================================================================

total_flows = int(
    validation_summary_df["Rows"]
    .sum()
)

total_benign = int(
    validation_summary_df["Benign_Flows"]
    .sum()
)

total_attacks = int(
    validation_summary_df["Attack_Flows"]
    .sum()
)

total_missing_timestamps = int(
    validation_summary_df["Missing_Timestamps"]
    .sum()
)

total_missing_attack_types = int(
    validation_summary_df["Missing_Attack_Types"]
    .sum()
)


print("\n" + "=" * 110)
print("GLOBAL FLOW-LEVEL TOTALS")
print("=" * 110)

print(
    f"\nTotal flows       : "
    f"{total_flows:,}"
)

print(
    f"Benign flows      : "
    f"{total_benign:,}"
)

print(
    f"Attack flows      : "
    f"{total_attacks:,}"
)

print(
    f"Missing timestamps: "
    f"{total_missing_timestamps:,}"
)

print(
    f"Missing labels    : "
    f"{total_missing_attack_types:,}"
)


# =============================================================================
# 8. GLOBAL MULTICLASS ATTACK DISTRIBUTION
# =============================================================================

global_label_series = pd.Series(
    global_label_counts
).sort_values(
    ascending=False
)


print("\n" + "=" * 110)
print("GLOBAL ATTACK-TYPE DISTRIBUTION")
print("=" * 110)


for label, count in global_label_series.items():

    percentage = (
        count
        / total_flows
        * 100
    )

    print(
        f"{label:<40}"
        f"{count:>12,}"
        f"   ({percentage:>7.3f}%)"
    )


# =============================================================================
# 9. GLOBAL BINARY DISTRIBUTION
# =============================================================================

print("\n" + "=" * 110)
print("GLOBAL BINARY ATTACK DISTRIBUTION")
print("=" * 110)


benign_percentage = (
    total_benign
    / total_flows
    * 100
)

attack_percentage = (
    total_attacks
    / total_flows
    * 100
)


print(
    f"\n0 (BENIGN) : "
    f"{total_benign:,} "
    f"({benign_percentage:.3f}%)"
)

print(
    f"1 (ATTACK) : "
    f"{total_attacks:,} "
    f"({attack_percentage:.3f}%)"
)


# =============================================================================
# 10. GLOBAL TEMPORAL RANGE
# =============================================================================

global_start = (
    validation_summary_df["Start_Time"]
    .min()
)

global_end = (
    validation_summary_df["End_Time"]
    .max()
)


print("\n" + "=" * 110)
print("GLOBAL TEMPORAL RANGE")
print("=" * 110)

print(
    "\nStart:",
    global_start
)

print(
    "End  :",
    global_end
)


# =============================================================================
# 11. EXPECTED ATTACK TYPES
# =============================================================================

expected_attack_types = {

    "BENIGN",

    "DDoS",

    "PortScan",

    "Bot",

    "Infiltration",

    "Web Attack - Brute Force",

    "Web Attack - XSS",

    "Web Attack - Sql Injection",

    "FTP-Patator",

    "SSH-Patator",

    "DoS Hulk",

    "DoS GoldenEye",

    "DoS slowloris",

    "DoS Slowhttptest",

    "Heartbleed"

}


observed_attack_types = set(
    global_label_series.index
)


missing_expected_labels = (
    expected_attack_types
    - observed_attack_types
)

unexpected_labels = (
    observed_attack_types
    - expected_attack_types
)


# =============================================================================
# 12. FINAL VALIDATION CHECKS
# =============================================================================

checks = {

    "Exactly 8 processed files":
        len(processed_files) == 8,

    "Exactly 2,830,743 flows":
        total_flows == 2_830_743,

    "Exactly 557,646 attack flows":
        total_attacks == 557_646,

    "Exactly 2,273,097 benign flows":
        total_benign == 2_273_097,

    "Exactly 15 attack categories":
        len(observed_attack_types) == 15,

    "All expected attack categories present":
        len(missing_expected_labels) == 0,

    "No unexpected attack categories":
        len(unexpected_labels) == 0,

    "No missing timestamps":
        total_missing_timestamps == 0,

    "No missing attack types":
        total_missing_attack_types == 0,

    "Binary labels valid in every file":
        validation_summary_df[
            "Binary_Label_Valid"
        ].all(),

    "BENIGN mapping valid in every file":
        validation_summary_df[
            "Benign_Mapping_Valid"
        ].all(),

    "Attack mapping valid in every file":
        validation_summary_df[
            "Attack_Mapping_Valid"
        ].all(),

    "Source traceability valid":
        validation_summary_df[
            "Source_File_Valid"
        ].all()

}


print("\n" + "=" * 110)
print("FINAL CELL 7 VALIDATION")
print("=" * 110)


for check_name, result in checks.items():

    print(
        f"{'✓' if result else '❌'} "
        f"{check_name}: {bool(result)}"
    )


# =============================================================================
# 13. REPORT LABEL DIFFERENCES IF ANY
# =============================================================================

if missing_expected_labels:

    print(
        "\n❌ Missing expected labels:"
    )

    for label in sorted(
        missing_expected_labels
    ):

        print(
            "  -",
            label
        )


if unexpected_labels:

    print(
        "\n❌ Unexpected labels:"
    )

    for label in sorted(
        unexpected_labels
    ):

        print(
            "  -",
            label
        )


# =============================================================================
# 14. FINAL RESULT
# =============================================================================

print("\n" + "=" * 110)


if all(
    bool(value)
    for value in checks.values()
):

    print(
        "✓ CELL 7 COMPLETED SUCCESSFULLY."
    )

    print()

    print(
        "The 8 processed CICIDS2017 flow files "
        "have been validated directly from disk."
    )

    print()

    print(
        "No giant canonical_df was created."
    )

    print()

    print(
        "The cleaned flow-level dataset is ready "
        "for temporal segmentation."
    )

    print()

    print(
        "NEXT: Cell 8 will construct the temporal "
        "timeline/segments in a RAM-safe way."
    )

else:

    print(
        "❌ CELL 7 VALIDATION FAILED."
    )

    print(
        "Do NOT continue to temporal segmentation "
        "until the failed checks are investigated."
    )


print("=" * 110)


# =============================================================================
# 15. CLEAN UP
# =============================================================================

gc.collect()

CELL 7 — PROCESSED FLOW DATASET VALIDATION

Processed directory:
/content/drive/MyDrive/Network-Attack-Forecasting/data/processed

Processed Parquet files found: 8

1. Friday-WorkingHours-Afternoon-DDos.pcap_ISCX_cleaned.parquet
   Size: 30.24 MB

2. Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX_cleaned.parquet
   Size: 21.86 MB

3. Friday-WorkingHours-Morning.pcap_ISCX_cleaned.parquet
   Size: 25.59 MB

4. Monday-WorkingHours.pcap_ISCX_cleaned.parquet
   Size: 80.33 MB

5. Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX_cleaned.parquet
   Size: 32.12 MB

6. Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX_cleaned.parquet
   Size: 23.00 MB

7. Tuesday-WorkingHours.pcap_ISCX_cleaned.parquet
   Size: 63.39 MB

8. Wednesday-workingHours.pcap_ISCX_cleaned.parquet
   Size: 90.07 MB



##############################################################################################################
VALIDATING FILE 1/8
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX_cleaned.parquet
####

,Processed_File,Rows,Benign_Flows,Attack_Flows,Unique_Attack_Types,Missing_Timestamps,Missing_Attack_Types,Binary_Label_Valid,Benign_Mapping_Valid,Attack_Mapping_Valid,Source_File_Valid,Start_Time,End_Time
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX_c...,225745,97718,128027,2,0,0,True,True,True,True,2017-07-07 03:30:00,2017-07-07 05:02:00
1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,286467,127537,158930,2,0,0,True,True,True,True,2017-07-07 01:00:00,2017-07-07 03:29:00
2,Friday-WorkingHours-Morning.pcap_ISCX_cleaned....,191033,189067,1966,2,0,0,True,True,True,True,2017-07-07 08:59:00,2017-07-07 12:59:00
3,Monday-WorkingHours.pcap_ISCX_cleaned.parquet,529918,529918,0,1,0,0,True,True,True,True,2017-07-03 01:00:01,2017-07-03 12:59:58
4,Thursday-WorkingHours-Afternoon-Infilteration....,288602,288566,36,2,0,0,True,True,True,True,2017-07-06 01:00:00,2017-07-06 05:04:00
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,170366,168186,2180,4,0,0,True,True,True,True,2017-07-06 08:59:00,2017-07-06 12:59:00
6,Tuesday-WorkingHours.pcap_ISCX_cleaned.parquet,445909,432074,13835,3,0,0,True,True,True,True,2017-07-04 01:00:00,2017-07-04 12:59:00
7,Wednesday-workingHours.pcap_ISCX_cleaned.parquet,692703,440031,252672,6,0,0,True,True,True,True,2017-07-05 01:00:00,2017-07-05 12:59:00



GLOBAL FLOW-LEVEL TOTALS

Total flows       : 2,830,743
Benign flows      : 2,273,097
Attack flows      : 557,646
Missing timestamps: 0
Missing labels    : 0

GLOBAL ATTACK-TYPE DISTRIBUTION
BENIGN                                     2,273,097   ( 80.300%)
DoS Hulk                                     231,073   (  8.163%)
PortScan                                     158,930   (  5.614%)
DDoS                                         128,027   (  4.523%)
DoS GoldenEye                                 10,293   (  0.364%)
FTP-Patator                                    7,938   (  0.280%)
SSH-Patator                                    5,897   (  0.208%)
DoS slowloris                                  5,796   (  0.205%)
DoS Slowhttptest                               5,499   (  0.194%)
Bot                                            1,966   (  0.069%)
Web Attack - Brute Force                       1,507   (  0.053%)
Web Attack - XSS                                 652   (  0.023%)
Infiltration    

0

In [13]:
# =============================================================================
# CELL 8 — RAM-SAFE TEMPORAL GAP ANALYSIS AND SEGMENT CONSTRUCTION
# =============================================================================
#
# What we are doing:
#
# CICIDS2017 consists of multiple network-capture periods spread across
# several days. These capture periods must NOT automatically be treated as
# one continuous time series.
#
# For forecasting, a history window and its future prediction window must
# belong to the SAME continuous observation period.
#
# Example of what we must prevent:
#
#       Friday capture ends
#              ↓
#        several-hour gap
#              ↓
#       next capture begins
#
# A forecasting window must never jump across such a gap.
#
#
# THIS CELL WILL:
#
#   1. Read ONLY the Timestamp column from each processed Parquet file.
#
#   2. Build a lightweight global timestamp table.
#
#   3. Sort timestamps chronologically.
#
#   4. Calculate the time difference between consecutive flows.
#
#   5. Inspect the gap distribution.
#
#   6. Display all meaningful large gaps.
#
#   7. Automatically identify capture discontinuities using a conservative
#      threshold.
#
#   8. Assign Segment_ID values.
#
#   9. Create and save a small segment metadata table.
#
#
# IMPORTANT:
#
# We are NOT loading the ~85 CICFlowMeter feature columns.
#
# We are NOT constructing minute-level features yet.
#
# We are NOT creating forecasting windows yet.
#
# We are NOT training models.
#
# =============================================================================


import os
import glob
import gc

import numpy as np
import pandas as pd


# =============================================================================
# 1. PROJECT PATHS
# =============================================================================

PROJECT_ROOT = (
    "/content/drive/MyDrive/"
    "Network-Attack-Forecasting"
)

PROCESSED_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed"
)

TEMPORAL_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "temporal"
)

os.makedirs(
    TEMPORAL_DIR,
    exist_ok=True
)


# =============================================================================
# 2. LOCATE PROCESSED FILES
# =============================================================================

processed_files = sorted(
    glob.glob(
        os.path.join(
            PROCESSED_DIR,
            "*_cleaned.parquet"
        )
    )
)

assert len(processed_files) == 8, (
    f"Expected 8 processed files, "
    f"found {len(processed_files)}."
)


print("=" * 110)
print("CELL 8 — TEMPORAL GAP ANALYSIS")
print("=" * 110)

print(
    f"\nProcessed files found: "
    f"{len(processed_files)}"
)


# =============================================================================
# 3. LOAD ONLY TIMESTAMP + SOURCE FILE
# =============================================================================
#
# This is intentionally tiny compared with loading the complete feature
# dataset.
#
# =============================================================================

timeline_parts = []


for i, path in enumerate(
    processed_files,
    start=1
):

    filename = os.path.basename(path)

    print(
        f"\nReading timestamps "
        f"{i}/8: {filename}"
    )

    temp = pd.read_parquet(
        path,
        columns=[
            "Timestamp",
            "Source_File"
        ]
    )

    timeline_parts.append(
        temp
    )


# =============================================================================
# 4. COMBINE LIGHTWEIGHT TIMELINE
# =============================================================================

timeline_df = pd.concat(
    timeline_parts,
    ignore_index=True
)

del timeline_parts
del temp

gc.collect()


print("\n" + "=" * 110)
print("LIGHTWEIGHT GLOBAL TIMELINE")
print("=" * 110)

print(
    f"\nRows: "
    f"{len(timeline_df):,}"
)

print(
    "Columns:",
    timeline_df.columns.tolist()
)


assert len(timeline_df) == 2_830_743


# =============================================================================
# 5. SORT CHRONOLOGICALLY
# =============================================================================
#
# Unlike the previous failed attempt, this dataframe contains only TWO
# columns rather than ~90 columns.
#
# =============================================================================

timeline_df = (
    timeline_df
    .sort_values(
        "Timestamp",
        kind="mergesort"
    )
    .reset_index(drop=True)
)


print("\n✓ Global timestamp timeline sorted.")


# =============================================================================
# 6. CALCULATE INTER-FLOW GAPS
# =============================================================================

timeline_df["Gap_Seconds"] = (
    timeline_df["Timestamp"]
    .diff()
    .dt.total_seconds()
)


# =============================================================================
# 7. BASIC GAP STATISTICS
# =============================================================================

valid_gaps = (
    timeline_df["Gap_Seconds"]
    .dropna()
)


print("\n" + "=" * 110)
print("INTER-FLOW GAP STATISTICS")
print("=" * 110)

print(
    valid_gaps.describe(
        percentiles=[
            0.50,
            0.90,
            0.95,
            0.99,
            0.999,
            0.9999
        ]
    )
)


# =============================================================================
# 8. GAP COUNTS AT IMPORTANT THRESHOLDS
# =============================================================================

gap_thresholds = [
    1,
    5,
    10,
    30,
    60,
    120,
    300,
    600,
    1800,
    3600
]


print("\n" + "=" * 110)
print("NUMBER OF GAPS ABOVE SELECTED THRESHOLDS")
print("=" * 110)


for threshold in gap_thresholds:

    count = int(
        (
            valid_gaps
            > threshold
        )
        .sum()
    )

    print(
        f"> {threshold:>5} seconds : "
        f"{count:>10,}"
    )


# =============================================================================
# 9. DISPLAY LARGE GAPS
# =============================================================================
#
# We inspect gaps greater than 5 minutes.
#
# These are candidates for true capture discontinuities rather than normal
# packet/flow inactivity.
#
# =============================================================================

large_gap_mask = (
    timeline_df["Gap_Seconds"]
    > 300
)


large_gaps = (
    timeline_df.loc[
        large_gap_mask,
        [
            "Timestamp",
            "Gap_Seconds",
            "Source_File"
        ]
    ]
    .copy()
)


# Previous timestamp before each large gap

large_gap_indices = (
    timeline_df.index[
        large_gap_mask
    ]
)


previous_timestamps = (
    timeline_df["Timestamp"]
    .shift(1)
    .loc[large_gap_indices]
    .values
)


previous_sources = (
    timeline_df["Source_File"]
    .shift(1)
    .loc[large_gap_indices]
    .values
)


large_gaps.insert(
    0,
    "Previous_Timestamp",
    previous_timestamps
)

large_gaps.insert(
    1,
    "Previous_Source_File",
    previous_sources
)


large_gaps = large_gaps.rename(
    columns={
        "Timestamp":
            "Next_Timestamp",

        "Source_File":
            "Next_Source_File"
    }
)


large_gaps["Gap_Minutes"] = (
    large_gaps["Gap_Seconds"]
    / 60
)


print("\n" + "=" * 110)
print("GAPS GREATER THAN 5 MINUTES")
print("=" * 110)

print(
    f"\nNumber of >5-minute gaps: "
    f"{len(large_gaps)}"
)

if len(large_gaps) > 0:

    display(
        large_gaps[
            [
                "Previous_Timestamp",
                "Next_Timestamp",
                "Gap_Minutes",
                "Previous_Source_File",
                "Next_Source_File"
            ]
        ]
    )


# =============================================================================
# 10. DEFINE TEMPORAL DISCONTINUITY THRESHOLD
# =============================================================================
#
# For our minute-level forecasting task, a gap greater than 5 minutes means
# there is not enough continuous observation to safely bridge that period.
#
# Therefore:
#
#       Gap <= 5 minutes
#           same temporal segment
#
#       Gap > 5 minutes
#           new temporal segment
#
# This threshold is explicitly stored so it can later be changed and tested
# in sensitivity analysis.
#
# =============================================================================

SEGMENT_GAP_THRESHOLD_SECONDS = 300


timeline_df["New_Segment"] = (

    timeline_df["Gap_Seconds"]
    .gt(
        SEGMENT_GAP_THRESHOLD_SECONDS
    )
    .fillna(False)

)


# =============================================================================
# 11. ASSIGN SEGMENT IDs
# =============================================================================

timeline_df["Segment_ID"] = (
    timeline_df["New_Segment"]
    .cumsum()
    .astype("int16")
)


segment_count = (
    timeline_df["Segment_ID"]
    .nunique()
)


print("\n" + "=" * 110)
print("TEMPORAL SEGMENT CONSTRUCTION")
print("=" * 110)

print(
    f"\nGap threshold : "
    f"{SEGMENT_GAP_THRESHOLD_SECONDS} seconds"
)

print(
    f"Segments found: "
    f"{segment_count}"
)


# =============================================================================
# 12. CREATE SEGMENT METADATA
# =============================================================================

segment_metadata = (

    timeline_df

    .groupby(
        "Segment_ID",
        as_index=False
    )

    .agg(

        Start_Time=(
            "Timestamp",
            "min"
        ),

        End_Time=(
            "Timestamp",
            "max"
        ),

        Flow_Count=(
            "Timestamp",
            "size"
        )

    )

)


segment_metadata["Duration_Minutes"] = (

    (
        segment_metadata["End_Time"]
        -
        segment_metadata["Start_Time"]
    )

    .dt.total_seconds()

    / 60

)


print("\n" + "=" * 110)
print("SEGMENT METADATA")
print("=" * 110)

display(
    segment_metadata
)


# =============================================================================
# 13. VERIFY SEGMENTS DO NOT CONTAIN LARGE INTERNAL GAPS
# =============================================================================

segment_gap_check = (

    timeline_df

    .groupby(
        "Segment_ID"
    )["Gap_Seconds"]

    .max()

)


# Segment 0 begins with NaN Gap_Seconds.
# Ignore NaN values for validation.

segment_gap_valid = bool(

    (
        segment_gap_check
        .dropna()
        <= SEGMENT_GAP_THRESHOLD_SECONDS
    )
    .all()

)


print("\n" + "=" * 110)
print("SEGMENT CONTINUITY CHECK")
print("=" * 110)

print(
    "\nAll internal segment gaps "
    "<= threshold:",
    segment_gap_valid
)


# =============================================================================
# 14. SAVE SEGMENT METADATA
# =============================================================================

metadata_path = os.path.join(
    TEMPORAL_DIR,
    "segment_metadata.parquet"
)


segment_metadata.to_parquet(
    metadata_path,
    index=False
)


print("\n✓ Segment metadata saved:")
print(metadata_path)


# =============================================================================
# 15. SAVE LARGE-GAP AUDIT
# =============================================================================

gap_audit_path = os.path.join(
    TEMPORAL_DIR,
    "large_gap_audit.parquet"
)


large_gaps.to_parquet(
    gap_audit_path,
    index=False
)


print("\n✓ Large-gap audit saved:")
print(gap_audit_path)


# =============================================================================
# 16. SAVE LIGHTWEIGHT TIMELINE
# =============================================================================
#
# We save only the columns required to reconstruct temporal segment
# membership later.
#
# =============================================================================

timeline_output = (

    timeline_df[
        [
            "Timestamp",
            "Source_File",
            "Segment_ID"
        ]
    ]

    .copy()

)


timeline_path = os.path.join(
    TEMPORAL_DIR,
    "flow_timeline.parquet"
)


timeline_output.to_parquet(
    timeline_path,
    index=False,
    compression="snappy"
)


print("\n✓ Lightweight flow timeline saved:")
print(timeline_path)


# =============================================================================
# 17. FINAL VALIDATION
# =============================================================================

checks = {

    "Expected flow count":
        len(timeline_df) == 2_830_743,

    "No missing timestamps":
        timeline_df["Timestamp"]
        .isna()
        .sum() == 0,

    "Chronological order valid":
        timeline_df["Timestamp"]
        .is_monotonic_increasing,

    "At least one temporal segment":
        segment_count >= 1,

    "Internal segment gaps valid":
        segment_gap_valid,

    "Metadata saved":
        os.path.exists(
            metadata_path
        ),

    "Gap audit saved":
        os.path.exists(
            gap_audit_path
        ),

    "Timeline saved":
        os.path.exists(
            timeline_path
        )

}


print("\n" + "=" * 110)
print("FINAL CELL 8 VALIDATION")
print("=" * 110)


for name, result in checks.items():

    print(
        f"{'✓' if bool(result) else '❌'} "
        f"{name}: {bool(result)}"
    )


print("\n" + "=" * 110)


if all(
    bool(value)
    for value in checks.values()
):

    print(
        "✓ CELL 8 COMPLETED SUCCESSFULLY."
    )

    print()

    print(
        "The CICIDS2017 capture timeline has been "
        "constructed without loading the full feature "
        "dataset into RAM."
    )

    print()

    print(
        f"Temporal segments found: "
        f"{segment_count}"
    )

    print()

    print(
        "NEXT: inspect the gap and segment output "
        "before constructing minute-level features."
    )

else:

    print(
        "❌ CELL 8 VALIDATION FAILED."
    )

    print(
        "Do NOT continue to minute-level aggregation."
    )


print("=" * 110)


# =============================================================================
# 18. RAM CLEANUP
# =============================================================================

del timeline_output

gc.collect()

CELL 8 — TEMPORAL GAP ANALYSIS

Processed files found: 8

Reading timestamps 1/8: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX_cleaned.parquet

Reading timestamps 2/8: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX_cleaned.parquet

Reading timestamps 3/8: Friday-WorkingHours-Morning.pcap_ISCX_cleaned.parquet

Reading timestamps 4/8: Monday-WorkingHours.pcap_ISCX_cleaned.parquet

Reading timestamps 5/8: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX_cleaned.parquet

Reading timestamps 6/8: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX_cleaned.parquet

Reading timestamps 7/8: Tuesday-WorkingHours.pcap_ISCX_cleaned.parquet

Reading timestamps 8/8: Wednesday-workingHours.pcap_ISCX_cleaned.parquet

LIGHTWEIGHT GLOBAL TIMELINE

Rows: 2,830,743
Columns: ['Timestamp', 'Source_File']

✓ Global timestamp timeline sorted.

INTER-FLOW GAP STATISTICS
count     2.830742e+06
mean      1.373276e-01
std       5.461584e+01
min       0.000000e+00
50%       0.000000e+00
90%       0.000000e

,Previous_Timestamp,Next_Timestamp,Gap_Minutes,Previous_Source_File,Next_Source_File
222799,2017-07-03 05:01:34,2017-07-03 08:55:58,234.400000,Monday-WorkingHours.pcap_ISCX.csv,Monday-WorkingHours.pcap_ISCX.csv
529918,2017-07-03 12:59:58,2017-07-04 01:00:00,720.033333,Monday-WorkingHours.pcap_ISCX.csv,Tuesday-WorkingHours.pcap_ISCX.csv
750684,2017-07-04 05:00:00,2017-07-04 08:53:00,233.000000,Tuesday-WorkingHours.pcap_ISCX.csv,Tuesday-WorkingHours.pcap_ISCX.csv
975827,2017-07-04 12:59:00,2017-07-05 01:00:00,721.000000,Tuesday-WorkingHours.pcap_ISCX.csv,Wednesday-workingHours.pcap_ISCX.csv
1203164,2017-07-05 05:10:00,2017-07-05 08:42:00,212.000000,Wednesday-workingHours.pcap_ISCX.csv,Wednesday-workingHours.pcap_ISCX.csv
1668530,2017-07-05 12:59:00,2017-07-06 01:00:00,721.000000,Wednesday-workingHours.pcap_ISCX.csv,Thursday-WorkingHours-Afternoon-Infilteration....
1957132,2017-07-06 05:04:00,2017-07-06 08:59:00,235.000000,Thursday-WorkingHours-Afternoon-Infilteration....,Thursday-WorkingHours-Morning-WebAttacks.pcap_...
2127498,2017-07-06 12:59:00,2017-07-07 01:00:00,721.000000,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...
2639710,2017-07-07 05:02:00,2017-07-07 08:59:00,237.000000,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Friday-WorkingHours-Morning.pcap_ISCX.csv



TEMPORAL SEGMENT CONSTRUCTION

Gap threshold : 300 seconds
Segments found: 10

SEGMENT METADATA


,Segment_ID,Start_Time,End_Time,Flow_Count,Duration_Minutes
0,0,2017-07-03 01:00:01,2017-07-03 05:01:34,222799,241.55
1,1,2017-07-03 08:55:58,2017-07-03 12:59:58,307119,244.00
2,2,2017-07-04 01:00:00,2017-07-04 05:00:00,220766,240.00
3,3,2017-07-04 08:53:00,2017-07-04 12:59:00,225143,246.00
4,4,2017-07-05 01:00:00,2017-07-05 05:10:00,227337,250.00
5,5,2017-07-05 08:42:00,2017-07-05 12:59:00,465366,257.00
6,6,2017-07-06 01:00:00,2017-07-06 05:04:00,288602,244.00
7,7,2017-07-06 08:59:00,2017-07-06 12:59:00,170366,240.00
8,8,2017-07-07 01:00:00,2017-07-07 05:02:00,512212,242.00
9,9,2017-07-07 08:59:00,2017-07-07 12:59:00,191033,240.00



SEGMENT CONTINUITY CHECK

All internal segment gaps <= threshold: False

✓ Segment metadata saved:
/content/drive/MyDrive/Network-Attack-Forecasting/data/temporal/segment_metadata.parquet

✓ Large-gap audit saved:
/content/drive/MyDrive/Network-Attack-Forecasting/data/temporal/large_gap_audit.parquet

✓ Lightweight flow timeline saved:
/content/drive/MyDrive/Network-Attack-Forecasting/data/temporal/flow_timeline.parquet

FINAL CELL 8 VALIDATION
✓ Expected flow count: True
✓ No missing timestamps: True
✓ Chronological order valid: True
✓ At least one temporal segment: True
❌ Internal segment gaps valid: False
✓ Metadata saved: True
✓ Gap audit saved: True
✓ Timeline saved: True

❌ CELL 8 VALIDATION FAILED.
Do NOT continue to minute-level aggregation.


46

In [14]:
# =============================================================================
# CELL 8B — CORRECT TEMPORAL SEGMENT CONTINUITY VALIDATION
# =============================================================================
#
# Why this cell exists:
#
# The original Cell 8 continuity check incorrectly counted the gap BEFORE
# the first row of each new segment as if that gap occurred INSIDE the
# segment.
#
# Example:
#
#   Segment 0 ends
#          |
#          | 234-minute capture gap
#          |
#   Segment 1 begins
#
# The first row of Segment 1 naturally stores that 234-minute Gap_Seconds
# value because Gap_Seconds was calculated before segmentation.
#
# That gap is the BOUNDARY BETWEEN segments, not an internal segment gap.
#
# Therefore, this validation recalculates time differences independently
# WITHIN each Segment_ID.
#
# =============================================================================


print("=" * 110)
print("CELL 8B — CORRECTED SEGMENT CONTINUITY VALIDATION")
print("=" * 110)


# =============================================================================
# 1. RECALCULATE GAPS WITHIN EACH SEGMENT
# =============================================================================

timeline_df["Internal_Gap_Seconds"] = (

    timeline_df
    .groupby("Segment_ID")["Timestamp"]
    .diff()
    .dt.total_seconds()

)


# =============================================================================
# 2. FIND MAXIMUM TRUE INTERNAL GAP PER SEGMENT
# =============================================================================

internal_gap_summary = (

    timeline_df

    .groupby("Segment_ID")

    .agg(

        Start_Time=(
            "Timestamp",
            "min"
        ),

        End_Time=(
            "Timestamp",
            "max"
        ),

        Flow_Count=(
            "Timestamp",
            "size"
        ),

        Max_Internal_Gap_Seconds=(
            "Internal_Gap_Seconds",
            "max"
        )

    )

    .reset_index()

)


internal_gap_summary["Max_Internal_Gap_Minutes"] = (

    internal_gap_summary[
        "Max_Internal_Gap_Seconds"
    ]

    / 60

)


# =============================================================================
# 3. DISPLAY RESULT
# =============================================================================

print("\nTRUE INTERNAL GAP SUMMARY")
print("-" * 110)

display(
    internal_gap_summary
)


# =============================================================================
# 4. VALIDATE
# =============================================================================

SEGMENT_GAP_THRESHOLD_SECONDS = 300


segment_gap_valid = bool(

    (
        internal_gap_summary[
            "Max_Internal_Gap_Seconds"
        ]
        <= SEGMENT_GAP_THRESHOLD_SECONDS
    )
    .all()

)


print("\n" + "=" * 110)
print("CORRECTED CONTINUITY CHECK")
print("=" * 110)

print(
    "\nAll TRUE internal segment gaps "
    "<= 300 seconds:",
    segment_gap_valid
)


# =============================================================================
# 5. ADDITIONAL SANITY CHECK
# =============================================================================

bad_segments = (

    internal_gap_summary[

        internal_gap_summary[
            "Max_Internal_Gap_Seconds"
        ]

        > SEGMENT_GAP_THRESHOLD_SECONDS

    ]

)


print(
    "\nSegments containing an invalid "
    "internal gap:",
    len(bad_segments)
)


if len(bad_segments) > 0:

    print(
        "\n❌ Unexpected large internal gaps found:"
    )

    display(
        bad_segments
    )

else:

    print(
        "✓ No segment contains a "
        ">5-minute internal discontinuity."
    )


# =============================================================================
# 6. VERIFY EXPECTED SEGMENT STRUCTURE
# =============================================================================

segment_count = (
    timeline_df["Segment_ID"]
    .nunique()
)


boundary_count = (

    timeline_df["Segment_ID"]
    .diff()
    .fillna(0)
    .ne(0)
    .sum()

)


print("\n" + "=" * 110)
print("SEGMENT STRUCTURE")
print("=" * 110)

print(
    f"\nSegments       : {segment_count}"
)

print(
    f"Boundaries     : {boundary_count}"
)

print(
    f"Expected gaps  : {segment_count - 1}"
)


structure_valid = (

    segment_count == 10
    and
    boundary_count == 9

)


# =============================================================================
# 7. FINAL RESULT
# =============================================================================

print("\n" + "=" * 110)
print("FINAL TEMPORAL SEGMENTATION VALIDATION")
print("=" * 110)


checks = {

    "Exactly 10 continuous capture segments":
        segment_count == 10,

    "Exactly 9 capture discontinuities":
        boundary_count == 9,

    "All internal gaps <= 5 minutes":
        segment_gap_valid,

    "Segment structure valid":
        structure_valid

}


for name, result in checks.items():

    print(
        f"{'✓' if result else '❌'} "
        f"{name}: {result}"
    )


print("\n" + "=" * 110)


if all(checks.values()):

    print(
        "✓ TEMPORAL SEGMENTATION FULLY VALIDATED."
    )

    print()

    print(
        "The CICIDS2017 dataset contains "
        "10 continuous capture segments separated "
        "by 9 large observation gaps."
    )

    print()

    print(
        "Forecasting windows can now be constructed "
        "WITHOUT crossing capture discontinuities."
    )

    print()

    print(
        "NEXT: construct the minute-level temporal "
        "feature dataset."
    )

else:

    print(
        "❌ Temporal segmentation still requires investigation."
    )


print("=" * 110)

CELL 8B — CORRECTED SEGMENT CONTINUITY VALIDATION

TRUE INTERNAL GAP SUMMARY
--------------------------------------------------------------------------------------------------------------


,Segment_ID,Start_Time,End_Time,Flow_Count,Max_Internal_Gap_Seconds,Max_Internal_Gap_Minutes
0,0,2017-07-03 01:00:01,2017-07-03 05:01:34,222799,13.0,0.216667
1,1,2017-07-03 08:55:58,2017-07-03 12:59:58,307119,24.0,0.400000
2,2,2017-07-04 01:00:00,2017-07-04 05:00:00,220766,60.0,1.000000
3,3,2017-07-04 08:53:00,2017-07-04 12:59:00,225143,60.0,1.000000
4,4,2017-07-05 01:00:00,2017-07-05 05:10:00,227337,60.0,1.000000
5,5,2017-07-05 08:42:00,2017-07-05 12:59:00,465366,60.0,1.000000
6,6,2017-07-06 01:00:00,2017-07-06 05:04:00,288602,60.0,1.000000
7,7,2017-07-06 08:59:00,2017-07-06 12:59:00,170366,60.0,1.000000
8,8,2017-07-07 01:00:00,2017-07-07 05:02:00,512212,60.0,1.000000
9,9,2017-07-07 08:59:00,2017-07-07 12:59:00,191033,60.0,1.000000



CORRECTED CONTINUITY CHECK

All TRUE internal segment gaps <= 300 seconds: True

Segments containing an invalid internal gap: 0
✓ No segment contains a >5-minute internal discontinuity.

SEGMENT STRUCTURE

Segments       : 10
Boundaries     : 9
Expected gaps  : 9

FINAL TEMPORAL SEGMENTATION VALIDATION
✓ Exactly 10 continuous capture segments: True
✓ Exactly 9 capture discontinuities: True
✓ All internal gaps <= 5 minutes: True
✓ Segment structure valid: True

✓ TEMPORAL SEGMENTATION FULLY VALIDATED.

The CICIDS2017 dataset contains 10 continuous capture segments separated by 9 large observation gaps.

Forecasting windows can now be constructed WITHOUT crossing capture discontinuities.

NEXT: construct the minute-level temporal feature dataset.


In [15]:
# =============================================================================
# CELL 9 — BUILD RAM-SAFE MINUTE-LEVEL TEMPORAL DATASET
# =============================================================================
#
# PROJECT:
# CICIDS2017 Network Attack Forecasting
#
#
# PURPOSE
# -------
#
# The raw CICIDS2017 data is FLOW LEVEL:
#
#       Flow 1 -> 09:00:01
#       Flow 2 -> 09:00:01
#       Flow 3 -> 09:00:02
#       ...
#
# Our forecasting models require a REGULAR temporal representation.
#
# We therefore convert the flow-level dataset into:
#
#       ONE ROW = ONE MINUTE OF NETWORK ACTIVITY
#
#
# Example:
#
#   Segment_ID | Minute | Flow_Count | Fwd_Packets_Sum | ...
#   -----------------------------------------------------------
#       0       | 09:00  |    500     |      12000      |
#       0       | 09:01  |    650     |      15400      |
#       0       | 09:02  |    430     |      10200      |
#
#
# ---------------------------------------------------------------------------
# VERY IMPORTANT: PREDICTORS VS GROUND TRUTH
# ---------------------------------------------------------------------------
#
# PREDICTOR FEATURES describe observed network behaviour:
#
#       flow volume
#       packet volume
#       byte volume
#       flow duration
#       packet sizes
#       inter-arrival times
#       flow rates
#       TCP flags
#       active/idle behaviour
#       etc.
#
#
# GROUND-TRUTH COLUMNS describe attacks:
#
#       Attack_State
#       Attack_Flow_Count
#       Attack_Ratio
#       Attack_Types
#
#
# ATTACK INFORMATION MUST NEVER BE INCLUDED AS MODEL INPUT.
#
# It will only be used later to construct:
#
#       Future_Attack
#       Lead_Time
#       Future_Attack_Type
#       Victim/target-related targets
#
#
# ---------------------------------------------------------------------------
# RAM-SAFE DESIGN
# ---------------------------------------------------------------------------
#
# We NEVER load all 2.83 million flows at once.
#
# For each cleaned Parquet file:
#
#       load one file
#            ↓
#       attach Segment_ID
#            ↓
#       aggregate flows into minutes
#            ↓
#       retain tiny minute table
#            ↓
#       delete flow dataframe
#
#
# After all files:
#
#       concatenate MINUTE tables only
#
#
# =============================================================================


import os
import glob
import gc
import warnings

import numpy as np
import pandas as pd


# =============================================================================
# 1. PROJECT PATHS
# =============================================================================

PROJECT_ROOT = (
    "/content/drive/MyDrive/"
    "Network-Attack-Forecasting"
)

PROCESSED_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed"
)

TEMPORAL_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "temporal"
)

FORECAST_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "forecasting"
)

os.makedirs(
    TEMPORAL_DIR,
    exist_ok=True
)

os.makedirs(
    FORECAST_DIR,
    exist_ok=True
)


# =============================================================================
# 2. LOCATE CLEANED FLOW FILES
# =============================================================================

processed_files = sorted(
    glob.glob(
        os.path.join(
            PROCESSED_DIR,
            "*_cleaned.parquet"
        )
    )
)


assert len(processed_files) == 8, (
    f"Expected 8 processed files, "
    f"found {len(processed_files)}."
)


print("=" * 115)
print("CELL 9 — MINUTE-LEVEL TEMPORAL DATASET CONSTRUCTION")
print("=" * 115)

print(
    f"\nProcessed flow files found: "
    f"{len(processed_files)}"
)


# =============================================================================
# 3. LOAD SEGMENT METADATA
# =============================================================================
#
# Cell 8 established 10 continuous capture segments.
#
# Instead of joining against a 2.83-million-row timeline, we can assign
# Segment_ID from each segment's validated Start_Time and End_Time.
#
# =============================================================================

segment_metadata_path = os.path.join(
    TEMPORAL_DIR,
    "segment_metadata.parquet"
)


assert os.path.exists(segment_metadata_path), (
    "segment_metadata.parquet not found. "
    "Run Cell 8 first."
)


segment_metadata = pd.read_parquet(
    segment_metadata_path
)


segment_metadata["Start_Time"] = pd.to_datetime(
    segment_metadata["Start_Time"]
)

segment_metadata["End_Time"] = pd.to_datetime(
    segment_metadata["End_Time"]
)


print(
    f"\nTemporal segments available: "
    f"{len(segment_metadata)}"
)


assert len(segment_metadata) == 10, (
    "Expected 10 validated temporal segments."
)


# =============================================================================
# 4. SELECT FLOW FEATURES
# =============================================================================
#
# These features represent major categories of CICFlowMeter behaviour.
#
# We deliberately DO NOT include:
#
#       Label
#       Attack_Type
#       Attack_Label
#
# as predictor features.
#
# Identity fields such as Source IP / Destination IP are also NOT aggregated
# numerically here. They will be handled separately later when constructing
# graph / victim / attacker representations.
#
# =============================================================================


SUM_FEATURES = [

    # -------------------------------------------------------------------------
    # Traffic volume
    # -------------------------------------------------------------------------

    "Total Fwd Packets",
    "Total Backward Packets",

    "Total Length of Fwd Packets",
    "Total Length of Bwd Packets",

    # -------------------------------------------------------------------------
    # TCP flag activity
    # -------------------------------------------------------------------------

    "FIN Flag Count",
    "SYN Flag Count",
    "RST Flag Count",
    "PSH Flag Count",
    "ACK Flag Count",
    "URG Flag Count",
    "CWE Flag Count",
    "ECE Flag Count",

    # -------------------------------------------------------------------------
    # Subflow traffic
    # -------------------------------------------------------------------------

    "Subflow Fwd Packets",
    "Subflow Fwd Bytes",

    "Subflow Bwd Packets",
    "Subflow Bwd Bytes",

    # -------------------------------------------------------------------------
    # Actual forward data packets
    # -------------------------------------------------------------------------

    "act_data_pkt_fwd"

]


MEAN_STD_MAX_FEATURES = [

    # -------------------------------------------------------------------------
    # Flow duration
    # -------------------------------------------------------------------------

    "Flow Duration",

    # -------------------------------------------------------------------------
    # Flow rates
    # -------------------------------------------------------------------------

    "Flow Bytes/s",
    "Flow Packets/s",

    "Fwd Packets/s",
    "Bwd Packets/s",

    # -------------------------------------------------------------------------
    # Packet lengths
    # -------------------------------------------------------------------------

    "Fwd Packet Length Mean",
    "Fwd Packet Length Std",

    "Bwd Packet Length Mean",
    "Bwd Packet Length Std",

    "Packet Length Mean",
    "Packet Length Std",

    # -------------------------------------------------------------------------
    # Inter-arrival behaviour
    # -------------------------------------------------------------------------

    "Flow IAT Mean",
    "Flow IAT Std",
    "Flow IAT Max",

    "Fwd IAT Mean",
    "Fwd IAT Std",
    "Fwd IAT Max",

    "Bwd IAT Mean",
    "Bwd IAT Std",
    "Bwd IAT Max",

    # -------------------------------------------------------------------------
    # Directional / packet-size behaviour
    # -------------------------------------------------------------------------

    "Down/Up Ratio",
    "Average Packet Size",

    "Avg Fwd Segment Size",
    "Avg Bwd Segment Size",

    # -------------------------------------------------------------------------
    # TCP window behaviour
    # -------------------------------------------------------------------------

    "Init_Win_bytes_forward",
    "Init_Win_bytes_backward",

    "min_seg_size_forward",

    # -------------------------------------------------------------------------
    # Active / idle behaviour
    # -------------------------------------------------------------------------

    "Active Mean",
    "Active Std",
    "Active Max",

    "Idle Mean",
    "Idle Std",
    "Idle Max"

]


# =============================================================================
# 5. COLUMNS REQUIRED FROM EACH FLOW FILE
# =============================================================================

required_flow_columns = [

    "Timestamp",

    "Source IP",
    "Destination IP",

    "Source Port",
    "Destination Port",

    "Protocol",

    "Attack_Type",
    "Attack_Label"

]


feature_columns = list(
    dict.fromkeys(
        SUM_FEATURES
        +
        MEAN_STD_MAX_FEATURES
    )
)


columns_to_load = list(
    dict.fromkeys(
        required_flow_columns
        +
        feature_columns
    )
)


print(
    f"\nFlow-level columns selected: "
    f"{len(columns_to_load)}"
)

print(
    f"Predictor source features: "
    f"{len(feature_columns)}"
)


# =============================================================================
# 6. SEGMENT ASSIGNMENT FUNCTION
# =============================================================================

def assign_segment_ids(df):

    """
    Assign each flow to one of the 10 validated continuous capture segments.

    Segment membership is based entirely on the timestamp ranges established
    in Cell 8.
    """

    segment_ids = np.full(
        len(df),
        -1,
        dtype=np.int16
    )


    timestamps = df["Timestamp"]


    for row in segment_metadata.itertuples():

        mask = (

            (timestamps >= row.Start_Time)
            &
            (timestamps <= row.End_Time)

        )

        segment_ids[mask.to_numpy()] = int(
            row.Segment_ID
        )


    return segment_ids


# =============================================================================
# 7. STORAGE FOR MINUTE TABLES ONLY
# =============================================================================

minute_parts = []


# =============================================================================
# 8. PROCESS EACH CLEANED FILE INDEPENDENTLY
# =============================================================================

for file_number, path in enumerate(
    processed_files,
    start=1
):

    filename = os.path.basename(path)


    print("\n\n")
    print("#" * 115)

    print(
        f"PROCESSING FILE "
        f"{file_number}/{len(processed_files)}"
    )

    print(filename)

    print("#" * 115)


    # =========================================================================
    # 8A. CHECK AVAILABLE COLUMNS
    # =========================================================================

    available_columns = pd.read_parquet(
        path
    ).columns.tolist()


    # Immediately release this temporary dataframe if pandas materialised it.

    gc.collect()


    missing_columns = [

        column

        for column in columns_to_load

        if column not in available_columns

    ]


    assert len(missing_columns) == 0, (
        f"{filename} missing required columns: "
        f"{missing_columns}"
    )


    # =========================================================================
    # 8B. LOAD ONLY SELECTED COLUMNS
    # =========================================================================

    df = pd.read_parquet(
        path,
        columns=columns_to_load
    )


    print(
        f"\nFlows loaded : "
        f"{len(df):,}"
    )

    print(
        f"Columns loaded: "
        f"{len(df.columns)}"
    )


    # =========================================================================
    # 8C. TIMESTAMP CHECK
    # =========================================================================

    df["Timestamp"] = pd.to_datetime(
        df["Timestamp"]
    )


    assert (
        df["Timestamp"]
        .isna()
        .sum()
        == 0
    )


    # =========================================================================
    # 8D. ASSIGN SEGMENT ID
    # =========================================================================

    df["Segment_ID"] = (
        assign_segment_ids(df)
    )


    unassigned = int(
        (df["Segment_ID"] == -1)
        .sum()
    )


    print(
        f"Unassigned flows: "
        f"{unassigned:,}"
    )


    assert unassigned == 0, (
        f"{filename} contains flows that could "
        f"not be assigned to a temporal segment."
    )


    # =========================================================================
    # 8E. CREATE MINUTE TIMESTAMP
    # =========================================================================

    df["Minute"] = (
        df["Timestamp"]
        .dt.floor("min")
    )


    # =========================================================================
    # 8F. SANITIZE NUMERIC FEATURES
    # =========================================================================
    #
    # CICIDS2017 contains occasional inf/-inf values in rate features.
    #
    # These cannot safely be propagated into model features.
    #
    # We replace +/- infinity with NaN here.
    #
    # Aggregations will ignore NaN values.
    #
    # Later we will explicitly audit missing minute features before training.
    #
    # =========================================================================

    df[feature_columns] = (

        df[feature_columns]

        .replace(
            [np.inf, -np.inf],
            np.nan
        )

    )


    # =========================================================================
    # 8G. BASE MINUTE TABLE
    # =========================================================================
    #
    # One row for every observed Segment_ID + Minute.
    #
    # =========================================================================

    grouped = df.groupby(
        [
            "Segment_ID",
            "Minute"
        ],
        sort=True,
        observed=True
    )


    minute_base = (

        grouped

        .size()

        .rename(
            "Flow_Count"
        )

        .reset_index()

    )


    # =========================================================================
    # 8H. ATTACK GROUND TRUTH
    # =========================================================================

    attack_counts = (

        grouped["Attack_Label"]

        .sum()

        .rename(
            "Attack_Flow_Count"
        )

        .reset_index()

    )


    minute_base = minute_base.merge(

        attack_counts,

        on=[
            "Segment_ID",
            "Minute"
        ],

        how="left"

    )


    minute_base["Attack_Flow_Count"] = (

        minute_base["Attack_Flow_Count"]
        .fillna(0)
        .astype("int32")

    )


    minute_base["Attack_State"] = (

        minute_base["Attack_Flow_Count"]
        .gt(0)
        .astype("int8")

    )


    minute_base["Attack_Ratio"] = (

        minute_base["Attack_Flow_Count"]

        /

        minute_base["Flow_Count"]

    )


    # =========================================================================
    # 8I. ATTACK TYPES OCCURRING DURING EACH MINUTE
    # =========================================================================
    #
    # A minute can theoretically contain multiple attack types.
    #
    # We therefore DO NOT force a single multiclass label here.
    #
    # Example:
    #
    #       "DDoS"
    #
    # or
    #
    #       "Web Attack - XSS|Web Attack - Brute Force"
    #
    # BENIGN is excluded from Attack_Types.
    #
    # =========================================================================

    attack_only = df.loc[
        df["Attack_Label"] == 1,
        [
            "Segment_ID",
            "Minute",
            "Attack_Type"
        ]
    ]


    if len(attack_only) > 0:

        attack_types = (

            attack_only

            .groupby(
                [
                    "Segment_ID",
                    "Minute"
                ],
                observed=True
            )["Attack_Type"]

            .agg(
                lambda values:
                    "|".join(
                        sorted(
                            set(
                                str(v)
                                for v in values
                                if pd.notna(v)
                            )
                        )
                    )
            )

            .rename(
                "Attack_Types"
            )

            .reset_index()

        )


        minute_base = minute_base.merge(

            attack_types,

            on=[
                "Segment_ID",
                "Minute"
            ],

            how="left"

        )

    else:

        minute_base["Attack_Types"] = pd.NA


    # =========================================================================
    # 8J. NETWORK IDENTITY / DIVERSITY FEATURES
    # =========================================================================
    #
    # These are useful behavioural features without directly feeding raw IP
    # strings into standard ML models.
    #
    # =========================================================================

    diversity = (

        grouped

        .agg(

            Unique_Source_IPs=(
                "Source IP",
                "nunique"
            ),

            Unique_Destination_IPs=(
                "Destination IP",
                "nunique"
            ),

            Unique_Source_Ports=(
                "Source Port",
                "nunique"
            ),

            Unique_Destination_Ports=(
                "Destination Port",
                "nunique"
            ),

            Unique_Protocols=(
                "Protocol",
                "nunique"
            )

        )

        .reset_index()

    )


    minute_base = minute_base.merge(

        diversity,

        on=[
            "Segment_ID",
            "Minute"
        ],

        how="left"

    )


    # =========================================================================
    # 8K. PROTOCOL COUNTS
    # =========================================================================

    protocol_series = (
        pd.to_numeric(
            df["Protocol"],
            errors="coerce"
        )
    )


    df["_Is_TCP"] = (
        protocol_series
        .eq(6)
        .astype("int8")
    )

    df["_Is_UDP"] = (
        protocol_series
        .eq(17)
        .astype("int8")
    )

    df["_Is_Other_Protocol"] = (

        (
            ~protocol_series.isin(
                [6, 17]
            )
        )

        .astype("int8")

    )


    protocol_features = (

        df

        .groupby(
            [
                "Segment_ID",
                "Minute"
            ],
            observed=True
        )

        .agg(

            TCP_Flow_Count=(
                "_Is_TCP",
                "sum"
            ),

            UDP_Flow_Count=(
                "_Is_UDP",
                "sum"
            ),

            Other_Protocol_Flow_Count=(
                "_Is_Other_Protocol",
                "sum"
            )

        )

        .reset_index()

    )


    minute_base = minute_base.merge(

        protocol_features,

        on=[
            "Segment_ID",
            "Minute"
        ],

        how="left"

    )


    # =========================================================================
    # 8L. SUM AGGREGATIONS
    # =========================================================================

    if len(SUM_FEATURES) > 0:

        sum_features = (

            df

            .groupby(
                [
                    "Segment_ID",
                    "Minute"
                ],
                observed=True
            )[SUM_FEATURES]

            .sum(
                min_count=1
            )

            .add_suffix(
                "_sum"
            )

            .reset_index()

        )


        minute_base = minute_base.merge(

            sum_features,

            on=[
                "Segment_ID",
                "Minute"
            ],

            how="left"

        )


    # =========================================================================
    # 8M. MEAN / STD / MAX AGGREGATIONS
    # =========================================================================
    #
    # We preserve three views of continuous flow behaviour:
    #
    #       mean -> typical behaviour
    #       std  -> variability
    #       max  -> extreme behaviour
    #
    # =========================================================================

    if len(MEAN_STD_MAX_FEATURES) > 0:

        numeric_group = (

            df

            .groupby(
                [
                    "Segment_ID",
                    "Minute"
                ],
                observed=True
            )[MEAN_STD_MAX_FEATURES]

        )


        mean_features = (

            numeric_group

            .mean()

            .add_suffix(
                "_mean"
            )

            .reset_index()

        )


        std_features = (

            numeric_group

            .std(
                ddof=0
            )

            .add_suffix(
                "_std"
            )

            .reset_index()

        )


        max_features = (

            numeric_group

            .max()

            .add_suffix(
                "_max"
            )

            .reset_index()

        )


        for feature_table in [

            mean_features,
            std_features,
            max_features

        ]:

            minute_base = minute_base.merge(

                feature_table,

                on=[
                    "Segment_ID",
                    "Minute"
                ],

                how="left"

            )


    # =========================================================================
    # 8N. SOURCE FILE TRACEABILITY
    # =========================================================================

    minute_base["Source_File"] = filename


    # =========================================================================
    # 8O. BASIC FILE-LEVEL AUDIT
    # =========================================================================

    print(
        f"\nMinute rows produced : "
        f"{len(minute_base):,}"
    )

    print(
        f"Attack minutes       : "
        f"{minute_base['Attack_State'].sum():,}"
    )

    print(
        f"Benign minutes       : "
        f"{(minute_base['Attack_State'] == 0).sum():,}"
    )

    print(
        f"Minute feature count : "
        f"{len(minute_base.columns):,}"
    )


    # =========================================================================
    # 8P. KEEP ONLY MINUTE TABLE
    # =========================================================================

    minute_parts.append(
        minute_base
    )


    # =========================================================================
    # 8Q. DELETE FLOW-LEVEL DATA FROM RAM
    # =========================================================================

    del df
    del grouped
    del minute_base
    del attack_counts
    del diversity
    del protocol_features

    if "attack_only" in locals():
        del attack_only

    if "attack_types" in locals():
        del attack_types

    if "sum_features" in locals():
        del sum_features

    if "numeric_group" in locals():
        del numeric_group

    if "mean_features" in locals():
        del mean_features

    if "std_features" in locals():
        del std_features

    if "max_features" in locals():
        del max_features

    gc.collect()


    print(
        "✓ Flow-level dataframe deleted from RAM."
    )


# =============================================================================
# 9. COMBINE MINUTE TABLES
# =============================================================================
#
# At this point we are combining only a few thousand minute rows, NOT
# millions of flows.
#
# =============================================================================

minute_df = pd.concat(
    minute_parts,
    ignore_index=True
)


del minute_parts

gc.collect()


# =============================================================================
# 10. SORT MINUTE DATA CHRONOLOGICALLY
# =============================================================================

minute_df = (

    minute_df

    .sort_values(
        [
            "Segment_ID",
            "Minute"
        ]
    )

    .reset_index(
        drop=True
    )

)


# =============================================================================
# 11. CHECK FOR DUPLICATE SEGMENT-MINUTE ROWS
# =============================================================================

duplicate_minutes = int(

    minute_df

    .duplicated(
        subset=[
            "Segment_ID",
            "Minute"
        ]
    )

    .sum()

)


# =============================================================================
# 12. CHECK MINUTE CONTINUITY
# =============================================================================
#
# Because each validated segment has no >5-minute flow gap, we inspect
# whether every minute in each segment is represented.
#
# Missing minutes are NOT silently fabricated here.
#
# =============================================================================

continuity_records = []


for segment_id, group in minute_df.groupby(
    "Segment_ID"
):

    start_minute = (
        group["Minute"]
        .min()
    )

    end_minute = (
        group["Minute"]
        .max()
    )

    expected_minutes = int(

        (
            end_minute
            - start_minute
        )

        .total_seconds()

        // 60

        + 1

    )


    observed_minutes = (
        group["Minute"]
        .nunique()
    )


    missing_minutes = (
        expected_minutes
        - observed_minutes
    )


    continuity_records.append({

        "Segment_ID":
            segment_id,

        "Start_Minute":
            start_minute,

        "End_Minute":
            end_minute,

        "Expected_Minutes":
            expected_minutes,

        "Observed_Minutes":
            observed_minutes,

        "Missing_Minutes":
            missing_minutes

    })


minute_continuity_df = pd.DataFrame(
    continuity_records
)


# =============================================================================
# 13. FEATURE / TARGET COLUMN SEPARATION
# =============================================================================

GROUND_TRUTH_COLUMNS = [

    "Attack_State",
    "Attack_Flow_Count",
    "Attack_Ratio",
    "Attack_Types"

]


IDENTIFIER_COLUMNS = [

    "Segment_ID",
    "Minute",
    "Source_File"

]


MODEL_FEATURE_COLUMNS = [

    column

    for column in minute_df.columns

    if column not in (
        GROUND_TRUTH_COLUMNS
        +
        IDENTIFIER_COLUMNS
    )

]


# =============================================================================
# 14. INFINITY AUDIT
# =============================================================================

numeric_model_data = (
    minute_df[
        MODEL_FEATURE_COLUMNS
    ]
    .select_dtypes(
        include=[
            np.number
        ]
    )
)


infinite_values = int(

    np.isinf(
        numeric_model_data
        .to_numpy()
    )
    .sum()

)


# =============================================================================
# 15. MISSING FEATURE AUDIT
# =============================================================================

missing_feature_counts = (

    minute_df[
        MODEL_FEATURE_COLUMNS
    ]

    .isna()

    .sum()

)


features_with_missing = (

    missing_feature_counts[
        missing_feature_counts > 0
    ]

    .sort_values(
        ascending=False
    )

)


# =============================================================================
# 16. DISPLAY GLOBAL RESULTS
# =============================================================================

print("\n\n")
print("=" * 115)
print("MINUTE-LEVEL DATASET SUMMARY")
print("=" * 115)

print(
    f"\nTotal minute rows : "
    f"{len(minute_df):,}"
)

print(
    f"Total columns     : "
    f"{len(minute_df.columns):,}"
)

print(
    f"Model features    : "
    f"{len(MODEL_FEATURE_COLUMNS):,}"
)

print(
    f"Attack minutes    : "
    f"{minute_df['Attack_State'].sum():,}"
)

print(
    f"Benign minutes    : "
    f"{(minute_df['Attack_State'] == 0).sum():,}"
)

print(
    f"Duplicate minutes : "
    f"{duplicate_minutes:,}"
)

print(
    f"Infinite values   : "
    f"{infinite_values:,}"
)


# =============================================================================
# 17. ATTACK-MINUTE DISTRIBUTION
# =============================================================================

print("\n" + "=" * 115)
print("ATTACK-STATE DISTRIBUTION")
print("=" * 115)

print(
    minute_df[
        "Attack_State"
    ]
    .value_counts()
    .sort_index()
)


# =============================================================================
# 18. ATTACK TYPE DISTRIBUTION AT MINUTE LEVEL
# =============================================================================

print("\n" + "=" * 115)
print("ATTACK TYPES PRESENT IN ATTACK MINUTES")
print("=" * 115)


attack_type_minute_counts = (

    minute_df.loc[
        minute_df["Attack_State"] == 1,
        "Attack_Types"
    ]

    .value_counts()

)


print(
    attack_type_minute_counts
)


# =============================================================================
# 19. CONTINUITY REPORT
# =============================================================================

print("\n" + "=" * 115)
print("MINUTE CONTINUITY BY SEGMENT")
print("=" * 115)

display(
    minute_continuity_df
)


total_missing_minutes = int(
    minute_continuity_df[
        "Missing_Minutes"
    ]
    .sum()
)


print(
    f"\nTotal missing minute bins: "
    f"{total_missing_minutes:,}"
)


# =============================================================================
# 20. MISSING FEATURE REPORT
# =============================================================================

print("\n" + "=" * 115)
print("MISSING MODEL-FEATURE VALUES")
print("=" * 115)


if len(features_with_missing) == 0:

    print(
        "\n✓ No model features contain NaN."
    )

else:

    print(
        f"\nFeatures containing NaN: "
        f"{len(features_with_missing)}"
    )

    print()

    print(
        features_with_missing
        .head(30)
    )


# =============================================================================
# 21. DISPLAY SAMPLE
# =============================================================================

print("\n" + "=" * 115)
print("MINUTE DATASET PREVIEW")
print("=" * 115)


preview_columns = [

    "Segment_ID",
    "Minute",

    "Flow_Count",

    "Unique_Source_IPs",
    "Unique_Destination_IPs",

    "TCP_Flow_Count",
    "UDP_Flow_Count",

    "Attack_State",
    "Attack_Flow_Count",
    "Attack_Ratio",
    "Attack_Types"

]


display(
    minute_df[
        preview_columns
    ]
    .head(20)
)


# =============================================================================
# 22. SAVE COMPLETE MINUTE DATASET
# =============================================================================

minute_dataset_path = os.path.join(
    TEMPORAL_DIR,
    "minute_level_dataset.parquet"
)


minute_df.to_parquet(
    minute_dataset_path,
    index=False,
    compression="snappy"
)


# =============================================================================
# 23. SAVE CONTINUITY AUDIT
# =============================================================================

continuity_path = os.path.join(
    TEMPORAL_DIR,
    "minute_continuity_audit.parquet"
)


minute_continuity_df.to_parquet(
    continuity_path,
    index=False
)


# =============================================================================
# 24. SAVE FEATURE MANIFEST
# =============================================================================
#
# This makes later notebooks reproducible.
#
# =============================================================================

feature_manifest = pd.DataFrame({

    "Feature": MODEL_FEATURE_COLUMNS

})


feature_manifest_path = os.path.join(
    TEMPORAL_DIR,
    "minute_model_feature_manifest.csv"
)


feature_manifest.to_csv(
    feature_manifest_path,
    index=False
)


print("\n" + "=" * 115)
print("SAVED OUTPUTS")
print("=" * 115)

print(
    "\nMinute dataset:"
)

print(
    minute_dataset_path
)

print(
    "\nContinuity audit:"
)

print(
    continuity_path
)

print(
    "\nFeature manifest:"
)

print(
    feature_manifest_path
)


# =============================================================================
# 25. FINAL VALIDATION
# =============================================================================

checks = {

    "Minute dataset is non-empty":
        len(minute_df) > 0,

    "Exactly 10 temporal segments":
        minute_df[
            "Segment_ID"
        ].nunique() == 10,

    "No duplicate Segment_ID + Minute":
        duplicate_minutes == 0,

    "No infinite model values":
        infinite_values == 0,

    "Attack_State contains only 0/1":
        set(
            minute_df[
                "Attack_State"
            ].unique()
        ).issubset(
            {0, 1}
        ),

    "Attack flow count never exceeds flow count":
        bool(
            (
                minute_df[
                    "Attack_Flow_Count"
                ]
                <=
                minute_df[
                    "Flow_Count"
                ]
            )
            .all()
        ),

    "Attack ratio is within [0,1]":
        bool(
            minute_df[
                "Attack_Ratio"
            ]
            .between(
                0,
                1
            )
            .all()
        ),

    "Attack state matches attack flow count":
        bool(
            (
                minute_df[
                    "Attack_State"
                ]
                ==
                minute_df[
                    "Attack_Flow_Count"
                ]
                .gt(0)
                .astype("int8")
            )
            .all()
        ),

    "Minute dataset saved":
        os.path.exists(
            minute_dataset_path
        ),

    "Feature manifest saved":
        os.path.exists(
            feature_manifest_path
        )

}


print("\n" + "=" * 115)
print("FINAL CELL 9 VALIDATION")
print("=" * 115)


for name, result in checks.items():

    print(
        f"{'✓' if bool(result) else '❌'} "
        f"{name}: {bool(result)}"
    )


print("\n" + "=" * 115)


if all(
    bool(value)
    for value in checks.values()
):

    print(
        "✓ CELL 9 COMPLETED SUCCESSFULLY."
    )

    print()

    print(
        "The flow-level CICIDS2017 data has been "
        "converted into a regular minute-level "
        "temporal representation."
    )

    print()

    print(
        "Attack ground truth is preserved separately "
        "from model predictor features."
    )

    print()

    print(
        "NEXT: inspect minute continuity and missing "
        "feature values before constructing forecasting "
        "history/future windows."
    )

else:

    print(
        "❌ CELL 9 VALIDATION FAILED."
    )

    print(
        "Do NOT construct forecasting windows until "
        "the failed checks are investigated."
    )


print("=" * 115)


# =============================================================================
# 26. CLEANUP
# =============================================================================

gc.collect()

CELL 9 — MINUTE-LEVEL TEMPORAL DATASET CONSTRUCTION

Processed flow files found: 8

Temporal segments available: 10

Flow-level columns selected: 58
Predictor source features: 50



###################################################################################################################
PROCESSING FILE 1/8
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX_cleaned.parquet
###################################################################################################################

Flows loaded : 225,745
Columns loaded: 58
Unassigned flows: 0

Minute rows produced : 93
Attack minutes       : 21
Benign minutes       : 72
Minute feature count : 132
✓ Flow-level dataframe deleted from RAM.



###################################################################################################################
PROCESSING FILE 2/8
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX_cleaned.parquet
###########################################################################################

,Segment_ID,Start_Minute,End_Minute,Expected_Minutes,Observed_Minutes,Missing_Minutes
0,0,2017-07-03 01:00:00,2017-07-03 05:01:00,242,242,0
1,1,2017-07-03 08:55:00,2017-07-03 12:59:00,245,245,0
2,2,2017-07-04 01:00:00,2017-07-04 05:00:00,241,241,0
3,3,2017-07-04 08:53:00,2017-07-04 12:59:00,247,247,0
4,4,2017-07-05 01:00:00,2017-07-05 05:10:00,251,251,0
5,5,2017-07-05 08:42:00,2017-07-05 12:59:00,258,258,0
6,6,2017-07-06 01:00:00,2017-07-06 05:04:00,245,245,0
7,7,2017-07-06 08:59:00,2017-07-06 12:59:00,241,241,0
8,8,2017-07-07 01:00:00,2017-07-07 05:02:00,243,243,0
9,9,2017-07-07 08:59:00,2017-07-07 12:59:00,241,241,0



Total missing minute bins: 0

MISSING MODEL-FEATURE VALUES

✓ No model features contain NaN.

MINUTE DATASET PREVIEW


,Segment_ID,Minute,Flow_Count,Unique_Source_IPs,Unique_Destination_IPs,TCP_Flow_Count,UDP_Flow_Count,Attack_State,Attack_Flow_Count,Attack_Ratio,Attack_Types
0,0,2017-07-03 01:00:00,171,31,47,134,36,0,0,0.0,NaN
1,0,2017-07-03 01:01:00,875,65,88,714,161,0,0,0.0,NaN
2,0,2017-07-03 01:02:00,1317,89,205,934,382,0,0,0.0,NaN
3,0,2017-07-03 01:03:00,1102,82,143,671,431,0,0,0.0,NaN
4,0,2017-07-03 01:04:00,633,94,91,476,155,0,0,0.0,NaN
5,0,2017-07-03 01:05:00,872,111,132,655,217,0,0,0.0,NaN
6,0,2017-07-03 01:06:00,619,50,89,422,196,0,0,0.0,NaN
7,0,2017-07-03 01:07:00,1258,72,116,927,331,0,0,0.0,NaN
8,0,2017-07-03 01:08:00,441,66,103,302,138,0,0,0.0,NaN
9,0,2017-07-03 01:09:00,503,55,106,295,208,0,0,0.0,NaN



SAVED OUTPUTS

Minute dataset:
/content/drive/MyDrive/Network-Attack-Forecasting/data/temporal/minute_level_dataset.parquet

Continuity audit:
/content/drive/MyDrive/Network-Attack-Forecasting/data/temporal/minute_continuity_audit.parquet

Feature manifest:
/content/drive/MyDrive/Network-Attack-Forecasting/data/temporal/minute_model_feature_manifest.csv

FINAL CELL 9 VALIDATION
✓ Minute dataset is non-empty: True
✓ Exactly 10 temporal segments: True
✓ No duplicate Segment_ID + Minute: True
✓ No infinite model values: True
✓ Attack_State contains only 0/1: True
✓ Attack flow count never exceeds flow count: True
✓ Attack ratio is within [0,1]: True
✓ Attack state matches attack flow count: True
✓ Minute dataset saved: True
✓ Feature manifest saved: True

✓ CELL 9 COMPLETED SUCCESSFULLY.

The flow-level CICIDS2017 data has been converted into a regular minute-level temporal representation.

Attack ground truth is preserved separately from model predictor features.

NEXT: inspect minute c

0

In [16]:
# =============================================================================
# CELL 10 — CONSTRUCT FORECASTING WINDOWS AND FUTURE TARGETS
# =============================================================================
#
# PROJECT:
# CICIDS2017 Network Attack Forecasting
#
#
# PURPOSE
# -------
#
# Cell 9 produced a regular temporal dataset:
#
#       ONE ROW = ONE MINUTE OF NETWORK BEHAVIOUR
#
# Cell 10 converts that temporal dataset into a FORECASTING problem:
#
#
#       <--------- OBSERVED HISTORY --------->   <---- FUTURE ---->
#
#       t-9  t-8  ...  t-2  t-1  t       |     t+1 ... t+5
#       -------------------------------     |     ------------
#                MODEL INPUT               |       TARGET
#
#
# We use:
#
#       HISTORY LENGTH   = 10 minutes
#       FORECAST HORIZON = 5 minutes
#
#
# For every valid forecasting sample we construct:
#
#   1. Future_Attack
#
#          0 = no attack in next 5 minutes
#          1 = at least one attack in next 5 minutes
#
#
#   2. Lead_Time_Minutes
#
#          Number of minutes from history end until the
#          FIRST attack minute in the future horizon.
#
#          Possible values:
#
#              1, 2, 3, 4, 5
#
#          NaN when Future_Attack = 0.
#
#
#   3. First_Future_Attack_Time
#
#          Timestamp of first future attack.
#
#
#   4. Future_Attack_Types
#
#          All attack types appearing anywhere in the
#          5-minute future horizon.
#
#
#   5. First_Future_Attack_Types
#
#          Attack type(s) present specifically at the FIRST
#          future attack minute.
#
#
# IMPORTANT
# ---------
#
# History and future MUST remain inside the same Segment_ID.
#
# We NEVER allow a forecasting window to cross one of the
# 9 CICIDS2017 capture discontinuities.
#
#
# ALSO IMPORTANT
# --------------
#
# We are NOT training a model here.
#
# This is still DATASET PREPARATION.
#
# =============================================================================


import os
import gc
import numpy as np
import pandas as pd


# =============================================================================
# 1. CONFIGURATION
# =============================================================================

HISTORY_MINUTES = 10
FORECAST_MINUTES = 5


PROJECT_ROOT = (
    "/content/drive/MyDrive/"
    "Network-Attack-Forecasting"
)

TEMPORAL_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "temporal"
)

FORECAST_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "forecasting"
)

os.makedirs(
    FORECAST_DIR,
    exist_ok=True
)


MINUTE_DATASET_PATH = os.path.join(
    TEMPORAL_DIR,
    "minute_level_dataset.parquet"
)


# =============================================================================
# 2. LOAD MINUTE-LEVEL DATASET
# =============================================================================

assert os.path.exists(
    MINUTE_DATASET_PATH
), (
    "minute_level_dataset.parquet not found. "
    "Run Cell 9 first."
)


minute_df = pd.read_parquet(
    MINUTE_DATASET_PATH
)


minute_df["Minute"] = pd.to_datetime(
    minute_df["Minute"]
)


minute_df = (
    minute_df
    .sort_values(
        [
            "Segment_ID",
            "Minute"
        ]
    )
    .reset_index(drop=True)
)


print("=" * 110)
print("CELL 10 — FORECASTING WINDOW + TARGET CONSTRUCTION")
print("=" * 110)

print(
    f"\nMinute rows loaded : "
    f"{len(minute_df):,}"
)

print(
    f"Columns loaded     : "
    f"{len(minute_df.columns):,}"
)

print(
    f"Segments           : "
    f"{minute_df['Segment_ID'].nunique()}"
)

print(
    f"\nHistory length     : "
    f"{HISTORY_MINUTES} minutes"
)

print(
    f"Forecast horizon   : "
    f"{FORECAST_MINUTES} minutes"
)


# =============================================================================
# 3. REQUIRED COLUMN CHECK
# =============================================================================

required_columns = [

    "Segment_ID",
    "Minute",

    "Attack_State",
    "Attack_Flow_Count",
    "Attack_Ratio",
    "Attack_Types"

]


missing_columns = [

    column
    for column in required_columns
    if column not in minute_df.columns

]


assert len(missing_columns) == 0, (
    f"Missing required columns: "
    f"{missing_columns}"
)


print(
    "\n✓ Required temporal and ground-truth "
    "columns exist."
)


# =============================================================================
# 4. VERIFY EACH SEGMENT IS MINUTE-CONTINUOUS
# =============================================================================
#
# Cell 9 already found zero missing minute bins.
#
# We verify it again because forecasting windows require
# exact minute-to-minute continuity.
#
# =============================================================================

continuity_failures = []


for segment_id, segment in minute_df.groupby(
    "Segment_ID",
    sort=True
):

    segment = (
        segment
        .sort_values("Minute")
        .reset_index(drop=True)
    )


    gaps = (
        segment["Minute"]
        .diff()
        .dropna()
    )


    invalid_gaps = gaps[
        gaps != pd.Timedelta(minutes=1)
    ]


    if len(invalid_gaps) > 0:

        continuity_failures.append(
            {
                "Segment_ID": segment_id,
                "Invalid_Gaps": len(invalid_gaps)
            }
        )


assert len(continuity_failures) == 0, (
    "Minute continuity failure detected. "
    "Do NOT construct forecasting windows."
)


print(
    "✓ Every segment is exactly minute-continuous."
)


# =============================================================================
# 5. IDENTIFY MODEL FEATURE COLUMNS
# =============================================================================
#
# These are the same 125 potential predictors from Cell 9.
#
# Ground-truth information is EXCLUDED.
#
# =============================================================================

GROUND_TRUTH_COLUMNS = [

    "Attack_State",
    "Attack_Flow_Count",
    "Attack_Ratio",
    "Attack_Types"

]


IDENTIFIER_COLUMNS = [

    "Segment_ID",
    "Minute",
    "Source_File"

]


MODEL_FEATURE_COLUMNS = [

    column
    for column in minute_df.columns

    if column not in (
        GROUND_TRUTH_COLUMNS
        +
        IDENTIFIER_COLUMNS
    )

]


print(
    f"\nPotential model features: "
    f"{len(MODEL_FEATURE_COLUMNS)}"
)


assert len(MODEL_FEATURE_COLUMNS) == 125, (
    "Expected 125 model features from Cell 9. "
    f"Found {len(MODEL_FEATURE_COLUMNS)}."
)


# =============================================================================
# 6. HELPER — PARSE ATTACK TYPE STRING
# =============================================================================
#
# Attack_Types can contain:
#
#       DDoS
#
# or theoretically:
#
#       Type_A|Type_B
#
# This function converts the stored representation into a set.
#
# =============================================================================

def parse_attack_types(value):

    if pd.isna(value):
        return set()

    value = str(value).strip()

    if value == "":
        return set()

    return {
        item.strip()
        for item in value.split("|")
        if item.strip()
    }


# =============================================================================
# 7. CONSTRUCT FORECASTING SAMPLE METADATA
# =============================================================================
#
# IMPORTANT:
#
# We do NOT duplicate all 125 × 10 history values into a giant dataframe yet.
#
# Instead, each forecasting sample stores:
#
#       Segment_ID
#       History_Start
#       History_End
#       Forecast_Start
#       Forecast_End
#       History_Start_Index
#       History_End_Index
#
# This allows later cells to reconstruct model inputs efficiently.
#
# =============================================================================

forecast_records = []


sample_id = 0


for segment_id, segment in minute_df.groupby(
    "Segment_ID",
    sort=True
):

    segment = (
        segment
        .sort_values("Minute")
        .reset_index()
        .rename(
            columns={
                "index": "Global_Row_Index"
            }
        )
    )


    n_minutes = len(segment)


    possible_samples = (

        n_minutes
        -
        HISTORY_MINUTES
        -
        FORECAST_MINUTES
        +
        1

    )


    if possible_samples <= 0:

        print(
            f"\nSegment {segment_id}: "
            f"too short for forecasting."
        )

        continue


    print(
        f"\nSegment {segment_id}: "
        f"{n_minutes} minutes -> "
        f"{possible_samples} candidate samples"
    )


    for history_start_pos in range(
        possible_samples
    ):

        history_end_pos = (
            history_start_pos
            +
            HISTORY_MINUTES
            -
            1
        )


        forecast_start_pos = (
            history_end_pos
            +
            1
        )


        forecast_end_pos = (
            forecast_start_pos
            +
            FORECAST_MINUTES
            -
            1
        )


        # ---------------------------------------------------------------------
        # HISTORY WINDOW
        # ---------------------------------------------------------------------

        history = segment.iloc[
            history_start_pos:
            history_end_pos + 1
        ]


        # ---------------------------------------------------------------------
        # FUTURE WINDOW
        # ---------------------------------------------------------------------

        future = segment.iloc[
            forecast_start_pos:
            forecast_end_pos + 1
        ]


        # ---------------------------------------------------------------------
        # STRICT WINDOW SIZE CHECK
        # ---------------------------------------------------------------------

        assert len(history) == HISTORY_MINUTES

        assert len(future) == FORECAST_MINUTES


        # ---------------------------------------------------------------------
        # STRICT TEMPORAL CHECK
        # ---------------------------------------------------------------------

        history_start_time = (
            history["Minute"].iloc[0]
        )

        history_end_time = (
            history["Minute"].iloc[-1]
        )

        forecast_start_time = (
            future["Minute"].iloc[0]
        )

        forecast_end_time = (
            future["Minute"].iloc[-1]
        )


        assert (
            forecast_start_time
            ==
            history_end_time
            +
            pd.Timedelta(minutes=1)
        )


        # ---------------------------------------------------------------------
        # FUTURE ATTACK TARGET
        # ---------------------------------------------------------------------

        future_attack_mask = (
            future["Attack_State"]
            .astype(int)
            .eq(1)
        )


        future_attack = int(
            future_attack_mask.any()
        )


        # ---------------------------------------------------------------------
        # DEFAULT NEGATIVE TARGET VALUES
        # ---------------------------------------------------------------------

        first_future_attack_time = pd.NaT

        lead_time_minutes = np.nan

        first_future_attack_types = None


        # ---------------------------------------------------------------------
        # ALL ATTACK TYPES IN FUTURE HORIZON
        # ---------------------------------------------------------------------

        all_future_attack_types = set()


        for value in future.loc[
            future_attack_mask,
            "Attack_Types"
        ]:

            all_future_attack_types.update(
                parse_attack_types(value)
            )


        if len(all_future_attack_types) > 0:

            future_attack_types = "|".join(
                sorted(
                    all_future_attack_types
                )
            )

        else:

            future_attack_types = None


        # ---------------------------------------------------------------------
        # POSITIVE SAMPLE:
        # FIND FIRST FUTURE ATTACK
        # ---------------------------------------------------------------------

        if future_attack == 1:

            first_attack_row = (

                future.loc[
                    future_attack_mask
                ]

                .iloc[0]

            )


            first_future_attack_time = (
                first_attack_row["Minute"]
            )


            # -----------------------------------------------------------------
            # LEAD TIME
            # -----------------------------------------------------------------
            #
            # History ends at t.
            #
            # If first attack is:
            #
            #       t+1 -> lead time = 1
            #       t+2 -> lead time = 2
            #       ...
            #       t+5 -> lead time = 5
            #
            # -----------------------------------------------------------------

            lead_time_minutes = (

                first_future_attack_time
                -
                history_end_time

            ).total_seconds() / 60.0


            # -----------------------------------------------------------------
            # ATTACK TYPE AT FIRST ATTACK MINUTE
            # -----------------------------------------------------------------

            first_types = parse_attack_types(
                first_attack_row[
                    "Attack_Types"
                ]
            )


            if len(first_types) > 0:

                first_future_attack_types = (
                    "|".join(
                        sorted(
                            first_types
                        )
                    )
                )


        # ---------------------------------------------------------------------
        # HISTORY ATTACK CONTEXT
        # ---------------------------------------------------------------------
        #
        # IMPORTANT:
        #
        # These columns are for AUDITING only.
        #
        # We are NOT using Attack_State as a model input.
        #
        # They help us later distinguish:
        #
        #   benign history -> future attack
        #
        # from
        #
        #   already-under-attack history -> continued attack
        #
        # This distinction matters enormously for genuine early forecasting.
        #
        # ---------------------------------------------------------------------

        history_contains_attack = int(
            history["Attack_State"]
            .astype(int)
            .eq(1)
            .any()
        )


        history_attack_minutes = int(
            history["Attack_State"]
            .astype(int)
            .sum()
        )


        # ---------------------------------------------------------------------
        # FUTURE ATTACK MINUTE COUNT
        # ---------------------------------------------------------------------

        future_attack_minutes = int(
            future_attack_mask.sum()
        )


        # ---------------------------------------------------------------------
        # STORE SAMPLE
        # ---------------------------------------------------------------------

        forecast_records.append(
            {

                "Sample_ID":
                    sample_id,

                "Segment_ID":
                    int(segment_id),

                # -------------------------------------------------------------
                # Window timestamps
                # -------------------------------------------------------------

                "History_Start":
                    history_start_time,

                "History_End":
                    history_end_time,

                "Forecast_Start":
                    forecast_start_time,

                "Forecast_End":
                    forecast_end_time,

                # -------------------------------------------------------------
                # Row references back into minute_df
                # -------------------------------------------------------------

                "History_Start_Index":
                    int(
                        history[
                            "Global_Row_Index"
                        ].iloc[0]
                    ),

                "History_End_Index":
                    int(
                        history[
                            "Global_Row_Index"
                        ].iloc[-1]
                    ),

                # -------------------------------------------------------------
                # Main forecasting targets
                # -------------------------------------------------------------

                "Future_Attack":
                    future_attack,

                "First_Future_Attack_Time":
                    first_future_attack_time,

                "Lead_Time_Minutes":
                    lead_time_minutes,

                "First_Future_Attack_Types":
                    first_future_attack_types,

                "Future_Attack_Types":
                    future_attack_types,

                # -------------------------------------------------------------
                # Audit metadata
                # -------------------------------------------------------------

                "History_Contains_Attack":
                    history_contains_attack,

                "History_Attack_Minutes":
                    history_attack_minutes,

                "Future_Attack_Minutes":
                    future_attack_minutes

            }
        )


        sample_id += 1


# =============================================================================
# 8. CREATE FORECASTING DATAFRAME
# =============================================================================

forecast_df = pd.DataFrame(
    forecast_records
)


assert len(forecast_df) > 0


print("\n")
print("=" * 110)
print("FORECASTING DATASET CONSTRUCTED")
print("=" * 110)

print(
    f"\nTotal forecasting samples: "
    f"{len(forecast_df):,}"
)


# =============================================================================
# 9. BASIC TARGET DISTRIBUTION
# =============================================================================

positive_samples = int(
    forecast_df[
        "Future_Attack"
    ].sum()
)


negative_samples = int(
    (
        forecast_df[
            "Future_Attack"
        ]
        == 0
    ).sum()
)


positive_percentage = (
    positive_samples
    /
    len(forecast_df)
    *
    100
)


print("\n" + "=" * 110)
print("FUTURE ATTACK TARGET DISTRIBUTION")
print("=" * 110)

print(
    f"\nPositive samples : "
    f"{positive_samples:,}"
)

print(
    f"Negative samples : "
    f"{negative_samples:,}"
)

print(
    f"Positive rate    : "
    f"{positive_percentage:.2f}%"
)


# =============================================================================
# 10. LEAD-TIME DISTRIBUTION
# =============================================================================

positive_df = forecast_df.loc[
    forecast_df["Future_Attack"] == 1
].copy()


print("\n" + "=" * 110)
print("LEAD-TIME DISTRIBUTION")
print("=" * 110)


print(
    positive_df[
        "Lead_Time_Minutes"
    ]
    .value_counts()
    .sort_index()
)


# =============================================================================
# 11. FUTURE ATTACK TYPE DISTRIBUTION
# =============================================================================

print("\n" + "=" * 110)
print("FIRST FUTURE ATTACK TYPE DISTRIBUTION")
print("=" * 110)


print(
    positive_df[
        "First_Future_Attack_Types"
    ]
    .value_counts(
        dropna=False
    )
)


# =============================================================================
# 12. MULTI-TYPE FUTURE HORIZON AUDIT
# =============================================================================
#
# We do NOT yet force attack type into a simple multiclass target.
#
# First we need to know how often a 5-minute horizon contains
# more than one attack type.
#
# =============================================================================

def count_types(value):

    if pd.isna(value):
        return 0

    return len(
        parse_attack_types(value)
    )


positive_df[
    "Future_Attack_Type_Count"
] = (

    positive_df[
        "Future_Attack_Types"
    ]

    .apply(
        count_types
    )

)


multi_type_samples = int(

    (
        positive_df[
            "Future_Attack_Type_Count"
        ]
        > 1
    )

    .sum()

)


print("\n" + "=" * 110)
print("MULTI-ATTACK-TYPE HORIZON AUDIT")
print("=" * 110)

print(
    f"\nPositive samples containing "
    f">1 future attack type: "
    f"{multi_type_samples:,}"
)


if positive_samples > 0:

    print(
        f"Percentage of positive samples: "
        f"{multi_type_samples / positive_samples * 100:.2f}%"
    )


# =============================================================================
# 13. HISTORY-ATTACK CONTAMINATION AUDIT
# =============================================================================
#
# This is VERY important.
#
# A sample may predict an attack in the next 5 minutes while the
# observed 10-minute history already contains attack traffic.
#
# Such samples may represent attack continuation rather than true
# pre-attack forecasting.
#
# We do NOT remove them yet.
#
# Cell 11 will inspect this carefully before we decide how the final
# forecasting task should be defined.
#
# =============================================================================

history_attack_table = pd.crosstab(

    forecast_df[
        "History_Contains_Attack"
    ],

    forecast_df[
        "Future_Attack"
    ],

    rownames=[
        "History_Contains_Attack"
    ],

    colnames=[
        "Future_Attack"
    ]

)


print("\n" + "=" * 110)
print("HISTORY ATTACK CONTEXT VS FUTURE ATTACK")
print("=" * 110)

print()

print(
    history_attack_table
)


clean_history_positive = int(

    (
        (
            forecast_df[
                "History_Contains_Attack"
            ]
            == 0
        )

        &

        (
            forecast_df[
                "Future_Attack"
            ]
            == 1
        )
    )

    .sum()

)


print(
    f"\nBenign-history → future-attack samples: "
    f"{clean_history_positive:,}"
)


# =============================================================================
# 14. TEMPORAL VALIDITY CHECKS
# =============================================================================

positive_mask = (
    forecast_df[
        "Future_Attack"
    ]
    == 1
)


negative_mask = (
    forecast_df[
        "Future_Attack"
    ]
    == 0
)


check_positive_has_timestamp = bool(

    forecast_df.loc[
        positive_mask,
        "First_Future_Attack_Time"
    ]

    .notna()

    .all()

)


check_positive_has_lead_time = bool(

    forecast_df.loc[
        positive_mask,
        "Lead_Time_Minutes"
    ]

    .notna()

    .all()

)


check_negative_no_lead_time = bool(

    forecast_df.loc[
        negative_mask,
        "Lead_Time_Minutes"
    ]

    .isna()

    .all()

)


check_attack_after_history = bool(

    (
        forecast_df.loc[
            positive_mask,
            "First_Future_Attack_Time"
        ]

        >

        forecast_df.loc[
            positive_mask,
            "History_End"
        ]
    )

    .all()

)


check_attack_inside_horizon = bool(

    (
        forecast_df.loc[
            positive_mask,
            "First_Future_Attack_Time"
        ]

        <=

        forecast_df.loc[
            positive_mask,
            "Forecast_End"
        ]
    )

    .all()

)


check_lead_time_range = bool(

    forecast_df.loc[
        positive_mask,
        "Lead_Time_Minutes"
    ]

    .between(
        1,
        FORECAST_MINUTES
    )

    .all()

)


# =============================================================================
# 15. EXPECTED SAMPLE COUNT CHECK
# =============================================================================
#
# Each segment contributes:
#
#       segment_minutes - history - forecast + 1
#
# =============================================================================

expected_samples = 0


for _, segment in minute_df.groupby(
    "Segment_ID"
):

    n = len(segment)

    expected_samples += max(
        0,
        n
        -
        HISTORY_MINUTES
        -
        FORECAST_MINUTES
        +
        1
    )


check_sample_count = (
    len(forecast_df)
    ==
    expected_samples
)


# =============================================================================
# 16. DUPLICATE SAMPLE CHECK
# =============================================================================

duplicate_samples = int(

    forecast_df

    .duplicated(
        subset=[
            "Segment_ID",
            "History_Start",
            "History_End"
        ]
    )

    .sum()

)


# =============================================================================
# 17. DISPLAY EXAMPLE POSITIVE WINDOWS
# =============================================================================

print("\n" + "=" * 110)
print("EXAMPLE POSITIVE FORECASTING WINDOWS")
print("=" * 110)


display_columns = [

    "Sample_ID",
    "Segment_ID",

    "History_Start",
    "History_End",

    "Forecast_Start",
    "Forecast_End",

    "Future_Attack",

    "First_Future_Attack_Time",
    "Lead_Time_Minutes",

    "First_Future_Attack_Types",
    "Future_Attack_Types",

    "History_Contains_Attack"

]


display(
    forecast_df.loc[
        forecast_df[
            "Future_Attack"
        ]
        == 1,
        display_columns
    ]
    .head(20)
)


# =============================================================================
# 18. SAVE FORECASTING SAMPLE METADATA
# =============================================================================

FORECAST_METADATA_PATH = os.path.join(
    FORECAST_DIR,
    "forecast_samples.parquet"
)


forecast_df.to_parquet(
    FORECAST_METADATA_PATH,
    index=False,
    compression="snappy"
)


# =============================================================================
# 19. SAVE MODEL FEATURE MANIFEST
# =============================================================================

FORECAST_FEATURE_MANIFEST_PATH = os.path.join(
    FORECAST_DIR,
    "forecast_model_features.csv"
)


pd.DataFrame(
    {
        "Feature":
            MODEL_FEATURE_COLUMNS
    }
).to_csv(
    FORECAST_FEATURE_MANIFEST_PATH,
    index=False
)


# =============================================================================
# 20. SAVE FORECASTING CONFIGURATION
# =============================================================================

FORECAST_CONFIG_PATH = os.path.join(
    FORECAST_DIR,
    "forecast_configuration.csv"
)


pd.DataFrame(
    {
        "Parameter": [
            "History_Minutes",
            "Forecast_Minutes",
            "Minute_Feature_Count",
            "Segment_Count"
        ],

        "Value": [
            HISTORY_MINUTES,
            FORECAST_MINUTES,
            len(MODEL_FEATURE_COLUMNS),
            minute_df[
                "Segment_ID"
            ].nunique()
        ]
    }
).to_csv(
    FORECAST_CONFIG_PATH,
    index=False
)


# =============================================================================
# 21. FINAL VALIDATION
# =============================================================================

checks = {

    "Forecast dataset non-empty":
        len(forecast_df) > 0,

    "Expected number of samples":
        check_sample_count,

    "No duplicate forecasting samples":
        duplicate_samples == 0,

    "Positive samples have attack timestamp":
        check_positive_has_timestamp,

    "Positive samples have lead time":
        check_positive_has_lead_time,

    "Negative samples have no lead time":
        check_negative_no_lead_time,

    "Future attack occurs after history":
        check_attack_after_history,

    "Future attack lies inside forecast horizon":
        check_attack_inside_horizon,

    "Lead time lies between 1 and 5 minutes":
        check_lead_time_range,

    "Forecast metadata saved":
        os.path.exists(
            FORECAST_METADATA_PATH
        ),

    "Feature manifest saved":
        os.path.exists(
            FORECAST_FEATURE_MANIFEST_PATH
        ),

    "Forecast configuration saved":
        os.path.exists(
            FORECAST_CONFIG_PATH
        )

}


print("\n" + "=" * 110)
print("FINAL CELL 10 VALIDATION")
print("=" * 110)


for name, result in checks.items():

    print(
        f"{'✓' if bool(result) else '❌'} "
        f"{name}: {bool(result)}"
    )


passed = sum(
    bool(value)
    for value in checks.values()
)


print(
    f"\nPassed: "
    f"{passed} / {len(checks)}"
)


print("\n" + "=" * 110)


if all(
    bool(value)
    for value in checks.values()
):

    print(
        "✓ CELL 10 COMPLETED SUCCESSFULLY."
    )

    print()

    print(
        "Forecasting windows and future targets "
        "have been constructed."
    )

    print()

    print(
        "IMPORTANT: DO NOT TRAIN MODELS YET."
    )

    print()

    print(
        "NEXT: Cell 11 will audit forecasting "
        "sample independence, attack episodes, "
        "lead-time behaviour, attack-type targets, "
        "and history contamination."
    )

else:

    print(
        "❌ CELL 10 VALIDATION FAILED."
    )

    print(
        "Do NOT proceed until failed checks "
        "are investigated."
    )


print("=" * 110)


# =============================================================================
# 22. OUTPUT PATHS
# =============================================================================

print("\nSaved forecasting metadata:")
print(
    FORECAST_METADATA_PATH
)

print("\nSaved feature manifest:")
print(
    FORECAST_FEATURE_MANIFEST_PATH
)

print("\nSaved forecasting configuration:")
print(
    FORECAST_CONFIG_PATH
)


# =============================================================================
# 23. CLEANUP
# =============================================================================

gc.collect()

CELL 10 — FORECASTING WINDOW + TARGET CONSTRUCTION

Minute rows loaded : 2,454
Columns loaded     : 132
Segments           : 10

History length     : 10 minutes
Forecast horizon   : 5 minutes

✓ Required temporal and ground-truth columns exist.
✓ Every segment is exactly minute-continuous.

Potential model features: 125

Segment 0: 242 minutes -> 228 candidate samples

Segment 1: 245 minutes -> 231 candidate samples

Segment 2: 241 minutes -> 227 candidate samples

Segment 3: 247 minutes -> 233 candidate samples

Segment 4: 251 minutes -> 237 candidate samples

Segment 5: 258 minutes -> 244 candidate samples

Segment 6: 245 minutes -> 231 candidate samples

Segment 7: 241 minutes -> 227 candidate samples

Segment 8: 243 minutes -> 229 candidate samples

Segment 9: 241 minutes -> 227 candidate samples


FORECASTING DATASET CONSTRUCTED

Total forecasting samples: 2,314

FUTURE ATTACK TARGET DISTRIBUTION

Positive samples : 682
Negative samples : 1,632
Positive rate    : 29.47%

LEAD-TIME

,Sample_ID,Segment_ID,History_Start,History_End,Forecast_Start,Forecast_End,Future_Attack,First_Future_Attack_Time,Lead_Time_Minutes,First_Future_Attack_Types,Future_Attack_Types,History_Contains_Attack
514,514,2,2017-07-04 01:55:00,2017-07-04 02:04:00,2017-07-04 02:05:00,2017-07-04 02:09:00,1,2017-07-04 02:09:00,5.0,SSH-Patator,SSH-Patator,0
515,515,2,2017-07-04 01:56:00,2017-07-04 02:05:00,2017-07-04 02:06:00,2017-07-04 02:10:00,1,2017-07-04 02:09:00,4.0,SSH-Patator,SSH-Patator,0
516,516,2,2017-07-04 01:57:00,2017-07-04 02:06:00,2017-07-04 02:07:00,2017-07-04 02:11:00,1,2017-07-04 02:09:00,3.0,SSH-Patator,SSH-Patator,0
517,517,2,2017-07-04 01:58:00,2017-07-04 02:07:00,2017-07-04 02:08:00,2017-07-04 02:12:00,1,2017-07-04 02:09:00,2.0,SSH-Patator,SSH-Patator,0
518,518,2,2017-07-04 01:59:00,2017-07-04 02:08:00,2017-07-04 02:09:00,2017-07-04 02:13:00,1,2017-07-04 02:09:00,1.0,SSH-Patator,SSH-Patator,0
519,519,2,2017-07-04 02:00:00,2017-07-04 02:09:00,2017-07-04 02:10:00,2017-07-04 02:14:00,1,2017-07-04 02:10:00,1.0,SSH-Patator,SSH-Patator,1
520,520,2,2017-07-04 02:01:00,2017-07-04 02:10:00,2017-07-04 02:11:00,2017-07-04 02:15:00,1,2017-07-04 02:11:00,1.0,SSH-Patator,SSH-Patator,1
521,521,2,2017-07-04 02:02:00,2017-07-04 02:11:00,2017-07-04 02:12:00,2017-07-04 02:16:00,1,2017-07-04 02:12:00,1.0,SSH-Patator,SSH-Patator,1
522,522,2,2017-07-04 02:03:00,2017-07-04 02:12:00,2017-07-04 02:13:00,2017-07-04 02:17:00,1,2017-07-04 02:13:00,1.0,SSH-Patator,SSH-Patator,1
523,523,2,2017-07-04 02:04:00,2017-07-04 02:13:00,2017-07-04 02:14:00,2017-07-04 02:18:00,1,2017-07-04 02:14:00,1.0,SSH-Patator,SSH-Patator,1



FINAL CELL 10 VALIDATION
✓ Forecast dataset non-empty: True
✓ Expected number of samples: True
✓ No duplicate forecasting samples: True
✓ Positive samples have attack timestamp: True
✓ Positive samples have lead time: True
✓ Negative samples have no lead time: True
✓ Future attack occurs after history: True
✓ Future attack lies inside forecast horizon: True
✓ Lead time lies between 1 and 5 minutes: True
✓ Forecast metadata saved: True
✓ Feature manifest saved: True
✓ Forecast configuration saved: True

Passed: 12 / 12

✓ CELL 10 COMPLETED SUCCESSFULLY.

Forecasting windows and future targets have been constructed.

IMPORTANT: DO NOT TRAIN MODELS YET.

NEXT: Cell 11 will audit forecasting sample independence, attack episodes, lead-time behaviour, attack-type targets, and history contamination.

Saved forecasting metadata:
/content/drive/MyDrive/Network-Attack-Forecasting/data/forecasting/forecast_samples.parquet

Saved feature manifest:
/content/drive/MyDrive/Network-Attack-Forecasting

0

In [17]:
# =============================================================================
# CELL 11 — ATTACK EPISODE / ONSET / FORECASTABILITY AUDIT
# =============================================================================
#
# PROJECT:
# CICIDS2017 Network Attack Forecasting
#
#
# PURPOSE
# -------
#
# Cell 10 created 2,314 overlapping forecasting windows.
#
# However:
#
#       MANY positive windows != MANY independent attacks
#
# Example:
#
#       02:09  ATTACK
#       02:10  ATTACK
#       02:11  ATTACK
#       ...
#
# A long attack can generate many positive sliding windows.
#
# Therefore, before training any forecasting model, we need to determine:
#
#   1. How many DISTINCT attack episodes exist?
#
#   2. When does each episode START and END?
#
#   3. Which attack type(s) occur in each episode?
#
#   4. How long does each episode last?
#
#   5. How many forecasting windows correspond to each attack onset?
#
#   6. How many episodes actually have attack-free history before onset?
#
#   7. Which episodes can genuinely be forecast:
#
#          1 minute before onset?
#          2 minutes before onset?
#          3 minutes before onset?
#          4 minutes before onset?
#          5 minutes before onset?
#
#   8. How many independent forecastable episodes exist PER ATTACK TYPE?
#
#
# IMPORTANT
# ---------
#
# This cell DOES NOT train anything.
#
# It also DOES NOT delete attack-continuation windows.
#
# It only audits the temporal structure so that we can decide the
# scientifically correct forecasting formulation later.
#
#
# EPISODE DEFINITION USED HERE
# ----------------------------
#
# At the minute level:
#
#       Attack_State = 1
#
# indicates that attack traffic occurred during that minute.
#
# Consecutive attack minutes within the SAME Segment_ID are initially
# treated as belonging to the same continuous attack episode.
#
# A new episode begins when:
#
#       previous minute = benign
#                   OR
#       Segment_ID changes
#
#
# NOTE:
# -----
#
# This is the strict "continuous attack-state" definition.
#
# Later in this cell we ALSO perform a gap-merging sensitivity audit
# to see whether short benign gaps split what may effectively be the
# same larger attack campaign.
#
# =============================================================================


import os
import gc
import numpy as np
import pandas as pd


# =============================================================================
# 1. CONFIGURATION
# =============================================================================

HISTORY_MINUTES = 10
FORECAST_MINUTES = 5


PROJECT_ROOT = (
    "/content/drive/MyDrive/"
    "Network-Attack-Forecasting"
)

TEMPORAL_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "temporal"
)

FORECAST_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "forecasting"
)

AUDIT_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "audits"
)


os.makedirs(
    AUDIT_DIR,
    exist_ok=True
)


MINUTE_DATASET_PATH = os.path.join(
    TEMPORAL_DIR,
    "minute_level_dataset.parquet"
)


FORECAST_METADATA_PATH = os.path.join(
    FORECAST_DIR,
    "forecast_samples.parquet"
)


# =============================================================================
# 2. LOAD DATASETS FROM DISK
# =============================================================================
#
# We reload from disk intentionally.
#
# This makes Cell 11 reproducible even if Colab runtime variables were deleted.
#
# =============================================================================

assert os.path.exists(
    MINUTE_DATASET_PATH
), (
    "minute_level_dataset.parquet not found."
)


assert os.path.exists(
    FORECAST_METADATA_PATH
), (
    "forecast_samples.parquet not found. "
    "Run Cell 10 first."
)


minute_df = pd.read_parquet(
    MINUTE_DATASET_PATH
)


forecast_df = pd.read_parquet(
    FORECAST_METADATA_PATH
)


minute_df["Minute"] = pd.to_datetime(
    minute_df["Minute"]
)


for column in [

    "History_Start",
    "History_End",
    "Forecast_Start",
    "Forecast_End",
    "First_Future_Attack_Time"

]:

    if column in forecast_df.columns:

        forecast_df[column] = pd.to_datetime(
            forecast_df[column]
        )


minute_df = (

    minute_df

    .sort_values(
        [
            "Segment_ID",
            "Minute"
        ]
    )

    .reset_index(drop=True)

)


forecast_df = (

    forecast_df

    .sort_values(
        [
            "Segment_ID",
            "History_Start"
        ]
    )

    .reset_index(drop=True)

)


print("=" * 120)
print("CELL 11 — ATTACK EPISODE / ONSET / FORECASTABILITY AUDIT")
print("=" * 120)


print(
    f"\nMinute observations       : "
    f"{len(minute_df):,}"
)


print(
    f"Forecasting samples       : "
    f"{len(forecast_df):,}"
)


print(
    f"Attack minutes            : "
    f"{int(minute_df['Attack_State'].sum()):,}"
)


print(
    f"Future-positive windows   : "
    f"{int(forecast_df['Future_Attack'].sum()):,}"
)


clean_positive_count = int(

    (
        (forecast_df["Future_Attack"] == 1)
        &
        (forecast_df["History_Contains_Attack"] == 0)
    )

    .sum()

)


print(
    f"Clean pre-attack windows  : "
    f"{clean_positive_count:,}"
)


# =============================================================================
# 3. HELPER — PARSE ATTACK TYPES
# =============================================================================

def parse_attack_types(value):

    if pd.isna(value):
        return set()

    value = str(value).strip()

    if value == "":
        return set()

    return {

        item.strip()

        for item in value.split("|")

        if item.strip()

    }


# =============================================================================
# 4. IDENTIFY STRICT CONTINUOUS ATTACK EPISODES
# =============================================================================
#
# Definition:
#
# An episode consists of consecutive Attack_State == 1 minutes.
#
# Example:
#
#       10:00 ATTACK
#       10:01 ATTACK
#       10:02 ATTACK
#       10:03 BENIGN
#       10:04 ATTACK
#
# gives:
#
#       Episode A = 10:00–10:02
#       Episode B = 10:04
#
# We will later test whether short benign gaps should instead be merged.
#
# =============================================================================

episode_records = []

episode_id = 0


for segment_id, segment in minute_df.groupby(
    "Segment_ID",
    sort=True
):

    segment = (

        segment

        .sort_values("Minute")

        .reset_index(drop=True)

    )


    attack_positions = np.where(

        segment[
            "Attack_State"
        ].astype(int).values == 1

    )[0]


    if len(attack_positions) == 0:

        continue


    # -------------------------------------------------------------------------
    # Split attack positions whenever they are not consecutive.
    # -------------------------------------------------------------------------

    split_locations = np.where(

        np.diff(
            attack_positions
        ) > 1

    )[0] + 1


    episode_position_groups = np.split(

        attack_positions,
        split_locations

    )


    for positions in episode_position_groups:

        if len(positions) == 0:

            continue


        episode_rows = segment.iloc[
            positions
        ]


        episode_start = (
            episode_rows[
                "Minute"
            ].iloc[0]
        )


        episode_end = (
            episode_rows[
                "Minute"
            ].iloc[-1]
        )


        duration_minutes = len(
            episode_rows
        )


        # ---------------------------------------------------------------------
        # Collect all attack types occurring during the episode.
        # ---------------------------------------------------------------------

        attack_types = set()


        for value in episode_rows[
            "Attack_Types"
        ]:

            attack_types.update(
                parse_attack_types(
                    value
                )
            )


        attack_types_string = (

            "|".join(
                sorted(
                    attack_types
                )
            )

            if len(attack_types) > 0

            else None

        )


        # ---------------------------------------------------------------------
        # Count attack flows represented by episode.
        # ---------------------------------------------------------------------

        total_attack_flows = int(

            episode_rows[
                "Attack_Flow_Count"
            ]

            .sum()

        )


        # ---------------------------------------------------------------------
        # Maximum attack ratio during episode.
        # ---------------------------------------------------------------------

        max_attack_ratio = float(

            episode_rows[
                "Attack_Ratio"
            ]

            .max()

        )


        mean_attack_ratio = float(

            episode_rows[
                "Attack_Ratio"
            ]

            .mean()

        )


        episode_records.append(
            {

                "Episode_ID":
                    episode_id,

                "Segment_ID":
                    int(segment_id),

                "Episode_Start":
                    episode_start,

                "Episode_End":
                    episode_end,

                "Duration_Minutes":
                    int(duration_minutes),

                "Attack_Types":
                    attack_types_string,

                "Attack_Type_Count":
                    len(attack_types),

                "Total_Attack_Flows":
                    total_attack_flows,

                "Mean_Attack_Ratio":
                    mean_attack_ratio,

                "Max_Attack_Ratio":
                    max_attack_ratio

            }
        )


        episode_id += 1


episode_df = pd.DataFrame(
    episode_records
)


assert len(episode_df) > 0


print("\n")
print("=" * 120)
print("STRICT CONTINUOUS ATTACK EPISODES")
print("=" * 120)


print(
    f"\nDistinct continuous attack episodes: "
    f"{len(episode_df):,}"
)


print(
    f"Attack minutes represented          : "
    f"{episode_df['Duration_Minutes'].sum():,}"
)


print(
    f"Original attack minutes             : "
    f"{int(minute_df['Attack_State'].sum()):,}"
)


assert (

    episode_df[
        "Duration_Minutes"
    ].sum()

    ==

    int(
        minute_df[
            "Attack_State"
        ].sum()
    )

)


print(
    "\n✓ Every attack minute belongs to exactly "
    "one strict attack episode."
)


# =============================================================================
# 5. EPISODE DURATION DISTRIBUTION
# =============================================================================

print("\n" + "=" * 120)
print("ATTACK EPISODE DURATION DISTRIBUTION")
print("=" * 120)


print(

    episode_df[
        "Duration_Minutes"
    ]

    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )

)


print(
    "\nSingle-minute episodes :",
    int(
        (
            episode_df[
                "Duration_Minutes"
            ]
            == 1
        )
        .sum()
    )
)


print(
    "Episodes >= 5 min     :",
    int(
        (
            episode_df[
                "Duration_Minutes"
            ]
            >= 5
        )
        .sum()
    )
)


print(
    "Episodes >= 10 min    :",
    int(
        (
            episode_df[
                "Duration_Minutes"
            ]
            >= 10
        )
        .sum()
    )
)


# =============================================================================
# 6. EPISODE DISTRIBUTION BY ATTACK TYPE
# =============================================================================
#
# If an episode contains multiple types, it appears once for each type
# in this exploded audit.
#
# =============================================================================

episode_type_records = []


for _, row in episode_df.iterrows():

    types = parse_attack_types(
        row[
            "Attack_Types"
        ]
    )


    if len(types) == 0:

        episode_type_records.append(
            {
                "Episode_ID":
                    row[
                        "Episode_ID"
                    ],

                "Attack_Type":
                    "UNKNOWN"
            }
        )

    else:

        for attack_type in types:

            episode_type_records.append(
                {
                    "Episode_ID":
                        row[
                            "Episode_ID"
                        ],

                    "Attack_Type":
                        attack_type
                }
            )


episode_type_df = pd.DataFrame(
    episode_type_records
)


episode_type_counts = (

    episode_type_df

    .groupby(
        "Attack_Type"
    )[
        "Episode_ID"
    ]

    .nunique()

    .sort_values(
        ascending=False
    )

)


print("\n" + "=" * 120)
print("STRICT EPISODES BY ATTACK TYPE")
print("=" * 120)

print()

print(
    episode_type_counts
)


# =============================================================================
# 7. MAP CLEAN POSITIVE FORECAST WINDOWS TO ATTACK EPISODES
# =============================================================================
#
# Clean positive window:
#
#       History_Contains_Attack = 0
#       Future_Attack = 1
#
# We map its First_Future_Attack_Time to the attack episode whose
# Episode_Start matches that onset.
#
# WHY START TIME?
#
# If the history is truly attack-free, the first future attack should
# represent the beginning of an attack episode.
#
# =============================================================================

clean_positive_df = forecast_df.loc[

    (
        forecast_df[
            "Future_Attack"
        ]
        == 1
    )

    &

    (
        forecast_df[
            "History_Contains_Attack"
        ]
        == 0
    )

].copy()


episode_start_lookup = (

    episode_df

    .set_index(
        [
            "Segment_ID",
            "Episode_Start"
        ]
    )[
        "Episode_ID"
    ]

    .to_dict()

)


def map_clean_window_to_episode(row):

    key = (

        int(
            row[
                "Segment_ID"
            ]
        ),

        row[
            "First_Future_Attack_Time"
        ]

    )

    return episode_start_lookup.get(
        key,
        np.nan
    )


clean_positive_df[
    "Episode_ID"
] = clean_positive_df.apply(

    map_clean_window_to_episode,
    axis=1

)


mapped_clean_windows = int(

    clean_positive_df[
        "Episode_ID"
    ]

    .notna()

    .sum()

)


unmapped_clean_windows = int(

    clean_positive_df[
        "Episode_ID"
    ]

    .isna()

    .sum()

)


print("\n" + "=" * 120)
print("CLEAN FORECAST WINDOW → ATTACK EPISODE MAPPING")
print("=" * 120)


print(
    f"\nClean positive windows : "
    f"{len(clean_positive_df):,}"
)


print(
    f"Mapped to onset episode: "
    f"{mapped_clean_windows:,}"
)


print(
    f"Unmapped               : "
    f"{unmapped_clean_windows:,}"
)


# =============================================================================
# 8. FORECASTABLE EPISODES
# =============================================================================
#
# An episode is forecastable if at least one clean positive window
# predicts its onset.
#
# =============================================================================

forecastable_episode_ids = set(

    clean_positive_df[
        "Episode_ID"
    ]

    .dropna()

    .astype(int)

    .unique()

)


episode_df[
    "Forecastable"
] = (

    episode_df[
        "Episode_ID"
    ]

    .isin(
        forecastable_episode_ids
    )

    .astype(int)

)


print("\n" + "=" * 120)
print("INDEPENDENT FORECASTABLE EPISODES")
print("=" * 120)


print(
    f"\nTotal strict episodes     : "
    f"{len(episode_df):,}"
)


print(
    f"Forecastable episodes     : "
    f"{int(episode_df['Forecastable'].sum()):,}"
)


print(
    f"Non-forecastable episodes : "
    f"{int((episode_df['Forecastable'] == 0).sum()):,}"
)


# =============================================================================
# 9. COUNT CLEAN WINDOWS PER EPISODE
# =============================================================================

windows_per_episode = (

    clean_positive_df

    .dropna(
        subset=[
            "Episode_ID"
        ]
    )

    .groupby(
        "Episode_ID"
    )

    .size()

    .rename(
        "Clean_Forecast_Windows"
    )

)


episode_df = episode_df.merge(

    windows_per_episode,

    how="left",

    left_on="Episode_ID",

    right_index=True

)


episode_df[
    "Clean_Forecast_Windows"
] = (

    episode_df[
        "Clean_Forecast_Windows"
    ]

    .fillna(0)

    .astype(int)

)


# =============================================================================
# 10. DETERMINE AVAILABLE WARNING HORIZONS PER EPISODE
# =============================================================================
#
# For each episode:
#
#       Warning_1_Min = 1
#
# means that at least one clean forecasting window exists where
# the episode onset occurs exactly 1 minute after history ends.
#
# Same for 2,3,4,5.
#
# =============================================================================

for lead in range(
    1,
    FORECAST_MINUTES + 1
):

    episode_df[
        f"Warning_{lead}_Min"
    ] = 0


for episode_id_value, group in (

    clean_positive_df

    .dropna(
        subset=[
            "Episode_ID"
        ]
    )

    .groupby(
        "Episode_ID"
    )

):

    episode_id_value = int(
        episode_id_value
    )


    available_leads = set(

        group[
            "Lead_Time_Minutes"
        ]

        .dropna()

        .astype(int)

        .tolist()

    )


    for lead in available_leads:

        if (
            1
            <= lead
            <= FORECAST_MINUTES
        ):

            episode_df.loc[

                episode_df[
                    "Episode_ID"
                ]
                == episode_id_value,

                f"Warning_{lead}_Min"

            ] = 1


# =============================================================================
# 11. WARNING-HORIZON SUMMARY
# =============================================================================

print("\n" + "=" * 120)
print("INDEPENDENT EPISODES BY AVAILABLE WARNING TIME")
print("=" * 120)


warning_summary = []


for lead in range(
    1,
    FORECAST_MINUTES + 1
):

    count = int(

        episode_df[
            f"Warning_{lead}_Min"
        ]

        .sum()

    )


    warning_summary.append(
        {
            "Lead_Time_Minutes":
                lead,

            "Independent_Episodes":
                count
        }
    )


warning_summary_df = pd.DataFrame(
    warning_summary
)


print()

print(
    warning_summary_df.to_string(
        index=False
    )
)


# =============================================================================
# 12. FORECASTABLE EPISODES BY ATTACK TYPE
# =============================================================================

forecastability_type_records = []


for _, episode in episode_df.iterrows():

    types = parse_attack_types(
        episode[
            "Attack_Types"
        ]
    )


    if len(types) == 0:

        types = {
            "UNKNOWN"
        }


    for attack_type in types:

        record = {

            "Episode_ID":
                int(
                    episode[
                        "Episode_ID"
                    ]
                ),

            "Attack_Type":
                attack_type,

            "Forecastable":
                int(
                    episode[
                        "Forecastable"
                    ]
                )

        }


        for lead in range(
            1,
            FORECAST_MINUTES + 1
        ):

            record[
                f"Warning_{lead}_Min"
            ] = int(

                episode[
                    f"Warning_{lead}_Min"
                ]

            )


        forecastability_type_records.append(
            record
        )


forecastability_type_df = pd.DataFrame(
    forecastability_type_records
)


type_summary_records = []


for attack_type, group in (

    forecastability_type_df

    .groupby(
        "Attack_Type"
    )

):

    record = {

        "Attack_Type":
            attack_type,

        "Total_Episodes":
            int(
                group[
                    "Episode_ID"
                ].nunique()
            ),

        "Forecastable_Episodes":
            int(
                group.loc[
                    group[
                        "Forecastable"
                    ]
                    == 1,
                    "Episode_ID"
                ].nunique()
            )

    }


    for lead in range(
        1,
        FORECAST_MINUTES + 1
    ):

        record[
            f"Forecastable_{lead}min"
        ] = int(

            group.loc[

                group[
                    f"Warning_{lead}_Min"
                ]
                == 1,

                "Episode_ID"

            ]

            .nunique()

        )


    type_summary_records.append(
        record
    )


type_summary_df = (

    pd.DataFrame(
        type_summary_records
    )

    .sort_values(
        [
            "Forecastable_Episodes",
            "Total_Episodes"
        ],
        ascending=False
    )

    .reset_index(
        drop=True
    )

)


print("\n" + "=" * 120)
print("FORECASTABILITY BY ATTACK TYPE")
print("=" * 120)

print()

print(
    type_summary_df.to_string(
        index=False
    )
)


# =============================================================================
# 13. CLEAN WINDOWS PER INDEPENDENT EPISODE
# =============================================================================

forecastable_only = episode_df.loc[

    episode_df[
        "Forecastable"
    ]
    == 1

].copy()


print("\n" + "=" * 120)
print("CLEAN WINDOWS PER FORECASTABLE EPISODE")
print("=" * 120)


if len(
    forecastable_only
) > 0:

    print()

    print(

        forecastable_only[
            "Clean_Forecast_Windows"
        ]

        .describe()

    )


# =============================================================================
# 14. ATTACK TYPE MIXING INSIDE EPISODES
# =============================================================================

multi_type_episode_count = int(

    (
        episode_df[
            "Attack_Type_Count"
        ]
        > 1
    )

    .sum()

)


print("\n" + "=" * 120)
print("MULTI-TYPE EPISODE AUDIT")
print("=" * 120)


print(
    f"\nEpisodes containing >1 attack type: "
    f"{multi_type_episode_count:,}"
)


if len(
    episode_df
) > 0:

    print(
        f"Percentage: "
        f"{multi_type_episode_count / len(episode_df) * 100:.2f}%"
    )


# =============================================================================
# 15. EPISODE GAP DISTRIBUTION
# =============================================================================
#
# We now inspect the BENIGN gap between consecutive strict attack episodes
# in the SAME segment.
#
# Example:
#
#       Episode A ends 10:10
#       Episode B starts 10:13
#
#       Missing attack minutes:
#
#           10:11
#           10:12
#
#       benign gap = 2 minutes
#
# This tells us whether strict episode counting is fragmenting attacks
# because of tiny benign interruptions.
#
# =============================================================================

gap_records = []


for segment_id, episodes in episode_df.groupby(
    "Segment_ID",
    sort=True
):

    episodes = (

        episodes

        .sort_values(
            "Episode_Start"
        )

        .reset_index(
            drop=True
        )

    )


    for i in range(
        1,
        len(episodes)
    ):

        previous = episodes.iloc[
            i - 1
        ]

        current = episodes.iloc[
            i
        ]


        raw_difference_minutes = (

            current[
                "Episode_Start"
            ]

            -

            previous[
                "Episode_End"
            ]

        ).total_seconds() / 60.0


        benign_gap_minutes = max(

            0,

            int(
                raw_difference_minutes
                - 1
            )

        )


        previous_types = parse_attack_types(
            previous[
                "Attack_Types"
            ]
        )


        current_types = parse_attack_types(
            current[
                "Attack_Types"
            ]
        )


        shared_attack_type = int(

            len(
                previous_types
                &
                current_types
            )
            > 0

        )


        gap_records.append(
            {

                "Segment_ID":
                    int(
                        segment_id
                    ),

                "Previous_Episode_ID":
                    int(
                        previous[
                            "Episode_ID"
                        ]
                    ),

                "Next_Episode_ID":
                    int(
                        current[
                            "Episode_ID"
                        ]
                    ),

                "Previous_End":
                    previous[
                        "Episode_End"
                    ],

                "Next_Start":
                    current[
                        "Episode_Start"
                    ],

                "Benign_Gap_Minutes":
                    benign_gap_minutes,

                "Shared_Attack_Type":
                    shared_attack_type

            }
        )


episode_gap_df = pd.DataFrame(
    gap_records
)


print("\n" + "=" * 120)
print("INTER-EPISODE BENIGN GAP DISTRIBUTION")
print("=" * 120)


if len(
    episode_gap_df
) > 0:

    print()

    print(

        episode_gap_df[
            "Benign_Gap_Minutes"
        ]

        .describe(
            percentiles=[
                0.25,
                0.50,
                0.75,
                0.90,
                0.95,
                0.99
            ]
        )

    )


    print(
        "\nGap <= 1 min :",
        int(
            (
                episode_gap_df[
                    "Benign_Gap_Minutes"
                ]
                <= 1
            )
            .sum()
        )
    )


    print(
        "Gap <= 2 min :",
        int(
            (
                episode_gap_df[
                    "Benign_Gap_Minutes"
                ]
                <= 2
            )
            .sum()
        )
    )


    print(
        "Gap <= 5 min :",
        int(
            (
                episode_gap_df[
                    "Benign_Gap_Minutes"
                ]
                <= 5
            )
            .sum()
        )
    )


# =============================================================================
# 16. GAP-MERGING SENSITIVITY ANALYSIS
# =============================================================================
#
# Strict consecutive attack minutes may over-fragment attacks.
#
# Therefore we calculate episode counts under several possible merging
# rules WITHOUT changing the dataset.
#
#
# MERGING RULE:
#
# Two neighbouring strict episodes are merged when:
#
#       benign gap <= threshold
#
# AND
#
#       they share at least one attack type.
#
#
# Thresholds tested:
#
#       0  = strict continuous definition
#       1
#       2
#       3
#       5 minutes
#
#
# This is only a SENSITIVITY ANALYSIS.
#
# We are NOT choosing a threshold yet.
#
# =============================================================================

MERGE_THRESHOLDS = [
    0,
    1,
    2,
    3,
    5
]


def count_merged_episodes(
    episode_table,
    gap_threshold
):

    merged_count = 0


    for segment_id, episodes in (

        episode_table

        .groupby(
            "Segment_ID",
            sort=True
        )

    ):

        episodes = (

            episodes

            .sort_values(
                "Episode_Start"
            )

            .reset_index(
                drop=True
            )

        )


        if len(
            episodes
        ) == 0:

            continue


        merged_count += 1


        current_end = episodes.loc[
            0,
            "Episode_End"
        ]


        current_types = parse_attack_types(

            episodes.loc[
                0,
                "Attack_Types"
            ]

        )


        for i in range(
            1,
            len(episodes)
        ):

            next_start = episodes.loc[
                i,
                "Episode_Start"
            ]


            next_end = episodes.loc[
                i,
                "Episode_End"
            ]


            next_types = parse_attack_types(

                episodes.loc[
                    i,
                    "Attack_Types"
                ]

            )


            raw_difference = (

                next_start
                -
                current_end

            ).total_seconds() / 60.0


            benign_gap = max(

                0,

                int(
                    raw_difference
                    - 1
                )

            )


            shares_type = (

                len(
                    current_types
                    &
                    next_types
                )
                > 0

            )


            if (

                benign_gap
                <= gap_threshold

                and

                shares_type

            ):

                current_end = max(
                    current_end,
                    next_end
                )


                current_types.update(
                    next_types
                )


            else:

                merged_count += 1

                current_end = (
                    next_end
                )

                current_types = set(
                    next_types
                )


    return merged_count


sensitivity_records = []


for threshold in MERGE_THRESHOLDS:

    merged_count = count_merged_episodes(

        episode_df,

        threshold

    )


    sensitivity_records.append(
        {

            "Allowed_Benign_Gap_Minutes":
                threshold,

            "Episode_Count":
                merged_count

        }
    )


sensitivity_df = pd.DataFrame(
    sensitivity_records
)


print("\n" + "=" * 120)
print("ATTACK EPISODE GAP-MERGING SENSITIVITY")
print("=" * 120)

print()

print(
    sensitivity_df.to_string(
        index=False
    )
)


# =============================================================================
# 17. DISPLAY FORECASTABLE EPISODES
# =============================================================================

print("\n" + "=" * 120)
print("EXAMPLE FORECASTABLE ATTACK EPISODES")
print("=" * 120)


episode_display_columns = [

    "Episode_ID",
    "Segment_ID",

    "Episode_Start",
    "Episode_End",

    "Duration_Minutes",

    "Attack_Types",

    "Total_Attack_Flows",

    "Clean_Forecast_Windows",

    "Warning_1_Min",
    "Warning_2_Min",
    "Warning_3_Min",
    "Warning_4_Min",
    "Warning_5_Min"

]


display(

    forecastable_only[
        episode_display_columns
    ]

    .head(30)

)


# =============================================================================
# 18. SAVE AUDIT TABLES
# =============================================================================

EPISODE_PATH = os.path.join(
    AUDIT_DIR,
    "attack_episodes_strict.parquet"
)


EPISODE_TYPE_SUMMARY_PATH = os.path.join(
    AUDIT_DIR,
    "attack_episode_type_forecastability.csv"
)


EPISODE_GAP_PATH = os.path.join(
    AUDIT_DIR,
    "attack_episode_gaps.csv"
)


EPISODE_SENSITIVITY_PATH = os.path.join(
    AUDIT_DIR,
    "attack_episode_gap_sensitivity.csv"
)


WARNING_SUMMARY_PATH = os.path.join(
    AUDIT_DIR,
    "attack_episode_warning_summary.csv"
)


CLEAN_WINDOW_MAPPING_PATH = os.path.join(
    AUDIT_DIR,
    "clean_forecast_window_episode_mapping.parquet"
)


episode_df.to_parquet(
    EPISODE_PATH,
    index=False,
    compression="snappy"
)


type_summary_df.to_csv(
    EPISODE_TYPE_SUMMARY_PATH,
    index=False
)


episode_gap_df.to_csv(
    EPISODE_GAP_PATH,
    index=False
)


sensitivity_df.to_csv(
    EPISODE_SENSITIVITY_PATH,
    index=False
)


warning_summary_df.to_csv(
    WARNING_SUMMARY_PATH,
    index=False
)


clean_positive_df.to_parquet(
    CLEAN_WINDOW_MAPPING_PATH,
    index=False,
    compression="snappy"
)


# =============================================================================
# 19. FINAL SANITY CHECKS
# =============================================================================

check_all_attack_minutes_accounted = bool(

    episode_df[
        "Duration_Minutes"
    ].sum()

    ==

    minute_df[
        "Attack_State"
    ].sum()

)


check_episode_times_valid = bool(

    (
        episode_df[
            "Episode_End"
        ]

        >=

        episode_df[
            "Episode_Start"
        ]
    )

    .all()

)


check_episode_segment_valid = bool(

    episode_df[
        "Segment_ID"
    ]

    .isin(
        minute_df[
            "Segment_ID"
        ].unique()
    )

    .all()

)


check_clean_windows_mapped = bool(

    unmapped_clean_windows
    == 0

)


check_forecastable_count_valid = bool(

    episode_df[
        "Forecastable"
    ].sum()

    ==

    len(
        forecastable_episode_ids
    )

)


check_warning_values = True


for lead in range(
    1,
    FORECAST_MINUTES + 1
):

    valid_values = set(

        episode_df[
            f"Warning_{lead}_Min"
        ]

        .dropna()

        .unique()

    )


    if not valid_values.issubset(
        {
            0,
            1
        }
    ):

        check_warning_values = False


checks = {

    "Every attack minute belongs to an episode":
        check_all_attack_minutes_accounted,

    "Episode start/end times valid":
        check_episode_times_valid,

    "Episode segment IDs valid":
        check_episode_segment_valid,

    "All clean positive windows map to an onset":
        check_clean_windows_mapped,

    "Forecastable episode count internally consistent":
        check_forecastable_count_valid,

    "Warning indicators are binary":
        check_warning_values,

    "Episode audit saved":
        os.path.exists(
            EPISODE_PATH
        ),

    "Attack-type forecastability summary saved":
        os.path.exists(
            EPISODE_TYPE_SUMMARY_PATH
        ),

    "Gap sensitivity audit saved":
        os.path.exists(
            EPISODE_SENSITIVITY_PATH
        )

}


print("\n" + "=" * 120)
print("FINAL CELL 11 VALIDATION")
print("=" * 120)


for name, result in checks.items():

    print(
        f"{'✓' if bool(result) else '❌'} "
        f"{name}: {bool(result)}"
    )


passed = sum(

    bool(
        value
    )

    for value in checks.values()

)


print(
    f"\nPassed: "
    f"{passed} / {len(checks)}"
)


print("\n" + "=" * 120)


if all(
    bool(value)
    for value in checks.values()
):

    print(
        "✓ CELL 11 COMPLETED SUCCESSFULLY."
    )

else:

    print(
        "⚠ CELL 11 completed, but one or more "
        "sanity checks require investigation."
    )


print()
print(
    "IMPORTANT: DO NOT TRAIN MODELS YET."
)

print()
print(
    "NEXT: interpret independent attack-event counts "
    "and decide the final forecasting formulation."
)

print("=" * 120)


# =============================================================================
# 20. OUTPUT PATHS
# =============================================================================

print("\nSaved strict episode table:")
print(
    EPISODE_PATH
)

print("\nSaved attack-type forecastability table:")
print(
    EPISODE_TYPE_SUMMARY_PATH
)

print("\nSaved episode-gap table:")
print(
    EPISODE_GAP_PATH
)

print("\nSaved gap sensitivity table:")
print(
    EPISODE_SENSITIVITY_PATH
)

print("\nSaved warning-time summary:")
print(
    WARNING_SUMMARY_PATH
)

print("\nSaved clean-window → episode mapping:")
print(
    CLEAN_WINDOW_MAPPING_PATH
)


# =============================================================================
# 21. MEMORY CLEANUP
# =============================================================================

gc.collect()

CELL 11 — ATTACK EPISODE / ONSET / FORECASTABILITY AUDIT

Minute observations       : 2,454
Forecasting samples       : 2,314
Attack minutes            : 520
Future-positive windows   : 682
Clean pre-attack windows  : 79


STRICT CONTINUOUS ATTACK EPISODES

Distinct continuous attack episodes: 82
Attack minutes represented          : 520
Original attack minutes             : 520

✓ Every attack minute belongs to exactly one strict attack episode.

ATTACK EPISODE DURATION DISTRIBUTION
count    82.000000
mean      6.341463
std      13.149867
min       1.000000
25%       1.000000
50%       2.000000
75%       4.000000
90%      16.000000
95%      23.850000
99%      64.330000
max      70.000000
Name: Duration_Minutes, dtype: float64

Single-minute episodes : 39
Episodes >= 5 min     : 17
Episodes >= 10 min    : 11

STRICT EPISODES BY ATTACK TYPE

Attack_Type
Infiltration                  25
Bot                           17
Heartbleed                    11
PortScan                       9
DoS

,Episode_ID,Segment_ID,Episode_Start,Episode_End,Duration_Minutes,Attack_Types,Total_Attack_Flows,Clean_Forecast_Windows,Warning_1_Min,Warning_2_Min,Warning_3_Min,Warning_4_Min,Warning_5_Min
0,0,2,2017-07-04 02:09:00,2017-07-04 03:11:00,63,SSH-Patator,5897,5,1,1,1,1,1
1,1,3,2017-07-04 09:17:00,2017-07-04 09:17:00,1,FTP-Patator,1,5,1,1,1,1,1
4,4,4,2017-07-05 02:24:00,2017-07-05 02:25:00,2,DoS slowloris,5,5,1,1,1,1,1
5,5,4,2017-07-05 03:12:00,2017-07-05 03:12:00,1,Heartbleed,1,5,1,1,1,1,1
16,16,5,2017-07-05 09:01:00,2017-07-05 09:01:00,1,DoS slowloris,1,5,1,1,1,1,1
17,17,5,2017-07-05 09:48:00,2017-07-05 10:11:00,24,DoS slowloris,5790,5,1,1,1,1,1
25,25,6,2017-07-06 02:19:00,2017-07-06 02:19:00,1,Infiltration,2,5,1,1,1,1,1
33,33,6,2017-07-06 03:04:00,2017-07-06 03:04:00,1,Infiltration,1,5,1,1,1,1,1
50,50,7,2017-07-06 09:15:00,2017-07-06 09:16:00,2,Web Attack - Brute Force,5,5,1,1,1,1,1
53,53,7,2017-07-06 10:15:00,2017-07-06 10:35:00,21,Web Attack - XSS,652,5,1,1,1,1,1



FINAL CELL 11 VALIDATION
✓ Every attack minute belongs to an episode: True
✓ Episode start/end times valid: True
✓ Episode segment IDs valid: True
✓ All clean positive windows map to an onset: True
✓ Forecastable episode count internally consistent: True
✓ Warning indicators are binary: True
✓ Episode audit saved: True
✓ Attack-type forecastability summary saved: True
✓ Gap sensitivity audit saved: True

Passed: 9 / 9

✓ CELL 11 COMPLETED SUCCESSFULLY.

IMPORTANT: DO NOT TRAIN MODELS YET.

NEXT: interpret independent attack-event counts and decide the final forecasting formulation.

Saved strict episode table:
/content/drive/MyDrive/Network-Attack-Forecasting/data/audits/attack_episodes_strict.parquet

Saved attack-type forecastability table:
/content/drive/MyDrive/Network-Attack-Forecasting/data/audits/attack_episode_type_forecastability.csv

Saved episode-gap table:
/content/drive/MyDrive/Network-Attack-Forecasting/data/audits/attack_episode_gaps.csv

Saved gap sensitivity table:
/c

0

In [18]:
# =============================================================================
# CELL 12 — GENERAL FUTURE ATTACK-STATE FORECASTING AUDIT
# =============================================================================
#
# PURPOSE
# -------
# We previously tested a very strict forecasting formulation:
#
#     "Can an attack be forecast using ONLY completely attack-free history?"
#
# That formulation produced very few independent attack onsets.
#
# We now test a broader but still genuine forecasting formulation:
#
#     Given the previous L minutes of observed network behaviour,
#     predict whether the network will be under attack H minutes
#     into the FUTURE.
#
#
# IMPORTANT DIFFERENCE FROM THE PREVIOUS FORMULATION
# --------------------------------------------------
#
# We DO NOT require the historical sequence to be completely benign.
#
# Example with:
#
#     History length L = 15 minutes
#     Forecast horizon H = 5 minutes
#
#     Input  : minutes t-14, ..., t
#     Target : Attack_State at t+5
#
# The model NEVER sees information from t+1 ... t+5.
#
# Therefore this is still forecasting rather than current-state detection.
#
#
# THIS CELL DOES NOT TRAIN ANY MODEL.
#
# It audits:
#
#   1. Number of usable forecasting sequences
#   2. Positive/negative target balance
#   3. Results for multiple forecast horizons
#   4. Whether target attacks are:
#         - new attack onsets
#         - continuation of an attack already visible in history
#   5. Attack-type coverage
#   6. Segment safety
#
# Forecasting windows are NEVER allowed to cross the capture
# discontinuities identified earlier.
#
# =============================================================================


import pandas as pd
import numpy as np


# =============================================================================
# 1. REQUIRE minute_df
# =============================================================================

assert "minute_df" in globals(), (
    "minute_df not found. Run the minute-level dataset construction "
    "cell first."
)

required_cols = {
    "Segment_ID",
    "Minute",
    "Attack_State"
}

missing = required_cols - set(minute_df.columns)

assert not missing, (
    f"minute_df is missing required columns: {missing}"
)


df = minute_df.copy()

df["Minute"] = pd.to_datetime(df["Minute"])

df = (
    df
    .sort_values(["Segment_ID", "Minute"])
    .reset_index(drop=True)
)


print("=" * 110)
print("CELL 12 — GENERAL FUTURE ATTACK-STATE FORECASTING AUDIT")
print("=" * 110)

print("\nMinute rows :", len(df))
print("Segments    :", df["Segment_ID"].nunique())

print("\nAttack-state distribution:")
print(df["Attack_State"].value_counts().sort_index())


# =============================================================================
# 2. PARAMETERS
# =============================================================================
#
# The paper uses a sequence length of 15.
#
# We will therefore begin with:
#
#     L = 15 minutes
#
# But instead of assuming one forecasting horizon, we audit several.
#
# =============================================================================

HISTORY_LENGTH = 15

FORECAST_HORIZONS = [
    1,
    5,
    10,
    15,
    30
]


# =============================================================================
# 3. DETERMINE ATTACK TYPE COLUMN IF AVAILABLE
# =============================================================================

attack_type_candidates = [
    "Attack_Type",
    "Dominant_Attack_Type",
    "Attack_Label_Name",
    "Label"
]

attack_type_col = None

for col in attack_type_candidates:
    if col in df.columns:
        attack_type_col = col
        break


print("\nHistory length :", HISTORY_LENGTH, "minutes")
print("Horizons       :", FORECAST_HORIZONS)

if attack_type_col is not None:
    print("Attack type col:", attack_type_col)
else:
    print(
        "Attack type col: not present in minute_df "
        "(binary forecasting audit will still work)"
    )


# =============================================================================
# 4. CREATE FORECASTING WINDOWS
# =============================================================================
#
# For every segment:
#
# history:
#
#     t-L+1 ... t
#
# target:
#
#     t+H
#
# We explicitly verify that the timestamps are contiguous.
#
# =============================================================================

all_samples = []

for horizon in FORECAST_HORIZONS:

    horizon_samples = []

    for segment_id, seg in df.groupby("Segment_ID"):

        seg = (
            seg
            .sort_values("Minute")
            .reset_index(drop=True)
        )

        n = len(seg)

        for end_idx in range(
            HISTORY_LENGTH - 1,
            n - horizon
        ):

            start_idx = end_idx - HISTORY_LENGTH + 1
            target_idx = end_idx + horizon

            history = seg.iloc[start_idx:end_idx + 1]
            target = seg.iloc[target_idx]

            # -------------------------------------------------------------
            # Expected timestamps
            # -------------------------------------------------------------

            expected_history_duration = pd.Timedelta(
                minutes=HISTORY_LENGTH - 1
            )

            actual_history_duration = (
                history["Minute"].iloc[-1]
                - history["Minute"].iloc[0]
            )

            # History itself must be continuous
            if actual_history_duration != expected_history_duration:
                continue

            # Target must really be H minutes after history end
            expected_target_time = (
                history["Minute"].iloc[-1]
                + pd.Timedelta(minutes=horizon)
            )

            if target["Minute"] != expected_target_time:
                continue

            # -------------------------------------------------------------
            # History attack information
            # -------------------------------------------------------------

            history_attack_minutes = int(
                history["Attack_State"].sum()
            )

            history_contains_attack = int(
                history_attack_minutes > 0
            )

            current_state = int(
                history["Attack_State"].iloc[-1]
            )

            target_state = int(
                target["Attack_State"]
            )

            # -------------------------------------------------------------
            # Is target attack a NEW onset?
            #
            # New onset means:
            #
            # target = attack
            # current observed minute = benign
            #
            # This is not necessarily equivalent to an independent
            # campaign, but it tells us whether the forecast transitions
            # from benign current state -> future attack state.
            # -------------------------------------------------------------

            new_onset_target = int(
                target_state == 1
                and current_state == 0
            )

            # -------------------------------------------------------------
            # Attack continuation
            #
            # Attack is already active at current time AND still active
            # H minutes later.
            # -------------------------------------------------------------

            continuation_target = int(
                target_state == 1
                and current_state == 1
            )

            sample = {
                "Segment_ID": segment_id,

                "History_Start":
                    history["Minute"].iloc[0],

                "History_End":
                    history["Minute"].iloc[-1],

                "Target_Time":
                    target["Minute"],

                "History_Length":
                    HISTORY_LENGTH,

                "Forecast_Horizon":
                    horizon,

                "History_Attack_Minutes":
                    history_attack_minutes,

                "History_Contains_Attack":
                    history_contains_attack,

                "Current_Attack_State":
                    current_state,

                "Future_Attack_State":
                    target_state,

                "New_Onset_Target":
                    new_onset_target,

                "Continuation_Target":
                    continuation_target
            }

            if attack_type_col is not None:
                sample["Future_Attack_Type"] = target[
                    attack_type_col
                ]

            horizon_samples.append(sample)

    horizon_df = pd.DataFrame(horizon_samples)

    all_samples.append(horizon_df)


forecast_audit_df = pd.concat(
    all_samples,
    ignore_index=True
)


# =============================================================================
# 5. BASIC SUMMARY BY HORIZON
# =============================================================================

summary_rows = []

for horizon in FORECAST_HORIZONS:

    hdf = forecast_audit_df[
        forecast_audit_df["Forecast_Horizon"] == horizon
    ]

    total = len(hdf)

    positives = int(
        hdf["Future_Attack_State"].sum()
    )

    negatives = total - positives

    onset = int(
        hdf["New_Onset_Target"].sum()
    )

    continuation = int(
        hdf["Continuation_Target"].sum()
    )

    history_attack = int(
        hdf["History_Contains_Attack"].sum()
    )

    history_clean = total - history_attack

    summary_rows.append({

        "Horizon_Minutes": horizon,

        "Total_Samples": total,

        "Future_Attack_Positive": positives,

        "Future_Benign_Negative": negatives,

        "Positive_Percent":
            100 * positives / total
            if total else np.nan,

        "Benign_To_Attack": onset,

        "Attack_Continuation_Positive":
            continuation,

        "Clean_History_Samples":
            history_clean,

        "Attack_In_History_Samples":
            history_attack
    })


forecast_summary_df = pd.DataFrame(summary_rows)


print("\n")
print("=" * 110)
print("FORECASTING SAMPLE SUMMARY")
print("=" * 110)

print(
    forecast_summary_df.to_string(
        index=False
    )
)


# =============================================================================
# 6. DECOMPOSE POSITIVE TARGETS
# =============================================================================
#
# Positive future targets can come from two different situations:
#
# A)
# current state = benign
# future state  = attack
#
#     -> future transition / onset
#
# B)
# current state = attack
# future state  = attack
#
#     -> attack continuation
#
# This distinction is VERY important.
#
# =============================================================================

print("\n")
print("=" * 110)
print("POSITIVE TARGET DECOMPOSITION")
print("=" * 110)


for horizon in FORECAST_HORIZONS:

    hdf = forecast_audit_df[
        forecast_audit_df["Forecast_Horizon"] == horizon
    ]

    positive = hdf[
        hdf["Future_Attack_State"] == 1
    ]

    print(f"\nHORIZON = {horizon} MINUTES")
    print("-" * 70)

    print(
        "Future attack samples :",
        len(positive)
    )

    print(
        "New-onset targets      :",
        int(positive["New_Onset_Target"].sum())
    )

    print(
        "Continuation targets   :",
        int(positive["Continuation_Target"].sum())
    )

    if len(positive) > 0:

        print(
            "Onset percentage     :",
            round(
                100
                * positive["New_Onset_Target"].mean(),
                2
            ),
            "%"
        )

        print(
            "Continuation percent  :",
            round(
                100
                * positive["Continuation_Target"].mean(),
                2
            ),
            "%"
        )


# =============================================================================
# 7. HISTORY CONTAMINATION AMONG POSITIVE FORECASTS
# =============================================================================
#
# This tells us how much of B's positive dataset contains some attack
# activity in the 15-minute history.
#
# =============================================================================

print("\n")
print("=" * 110)
print("HISTORICAL ATTACK CONTENT AMONG POSITIVE TARGETS")
print("=" * 110)


history_rows = []

for horizon in FORECAST_HORIZONS:

    hdf = forecast_audit_df[
        (forecast_audit_df["Forecast_Horizon"] == horizon)
        &
        (forecast_audit_df["Future_Attack_State"] == 1)
    ]

    if len(hdf) == 0:
        continue

    completely_clean = int(
        (hdf["History_Attack_Minutes"] == 0).sum()
    )

    partially_or_fully_attack = int(
        (hdf["History_Attack_Minutes"] > 0).sum()
    )

    history_rows.append({

        "Horizon_Minutes":
            horizon,

        "Positive_Targets":
            len(hdf),

        "Completely_Clean_History":
            completely_clean,

        "Attack_Present_In_History":
            partially_or_fully_attack,

        "Clean_History_Percent":
            100 * completely_clean / len(hdf),

        "Attack_History_Percent":
            100 * partially_or_fully_attack / len(hdf)
    })


history_summary_df = pd.DataFrame(history_rows)

print(
    history_summary_df.to_string(
        index=False
    )
)


# =============================================================================
# 8. ATTACK TYPE COVERAGE
# =============================================================================

if attack_type_col is not None:

    print("\n")
    print("=" * 110)
    print("FUTURE ATTACK TYPE COVERAGE")
    print("=" * 110)

    for horizon in FORECAST_HORIZONS:

        hdf = forecast_audit_df[
            (forecast_audit_df["Forecast_Horizon"] == horizon)
            &
            (forecast_audit_df["Future_Attack_State"] == 1)
        ]

        print(
            f"\nHORIZON = {horizon} MINUTES"
        )

        print("-" * 70)

        if len(hdf) == 0:
            print("No positive samples.")
            continue

        print(
            hdf["Future_Attack_Type"]
            .value_counts(dropna=False)
            .to_string()
        )


# =============================================================================
# 9. CURRENT -> FUTURE TRANSITION MATRIX
# =============================================================================

print("\n")
print("=" * 110)
print("CURRENT STATE -> FUTURE STATE")
print("=" * 110)


for horizon in FORECAST_HORIZONS:

    hdf = forecast_audit_df[
        forecast_audit_df["Forecast_Horizon"] == horizon
    ]

    transition = pd.crosstab(
        hdf["Current_Attack_State"],
        hdf["Future_Attack_State"],
        rownames=["Current"],
        colnames=["Future"]
    )

    print(
        f"\nHORIZON = {horizon} MINUTES"
    )

    print(transition)


# =============================================================================
# 10. SEGMENT SAFETY CHECK
# =============================================================================

print("\n")
print("=" * 110)
print("SEGMENT / TEMPORAL SAFETY VALIDATION")
print("=" * 110)


invalid_history = (
    (
        forecast_audit_df["History_End"]
        -
        forecast_audit_df["History_Start"]
    )
    != pd.Timedelta(
        minutes=HISTORY_LENGTH - 1
    )
).sum()


invalid_target = (
    (
        forecast_audit_df["Target_Time"]
        -
        forecast_audit_df["History_End"]
    )
    != pd.to_timedelta(
        forecast_audit_df["Forecast_Horizon"],
        unit="m"
    )
).sum()


print(
    "Invalid history windows :",
    invalid_history
)

print(
    "Invalid target horizons :",
    invalid_target
)

print(
    "Segment IDs represented :",
    forecast_audit_df["Segment_ID"].nunique()
)


assert invalid_history == 0
assert invalid_target == 0


# =============================================================================
# 11. FINAL INTERPRETATION DATA
# =============================================================================

print("\n")
print("=" * 110)
print("CELL 12 COMPLETE")
print("=" * 110)

print(
    """
This cell has NOT trained a model.

It has measured how much data exists for the broader forecasting task:

    past 15-minute network sequence
                 ↓
       future attack state at t + H

Unlike the strict onset experiment, histories containing attack
activity are retained.

The next decision should be based on:

1. How many future-positive samples exist.
2. How many represent benign -> attack transitions.
3. How many merely represent attack -> attack continuation.
4. How this changes as the forecasting horizon increases.
5. Whether attack-type coverage remains usable.

DO NOT create the final train/test split yet.
First inspect these results.
"""
)

CELL 12 — GENERAL FUTURE ATTACK-STATE FORECASTING AUDIT

Minute rows : 2454
Segments    : 10

Attack-state distribution:
Attack_State
0    1934
1     520
Name: count, dtype: int64

History length : 15 minutes
Horizons       : [1, 5, 10, 15, 30]
Attack type col: not present in minute_df (binary forecasting audit will still work)


FORECASTING SAMPLE SUMMARY
 Horizon_Minutes  Total_Samples  Future_Attack_Positive  Future_Benign_Negative  Positive_Percent  Benign_To_Attack  Attack_Continuation_Positive  Clean_History_Samples  Attack_In_History_Samples
               1           2304                     518                    1786         22.482639                81                           437                   1452                        852
               5           2264                     516                    1748         22.791519               107                           409                   1416                        848
              10           2214                     5

In [19]:
# =============================================================================
# CELL 13 — ADD PER-ATTACK-TYPE COUNTS TO THE MINUTE-LEVEL DATASET
# =============================================================================
#
# PURPOSE
# -------
#
# Cell 12 showed that the broader forecasting formulation is viable:
#
#       previous 15 minutes
#               ↓
#       future attack state at t + H
#
# for horizons:
#
#       H = 1, 5, 10, 15, 30 minutes
#
#
# HOWEVER:
#
# Our current minute-level dataset contains:
#
#       Attack_State
#       Attack_Flow_Count
#
# but does NOT preserve which attack type occurred during each minute.
#
#
# This cell fixes that.
#
#
# FOR EVERY MINUTE WE WILL CREATE:
#
#       BENIGN_Count
#       Bot_Count
#       DDoS_Count
#       DoS_GoldenEye_Count
#       DoS_Hulk_Count
#       DoS_Slowhttptest_Count
#       DoS_slowloris_Count
#       FTP_Patator_Count
#       Heartbleed_Count
#       Infiltration_Count
#       PortScan_Count
#       SSH_Patator_Count
#       Web_Attack_Brute_Force_Count
#       Web_Attack_Sql_Injection_Count
#       Web_Attack_XSS_Count
#
#
# We will ALSO preserve:
#
#       Dominant_Attack_Type
#
# which is the attack type having the largest number of attack flows
# during that minute.
#
#
# IMPORTANT
# ---------
#
# We DO NOT load the entire ~2.83 million-flow dataset into RAM.
#
# The eight raw GeneratedLabelledFlows / TrafficLabelling CSVs are
# processed in chunks.
#
# Only:
#
#       Timestamp
#       Label
#
# are loaded because those are the only columns required for this task.
#
#
# NO MODEL TRAINING OCCURS HERE.
#
# =============================================================================


import os
import gc
import re
import numpy as np
import pandas as pd


# =============================================================================
# 1. CONFIGURATION
# =============================================================================

PROJECT_ROOT = (
    "/content/drive/MyDrive/"
    "Network-Attack-Forecasting"
)

TEMPORAL_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "temporal"
)

AUDIT_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "audits"
)

os.makedirs(
    TEMPORAL_DIR,
    exist_ok=True
)

os.makedirs(
    AUDIT_DIR,
    exist_ok=True
)


MINUTE_DATASET_PATH = os.path.join(
    TEMPORAL_DIR,
    "minute_level_dataset.parquet"
)


ENRICHED_MINUTE_DATASET_PATH = os.path.join(
    TEMPORAL_DIR,
    "minute_level_dataset_with_attack_types.parquet"
)


ATTACK_TYPE_AUDIT_PATH = os.path.join(
    AUDIT_DIR,
    "minute_attack_type_counts.csv"
)


# =============================================================================
# 2. LOCATE RAW CICIDS2017 FILES
# =============================================================================
#
# Earlier in the notebook our raw files were stored as:
#
#       raw_files
#
# with compatibility alias:
#
#       glf_files
#
# If the runtime restarted, reconstruct the paths from Google Drive.
#
# NOTE:
# Your actual Drive folder has a trailing space:
#
#       "TrafficLabelling "
#
# Keep that exactly as it exists in your Drive.
#
# =============================================================================

RAW_DATA_DIR = (
    "/content/drive/MyDrive/"
    "TrafficLabelling "
)


if "raw_files" in globals():

    source_files = raw_files

elif "glf_files" in globals():

    source_files = glf_files

else:

    assert os.path.isdir(
        RAW_DATA_DIR
    ), (
        f"Raw data directory not found:\n"
        f"{RAW_DATA_DIR}"
    )

    source_files = sorted(
        [
            os.path.join(
                RAW_DATA_DIR,
                filename
            )
            for filename in os.listdir(
                RAW_DATA_DIR
            )
            if filename.lower().endswith(
                ".csv"
            )
        ]
    )


assert len(source_files) == 8, (
    f"Expected 8 CICIDS2017 CSV files, "
    f"but found {len(source_files)}."
)


print("=" * 115)
print("CELL 13 — PER-ATTACK-TYPE MINUTE ENRICHMENT")
print("=" * 115)

print(
    f"\nRaw CSV files found : "
    f"{len(source_files)}"
)

for i, path in enumerate(
    source_files,
    1
):

    print(
        f"{i}. {os.path.basename(path)}"
    )


# =============================================================================
# 3. LOAD EXISTING MINUTE DATASET
# =============================================================================

assert os.path.exists(
    MINUTE_DATASET_PATH
), (
    "Existing minute-level dataset not found:\n"
    f"{MINUTE_DATASET_PATH}\n\n"
    "Run the minute-level construction cell first."
)


minute_base = pd.read_parquet(
    MINUTE_DATASET_PATH
)


minute_base["Minute"] = pd.to_datetime(
    minute_base["Minute"]
)


minute_base = (
    minute_base
    .sort_values(
        [
            "Segment_ID",
            "Minute"
        ]
    )
    .reset_index(
        drop=True
    )
)


print("\nExisting minute dataset:")
print(
    f"Rows     : {len(minute_base):,}"
)

print(
    f"Columns  : {len(minute_base.columns):,}"
)

print(
    f"Segments : "
    f"{minute_base['Segment_ID'].nunique()}"
)

print(
    f"Attack minutes : "
    f"{int(minute_base['Attack_State'].sum()):,}"
)


# =============================================================================
# 4. CANONICAL CICIDS2017 ATTACK LABELS
# =============================================================================
#
# We explicitly define the labels we expect.
#
# This protects us from silently losing a class because of malformed
# text, whitespace or the known Web Attack encoding issue.
#
# =============================================================================

EXPECTED_ATTACK_TYPES = [

    "Bot",
    "DDoS",

    "DoS GoldenEye",
    "DoS Hulk",
    "DoS Slowhttptest",
    "DoS slowloris",

    "FTP-Patator",
    "Heartbleed",
    "Infiltration",
    "PortScan",
    "SSH-Patator",

    "Web Attack - Brute Force",
    "Web Attack - Sql Injection",
    "Web Attack - XSS"
]


EXPECTED_LABELS = (
    ["BENIGN"]
    +
    EXPECTED_ATTACK_TYPES
)


# =============================================================================
# 5. NORMALIZE LABEL FUNCTION
# =============================================================================
#
# CICIDS2017's Web Attack file is known to contain unusual characters /
# encoding artifacts in some versions.
#
# Earlier we encountered a UnicodeDecodeError in this exact file.
#
# Therefore:
#
#   - CSV is read using latin1
#   - whitespace is stripped
#   - common malformed Web Attack separators are normalized
#
# =============================================================================

def normalize_attack_label(value):

    if pd.isna(value):
        return None

    value = str(value).strip()

    # -------------------------------------------------------------
    # Normalize common encoding / punctuation variants.
    # -------------------------------------------------------------

    value = (
        value
        .replace("–", "-")
        .replace("—", "-")
        .replace("�", "-")
        .replace("\x96", "-")
    )

    # Collapse repeated whitespace.
    value = re.sub(
        r"\s+",
        " ",
        value
    )

    # -------------------------------------------------------------
    # Normalize Web Attack labels based on their semantic suffix.
    # -------------------------------------------------------------

    lower = value.lower()

    if (
        "web attack" in lower
        and
        "brute force" in lower
    ):
        return "Web Attack - Brute Force"

    if (
        "web attack" in lower
        and
        (
            "sql injection" in lower
            or
            "sql" in lower
        )
    ):
        return "Web Attack - Sql Injection"

    if (
        "web attack" in lower
        and
        "xss" in lower
    ):
        return "Web Attack - XSS"

    # -------------------------------------------------------------
    # Normalize other known labels case-insensitively.
    # -------------------------------------------------------------

    canonical_lookup = {
        label.lower(): label
        for label in EXPECTED_LABELS
    }

    if lower in canonical_lookup:
        return canonical_lookup[lower]

    # Preserve unexpected values so the audit catches them.
    return value


# =============================================================================
# 6. HELPER — SAFE COLUMN DETECTION
# =============================================================================

def detect_required_columns(path):

    header = pd.read_csv(
        path,
        nrows=0,
        encoding="latin1"
    )

    original_columns = list(
        header.columns
    )

    stripped_lookup = {
        col.strip(): col
        for col in original_columns
    }

    assert "Timestamp" in stripped_lookup, (
        f"Timestamp column not found in:\n{path}"
    )

    assert "Label" in stripped_lookup, (
        f"Label column not found in:\n{path}"
    )

    return (
        stripped_lookup["Timestamp"],
        stripped_lookup["Label"]
    )


# =============================================================================
# 7. PROCESS RAW FILES IN CHUNKS
# =============================================================================
#
# Memory-safe strategy:
#
# Raw flow rows
#      ↓
# read only Timestamp + Label
#      ↓
# floor timestamp to minute
#      ↓
# count labels per minute
#      ↓
# discard raw chunk
#
#
# We intentionally DO NOT concatenate millions of flow rows.
#
# =============================================================================

CHUNK_SIZE = 150_000

minute_count_parts = []

raw_rows_processed = 0


print("\n" + "=" * 115)
print("PROCESSING RAW FILES IN MEMORY-SAFE CHUNKS")
print("=" * 115)


for file_number, path in enumerate(
    source_files,
    1
):

    filename = os.path.basename(
        path
    )

    print(
        f"\n[{file_number}/8] {filename}"
    )

    timestamp_col, label_col = (
        detect_required_columns(
            path
        )
    )

    file_rows = 0
    file_valid_timestamps = 0

    file_minute_parts = []


    reader = pd.read_csv(
        path,

        usecols=[
            timestamp_col,
            label_col
        ],

        chunksize=CHUNK_SIZE,

        encoding="latin1",

        low_memory=False
    )


    for chunk_number, chunk in enumerate(
        reader,
        1
    ):

        file_rows += len(
            chunk
        )

        raw_rows_processed += len(
            chunk
        )

        # ---------------------------------------------------------
        # Standardize names.
        # ---------------------------------------------------------

        chunk = chunk.rename(
            columns={
                timestamp_col:
                    "Timestamp",

                label_col:
                    "Label"
            }
        )


        # ---------------------------------------------------------
        # Normalize labels.
        # ---------------------------------------------------------

        chunk["Label"] = (
            chunk["Label"]
            .map(
                normalize_attack_label
            )
        )


        # ---------------------------------------------------------
        # Parse timestamps.
        #
        # CICIDS2017 commonly uses day/month/year timestamps.
        # dayfirst=True protects against interpreting:
        #
        #       04/07/2017
        #
        # as April 7 instead of July 4.
        # ---------------------------------------------------------

        chunk["Timestamp"] = pd.to_datetime(
            chunk["Timestamp"],
            errors="coerce",
            dayfirst=True
        )


        valid_timestamp_mask = (
            chunk["Timestamp"].notna()
        )

        file_valid_timestamps += int(
            valid_timestamp_mask.sum()
        )

        chunk = chunk.loc[
            valid_timestamp_mask
        ].copy()


        # ---------------------------------------------------------
        # Convert each flow timestamp to its one-minute bin.
        # ---------------------------------------------------------

        chunk["Minute"] = (
            chunk["Timestamp"]
            .dt.floor("min")
        )


        # ---------------------------------------------------------
        # Count each label inside each minute.
        # ---------------------------------------------------------

        chunk_counts = (
            chunk
            .groupby(
                [
                    "Minute",
                    "Label"
                ],
                dropna=False
            )
            .size()
            .rename(
                "Count"
            )
            .reset_index()
        )


        file_minute_parts.append(
            chunk_counts
        )


        del chunk
        del chunk_counts

        gc.collect()


    # -------------------------------------------------------------------------
    # Combine only this file's tiny minute-level count tables.
    # -------------------------------------------------------------------------

    file_counts = pd.concat(
        file_minute_parts,
        ignore_index=True
    )


    file_counts = (
        file_counts
        .groupby(
            [
                "Minute",
                "Label"
            ],
            as_index=False
        )["Count"]
        .sum()
    )


    minute_count_parts.append(
        file_counts
    )


    print(
        f"    Rows processed       : "
        f"{file_rows:,}"
    )

    print(
        f"    Valid timestamps     : "
        f"{file_valid_timestamps:,}"
    )

    print(
        f"    Minute-label records : "
        f"{len(file_counts):,}"
    )


    del file_minute_parts
    del file_counts

    gc.collect()


# =============================================================================
# 8. COMBINE MINUTE COUNTS ACROSS ALL EIGHT FILES
# =============================================================================

all_minute_counts = pd.concat(
    minute_count_parts,
    ignore_index=True
)


all_minute_counts = (
    all_minute_counts
    .groupby(
        [
            "Minute",
            "Label"
        ],
        as_index=False
    )["Count"]
    .sum()
)


print("\n" + "=" * 115)
print("RAW LABEL AUDIT")
print("=" * 115)


label_totals = (
    all_minute_counts
    .groupby(
        "Label"
    )["Count"]
    .sum()
    .sort_values(
        ascending=False
    )
)


print("\nLabels discovered:")

print(
    label_totals.to_string()
)


# =============================================================================
# 9. CHECK FOR UNEXPECTED LABELS
# =============================================================================

discovered_labels = set(
    all_minute_counts[
        "Label"
    ]
    .dropna()
    .unique()
)


unexpected_labels = (
    discovered_labels
    -
    set(
        EXPECTED_LABELS
    )
)


missing_expected_attacks = (
    set(
        EXPECTED_ATTACK_TYPES
    )
    -
    discovered_labels
)


print("\nUnexpected labels:")
print(
    unexpected_labels
    if unexpected_labels
    else "None"
)


print("\nExpected attack types not discovered:")
print(
    missing_expected_attacks
    if missing_expected_attacks
    else "None"
)


# =============================================================================
# 10. PIVOT LABEL COUNTS INTO ONE ROW PER MINUTE
# =============================================================================

minute_label_wide = (
    all_minute_counts
    .pivot_table(
        index="Minute",
        columns="Label",
        values="Count",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)


minute_label_wide.columns.name = None


# =============================================================================
# 11. ENSURE ALL EXPECTED LABEL COLUMNS EXIST
# =============================================================================

for label in EXPECTED_LABELS:

    if label not in minute_label_wide.columns:

        minute_label_wide[
            label
        ] = 0


# =============================================================================
# 12. CREATE SAFE COLUMN NAMES
# =============================================================================
#
# Example:
#
#       "Web Attack - Brute Force"
#
# becomes:
#
#       Web_Attack_Brute_Force_Count
#
# =============================================================================

def label_to_count_column(label):

    safe = re.sub(
        r"[^A-Za-z0-9]+",
        "_",
        label
    )

    safe = safe.strip("_")

    return (
        safe
        +
        "_Count"
    )


rename_map = {
    label:
        label_to_count_column(
            label
        )
    for label in EXPECTED_LABELS
}


minute_label_wide = (
    minute_label_wide
    .rename(
        columns=rename_map
    )
)


attack_count_columns = [

    label_to_count_column(
        attack_type
    )

    for attack_type in EXPECTED_ATTACK_TYPES

]


benign_count_column = (
    label_to_count_column(
        "BENIGN"
    )
)


# =============================================================================
# 13. CONVERT COUNTS TO INTEGER
# =============================================================================

for column in (
    [benign_count_column]
    +
    attack_count_columns
):

    minute_label_wide[
        column
    ] = (
        minute_label_wide[
            column
        ]
        .fillna(0)
        .astype("int64")
    )


# =============================================================================
# 14. CALCULATE TOTAL ATTACK FLOWS FROM ATTACK-TYPE COUNTS
# =============================================================================

minute_label_wide[
    "Attack_Type_Count_Total"
] = (
    minute_label_wide[
        attack_count_columns
    ]
    .sum(
        axis=1
    )
)


# =============================================================================
# 15. DOMINANT ATTACK TYPE
# =============================================================================
#
# For benign minutes:
#
#       Dominant_Attack_Type = None
#
# For attack minutes:
#
#       attack type with largest flow count
#
# =============================================================================

count_to_attack_type = {

    label_to_count_column(
        attack_type
    ):
        attack_type

    for attack_type in EXPECTED_ATTACK_TYPES

}


def get_dominant_attack_type(row):

    attack_counts = row[
        attack_count_columns
    ]

    maximum = attack_counts.max()

    if maximum <= 0:
        return None

    winning_column = (
        attack_counts.idxmax()
    )

    return count_to_attack_type[
        winning_column
    ]


minute_label_wide[
    "Dominant_Attack_Type"
] = minute_label_wide.apply(
    get_dominant_attack_type,
    axis=1
)


# =============================================================================
# 16. NUMBER OF ATTACK TYPES ACTIVE IN EACH MINUTE
# =============================================================================

minute_label_wide[
    "Active_Attack_Type_Count"
] = (
    (
        minute_label_wide[
            attack_count_columns
        ]
        > 0
    )
    .sum(
        axis=1
    )
    .astype(int)
)


# =============================================================================
# 17. MERGE WITH EXISTING minute_df
# =============================================================================
#
# IMPORTANT:
#
# We use the EXISTING minute dataset as the authoritative temporal
# structure because it already contains:
#
#       Segment_ID
#       Attack_State
#       Flow_Count
#       Attack_Flow_Count
#       engineered temporal features
#
# We are ONLY enriching it with attack-type information.
#
# =============================================================================

columns_to_add = [

    "Minute",
    benign_count_column,

    *attack_count_columns,

    "Attack_Type_Count_Total",
    "Dominant_Attack_Type",
    "Active_Attack_Type_Count"

]


minute_enriched = minute_base.merge(

    minute_label_wide[
        columns_to_add
    ],

    on="Minute",

    how="left",

    validate="one_to_one"
)


# =============================================================================
# 18. FILL MINUTES WITH NO LABEL COUNT RECORD
# =============================================================================

all_count_columns = (

    [benign_count_column]
    +
    attack_count_columns
    +
    [
        "Attack_Type_Count_Total",
        "Active_Attack_Type_Count"
    ]

)


for column in all_count_columns:

    minute_enriched[
        column
    ] = (
        minute_enriched[
            column
        ]
        .fillna(0)
        .astype("int64")
    )


# =============================================================================
# 19. VALIDATION — ATTACK FLOW COUNTS MUST MATCH
# =============================================================================
#
# This is the most important check in the cell.
#
# Existing:
#
#       Attack_Flow_Count
#
# should equal:
#
#       sum(all 14 attack-type counts)
#
# for EVERY minute.
#
# =============================================================================

minute_enriched[
    "Attack_Count_Difference"
] = (

    minute_enriched[
        "Attack_Type_Count_Total"
    ]

    -

    minute_enriched[
        "Attack_Flow_Count"
    ]

)


mismatch_mask = (

    minute_enriched[
        "Attack_Count_Difference"
    ]
    != 0

)


mismatch_count = int(
    mismatch_mask.sum()
)


print("\n" + "=" * 115)
print("ATTACK-FLOW COUNT CONSISTENCY")
print("=" * 115)


print(
    f"\nMinutes with count mismatch : "
    f"{mismatch_count:,}"
)


if mismatch_count > 0:

    print(
        "\nFirst mismatches:"
    )

    display(

        minute_enriched.loc[
            mismatch_mask,
            [
                "Segment_ID",
                "Minute",
                "Attack_Flow_Count",
                "Attack_Type_Count_Total",
                "Attack_Count_Difference"
            ]
        ]
        .head(20)

    )


# =============================================================================
# 20. VALIDATION — BINARY ATTACK STATE
# =============================================================================

derived_attack_state = (

    minute_enriched[
        "Attack_Type_Count_Total"
    ]
    > 0

).astype(int)


state_mismatch_count = int(

    (
        derived_attack_state

        !=

        minute_enriched[
            "Attack_State"
        ].astype(int)

    )

    .sum()

)


print(
    f"Attack-state mismatches    : "
    f"{state_mismatch_count:,}"
)


# =============================================================================
# 21. ATTACK MINUTES BY TYPE
# =============================================================================

attack_type_summary_records = []


for attack_type in EXPECTED_ATTACK_TYPES:

    column = label_to_count_column(
        attack_type
    )

    attack_type_summary_records.append(
        {
            "Attack_Type":
                attack_type,

            "Total_Flows":
                int(
                    minute_enriched[
                        column
                    ].sum()
                ),

            "Active_Minutes":
                int(
                    (
                        minute_enriched[
                            column
                        ]
                        > 0
                    )
                    .sum()
                ),

            "Dominant_Minutes":
                int(
                    (
                        minute_enriched[
                            "Dominant_Attack_Type"
                        ]
                        == attack_type
                    )
                    .sum()
                )
        }
    )


attack_type_summary_df = (

    pd.DataFrame(
        attack_type_summary_records
    )

    .sort_values(
        "Total_Flows",
        ascending=False
    )

    .reset_index(
        drop=True
    )

)


print("\n" + "=" * 115)
print("ATTACK-TYPE TEMPORAL COVERAGE")
print("=" * 115)

print()

print(
    attack_type_summary_df.to_string(
        index=False
    )
)


# =============================================================================
# 22. MULTI-ATTACK MINUTES
# =============================================================================

multi_attack_minutes = int(

    (
        minute_enriched[
            "Active_Attack_Type_Count"
        ]
        > 1
    )

    .sum()

)


print("\n" + "=" * 115)
print("MULTI-ATTACK MINUTE AUDIT")
print("=" * 115)


print(
    f"\nMinutes with >1 attack type : "
    f"{multi_attack_minutes:,}"
)


print(
    f"Total attack minutes        : "
    f"{int(minute_enriched['Attack_State'].sum()):,}"
)


# =============================================================================
# 23. DISPLAY EXAMPLE ATTACK MINUTES
# =============================================================================

display_columns = [

    "Segment_ID",
    "Minute",

    "Attack_State",
    "Flow_Count",
    "Attack_Flow_Count",

    "Dominant_Attack_Type",
    "Active_Attack_Type_Count",

    *attack_count_columns

]


print("\n" + "=" * 115)
print("EXAMPLE ENRICHED ATTACK MINUTES")
print("=" * 115)

display(

    minute_enriched.loc[
        minute_enriched[
            "Attack_State"
        ]
        == 1,
        display_columns
    ]
    .head(20)

)


# =============================================================================
# 24. REMOVE TEMPORARY VALIDATION COLUMN
# =============================================================================

minute_enriched = (
    minute_enriched
    .drop(
        columns=[
            "Attack_Count_Difference"
        ]
    )
)


# =============================================================================
# 25. SAVE ENRICHED MINUTE DATASET
# =============================================================================

minute_enriched.to_parquet(

    ENRICHED_MINUTE_DATASET_PATH,

    index=False,

    compression="snappy"
)


attack_type_summary_df.to_csv(

    ATTACK_TYPE_AUDIT_PATH,

    index=False
)


# =============================================================================
# 26. FINAL VALIDATION
# =============================================================================

checks = {

    "Original minute row count preserved":
        len(minute_enriched)
        ==
        len(minute_base),

    "Segment count preserved":
        minute_enriched[
            "Segment_ID"
        ].nunique()
        ==
        minute_base[
            "Segment_ID"
        ].nunique(),

    "No attack-flow count mismatches":
        mismatch_count == 0,

    "No binary attack-state mismatches":
        state_mismatch_count == 0,

    "All 14 attack-count columns created":
        len(
            attack_count_columns
        )
        == 14
        and
        all(
            col in minute_enriched.columns
            for col in attack_count_columns
        ),

    "Dominant attack type created":
        "Dominant_Attack_Type"
        in minute_enriched.columns,

    "Enriched dataset saved":
        os.path.exists(
            ENRICHED_MINUTE_DATASET_PATH
        ),

    "Attack-type audit saved":
        os.path.exists(
            ATTACK_TYPE_AUDIT_PATH
        )
}


print("\n" + "=" * 115)
print("FINAL CELL 13 VALIDATION")
print("=" * 115)


for name, result in checks.items():

    print(
        f"{'✓' if bool(result) else '❌'} "
        f"{name}: {bool(result)}"
    )


passed = sum(
    bool(value)
    for value in checks.values()
)


print(
    f"\nPassed: "
    f"{passed} / {len(checks)}"
)


# =============================================================================
# 27. FINAL DATASET SUMMARY
# =============================================================================

print("\n" + "=" * 115)
print("ENRICHED MINUTE DATASET")
print("=" * 115)


print(
    f"\nRows       : "
    f"{len(minute_enriched):,}"
)

print(
    f"Columns    : "
    f"{len(minute_enriched.columns):,}"
)

print(
    f"Segments   : "
    f"{minute_enriched['Segment_ID'].nunique():,}"
)

print(
    f"Attack min : "
    f"{int(minute_enriched['Attack_State'].sum()):,}"
)


print("\nSaved:")
print(
    ENRICHED_MINUTE_DATASET_PATH
)


print("\n" + "=" * 115)


if all(
    bool(value)
    for value in checks.values()
):

    print(
        "✓ CELL 13 COMPLETED SUCCESSFULLY."
    )

    print()
    print(
        "The minute-level dataset now preserves the "
        "full multiclass attack composition."
    )

    print()
    print(
        "NEXT: audit future ATTACK TYPES at "
        "1/5/10/15/30-minute forecasting horizons."
    )

else:

    print(
        "⚠ DO NOT CONTINUE."
    )

    print(
        "One or more attack-type reconstruction checks failed."
    )


print("=" * 115)


# =============================================================================
# 28. OPTIONAL MEMORY CLEANUP
# =============================================================================

del all_minute_counts
del minute_label_wide
del minute_count_parts

gc.collect()

CELL 13 — PER-ATTACK-TYPE MINUTE ENRICHMENT

Raw CSV files found : 8
1. Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
2. Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
3. Friday-WorkingHours-Morning.pcap_ISCX.csv
4. Monday-WorkingHours.pcap_ISCX.csv
5. Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
6. Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
7. Tuesday-WorkingHours.pcap_ISCX.csv
8. Wednesday-workingHours.pcap_ISCX.csv

Existing minute dataset:
Rows     : 2,454
Columns  : 132
Segments : 10
Attack minutes : 520

PROCESSING RAW FILES IN MEMORY-SAFE CHUNKS

[1/8] Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
    Rows processed       : 225,745
    Valid timestamps     : 225,745
    Minute-label records : 114

[2/8] Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
    Rows processed       : 286,467
    Valid timestamps     : 286,467
    Minute-label records : 177

[3/8] Friday-WorkingHours-Morning.pcap_ISCX.csv
    Rows processed       : 191,033
   

,Segment_ID,Minute,Attack_State,Flow_Count,Attack_Flow_Count,Dominant_Attack_Type,Active_Attack_Type_Count,Bot_Count,DDoS_Count,DoS_GoldenEye_Count,...,DoS_Slowhttptest_Count,DoS_slowloris_Count,FTP_Patator_Count,Heartbleed_Count,Infiltration_Count,PortScan_Count,SSH_Patator_Count,Web_Attack_Brute_Force_Count,Web_Attack_Sql_Injection_Count,Web_Attack_XSS_Count
556,2,2017-07-04 02:09:00,1,515,89,SSH-Patator,1,0,0,0,...,0,0,0,0,0,0,89,0,0,0
557,2,2017-07-04 02:10:00,1,790,84,SSH-Patator,1,0,0,0,...,0,0,0,0,0,0,84,0,0,0
558,2,2017-07-04 02:11:00,1,701,95,SSH-Patator,1,0,0,0,...,0,0,0,0,0,0,95,0,0,0
559,2,2017-07-04 02:12:00,1,848,100,SSH-Patator,1,0,0,0,...,0,0,0,0,0,0,100,0,0,0
560,2,2017-07-04 02:13:00,1,982,102,SSH-Patator,1,0,0,0,...,0,0,0,0,0,0,102,0,0,0
561,2,2017-07-04 02:14:00,1,1741,98,SSH-Patator,1,0,0,0,...,0,0,0,0,0,0,98,0,0,0
562,2,2017-07-04 02:15:00,1,1277,79,SSH-Patator,1,0,0,0,...,0,0,0,0,0,0,79,0,0,0
563,2,2017-07-04 02:16:00,1,2605,84,SSH-Patator,1,0,0,0,...,0,0,0,0,0,0,84,0,0,0
564,2,2017-07-04 02:17:00,1,2667,89,SSH-Patator,1,0,0,0,...,0,0,0,0,0,0,89,0,0,0
565,2,2017-07-04 02:18:00,1,1816,93,SSH-Patator,1,0,0,0,...,0,0,0,0,0,0,93,0,0,0



FINAL CELL 13 VALIDATION
✓ Original minute row count preserved: True
✓ Segment count preserved: True
✓ No attack-flow count mismatches: True
✓ No binary attack-state mismatches: True
✓ All 14 attack-count columns created: True
✓ Dominant attack type created: True
✓ Enriched dataset saved: True
✓ Attack-type audit saved: True

Passed: 8 / 8

ENRICHED MINUTE DATASET

Rows       : 2,454
Columns    : 150
Segments   : 10
Attack min : 520

Saved:
/content/drive/MyDrive/Network-Attack-Forecasting/data/temporal/minute_level_dataset_with_attack_types.parquet

✓ CELL 13 COMPLETED SUCCESSFULLY.

The minute-level dataset now preserves the full multiclass attack composition.

NEXT: audit future ATTACK TYPES at 1/5/10/15/30-minute forecasting horizons.


0

In [20]:
# =============================================================================
# CELL 14 — FUTURE ATTACK-TYPE FORECASTING AUDIT
# =============================================================================
#
# PURPOSE
# -------
#
# Cell 12 established that the dataset supports the broader binary
# forecasting formulation:
#
#       previous 15 minutes
#               ↓
#       attack state at t + H
#
# Cell 13 then restored the complete multiclass attack information at
# minute level.
#
# We now audit whether ATTACK TYPE can also be forecast at the same
# future horizons:
#
#       H = 1, 5, 10, 15, 30 minutes
#
#
# IMPORTANT
# ---------
#
# This cell DOES NOT train a model.
#
# It answers:
#
#   1. How many future targets exist for every attack type?
#   2. How does attack-type availability change with horizon?
#   3. How many future attack-type targets come from:
#
#          current BENIGN -> future ATTACK
#
#      versus:
#
#          current ATTACK -> future ATTACK
#
#   4. Does the future attack type match the currently active attack?
#   5. Which attack types have enough temporal observations to be
#      realistic forecasting targets?
#
#
# We preserve the 10 validated temporal segments and NEVER construct
# a forecasting pair across a capture discontinuity.
#
# =============================================================================


import os
import pandas as pd
import numpy as np


# =============================================================================
# 1. LOAD ENRICHED MINUTE DATASET
# =============================================================================

PROJECT_ROOT = (
    "/content/drive/MyDrive/"
    "Network-Attack-Forecasting"
)

ENRICHED_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "temporal",
    "minute_level_dataset_with_attack_types.parquet"
)

assert os.path.exists(ENRICHED_PATH), (
    "Enriched minute dataset not found.\n"
    "Run Cell 13 first."
)


df = pd.read_parquet(ENRICHED_PATH)

df["Minute"] = pd.to_datetime(df["Minute"])

df = (
    df
    .sort_values(["Segment_ID", "Minute"])
    .reset_index(drop=True)
)


print("=" * 115)
print("CELL 14 — FUTURE ATTACK-TYPE FORECASTING AUDIT")
print("=" * 115)

print(f"\nRows          : {len(df):,}")
print(f"Columns       : {len(df.columns):,}")
print(f"Segments      : {df['Segment_ID'].nunique():,}")
print(f"Attack minutes: {int(df['Attack_State'].sum()):,}")


# =============================================================================
# 2. PARAMETERS
# =============================================================================

HISTORY_LENGTH = 15

FORECAST_HORIZONS = [
    1,
    5,
    10,
    15,
    30
]


ATTACK_TYPES = [

    "Bot",
    "DDoS",

    "DoS GoldenEye",
    "DoS Hulk",
    "DoS Slowhttptest",
    "DoS slowloris",

    "FTP-Patator",
    "Heartbleed",
    "Infiltration",
    "PortScan",
    "SSH-Patator",

    "Web Attack - Brute Force",
    "Web Attack - Sql Injection",
    "Web Attack - XSS"
]


print(f"\nHistory length : {HISTORY_LENGTH} minutes")
print(f"Horizons       : {FORECAST_HORIZONS}")
print(f"Attack types   : {len(ATTACK_TYPES)}")


# =============================================================================
# 3. VALIDATE ATTACK-TYPE INFORMATION
# =============================================================================

assert "Dominant_Attack_Type" in df.columns

assert "Attack_State" in df.columns


# Every attack minute should now have an attack type.

missing_type_attack_minutes = (

    (
        (df["Attack_State"] == 1)
        &
        (df["Dominant_Attack_Type"].isna())
    )

    .sum()
)


# Every benign minute should have no dominant attack type.

type_on_benign_minutes = (

    (
        (df["Attack_State"] == 0)
        &
        (df["Dominant_Attack_Type"].notna())
    )

    .sum()
)


print("\nAttack-type consistency:")

print(
    "Attack minutes missing type :",
    int(missing_type_attack_minutes)
)

print(
    "Benign minutes having type  :",
    int(type_on_benign_minutes)
)


assert missing_type_attack_minutes == 0
assert type_on_benign_minutes == 0


# =============================================================================
# 4. CREATE A SINGLE STATE LABEL
# =============================================================================
#
# Instead of storing:
#
#       Attack_State = 0
#       Dominant_Attack_Type = None
#
# we create:
#
#       Network_State = "BENIGN"
#
# Attack minutes become their attack type:
#
#       Network_State = "DDoS"
#       Network_State = "PortScan"
#       ...
#
# This gives us a clean 15-class future-state target:
#
#       BENIGN + 14 attack types
#
# =============================================================================

df["Network_State"] = np.where(

    df["Attack_State"].astype(int) == 0,

    "BENIGN",

    df["Dominant_Attack_Type"]
)


print("\nCurrent network-state distribution:")

print(
    df["Network_State"]
    .value_counts()
    .to_string()
)


# =============================================================================
# 5. CONSTRUCT FORECAST PAIRS
# =============================================================================
#
# For each valid historical sequence:
#
#       [t-14, ..., t]
#
# we inspect:
#
#       state at t+H
#
#
# IMPORTANT:
#
# These are NOT yet final model tensors.
#
# We only create a compact audit table containing the current and
# future states.
#
# =============================================================================

records = []


for horizon in FORECAST_HORIZONS:

    for segment_id, seg in df.groupby("Segment_ID"):

        seg = (
            seg
            .sort_values("Minute")
            .reset_index(drop=True)
        )

        n = len(seg)


        for end_idx in range(
            HISTORY_LENGTH - 1,
            n - horizon
        ):

            start_idx = (
                end_idx
                -
                HISTORY_LENGTH
                +
                1
            )

            target_idx = (
                end_idx
                +
                horizon
            )


            history = seg.iloc[
                start_idx:end_idx + 1
            ]

            current = seg.iloc[
                end_idx
            ]

            target = seg.iloc[
                target_idx
            ]


            # -------------------------------------------------------------
            # TEMPORAL VALIDATION
            # -------------------------------------------------------------

            history_start = (
                history["Minute"].iloc[0]
            )

            history_end = (
                history["Minute"].iloc[-1]
            )

            target_time = (
                target["Minute"]
            )


            expected_history_duration = (
                pd.Timedelta(
                    minutes=HISTORY_LENGTH - 1
                )
            )

            if (
                history_end
                -
                history_start
                !=
                expected_history_duration
            ):
                continue


            expected_target_time = (
                history_end
                +
                pd.Timedelta(
                    minutes=horizon
                )
            )

            if target_time != expected_target_time:
                continue


            # -------------------------------------------------------------
            # HISTORY INFORMATION
            # -------------------------------------------------------------

            history_attack_minutes = int(
                history[
                    "Attack_State"
                ].sum()
            )


            history_attack_types = (

                history.loc[
                    history["Attack_State"] == 1,
                    "Dominant_Attack_Type"
                ]

                .dropna()

                .unique()

                .tolist()
            )


            current_state = (
                current["Network_State"]
            )

            future_state = (
                target["Network_State"]
            )


            current_is_attack = int(
                current_state != "BENIGN"
            )

            future_is_attack = int(
                future_state != "BENIGN"
            )


            # -------------------------------------------------------------
            # TRANSITION TYPE
            # -------------------------------------------------------------

            if (
                current_state == "BENIGN"
                and
                future_state == "BENIGN"
            ):

                transition_type = (
                    "BENIGN -> BENIGN"
                )


            elif (
                current_state == "BENIGN"
                and
                future_state != "BENIGN"
            ):

                transition_type = (
                    "BENIGN -> ATTACK"
                )


            elif (
                current_state != "BENIGN"
                and
                future_state == "BENIGN"
            ):

                transition_type = (
                    "ATTACK -> BENIGN"
                )


            else:

                transition_type = (
                    "ATTACK -> ATTACK"
                )


            # -------------------------------------------------------------
            # SAME ATTACK CONTINUATION
            # -------------------------------------------------------------
            #
            # Example:
            #
            #       current = DDoS
            #       future  = DDoS
            #
            # This is attack-type continuation.
            #
            # -------------------------------------------------------------

            same_attack_type = int(

                current_state != "BENIGN"

                and

                future_state != "BENIGN"

                and

                current_state == future_state
            )


            # -------------------------------------------------------------
            # DIFFERENT ATTACK TYPE
            # -------------------------------------------------------------

            different_attack_type = int(

                current_state != "BENIGN"

                and

                future_state != "BENIGN"

                and

                current_state != future_state
            )


            # -------------------------------------------------------------
            # WAS FUTURE TYPE ALREADY PRESENT ANYWHERE IN HISTORY?
            # -------------------------------------------------------------

            future_type_seen_in_history = int(

                future_state != "BENIGN"

                and

                future_state
                in
                history_attack_types
            )


            records.append(
                {
                    "Segment_ID":
                        segment_id,

                    "History_Start":
                        history_start,

                    "History_End":
                        history_end,

                    "Target_Time":
                        target_time,

                    "Forecast_Horizon":
                        horizon,

                    "History_Attack_Minutes":
                        history_attack_minutes,

                    "History_Contains_Attack":
                        int(
                            history_attack_minutes > 0
                        ),

                    "History_Attack_Types":
                        "|".join(
                            sorted(
                                history_attack_types
                            )
                        ),

                    "Current_State":
                        current_state,

                    "Future_State":
                        future_state,

                    "Current_Is_Attack":
                        current_is_attack,

                    "Future_Is_Attack":
                        future_is_attack,

                    "Transition_Type":
                        transition_type,

                    "Same_Attack_Type":
                        same_attack_type,

                    "Different_Attack_Type":
                        different_attack_type,

                    "Future_Type_Seen_In_History":
                        future_type_seen_in_history
                }
            )


type_forecast_audit_df = pd.DataFrame(
    records
)


print("\n")
print("=" * 115)
print("FORECAST PAIRS CREATED")
print("=" * 115)

print(
    f"\nTotal horizon-specific pairs : "
    f"{len(type_forecast_audit_df):,}"
)


# =============================================================================
# 6. FUTURE STATE DISTRIBUTION BY HORIZON
# =============================================================================

print("\n")
print("=" * 115)
print("FUTURE NETWORK STATE DISTRIBUTION BY HORIZON")
print("=" * 115)


future_state_tables = {}


for horizon in FORECAST_HORIZONS:

    hdf = type_forecast_audit_df[
        type_forecast_audit_df[
            "Forecast_Horizon"
        ]
        ==
        horizon
    ]


    counts = (
        hdf[
            "Future_State"
        ]
        .value_counts()
        .reindex(
            ["BENIGN"] + ATTACK_TYPES,
            fill_value=0
        )
    )


    percentages = (
        100
        *
        counts
        /
        len(hdf)
    )


    table = pd.DataFrame(
        {
            "Samples":
                counts,

            "Percent":
                percentages.round(3)
        }
    )


    future_state_tables[
        horizon
    ] = table


    print(
        f"\nHORIZON = {horizon} MINUTES"
    )

    print("-" * 75)

    print(
        table.to_string()
    )


# =============================================================================
# 7. ATTACK-ONLY HORIZON MATRIX
# =============================================================================
#
# Easier comparison:
#
# rows    = attack type
# columns = forecasting horizon
#
# =============================================================================

attack_horizon_matrix = pd.DataFrame(
    index=ATTACK_TYPES
)


for horizon in FORECAST_HORIZONS:

    hdf = type_forecast_audit_df[
        (
            type_forecast_audit_df[
                "Forecast_Horizon"
            ]
            ==
            horizon
        )
        &
        (
            type_forecast_audit_df[
                "Future_Is_Attack"
            ]
            ==
            1
        )
    ]


    counts = (
        hdf[
            "Future_State"
        ]
        .value_counts()
        .reindex(
            ATTACK_TYPES,
            fill_value=0
        )
    )


    attack_horizon_matrix[
        f"H_{horizon}_min"
    ] = counts


print("\n")
print("=" * 115)
print("ATTACK-TYPE TARGET COUNTS BY FORECAST HORIZON")
print("=" * 115)

print()

print(
    attack_horizon_matrix.to_string()
)


# =============================================================================
# 8. TRANSITION DISTRIBUTION
# =============================================================================

print("\n")
print("=" * 115)
print("CURRENT -> FUTURE TRANSITION TYPES")
print("=" * 115)


transition_rows = []


for horizon in FORECAST_HORIZONS:

    hdf = type_forecast_audit_df[
        type_forecast_audit_df[
            "Forecast_Horizon"
        ]
        ==
        horizon
    ]


    counts = (
        hdf[
            "Transition_Type"
        ]
        .value_counts()
    )


    row = {
        "Horizon_Minutes":
            horizon
    }


    for transition in [

        "BENIGN -> BENIGN",
        "BENIGN -> ATTACK",
        "ATTACK -> BENIGN",
        "ATTACK -> ATTACK"

    ]:

        row[
            transition
        ] = int(
            counts.get(
                transition,
                0
            )
        )


    transition_rows.append(
        row
    )


transition_summary_df = pd.DataFrame(
    transition_rows
)


print()

print(
    transition_summary_df.to_string(
        index=False
    )
)


# =============================================================================
# 9. FUTURE ATTACK TYPE — WAS IT ALREADY PRESENT IN HISTORY?
# =============================================================================
#
# This is important.
#
# A future DDoS prediction is much easier if DDoS was already active
# during the historical input.
#
# We therefore separate:
#
#       future type already visible in history
#
# from:
#
#       future type NOT visible in history
#
# =============================================================================

print("\n")
print("=" * 115)
print("FUTURE ATTACK TYPE VISIBILITY IN HISTORY")
print("=" * 115)


visibility_rows = []


for horizon in FORECAST_HORIZONS:

    positive = type_forecast_audit_df[
        (
            type_forecast_audit_df[
                "Forecast_Horizon"
            ]
            ==
            horizon
        )
        &
        (
            type_forecast_audit_df[
                "Future_Is_Attack"
            ]
            ==
            1
        )
    ]


    total = len(
        positive
    )


    already_seen = int(
        positive[
            "Future_Type_Seen_In_History"
        ].sum()
    )


    not_seen = (
        total
        -
        already_seen
    )


    visibility_rows.append(
        {
            "Horizon_Minutes":
                horizon,

            "Future_Attack_Targets":
                total,

            "Future_Type_Already_In_History":
                already_seen,

            "Future_Type_Not_In_History":
                not_seen,

            "Already_Seen_Percent":
                (
                    100
                    *
                    already_seen
                    /
                    total
                )
                if total
                else np.nan,

            "Not_Seen_Percent":
                (
                    100
                    *
                    not_seen
                    /
                    total
                )
                if total
                else np.nan
        }
    )


visibility_summary_df = pd.DataFrame(
    visibility_rows
)


print()

print(
    visibility_summary_df.to_string(
        index=False
    )
)


# =============================================================================
# 10. BENIGN -> ATTACK COUNTS BY FUTURE ATTACK TYPE
# =============================================================================
#
# These are especially interesting because the CURRENT minute is benign
# while the FUTURE target is a specific attack.
#
# Again, these are temporal transition samples, NOT necessarily
# independent attack campaigns.
#
# =============================================================================

print("\n")
print("=" * 115)
print("BENIGN -> FUTURE ATTACK TYPE COUNTS")
print("=" * 115)


benign_to_attack_matrix = pd.DataFrame(
    index=ATTACK_TYPES
)


for horizon in FORECAST_HORIZONS:

    hdf = type_forecast_audit_df[
        (
            type_forecast_audit_df[
                "Forecast_Horizon"
            ]
            ==
            horizon
        )
        &
        (
            type_forecast_audit_df[
                "Current_State"
            ]
            ==
            "BENIGN"
        )
        &
        (
            type_forecast_audit_df[
                "Future_Is_Attack"
            ]
            ==
            1
        )
    ]


    counts = (
        hdf[
            "Future_State"
        ]
        .value_counts()
        .reindex(
            ATTACK_TYPES,
            fill_value=0
        )
    )


    benign_to_attack_matrix[
        f"H_{horizon}_min"
    ] = counts


print()

print(
    benign_to_attack_matrix.to_string()
)


# =============================================================================
# 11. SAME ATTACK CONTINUATION
# =============================================================================

print("\n")
print("=" * 115)
print("ATTACK -> SAME ATTACK TYPE CONTINUATION")
print("=" * 115)


continuation_rows = []


for horizon in FORECAST_HORIZONS:

    hdf = type_forecast_audit_df[
        (
            type_forecast_audit_df[
                "Forecast_Horizon"
            ]
            ==
            horizon
        )
        &
        (
            type_forecast_audit_df[
                "Future_Is_Attack"
            ]
            ==
            1
        )
    ]


    same = int(
        hdf[
            "Same_Attack_Type"
        ].sum()
    )


    different = int(
        hdf[
            "Different_Attack_Type"
        ].sum()
    )


    current_benign = int(
        (
            hdf[
                "Current_State"
            ]
            ==
            "BENIGN"
        )
        .sum()
    )


    continuation_rows.append(
        {
            "Horizon_Minutes":
                horizon,

            "Future_Attack_Targets":
                len(hdf),

            "Same_Attack_Continuation":
                same,

            "Different_Attack_To_Attack":
                different,

            "Current_Benign_To_Attack":
                current_benign
        }
    )


continuation_summary_df = pd.DataFrame(
    continuation_rows
)


print()

print(
    continuation_summary_df.to_string(
        index=False
    )
)


# =============================================================================
# 12. TEMPORAL SAFETY VALIDATION
# =============================================================================

invalid_history = (

    (
        type_forecast_audit_df[
            "History_End"
        ]
        -
        type_forecast_audit_df[
            "History_Start"
        ]
    )

    !=

    pd.Timedelta(
        minutes=HISTORY_LENGTH - 1
    )

).sum()


invalid_horizon = (

    (
        type_forecast_audit_df[
            "Target_Time"
        ]
        -
        type_forecast_audit_df[
            "History_End"
        ]
    )

    !=

    pd.to_timedelta(
        type_forecast_audit_df[
            "Forecast_Horizon"
        ],
        unit="m"
    )

).sum()


print("\n")
print("=" * 115)
print("TEMPORAL SAFETY VALIDATION")
print("=" * 115)


print(
    "\nInvalid historical windows :",
    int(invalid_history)
)

print(
    "Invalid forecast horizons  :",
    int(invalid_horizon)
)


assert invalid_history == 0
assert invalid_horizon == 0


# =============================================================================
# 13. SAVE AUDIT TABLES
# =============================================================================

AUDIT_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "audits"
)

os.makedirs(
    AUDIT_DIR,
    exist_ok=True
)


type_forecast_audit_df.to_parquet(
    os.path.join(
        AUDIT_DIR,
        "attack_type_forecasting_pairs.parquet"
    ),
    index=False
)


attack_horizon_matrix.to_csv(
    os.path.join(
        AUDIT_DIR,
        "attack_type_horizon_counts.csv"
    )
)


benign_to_attack_matrix.to_csv(
    os.path.join(
        AUDIT_DIR,
        "benign_to_attack_type_horizon_counts.csv"
    )
)


transition_summary_df.to_csv(
    os.path.join(
        AUDIT_DIR,
        "forecast_transition_summary.csv"
    ),
    index=False
)


visibility_summary_df.to_csv(
    os.path.join(
        AUDIT_DIR,
        "future_type_history_visibility.csv"
    ),
    index=False
)


# =============================================================================
# 14. FINAL SUMMARY
# =============================================================================

print("\n")
print("=" * 115)
print("CELL 14 COMPLETE")
print("=" * 115)


print(

)

print("=" * 115)

CELL 14 — FUTURE ATTACK-TYPE FORECASTING AUDIT

Rows          : 2,454
Columns       : 150
Segments      : 10
Attack minutes: 520

History length : 15 minutes
Horizons       : [1, 5, 10, 15, 30]
Attack types   : 14

Attack-type consistency:
Attack minutes missing type : 0
Benign minutes having type  : 0

Current network-state distribution:
Network_State
BENIGN                        1934
Bot                            163
FTP-Patator                     64
SSH-Patator                     63
Web Attack - Brute Force        44
Infiltration                    28
DoS slowloris                   27
PortScan                        27
DDoS                            21
DoS Hulk                        21
Web Attack - XSS                21
DoS Slowhttptest                19
Heartbleed                      11
DoS GoldenEye                    8
Web Attack - Sql Injection       3


FORECAST PAIRS CREATED

Total horizon-specific pairs : 10,960


FUTURE NETWORK STATE DISTRIBUTION BY HORIZON

HORIZON 

In [21]:
# =============================================================================
# CELL 15 — ATTACK EPISODE + SEGMENT COVERAGE AUDIT
# =============================================================================
#
# PURPOSE
# -------
#
# Cells 12–14 counted forecasting WINDOWS and attack MINUTES.
#
# However:
#
#       100 attack minutes != 100 independent attacks
#
# Example:
#
#       10:00 Bot
#       10:01 Bot
#       10:02 Bot
#       ...
#       10:30 Bot
#
# These may represent ONE continuous Bot episode rather than
# 31 independent attack events.
#
#
# This cell therefore identifies ATTACK EPISODES.
#
#
# DEFINITION USED HERE
# --------------------
#
# A new episode begins when:
#
#   1. attack type changes, OR
#   2. attack traffic disappears and later reappears, OR
#   3. a capture-segment boundary is crossed.
#
#
# We calculate episodes under multiple benign-gap tolerances:
#
#       0 minutes
#       1 minute
#       2 minutes
#       5 minutes
#
# Why?
#
# Short benign interruptions may fragment one underlying campaign into
# many artificial "episodes".
#
# Example:
#
#       DDoS
#       DDoS
#       BENIGN
#       DDoS
#
# Under strict 0-gap definition:
#
#       episode 1
#       BENIGN
#       episode 2
#
# Under a 1-minute gap tolerance:
#
#       one DDoS episode
#
#
# THIS CELL DOES NOT TRAIN A MODEL.
#
# =============================================================================


import os
import pandas as pd
import numpy as np


# =============================================================================
# 1. LOAD ENRICHED MINUTE DATASET
# =============================================================================

PROJECT_ROOT = (
    "/content/drive/MyDrive/"
    "Network-Attack-Forecasting"
)

ENRICHED_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "temporal",
    "minute_level_dataset_with_attack_types.parquet"
)

AUDIT_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "audits"
)

os.makedirs(
    AUDIT_DIR,
    exist_ok=True
)


assert os.path.exists(
    ENRICHED_PATH
), (
    "Enriched minute dataset not found. "
    "Run Cell 13 first."
)


df = pd.read_parquet(
    ENRICHED_PATH
)


df["Minute"] = pd.to_datetime(
    df["Minute"]
)


df = (
    df
    .sort_values(
        [
            "Segment_ID",
            "Minute"
        ]
    )
    .reset_index(
        drop=True
    )
)


print("=" * 115)
print("CELL 15 — ATTACK EPISODE + SEGMENT COVERAGE AUDIT")
print("=" * 115)


print(
    f"\nRows          : {len(df):,}"
)

print(
    f"Segments      : {df['Segment_ID'].nunique():,}"
)

print(
    f"Attack minutes: {int(df['Attack_State'].sum()):,}"
)


# =============================================================================
# 2. ATTACK TYPES
# =============================================================================

ATTACK_TYPES = sorted(

    df.loc[
        df["Attack_State"] == 1,
        "Dominant_Attack_Type"
    ]

    .dropna()

    .unique()

    .tolist()

)


print(
    f"Attack types  : {len(ATTACK_TYPES)}"
)


print("\nAttack types discovered:")

for attack in ATTACK_TYPES:

    print(
        f"  - {attack}"
    )


# =============================================================================
# 3. BASIC ATTACK-TYPE SEGMENT COVERAGE
# =============================================================================
#
# Before episode detection, determine:
#
#       which attack occurs in which segment?
#
# =============================================================================

attack_rows = df[
    df["Attack_State"] == 1
].copy()


segment_coverage = (

    attack_rows

    .groupby(
        "Dominant_Attack_Type"
    )

    .agg(

        Attack_Minutes=(
            "Minute",
            "size"
        ),

        Segments_Present=(
            "Segment_ID",
            "nunique"
        )

    )

    .reset_index()

    .rename(
        columns={
            "Dominant_Attack_Type":
                "Attack_Type"
        }
    )

)


segment_lists = (

    attack_rows

    .groupby(
        "Dominant_Attack_Type"
    )["Segment_ID"]

    .apply(
        lambda x:
            sorted(
                x.unique().tolist()
            )
    )

)


segment_coverage[
    "Segment_IDs"
] = (

    segment_coverage[
        "Attack_Type"
    ]

    .map(
        segment_lists
    )

)


segment_coverage = (

    segment_coverage

    .sort_values(
        [
            "Segments_Present",
            "Attack_Minutes"
        ],
        ascending=[
            True,
            False
        ]
    )

    .reset_index(
        drop=True
    )

)


print("\n")
print("=" * 115)
print("ATTACK TYPE — SEGMENT COVERAGE")
print("=" * 115)

print()

print(
    segment_coverage.to_string(
        index=False
    )
)


# =============================================================================
# 4. FUNCTION TO BUILD ATTACK EPISODES
# =============================================================================
#
# GAP_TOLERANCE controls how many BENIGN minutes may occur between two
# observations of the SAME attack type while still treating them as
# one episode.
#
#
# IMPORTANT:
#
# Episodes can NEVER cross Segment_ID boundaries.
#
# Attack types can NEVER be merged with a different attack type.
#
# =============================================================================

def build_attack_episodes(
    minute_df,
    gap_tolerance_minutes=0
):

    episodes = []

    episode_id = 0


    # -------------------------------------------------------------------------
    # Process each capture segment independently.
    # -------------------------------------------------------------------------

    for segment_id, seg in minute_df.groupby(
        "Segment_ID"
    ):

        seg = (
            seg
            .sort_values(
                "Minute"
            )
            .reset_index(
                drop=True
            )
        )


        # ---------------------------------------------------------------------
        # We build episodes separately for every attack type.
        # ---------------------------------------------------------------------

        for attack_type in ATTACK_TYPES:

            attack_times = (

                seg.loc[
                    (
                        seg[
                            "Attack_State"
                        ]
                        ==
                        1
                    )
                    &
                    (
                        seg[
                            "Dominant_Attack_Type"
                        ]
                        ==
                        attack_type
                    ),
                    "Minute"
                ]

                .sort_values()

                .tolist()

            )


            if len(
                attack_times
            ) == 0:

                continue


            current_start = (
                attack_times[0]
            )

            current_end = (
                attack_times[0]
            )

            attack_minute_count = 1


            # -----------------------------------------------------------------
            # Examine subsequent attack minutes.
            # -----------------------------------------------------------------

            for timestamp in attack_times[1:]:

                gap_minutes = int(

                    (
                        timestamp
                        -
                        current_end
                    )

                    .total_seconds()

                    /
                    60

                ) - 1


                # -------------------------------------------------------------
                # Merge when benign interruption <= tolerance.
                # -------------------------------------------------------------

                if (
                    gap_minutes
                    <=
                    gap_tolerance_minutes
                ):

                    current_end = timestamp

                    attack_minute_count += 1


                else:

                    episode_id += 1


                    total_span_minutes = int(

                        (
                            current_end
                            -
                            current_start
                        )

                        .total_seconds()

                        /
                        60

                    ) + 1


                    episodes.append(
                        {
                            "Episode_ID":
                                episode_id,

                            "Segment_ID":
                                segment_id,

                            "Attack_Type":
                                attack_type,

                            "Start_Time":
                                current_start,

                            "End_Time":
                                current_end,

                            "Attack_Minutes":
                                attack_minute_count,

                            "Episode_Span_Minutes":
                                total_span_minutes,

                            "Internal_Benign_Minutes":
                                (
                                    total_span_minutes
                                    -
                                    attack_minute_count
                                ),

                            "Gap_Tolerance":
                                gap_tolerance_minutes
                        }
                    )


                    # Start new episode.

                    current_start = timestamp

                    current_end = timestamp

                    attack_minute_count = 1


            # -----------------------------------------------------------------
            # Save final episode.
            # -----------------------------------------------------------------

            episode_id += 1


            total_span_minutes = int(

                (
                    current_end
                    -
                    current_start
                )

                .total_seconds()

                /
                60

            ) + 1


            episodes.append(
                {
                    "Episode_ID":
                        episode_id,

                    "Segment_ID":
                        segment_id,

                    "Attack_Type":
                        attack_type,

                    "Start_Time":
                        current_start,

                    "End_Time":
                        current_end,

                    "Attack_Minutes":
                        attack_minute_count,

                    "Episode_Span_Minutes":
                        total_span_minutes,

                    "Internal_Benign_Minutes":
                        (
                            total_span_minutes
                            -
                            attack_minute_count
                        ),

                    "Gap_Tolerance":
                        gap_tolerance_minutes
                }
            )


    return pd.DataFrame(
        episodes
    )


# =============================================================================
# 5. BUILD EPISODES USING MULTIPLE GAP DEFINITIONS
# =============================================================================

GAP_TOLERANCES = [
    0,
    1,
    2,
    5
]


episode_tables = {}


for tolerance in GAP_TOLERANCES:

    episode_tables[
        tolerance
    ] = build_attack_episodes(
        df,
        gap_tolerance_minutes=tolerance
    )


# =============================================================================
# 6. TOTAL EPISODE COUNTS UNDER EACH DEFINITION
# =============================================================================

episode_definition_rows = []


for tolerance in GAP_TOLERANCES:

    edf = episode_tables[
        tolerance
    ]


    episode_definition_rows.append(
        {
            "Gap_Tolerance_Minutes":
                tolerance,

            "Total_Episodes":
                len(edf),

            "Attack_Types":
                edf[
                    "Attack_Type"
                ].nunique(),

            "Segments":
                edf[
                    "Segment_ID"
                ].nunique(),

            "Median_Episode_Attack_Minutes":
                edf[
                    "Attack_Minutes"
                ].median(),

            "Median_Episode_Span_Minutes":
                edf[
                    "Episode_Span_Minutes"
                ].median(),

            "Maximum_Episode_Span_Minutes":
                edf[
                    "Episode_Span_Minutes"
                ].max()
        }
    )


episode_definition_summary = pd.DataFrame(
    episode_definition_rows
)


print("\n")
print("=" * 115)
print("EPISODE COUNT SENSITIVITY TO BENIGN-GAP TOLERANCE")
print("=" * 115)

print()

print(
    episode_definition_summary.to_string(
        index=False
    )
)


# =============================================================================
# 7. EPISODES PER ATTACK TYPE
# =============================================================================

episode_count_matrix = pd.DataFrame(
    index=ATTACK_TYPES
)


for tolerance in GAP_TOLERANCES:

    edf = episode_tables[
        tolerance
    ]


    counts = (

        edf[
            "Attack_Type"
        ]

        .value_counts()

        .reindex(
            ATTACK_TYPES,
            fill_value=0
        )

    )


    episode_count_matrix[
        f"Gap_{tolerance}_min"
    ] = counts


print("\n")
print("=" * 115)
print("EPISODES PER ATTACK TYPE")
print("=" * 115)

print()

print(
    episode_count_matrix.to_string()
)


# =============================================================================
# 8. USE 2-MINUTE GAP TOLERANCE FOR DETAILED AUDIT
# =============================================================================
#
# This is NOT yet a permanent scientific decision.
#
# We use 2 minutes here as a middle-ground audit view.
#
# We will inspect sensitivity across all tolerances before choosing
# an episode definition for final evaluation.
#
# =============================================================================

AUDIT_TOLERANCE = 2


episodes = episode_tables[
    AUDIT_TOLERANCE
].copy()


# =============================================================================
# 9. DETAILED ATTACK-TYPE EPISODE SUMMARY
# =============================================================================

episode_summary = (

    episodes

    .groupby(
        "Attack_Type"
    )

    .agg(

        Episodes=(
            "Episode_ID",
            "count"
        ),

        Segments=(
            "Segment_ID",
            "nunique"
        ),

        Total_Attack_Minutes=(
            "Attack_Minutes",
            "sum"
        ),

        Median_Attack_Minutes=(
            "Attack_Minutes",
            "median"
        ),

        Max_Attack_Minutes=(
            "Attack_Minutes",
            "max"
        ),

        Median_Span_Minutes=(
            "Episode_Span_Minutes",
            "median"
        ),

        Max_Span_Minutes=(
            "Episode_Span_Minutes",
            "max"
        )

    )

    .reset_index()

)


episode_summary[
    "Attack_Minutes_Per_Episode"
] = (

    episode_summary[
        "Total_Attack_Minutes"
    ]

    /

    episode_summary[
        "Episodes"
    ]

)


episode_summary = (

    episode_summary

    .sort_values(
        "Episodes",
        ascending=False
    )

    .reset_index(
        drop=True
    )

)


print("\n")
print("=" * 115)
print(
    f"DETAILED EPISODE SUMMARY — "
    f"{AUDIT_TOLERANCE}-MINUTE GAP TOLERANCE"
)
print("=" * 115)

print()

print(
    episode_summary.to_string(
        index=False
    )
)


# =============================================================================
# 10. INDIVIDUAL EPISODE TABLE
# =============================================================================

print("\n")
print("=" * 115)
print(
    f"INDIVIDUAL ATTACK EPISODES — "
    f"{AUDIT_TOLERANCE}-MINUTE GAP TOLERANCE"
)
print("=" * 115)

print()

print(
    episodes[
        [
            "Episode_ID",
            "Segment_ID",
            "Attack_Type",
            "Start_Time",
            "End_Time",
            "Attack_Minutes",
            "Episode_Span_Minutes",
            "Internal_Benign_Minutes"
        ]
    ]

    .sort_values(
        [
            "Segment_ID",
            "Start_Time"
        ]
    )

    .to_string(
        index=False
    )
)


# =============================================================================
# 11. HOW CONCENTRATED ARE ATTACK TYPES IN SEGMENTS?
# =============================================================================
#
# If an attack exists in only one segment, then segment-held-out
# evaluation cannot place that attack in BOTH train and test.
#
# =============================================================================

single_segment_attacks = (

    segment_coverage.loc[
        segment_coverage[
            "Segments_Present"
        ]
        ==
        1,
        "Attack_Type"
    ]

    .tolist()

)


multi_segment_attacks = (

    segment_coverage.loc[
        segment_coverage[
            "Segments_Present"
        ]
        >
        1,
        "Attack_Type"
    ]

    .tolist()

)


print("\n")
print("=" * 115)
print("SEGMENT GENERALIZATION AUDIT")
print("=" * 115)


print(
    f"\nAttack types appearing in ONE segment only: "
    f"{len(single_segment_attacks)}"
)

for attack in single_segment_attacks:

    print(
        f"  - {attack}"
    )


print(
    f"\nAttack types appearing in MULTIPLE segments: "
    f"{len(multi_segment_attacks)}"
)

for attack in multi_segment_attacks:

    print(
        f"  - {attack}"
    )


# =============================================================================
# 12. SPARSITY FLAGS
# =============================================================================
#
# These are descriptive audit flags, NOT automatic decisions about
# whether a class must be removed.
#
# We flag classes with:
#
#       < 5 episodes
#
# or:
#
#       only 1 segment
#
# because such classes will be particularly difficult to evaluate
# independently.
#
# =============================================================================

sparsity_audit = (
    episode_summary[
        [
            "Attack_Type",
            "Episodes",
            "Segments",
            "Total_Attack_Minutes"
        ]
    ]
    .copy()
)


sparsity_audit[
    "Few_Episodes_LT5"
] = (

    sparsity_audit[
        "Episodes"
    ]
    <
    5

)


sparsity_audit[
    "Single_Segment"
] = (

    sparsity_audit[
        "Segments"
    ]
    ==
    1

)


sparsity_audit[
    "Sparse_For_Independent_Type_Evaluation"
] = (

    sparsity_audit[
        "Few_Episodes_LT5"
    ]

    |

    sparsity_audit[
        "Single_Segment"
    ]

)


print("\n")
print("=" * 115)
print("ATTACK-TYPE SPARSITY AUDIT")
print("=" * 115)

print()

print(
    sparsity_audit.to_string(
        index=False
    )
)


# =============================================================================
# 13. OVERALL DATASET-LEVEL EPISODE STATISTICS
# =============================================================================

total_episodes = len(
    episodes
)


attack_types_with_lt5_episodes = int(

    (
        episode_summary[
            "Episodes"
        ]
        <
        5
    )

    .sum()

)


single_segment_count = len(
    single_segment_attacks
)


print("\n")
print("=" * 115)
print("OVERALL INDEPENDENCE / SPARSITY SUMMARY")
print("=" * 115)


print(
    f"\nTotal attack minutes             : "
    f"{int(df['Attack_State'].sum()):,}"
)

print(
    f"Attack episodes (2-min tolerance): "
    f"{total_episodes:,}"
)

print(
    f"Attack types                     : "
    f"{len(ATTACK_TYPES):,}"
)

print(
    f"Types with <5 episodes           : "
    f"{attack_types_with_lt5_episodes:,}"
)

print(
    f"Types present in only 1 segment  : "
    f"{single_segment_count:,}"
)


# =============================================================================
# 14. SAVE AUDIT OUTPUTS
# =============================================================================

segment_coverage.to_csv(

    os.path.join(
        AUDIT_DIR,
        "attack_type_segment_coverage.csv"
    ),

    index=False
)


episode_definition_summary.to_csv(

    os.path.join(
        AUDIT_DIR,
        "episode_gap_sensitivity.csv"
    ),

    index=False
)


episode_count_matrix.to_csv(

    os.path.join(
        AUDIT_DIR,
        "attack_episode_counts_by_type.csv"
    )
)


episode_summary.to_csv(

    os.path.join(
        AUDIT_DIR,
        "attack_episode_summary.csv"
    ),

    index=False
)


episodes.to_parquet(

    os.path.join(
        AUDIT_DIR,
        "attack_episodes_gap2.parquet"
    ),

    index=False
)


sparsity_audit.to_csv(

    os.path.join(
        AUDIT_DIR,
        "attack_type_sparsity_audit.csv"
    ),

    index=False
)


# =============================================================================
# 15. FINAL VALIDATION
# =============================================================================

checks = {

    "All 14 attack types represented":
        len(ATTACK_TYPES) == 14,

    "All 10 capture segments preserved":
        df["Segment_ID"].nunique() == 10,

    "Every attack episode stays inside one segment":
        episodes[
            "Segment_ID"
        ].notna().all(),

    "Every episode has positive attack minutes":
        (
            episodes[
                "Attack_Minutes"
            ]
            >
            0
        ).all(),

    "Every episode has valid duration":
        (
            episodes[
                "Episode_Span_Minutes"
            ]
            >=
            episodes[
                "Attack_Minutes"
            ]
        ).all()
}


print("\n")
print("=" * 115)
print("FINAL CELL 15 VALIDATION")
print("=" * 115)


for name, result in checks.items():

    print(
        f"{'✓' if bool(result) else '❌'} "
        f"{name}: {bool(result)}"
    )


print("\n" + "=" * 115)
print("CELL 15 COMPLETE")
print("=" * 115)


print(
    """
This audit distinguishes:

    ATTACK MINUTES

from:

    TEMPORALLY DISTINCT ATTACK EPISODES.

This distinction is critical because overlapping sliding windows from
one long attack campaign are NOT independent attack examples.

NEXT:

Use these results to decide:

    1. Whether CICIDS2017 contains enough independent attack episodes
       for the desired forecasting tasks.

    2. Which attack-type targets are defensible.

    3. Whether attack types need to be grouped into broader families.

    4. How train / validation / test splits must be constructed without
       temporal or campaign leakage.

DO NOT TRAIN A MODEL YET.
"""
)

print("=" * 115)

CELL 15 — ATTACK EPISODE + SEGMENT COVERAGE AUDIT

Rows          : 2,454
Segments      : 10
Attack minutes: 520
Attack types  : 14

Attack types discovered:
  - Bot
  - DDoS
  - DoS GoldenEye
  - DoS Hulk
  - DoS Slowhttptest
  - DoS slowloris
  - FTP-Patator
  - Heartbleed
  - Infiltration
  - PortScan
  - SSH-Patator
  - Web Attack - Brute Force
  - Web Attack - Sql Injection
  - Web Attack - XSS


ATTACK TYPE — SEGMENT COVERAGE

               Attack_Type  Attack_Minutes  Segments_Present Segment_IDs
                       Bot             163                 1         [9]
               FTP-Patator              64                 1         [3]
               SSH-Patator              63                 1         [2]
  Web Attack - Brute Force              44                 1         [7]
              Infiltration              28                 1         [6]
                  PortScan              27                 1         [8]
                      DDoS              21           

In [22]:
# =============================================================================
# CELL 16 — MULTI-RESOLUTION TEMPORAL FEASIBILITY AUDIT
# =============================================================================
#
# PURPOSE
# -------
#
# Our current temporal dataset uses:
#
#       60 seconds -> one network-state observation
#
# This produced only 2,454 temporal rows from the much larger raw
# CICIDS2017 flow dataset.
#
# Before accepting 60 seconds as the final resolution, we compare:
#
#       10 seconds
#       30 seconds
#       60 seconds
#
# IMPORTANT:
#
# We are NOT creating synthetic data.
#
# Every temporal bin is constructed from REAL CICIDS2017 flows.
#
# This cell asks:
#
#   1. How many temporal states exist at each resolution?
#   2. How many contain attack traffic?
#   3. How many contain benign traffic?
#   4. How many flows occur per bin?
#   5. How many bins are empty?
#   6. How many forecasting sequences could be created?
#   7. How much attack-state temporal coverage exists?
#   8. Does finer resolution actually give useful additional temporal
#      structure, or merely many nearly-empty observations?
#
# RAM SAFETY
# ----------
#
# We DO NOT load all ~2.8 million raw flows into RAM simultaneously.
#
# Each raw CSV is processed in chunks and immediately aggregated.
#
# FIX IN THIS VERSION
# -------------------
#
# The previous version could crash on the WebAttacks file because
# Attack_Type was not guaranteed to have a string dtype before using
# pandas .str operations.
#
# This version:
#
#   1. Explicitly converts normalized labels to pandas StringDtype.
#   2. Handles missing labels BEFORE binary attack-label creation.
#   3. Never silently converts a missing label into an attack.
#   4. Keeps latin1 decoding for the malformed WebAttack characters.
#
# =============================================================================


import os
import gc
import pandas as pd
import numpy as np


# =============================================================================
# 1. PROJECT PATHS
# =============================================================================

PROJECT_ROOT = (
    "/content/drive/MyDrive/"
    "Network-Attack-Forecasting"
)

RAW_DIR = (
    "/content/drive/MyDrive/"
    "TrafficLabelling "
)

AUDIT_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "audits"
)

TEMPORAL_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "temporal"
)

os.makedirs(AUDIT_DIR, exist_ok=True)
os.makedirs(TEMPORAL_DIR, exist_ok=True)


raw_files = sorted([
    os.path.join(RAW_DIR, f)
    for f in os.listdir(RAW_DIR)
    if f.lower().endswith(".csv")
])


assert len(raw_files) == 8, (
    f"Expected 8 raw CSV files, found {len(raw_files)}"
)


print("=" * 115)
print("CELL 16 — MULTI-RESOLUTION TEMPORAL FEASIBILITY AUDIT")
print("=" * 115)

print(f"\nRaw files found : {len(raw_files)}")


# =============================================================================
# 2. RESOLUTIONS TO TEST
# =============================================================================

RESOLUTIONS = {
    "10s": 10,
    "30s": 30,
    "60s": 60
}

print("\nTemporal resolutions:")

for name, seconds in RESOLUTIONS.items():
    print(f"  {name:>3} -> {seconds} seconds")


# =============================================================================
# 3. COLUMNS REQUIRED FOR THIS AUDIT
# =============================================================================
#
# Only the columns required for the temporal feasibility audit are read.
# This keeps RAM usage low.
#
# =============================================================================

USECOLS = [
    "Timestamp",
    "Label",
    "Flow ID"
]


# =============================================================================
# 4. LABEL NORMALIZATION
# =============================================================================

def normalize_attack_label(label):

    if pd.isna(label):
        return pd.NA

    label = str(label).strip()

    # Remove malformed characters occasionally present in WebAttack labels.
    label = (
        label
        .replace("�", "")
        .replace("–", "-")
        .replace("—", "-")
        .replace("Â", "")
        .replace("Ã", "")
    )

    label = " ".join(label.split())

    lower = label.lower()

    # -------------------------------------------------------------------------
    # Standardize CICIDS2017 Web Attack labels.
    # -------------------------------------------------------------------------

    if "web attack" in lower:

        if "brute force" in lower:
            return "Web Attack - Brute Force"

        if "sql injection" in lower:
            return "Web Attack - Sql Injection"

        if "xss" in lower:
            return "Web Attack - XSS"

    return label


# =============================================================================
# 5. ROBUST CSV ENCODING / CHUNK SIZE
# =============================================================================
#
# latin1 is intentional.
#
# The CICIDS2017 WebAttack CSV contains bytes that previously caused
# UTF-8 decoding errors.
#
# =============================================================================

CSV_ENCODING = "latin1"

CHUNK_SIZE = 150_000


# =============================================================================
# 6. TEMPORARY AGGREGATION STORAGE
# =============================================================================

resolution_chunks = {
    name: []
    for name in RESOLUTIONS
}


# Track malformed/missing labels for validation.

total_missing_labels = 0


# =============================================================================
# 7. PROCESS RAW FILES ONE AT A TIME
# =============================================================================

for file_number, path in enumerate(raw_files, start=1):

    filename = os.path.basename(path)

    print("\n" + "=" * 115)
    print(
        f"FILE {file_number}/8 — {filename}"
    )
    print("=" * 115)


    file_resolution_parts = {
        name: []
        for name in RESOLUTIONS
    }


    # -------------------------------------------------------------------------
    # Read raw CSV in chunks.
    #
    # NOTE:
    # We deliberately do NOT specify dtype={"Label": ...} here because the
    # original raw column name may contain leading whitespace.
    #
    # Instead, we normalize the column names first and explicitly convert
    # Attack_Type to pandas StringDtype afterward.
    # -------------------------------------------------------------------------

    reader = pd.read_csv(
        path,
        encoding=CSV_ENCODING,
        usecols=lambda c: c.strip() in USECOLS,
        chunksize=CHUNK_SIZE,
        low_memory=False
    )


    processed_rows = 0


    for chunk_number, chunk in enumerate(reader, start=1):

        # ---------------------------------------------------------------------
        # Normalize column names.
        # ---------------------------------------------------------------------

        chunk.columns = [
            c.strip()
            for c in chunk.columns
        ]


        # ---------------------------------------------------------------------
        # Parse timestamp.
        #
        # CICIDS2017 commonly contains:
        #
        #       dd/mm/yyyy HH:MM
        #       dd/mm/yyyy HH:MM:SS
        #
        # ---------------------------------------------------------------------

        chunk["Timestamp"] = pd.to_datetime(
            chunk["Timestamp"],
            dayfirst=True,
            errors="coerce",
            format="mixed"
        )


        # Remove rows without usable timestamps.

        chunk = chunk[
            chunk["Timestamp"].notna()
        ].copy()


        # =====================================================================
        # LABEL NORMALIZATION — CORRECTED VERSION
        # =====================================================================
        #
        # This is the important fix.
        #
        # Explicit StringDtype conversion guarantees that .str operations are
        # valid even if pandas inferred a strange dtype in a particular chunk.
        #
        # =====================================================================

        chunk["Attack_Type"] = (
            chunk["Label"]
            .map(normalize_attack_label)
            .astype("string")
            .str.strip()
        )


        # ---------------------------------------------------------------------
        # Detect missing labels BEFORE binary encoding.
        #
        # We must NOT do:
        #
        #     Attack_Type != "BENIGN"
        #
        # on missing values and accidentally treat them as attacks.
        # ---------------------------------------------------------------------

        missing_label_mask = (
            chunk["Attack_Type"].isna()
            |
            chunk["Attack_Type"].eq("")
        )


        missing_count = int(
            missing_label_mask.sum()
        )


        if missing_count > 0:

            total_missing_labels += missing_count

            print(
                f"  ⚠ chunk {chunk_number}: "
                f"{missing_count:,} rows have missing/empty labels "
                f"and will be excluded."
            )


            chunk = chunk.loc[
                ~missing_label_mask
            ].copy()


        # ---------------------------------------------------------------------
        # Binary attack indicator.
        #
        #     0 = BENIGN
        #     1 = any attack
        #
        # ---------------------------------------------------------------------

        chunk["Attack_Label"] = (
            chunk["Attack_Type"]
            .str.upper()
            .ne("BENIGN")
            .astype("int8")
        )


        # ---------------------------------------------------------------------
        # Aggregate separately for every candidate resolution.
        # ---------------------------------------------------------------------

        for resolution_name, seconds in RESOLUTIONS.items():

            freq = f"{seconds}s"


            temp = chunk[
                [
                    "Timestamp",
                    "Attack_Type",
                    "Attack_Label"
                ]
            ].copy()


            temp["Time_Bin"] = (
                temp["Timestamp"]
                .dt.floor(freq)
            )


            # -----------------------------------------------------------------
            # Base flow counts.
            # -----------------------------------------------------------------

            base = (
                temp
                .groupby(
                    "Time_Bin",
                    sort=False
                )
                .agg(
                    Flow_Count=(
                        "Timestamp",
                        "size"
                    ),

                    Attack_Flow_Count=(
                        "Attack_Label",
                        "sum"
                    )
                )
            )


            base["Benign_Flow_Count"] = (
                base["Flow_Count"]
                -
                base["Attack_Flow_Count"]
            )


            # -----------------------------------------------------------------
            # Number of distinct attack types in each bin.
            # -----------------------------------------------------------------

            attack_only = temp[
                temp["Attack_Label"] == 1
            ]


            if len(attack_only) > 0:

                attack_stats = (
                    attack_only
                    .groupby(
                        "Time_Bin"
                    )
                    .agg(
                        Unique_Attack_Types=(
                            "Attack_Type",
                            "nunique"
                        )
                    )
                )


                base = base.join(
                    attack_stats,
                    how="left"
                )


            else:

                base[
                    "Unique_Attack_Types"
                ] = 0


            base[
                "Unique_Attack_Types"
            ] = (
                base[
                    "Unique_Attack_Types"
                ]
                .fillna(0)
                .astype("int16")
            )


            base = (
                base
                .reset_index()
            )


            file_resolution_parts[
                resolution_name
            ].append(base)


        processed_rows += len(chunk)


        print(
            f"  chunk {chunk_number:<3} "
            f"processed rows so far: "
            f"{processed_rows:,}"
        )


        del chunk
        gc.collect()


    # =========================================================================
    # 8. RE-AGGREGATE CHUNK BOUNDARIES
    # =========================================================================
    #
    # A temporal bin may be split across two pandas chunks.
    #
    # Therefore duplicate Time_Bin rows must be merged after each source file.
    #
    # =========================================================================

    for resolution_name in RESOLUTIONS:

        file_df = pd.concat(
            file_resolution_parts[
                resolution_name
            ],
            ignore_index=True
        )


        file_df = (
            file_df
            .groupby(
                "Time_Bin",
                as_index=False
            )
            .agg(
                Flow_Count=(
                    "Flow_Count",
                    "sum"
                ),

                Attack_Flow_Count=(
                    "Attack_Flow_Count",
                    "sum"
                ),

                Benign_Flow_Count=(
                    "Benign_Flow_Count",
                    "sum"
                ),

                # Approximate audit statistic only.
                Unique_Attack_Types=(
                    "Unique_Attack_Types",
                    "max"
                )
            )
        )


        file_df[
            "Source_File"
        ] = filename


        resolution_chunks[
            resolution_name
        ].append(file_df)


        del file_df


    del file_resolution_parts
    gc.collect()


# =============================================================================
# LABEL VALIDATION RESULT
# =============================================================================

print("\n")
print("=" * 115)
print("LABEL VALIDATION")
print("=" * 115)

print(
    f"\nMissing/empty labels excluded : "
    f"{total_missing_labels:,}"
)

if total_missing_labels == 0:
    print("✓ No missing labels detected.")
else:
    print(
        "⚠ Missing labels were detected and safely excluded "
        "instead of being classified as attacks."
    )


# =============================================================================
# 9. COMBINE FILE-LEVEL AGGREGATES
# =============================================================================

resolution_datasets = {}


for resolution_name in RESOLUTIONS:

    rdf = pd.concat(
        resolution_chunks[
            resolution_name
        ],
        ignore_index=True
    )


    rdf = (
        rdf
        .sort_values(
            [
                "Time_Bin",
                "Source_File"
            ]
        )
        .reset_index(drop=True)
    )


    rdf[
        "Attack_State"
    ] = (
        rdf[
            "Attack_Flow_Count"
        ]
        >
        0
    ).astype("int8")


    resolution_datasets[
        resolution_name
    ] = rdf


    del rdf
    gc.collect()


# =============================================================================
# 10. CREATE CONTINUOUS CAPTURE SEGMENTS FOR EACH RESOLUTION
# =============================================================================
#
# We must NEVER treat overnight/lunch capture gaps as continuous time.
#
# A new segment is created whenever the temporal gap exceeds 300 seconds.
#
# =============================================================================

for resolution_name, seconds in RESOLUTIONS.items():

    rdf = resolution_datasets[
        resolution_name
    ].copy()


    rdf = (
        rdf
        .sort_values("Time_Bin")
        .reset_index(drop=True)
    )


    time_gap = (
        rdf[
            "Time_Bin"
        ]
        .diff()
        .dt.total_seconds()
    )


    new_segment = (
        time_gap.isna()
        |
        (time_gap > 300)
    )


    rdf[
        "Audit_Segment_ID"
    ] = (
        new_segment
        .cumsum()
        -
        1
    ).astype("int16")


    resolution_datasets[
        resolution_name
    ] = rdf


# =============================================================================
# 11. RESOLUTION SUMMARY
# =============================================================================

summary_rows = []


for resolution_name, seconds in RESOLUTIONS.items():

    rdf = resolution_datasets[
        resolution_name
    ]


    total_bins = len(rdf)

    attack_bins = int(
        rdf[
            "Attack_State"
        ].sum()
    )

    benign_bins = (
        total_bins
        -
        attack_bins
    )


    # -------------------------------------------------------------------------
    # Estimate number of possible clock bins inside each continuous segment.
    # -------------------------------------------------------------------------

    expected_bins = 0


    for segment_id, seg in rdf.groupby(
        "Audit_Segment_ID"
    ):

        start = seg[
            "Time_Bin"
        ].min()

        end = seg[
            "Time_Bin"
        ].max()


        expected_bins += int(
            (
                end
                -
                start
            )
            .total_seconds()
            //
            seconds
        ) + 1


    empty_bins = (
        expected_bins
        -
        total_bins
    )


    empty_percent = (
        100
        *
        empty_bins
        /
        expected_bins

        if expected_bins > 0

        else np.nan
    )


    summary_rows.append(
        {
            "Resolution":
                resolution_name,

            "Seconds_Per_Bin":
                seconds,

            "Observed_Bins":
                total_bins,

            "Expected_Bins":
                expected_bins,

            "Empty_Bins":
                empty_bins,

            "Empty_Percent":
                empty_percent,

            "Attack_Bins":
                attack_bins,

            "Benign_Bins":
                benign_bins,

            "Attack_Bin_Percent":
                (
                    100
                    *
                    attack_bins
                    /
                    total_bins
                ),

            "Median_Flows_Per_Bin":
                rdf[
                    "Flow_Count"
                ].median(),

            "Mean_Flows_Per_Bin":
                rdf[
                    "Flow_Count"
                ].mean(),

            "P10_Flows_Per_Bin":
                rdf[
                    "Flow_Count"
                ].quantile(0.10),

            "P90_Flows_Per_Bin":
                rdf[
                    "Flow_Count"
                ].quantile(0.90),

            "Maximum_Flows_Per_Bin":
                rdf[
                    "Flow_Count"
                ].max(),

            "Audit_Segments":
                rdf[
                    "Audit_Segment_ID"
                ].nunique()
        }
    )


resolution_summary = pd.DataFrame(
    summary_rows
)


print("\n")
print("=" * 115)
print("TEMPORAL RESOLUTION SUMMARY")
print("=" * 115)

print()

print(
    resolution_summary.to_string(
        index=False
    )
)


# =============================================================================
# 12. FORECASTING SEQUENCE CAPACITY
# =============================================================================
#
# Compare the SAME physical history durations:
#
#       5 minutes
#       10 minutes
#       15 minutes
#
# and horizons:
#
#       1 minute
#       5 minutes
#       10 minutes
#       15 minutes
#       30 minutes
#
# =============================================================================

HISTORY_MINUTES = [
    5,
    10,
    15
]

HORIZON_MINUTES = [
    1,
    5,
    10,
    15,
    30
]


sequence_rows = []


for resolution_name, seconds in RESOLUTIONS.items():

    rdf = resolution_datasets[
        resolution_name
    ]


    for history_minutes in HISTORY_MINUTES:

        history_steps = int(
            history_minutes
            *
            60
            /
            seconds
        )


        for horizon_minutes in HORIZON_MINUTES:

            horizon_steps = int(
                horizon_minutes
                *
                60
                /
                seconds
            )


            valid_sequences = 0
            positive_targets = 0
            benign_targets = 0


            for segment_id, seg in rdf.groupby(
                "Audit_Segment_ID"
            ):

                seg = (
                    seg
                    .sort_values(
                        "Time_Bin"
                    )
                    .reset_index(
                        drop=True
                    )
                )


                n = len(seg)


                if n < (
                    history_steps
                    +
                    horizon_steps
                ):
                    continue


                times = seg[
                    "Time_Bin"
                ].to_numpy()


                states = seg[
                    "Attack_State"
                ].to_numpy()


                # -------------------------------------------------------------
                # Sliding windows.
                #
                # Real elapsed time is explicitly validated so missing bins
                # cannot silently become fake continuous observations.
                # -------------------------------------------------------------

                for end_idx in range(
                    history_steps - 1,
                    n - horizon_steps
                ):

                    start_idx = (
                        end_idx
                        -
                        history_steps
                        +
                        1
                    )

                    target_idx = (
                        end_idx
                        +
                        horizon_steps
                    )


                    actual_history_seconds = (
                        times[end_idx]
                        -
                        times[start_idx]
                    ) / np.timedelta64(
                        1,
                        "s"
                    )


                    expected_history_seconds = (
                        (
                            history_steps
                            -
                            1
                        )
                        *
                        seconds
                    )


                    if (
                        actual_history_seconds
                        !=
                        expected_history_seconds
                    ):
                        continue


                    actual_horizon_seconds = (
                        times[target_idx]
                        -
                        times[end_idx]
                    ) / np.timedelta64(
                        1,
                        "s"
                    )


                    expected_horizon_seconds = (
                        horizon_steps
                        *
                        seconds
                    )


                    if (
                        actual_horizon_seconds
                        !=
                        expected_horizon_seconds
                    ):
                        continue


                    valid_sequences += 1


                    if (
                        states[target_idx]
                        ==
                        1
                    ):
                        positive_targets += 1

                    else:
                        benign_targets += 1


            sequence_rows.append(
                {
                    "Resolution":
                        resolution_name,

                    "History_Minutes":
                        history_minutes,

                    "History_Steps":
                        history_steps,

                    "Horizon_Minutes":
                        horizon_minutes,

                    "Horizon_Steps":
                        horizon_steps,

                    "Valid_Sequences":
                        valid_sequences,

                    "Future_Attack_Targets":
                        positive_targets,

                    "Future_Benign_Targets":
                        benign_targets,

                    "Positive_Percent":
                        (
                            100
                            *
                            positive_targets
                            /
                            valid_sequences
                        )
                        if valid_sequences
                        else np.nan
                }
            )


sequence_capacity = pd.DataFrame(
    sequence_rows
)


print("\n")
print("=" * 115)
print("FORECASTING SEQUENCE CAPACITY")
print("=" * 115)

print()

print(
    sequence_capacity.to_string(
        index=False
    )
)


# =============================================================================
# 13. ATTACK/BENIGN TRANSITIONS
# =============================================================================
#
# Count:
#
#       benign -> attack
#       attack -> benign
#
# =============================================================================

transition_rows = []


for resolution_name in RESOLUTIONS:

    rdf = resolution_datasets[
        resolution_name
    ]


    benign_to_attack = 0
    attack_to_benign = 0


    for segment_id, seg in rdf.groupby(
        "Audit_Segment_ID"
    ):

        seg = (
            seg
            .sort_values(
                "Time_Bin"
            )
            .reset_index(drop=True)
        )


        previous = (
            seg[
                "Attack_State"
            ]
            .shift(1)
        )


        current = seg[
            "Attack_State"
        ]


        benign_to_attack += int(
            (
                (previous == 0)
                &
                (current == 1)
            )
            .sum()
        )


        attack_to_benign += int(
            (
                (previous == 1)
                &
                (current == 0)
            )
            .sum()
        )


    transition_rows.append(
        {
            "Resolution":
                resolution_name,

            "Benign_to_Attack_Transitions":
                benign_to_attack,

            "Attack_to_Benign_Transitions":
                attack_to_benign,

            "Total_State_Transitions":
                (
                    benign_to_attack
                    +
                    attack_to_benign
                )
        }
    )


transition_summary = pd.DataFrame(
    transition_rows
)


print("\n")
print("=" * 115)
print("ATTACK-STATE TRANSITION SUMMARY")
print("=" * 115)

print()

print(
    transition_summary.to_string(
        index=False
    )
)


# =============================================================================
# 14. TEMPORAL AUTOCORRELATION AUDIT
# =============================================================================

autocorr_rows = []


for resolution_name in RESOLUTIONS:

    rdf = resolution_datasets[
        resolution_name
    ]


    flow_autocorrs = []
    attack_autocorrs = []


    for segment_id, seg in rdf.groupby(
        "Audit_Segment_ID"
    ):

        seg = (
            seg
            .sort_values(
                "Time_Bin"
            )
        )


        if len(seg) < 3:
            continue


        flow_ac = (
            seg[
                "Flow_Count"
            ]
            .autocorr(
                lag=1
            )
        )


        attack_ac = (
            seg[
                "Attack_Flow_Count"
            ]
            .autocorr(
                lag=1
            )
        )


        if pd.notna(flow_ac):
            flow_autocorrs.append(
                flow_ac
            )


        if pd.notna(attack_ac):
            attack_autocorrs.append(
                attack_ac
            )


    autocorr_rows.append(
        {
            "Resolution":
                resolution_name,

            "Mean_Lag1_FlowCount_Autocorr":
                (
                    np.mean(
                        flow_autocorrs
                    )
                    if flow_autocorrs
                    else np.nan
                ),

            "Mean_Lag1_AttackCount_Autocorr":
                (
                    np.mean(
                        attack_autocorrs
                    )
                    if attack_autocorrs
                    else np.nan
                )
        }
    )


autocorr_summary = pd.DataFrame(
    autocorr_rows
)


print("\n")
print("=" * 115)
print("LAG-1 TEMPORAL AUTOCORRELATION")
print("=" * 115)

print()

print(
    autocorr_summary.to_string(
        index=False
    )
)


# =============================================================================
# 15. SAVE AUDIT RESULTS
# =============================================================================

resolution_summary.to_csv(
    os.path.join(
        AUDIT_DIR,
        "temporal_resolution_summary.csv"
    ),
    index=False
)


sequence_capacity.to_csv(
    os.path.join(
        AUDIT_DIR,
        "temporal_resolution_sequence_capacity.csv"
    ),
    index=False
)


transition_summary.to_csv(
    os.path.join(
        AUDIT_DIR,
        "temporal_resolution_transitions.csv"
    ),
    index=False
)


autocorr_summary.to_csv(
    os.path.join(
        AUDIT_DIR,
        "temporal_resolution_autocorrelation.csv"
    ),
    index=False
)


# =============================================================================
# 16. SAVE LIGHTWEIGHT CANDIDATE TEMPORAL TABLES
# =============================================================================
#
# These are NOT final model datasets.
#
# =============================================================================

for resolution_name, rdf in resolution_datasets.items():

    save_path = os.path.join(
        TEMPORAL_DIR,
        f"candidate_{resolution_name}_audit.parquet"
    )


    rdf.to_parquet(
        save_path,
        index=False
    )


    print(
        f"\nSaved {resolution_name} candidate:"
        f"\n  {save_path}"
    )


# =============================================================================
# 17. FINAL SUMMARY
# =============================================================================

print("\n")
print("=" * 115)
print("CELL 16 COMPLETE")
print("=" * 115)


print(
    """
No model has been trained.

No synthetic network traffic has been generated.

No final temporal resolution has been selected.

We have compared 10-second, 30-second and 60-second representations
using the REAL CICIDS2017 flow records.

NEXT DECISION:

Choose the temporal resolution using:

    1. number of usable temporal states
    2. empty-bin percentage
    3. flows per state
    4. attack-bin coverage
    5. number of valid forecasting sequences
    6. benign -> attack transitions
    7. temporal redundancy / autocorrelation

DO NOT rebuild the final modelling dataset yet.
DO NOT start the model-training notebook yet.
"""
)

print("=" * 115)

CELL 16 — MULTI-RESOLUTION TEMPORAL FEASIBILITY AUDIT

Raw files found : 8

Temporal resolutions:
  10s -> 10 seconds
  30s -> 30 seconds
  60s -> 60 seconds

FILE 1/8 — Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
  chunk 1   processed rows so far: 150,000
  chunk 2   processed rows so far: 225,745

FILE 2/8 — Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
  chunk 1   processed rows so far: 150,000
  chunk 2   processed rows so far: 286,467

FILE 3/8 — Friday-WorkingHours-Morning.pcap_ISCX.csv
  chunk 1   processed rows so far: 150,000
  chunk 2   processed rows so far: 191,033

FILE 4/8 — Monday-WorkingHours.pcap_ISCX.csv
  chunk 1   processed rows so far: 150,000
  chunk 2   processed rows so far: 300,000
  chunk 3   processed rows so far: 450,000
  chunk 4   processed rows so far: 529,918

FILE 5/8 — Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  chunk 1   processed rows so far: 150,000
  chunk 2   processed rows so far: 288,602

FILE 6/8 — Thursday-Working

/usr/local/lib/python3.13/dist-packages/numpy/lib/_function_base_impl.py:2999: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]



Saved 10s candidate:
  /content/drive/MyDrive/Network-Attack-Forecasting/data/temporal/candidate_10s_audit.parquet

Saved 30s candidate:
  /content/drive/MyDrive/Network-Attack-Forecasting/data/temporal/candidate_30s_audit.parquet

Saved 60s candidate:
  /content/drive/MyDrive/Network-Attack-Forecasting/data/temporal/candidate_60s_audit.parquet


CELL 16 COMPLETE

No model has been trained.

No synthetic network traffic has been generated.

No final temporal resolution has been selected.

We have compared 10-second, 30-second and 60-second representations
using the REAL CICIDS2017 flow records.

NEXT DECISION:

Choose the temporal resolution using:

    1. number of usable temporal states
    2. empty-bin percentage
    3. flows per state
    4. attack-bin coverage
    5. number of valid forecasting sequences
    6. benign -> attack transitions
    7. temporal redundancy / autocorrelation

DO NOT rebuild the final modelling dataset yet.
DO NOT start the model-training notebook yet.



In [23]:
# =============================================================================
# CELL 17 — RAW TIMESTAMP PRECISION AUDIT
# =============================================================================
#
# PURPOSE
# -------
#
# Cell 16 produced an important result:
#
#   10-second bins  -> many empty intervals
#   30-second bins  -> many empty intervals
#   60-second bins  -> no empty intervals
#
# Even more importantly, the number of attack-containing bins was exactly
# the same at all three resolutions.
#
# Before deciding that 60 seconds is the finest usable resolution, we need
# to inspect the ORIGINAL timestamp values in the raw CICIDS2017
# TrafficLabelling CSV files.
#
# THIS CELL ANSWERS:
#
#   1. What do the original timestamp strings look like?
#   2. Do they contain seconds?
#   3. Are timestamps effectively rounded/quantized to whole minutes?
#   4. Which second-of-minute values actually occur?
#   5. Does timestamp precision differ between benign and attack flows?
#   6. Does timestamp precision differ between the 8 source files?
#   7. How many unique timestamps exist compared with the number of flows?
#
# IMPORTANT
# ---------
#
# This cell:
#
#   - DOES NOT modify the dataset.
#   - DOES NOT create temporal bins.
#   - DOES NOT train a model.
#   - DOES NOT generate synthetic data.
#
# It only audits the raw timestamps exactly as stored in the CSV files.
#
# RAM SAFETY
# ----------
#
# Only:
#
#       Timestamp
#       Label
#
# are loaded, and files are processed in chunks.
#
# =============================================================================


import os
import gc
from collections import Counter

import pandas as pd
import numpy as np


# =============================================================================
# 1. PATHS
# =============================================================================

PROJECT_ROOT = (
    "/content/drive/MyDrive/"
    "Network-Attack-Forecasting"
)

RAW_DIR = (
    "/content/drive/MyDrive/"
    "TrafficLabelling "
)

AUDIT_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "audits"
)

os.makedirs(
    AUDIT_DIR,
    exist_ok=True
)


raw_files = sorted([
    os.path.join(RAW_DIR, f)
    for f in os.listdir(RAW_DIR)
    if f.lower().endswith(".csv")
])


assert len(raw_files) == 8, (
    f"Expected 8 raw CSV files, found {len(raw_files)}"
)


print("=" * 120)
print("CELL 17 — RAW CICIDS2017 TIMESTAMP PRECISION AUDIT")
print("=" * 120)

print(
    f"\nRaw files found : {len(raw_files)}"
)


# =============================================================================
# 2. SETTINGS
# =============================================================================

CSV_ENCODING = "latin1"

CHUNK_SIZE = 150_000

USECOLS = [
    "Timestamp",
    "Label"
]


# =============================================================================
# 3. STORAGE FOR RESULTS
# =============================================================================

file_summaries = []

global_second_counter = Counter()

global_attack_second_counter = Counter()

global_benign_second_counter = Counter()

raw_timestamp_examples = []

parsed_timestamp_examples = []


# =============================================================================
# 4. PROCESS EACH RAW FILE
# =============================================================================

for file_number, path in enumerate(
    raw_files,
    start=1
):

    filename = os.path.basename(path)

    print("\n" + "=" * 120)

    print(
        f"FILE {file_number}/8 — {filename}"
    )

    print("=" * 120)


    # -------------------------------------------------------------------------
    # File-level counters
    # -------------------------------------------------------------------------

    total_rows = 0

    valid_timestamps = 0

    invalid_timestamps = 0

    attack_rows = 0

    benign_rows = 0


    second_counter = Counter()

    attack_second_counter = Counter()

    benign_second_counter = Counter()


    unique_timestamp_values = set()


    file_raw_examples = []


    # -------------------------------------------------------------------------
    # Read file in chunks
    # -------------------------------------------------------------------------

    reader = pd.read_csv(
        path,
        encoding=CSV_ENCODING,
        usecols=lambda c: c.strip() in USECOLS,
        chunksize=CHUNK_SIZE,
        low_memory=False
    )


    for chunk_number, chunk in enumerate(
        reader,
        start=1
    ):

        # ---------------------------------------------------------------------
        # Normalize column names
        # ---------------------------------------------------------------------

        chunk.columns = [
            c.strip()
            for c in chunk.columns
        ]


        # ---------------------------------------------------------------------
        # Preserve ORIGINAL timestamp text before parsing
        # ---------------------------------------------------------------------

        raw_timestamp = (
            chunk["Timestamp"]
            .astype("string")
            .str.strip()
        )


        # Save a few raw examples from this file.

        if len(file_raw_examples) < 10:

            examples_needed = (
                10
                -
                len(file_raw_examples)
            )

            examples = (
                raw_timestamp
                .dropna()
                .drop_duplicates()
                .head(examples_needed)
                .tolist()
            )

            file_raw_examples.extend(
                examples
            )


        # ---------------------------------------------------------------------
        # Normalize labels only enough to distinguish benign vs attack
        # ---------------------------------------------------------------------

        label = (
            chunk["Label"]
            .astype("string")
            .str.strip()
        )


        benign_mask = (
            label
            .str.upper()
            .eq("BENIGN")
        )


        attack_mask = (
            label.notna()
            &
            ~benign_mask
        )


        # ---------------------------------------------------------------------
        # Parse timestamps
        # ---------------------------------------------------------------------

        parsed_time = pd.to_datetime(
            raw_timestamp,
            dayfirst=True,
            errors="coerce",
            format="mixed"
        )


        valid_mask = (
            parsed_time.notna()
        )


        # ---------------------------------------------------------------------
        # Basic counts
        # ---------------------------------------------------------------------

        total_rows += len(chunk)

        valid_timestamps += int(
            valid_mask.sum()
        )

        invalid_timestamps += int(
            (~valid_mask).sum()
        )


        attack_rows += int(
            (
                attack_mask
                &
                valid_mask
            ).sum()
        )


        benign_rows += int(
            (
                benign_mask
                &
                valid_mask
            ).sum()
        )


        # ---------------------------------------------------------------------
        # SECOND-OF-MINUTE DISTRIBUTION
        # ---------------------------------------------------------------------

        valid_seconds = (
            parsed_time.loc[
                valid_mask
            ]
            .dt.second
        )


        second_counter.update(
            valid_seconds
            .astype(int)
            .tolist()
        )


        global_second_counter.update(
            valid_seconds
            .astype(int)
            .tolist()
        )


        # ---------------------------------------------------------------------
        # ATTACK SECOND DISTRIBUTION
        # ---------------------------------------------------------------------

        attack_valid_mask = (
            attack_mask
            &
            valid_mask
        )


        attack_seconds = (
            parsed_time.loc[
                attack_valid_mask
            ]
            .dt.second
        )


        attack_second_counter.update(
            attack_seconds
            .astype(int)
            .tolist()
        )


        global_attack_second_counter.update(
            attack_seconds
            .astype(int)
            .tolist()
        )


        # ---------------------------------------------------------------------
        # BENIGN SECOND DISTRIBUTION
        # ---------------------------------------------------------------------

        benign_valid_mask = (
            benign_mask
            &
            valid_mask
        )


        benign_seconds = (
            parsed_time.loc[
                benign_valid_mask
            ]
            .dt.second
        )


        benign_second_counter.update(
            benign_seconds
            .astype(int)
            .tolist()
        )


        global_benign_second_counter.update(
            benign_seconds
            .astype(int)
            .tolist()
        )


        # ---------------------------------------------------------------------
        # UNIQUE TIMESTAMP COUNT
        #
        # This lets us see whether thousands of flows share exactly the same
        # timestamp.
        # ---------------------------------------------------------------------

        unique_timestamp_values.update(
            parsed_time.loc[
                valid_mask
            ]
            .astype("int64")
            .tolist()
        )


        # ---------------------------------------------------------------------
        # Save a few parsed examples
        # ---------------------------------------------------------------------

        if len(parsed_timestamp_examples) < 50:

            temp_examples = pd.DataFrame(
                {
                    "Source_File":
                        filename,

                    "Raw_Timestamp":
                        raw_timestamp,

                    "Parsed_Timestamp":
                        parsed_time,

                    "Label":
                        label
                }
            )


            temp_examples = (
                temp_examples
                .loc[
                    temp_examples[
                        "Parsed_Timestamp"
                    ].notna()
                ]
                .head(10)
            )


            parsed_timestamp_examples.append(
                temp_examples
            )


        print(
            f"  chunk {chunk_number:<3} "
            f"rows processed: "
            f"{total_rows:,}"
        )


        del chunk
        gc.collect()


    # =========================================================================
    # 5. FILE-LEVEL ANALYSIS
    # =========================================================================

    distinct_seconds = sorted(
        second_counter.keys()
    )


    attack_distinct_seconds = sorted(
        attack_second_counter.keys()
    )


    benign_distinct_seconds = sorted(
        benign_second_counter.keys()
    )


    rows_at_second_zero = (
        second_counter.get(
            0,
            0
        )
    )


    percent_second_zero = (
        100
        *
        rows_at_second_zero
        /
        valid_timestamps

        if valid_timestamps > 0

        else np.nan
    )


    attack_at_second_zero = (
        attack_second_counter.get(
            0,
            0
        )
    )


    attack_second_zero_percent = (
        100
        *
        attack_at_second_zero
        /
        attack_rows

        if attack_rows > 0

        else np.nan
    )


    benign_at_second_zero = (
        benign_second_counter.get(
            0,
            0
        )
    )


    benign_second_zero_percent = (
        100
        *
        benign_at_second_zero
        /
        benign_rows

        if benign_rows > 0

        else np.nan
    )


    unique_timestamp_count = len(
        unique_timestamp_values
    )


    flows_per_unique_timestamp = (
        valid_timestamps
        /
        unique_timestamp_count

        if unique_timestamp_count > 0

        else np.nan
    )


    # -------------------------------------------------------------------------
    # Print file-level result
    # -------------------------------------------------------------------------

    print("\nRAW TIMESTAMP EXAMPLES:")

    for example in file_raw_examples:
        print(
            f"  {example}"
        )


    print(
        "\nDistinct second-of-minute values:"
    )

    print(
        distinct_seconds
    )


    print(
        f"\nDistinct seconds count : "
        f"{len(distinct_seconds)}"
    )


    print(
        f"Rows at second :00    : "
        f"{rows_at_second_zero:,} "
        f"({percent_second_zero:.4f}%)"
    )


    print(
        f"\nUnique timestamps      : "
        f"{unique_timestamp_count:,}"
    )


    print(
        f"Valid flow rows        : "
        f"{valid_timestamps:,}"
    )


    print(
        f"Flows / unique timestamp: "
        f"{flows_per_unique_timestamp:.2f}"
    )


    if attack_rows > 0:

        print(
            "\nATTACK TIMESTAMP PRECISION:"
        )

        print(
            f"Attack rows             : "
            f"{attack_rows:,}"
        )

        print(
            f"Distinct attack seconds : "
            f"{attack_distinct_seconds}"
        )

        print(
            f"Attack rows at :00      : "
            f"{attack_at_second_zero:,} "
            f"({attack_second_zero_percent:.4f}%)"
        )


    print(
        "\nBENIGN TIMESTAMP PRECISION:"
    )

    print(
        f"Benign rows             : "
        f"{benign_rows:,}"
    )

    print(
        f"Distinct benign seconds : "
        f"{benign_distinct_seconds}"
    )

    print(
        f"Benign rows at :00      : "
        f"{benign_at_second_zero:,} "
        f"({benign_second_zero_percent:.4f}%)"
    )


    # -------------------------------------------------------------------------
    # Store file summary
    # -------------------------------------------------------------------------

    file_summaries.append(
        {
            "Source_File":
                filename,

            "Total_Rows":
                total_rows,

            "Valid_Timestamps":
                valid_timestamps,

            "Invalid_Timestamps":
                invalid_timestamps,

            "Unique_Timestamps":
                unique_timestamp_count,

            "Flows_Per_Unique_Timestamp":
                flows_per_unique_timestamp,

            "Distinct_Second_Values":
                len(distinct_seconds),

            "Second_Values":
                ",".join(
                    map(
                        str,
                        distinct_seconds
                    )
                ),

            "Rows_At_Second_00":
                rows_at_second_zero,

            "Percent_At_Second_00":
                percent_second_zero,

            "Attack_Rows":
                attack_rows,

            "Attack_Distinct_Second_Values":
                len(
                    attack_distinct_seconds
                ),

            "Attack_Second_Values":
                ",".join(
                    map(
                        str,
                        attack_distinct_seconds
                    )
                ),

            "Attack_Percent_At_Second_00":
                attack_second_zero_percent,

            "Benign_Rows":
                benign_rows,

            "Benign_Distinct_Second_Values":
                len(
                    benign_distinct_seconds
                ),

            "Benign_Second_Values":
                ",".join(
                    map(
                        str,
                        benign_distinct_seconds
                    )
                ),

            "Benign_Percent_At_Second_00":
                benign_second_zero_percent
        }
    )


    # Save raw examples.

    for example in file_raw_examples:

        raw_timestamp_examples.append(
            {
                "Source_File":
                    filename,

                "Raw_Timestamp":
                    example
            }
        )


    # Clean up file-specific memory.

    del unique_timestamp_values
    gc.collect()


# =============================================================================
# 6. FILE-BY-FILE SUMMARY
# =============================================================================

timestamp_summary = pd.DataFrame(
    file_summaries
)


print("\n\n")
print("=" * 120)
print("FILE-BY-FILE TIMESTAMP PRECISION SUMMARY")
print("=" * 120)

print()

display_columns = [
    "Source_File",
    "Total_Rows",
    "Unique_Timestamps",
    "Flows_Per_Unique_Timestamp",
    "Distinct_Second_Values",
    "Percent_At_Second_00",
    "Attack_Rows",
    "Attack_Distinct_Second_Values",
    "Attack_Percent_At_Second_00"
]


print(
    timestamp_summary[
        display_columns
    ].to_string(
        index=False
    )
)


# =============================================================================
# 7. GLOBAL SECOND-OF-MINUTE DISTRIBUTION
# =============================================================================

global_seconds_df = pd.DataFrame(
    {
        "Second":
            list(
                range(60)
            )
    }
)


global_seconds_df[
    "All_Flows"
] = (
    global_seconds_df[
        "Second"
    ]
    .map(
        global_second_counter
    )
    .fillna(0)
    .astype("int64")
)


global_seconds_df[
    "Attack_Flows"
] = (
    global_seconds_df[
        "Second"
    ]
    .map(
        global_attack_second_counter
    )
    .fillna(0)
    .astype("int64")
)


global_seconds_df[
    "Benign_Flows"
] = (
    global_seconds_df[
        "Second"
    ]
    .map(
        global_benign_second_counter
    )
    .fillna(0)
    .astype("int64")
)


print("\n\n")
print("=" * 120)
print("GLOBAL SECOND-OF-MINUTE DISTRIBUTION")
print("=" * 120)

print()

print(
    global_seconds_df.to_string(
        index=False
    )
)


# =============================================================================
# 8. GLOBAL PRECISION RESULT
# =============================================================================

global_total = int(
    global_seconds_df[
        "All_Flows"
    ].sum()
)


global_attack_total = int(
    global_seconds_df[
        "Attack_Flows"
    ].sum()
)


global_benign_total = int(
    global_seconds_df[
        "Benign_Flows"
    ].sum()
)


global_second_zero = int(
    global_seconds_df.loc[
        global_seconds_df[
            "Second"
        ]
        ==
        0,

        "All_Flows"
    ].iloc[0]
)


global_attack_second_zero = int(
    global_seconds_df.loc[
        global_seconds_df[
            "Second"
        ]
        ==
        0,

        "Attack_Flows"
    ].iloc[0]
)


global_benign_second_zero = int(
    global_seconds_df.loc[
        global_seconds_df[
            "Second"
        ]
        ==
        0,

        "Benign_Flows"
    ].iloc[0]
)


global_second_zero_percent = (
    100
    *
    global_second_zero
    /
    global_total
)


global_attack_second_zero_percent = (
    100
    *
    global_attack_second_zero
    /
    global_attack_total

    if global_attack_total > 0

    else np.nan
)


global_benign_second_zero_percent = (
    100
    *
    global_benign_second_zero
    /
    global_benign_total
)


global_distinct_seconds = sorted(
    [
        second
        for second, count
        in global_second_counter.items()
        if count > 0
    ]
)


global_attack_distinct_seconds = sorted(
    [
        second
        for second, count
        in global_attack_second_counter.items()
        if count > 0
    ]
)


print("\n")
print("=" * 120)
print("GLOBAL TIMESTAMP PRECISION RESULT")
print("=" * 120)


print(
    f"\nTotal valid flow timestamps : "
    f"{global_total:,}"
)


print(
    f"Distinct second values      : "
    f"{global_distinct_seconds}"
)


print(
    f"Number of second values     : "
    f"{len(global_distinct_seconds)} / 60"
)


print(
    f"\nAll flows at second :00     : "
    f"{global_second_zero:,} "
    f"({global_second_zero_percent:.4f}%)"
)


print(
    f"Attack flows at second :00  : "
    f"{global_attack_second_zero:,} "
    f"({global_attack_second_zero_percent:.4f}%)"
)


print(
    f"Benign flows at second :00  : "
    f"{global_benign_second_zero:,} "
    f"({global_benign_second_zero_percent:.4f}%)"
)


print(
    f"\nAttack second values        : "
    f"{global_attack_distinct_seconds}"
)


# =============================================================================
# 9. AUTOMATIC INTERPRETATION
# =============================================================================

print("\n")
print("=" * 120)
print("AUTOMATIC INTERPRETATION")
print("=" * 120)


if (
    len(global_distinct_seconds) == 1
    and
    global_distinct_seconds[0] == 0
):

    print(
        """
RESULT:

All parsed timestamps occur at second :00.

This means the TrafficLabelling CSV representation is effectively
minute-resolution with respect to its Timestamp field.

Therefore:

    10-second temporal bins are NOT genuine additional observations.
    30-second temporal bins are NOT genuine additional observations.

Sub-minute forecasting states cannot be reconstructed from this
Timestamp column alone.

The next step should be to investigate the original packet captures
(PCAPs) if genuine sub-minute temporal resolution is required.
"""
    )


elif (
    global_second_zero_percent
    >
    95
):

    print(
        f"""
RESULT:

{global_second_zero_percent:.2f}% of all flows occur at second :00.

The timestamp field is overwhelmingly concentrated at whole-minute
boundaries.

Sub-minute temporal aggregation from this CSV is therefore highly
questionable.

Before choosing 10s or 30s resolution, inspect whether the PCAP data
contains the required packet-level timestamp precision.
"""
    )


else:

    print(
        f"""
RESULT:




{global_distinct_seconds}

The raw CICIDS2017 CSV files contain heterogeneous timestamp precision.

Monday benign traffic preserves second-level timestamps across all
60 second-of-minute positions.

However, ALL labelled attack flows occur at second :00, and the
attack-bearing CSV files are effectively minute-quantized.

Therefore:

1. Sub-minute attack timing cannot be reliably reconstructed from
   the CICIDS2017 TrafficLabelling CSV files.

2. The 10-second and 30-second representations are not suitable for
   attack forecasting from these CSVs because they would imply
   temporal precision that the attack records do not contain.

3. The 60-second representation is the finest defensible common
   temporal resolution for CSV-based attack forecasting.

4. Finer attack forecasting would require returning to packet-level
   PCAP timestamps.

FINAL DECISION:
Use 60-second temporal states for CICIDS2017 CSV forecasting."""

    )


# =============================================================================
# 10. SAVE RESULTS
# =============================================================================

summary_path = os.path.join(
    AUDIT_DIR,
    "raw_timestamp_precision_by_file.csv"
)


seconds_path = os.path.join(
    AUDIT_DIR,
    "raw_timestamp_second_distribution.csv"
)


examples_path = os.path.join(
    AUDIT_DIR,
    "raw_timestamp_examples.csv"
)


timestamp_summary.to_csv(
    summary_path,
    index=False
)


global_seconds_df.to_csv(
    seconds_path,
    index=False
)


pd.DataFrame(
    raw_timestamp_examples
).to_csv(
    examples_path,
    index=False
)


# =============================================================================
# 11. FINAL
# =============================================================================

print("\n")
print("=" * 120)
print("CELL 17 COMPLETE")
print("=" * 120)


print(
    f"""
Saved:

1. {summary_path}

2. {seconds_path}

3. {examples_path}


NO DATASET HAS BEEN MODIFIED.

NO FINAL RESOLUTION HAS BEEN CHOSEN.

NO MODEL HAS BEEN TRAINED.

NEXT:

Use this audit to decide whether:

    A. the TrafficLabelling CSV genuinely supports sub-minute states,

or

    B. we need the original CICIDS2017 PCAP timestamps for finer
       temporal forecasting.
"""
)

print("=" * 120)

CELL 17 — RAW CICIDS2017 TIMESTAMP PRECISION AUDIT

Raw files found : 8

FILE 1/8 — Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
  chunk 1   rows processed: 150,000
  chunk 2   rows processed: 225,745

RAW TIMESTAMP EXAMPLES:
  7/7/2017 3:30
  7/7/2017 3:31
  7/7/2017 3:32
  7/7/2017 3:33
  7/7/2017 3:34
  7/7/2017 3:35
  7/7/2017 3:36
  7/7/2017 3:37
  7/7/2017 3:38
  7/7/2017 3:39

Distinct second-of-minute values:
[0]

Distinct seconds count : 1
Rows at second :00    : 225,745 (100.0000%)

Unique timestamps      : 93
Valid flow rows        : 225,745
Flows / unique timestamp: 2427.37

ATTACK TIMESTAMP PRECISION:
Attack rows             : 128,027
Distinct attack seconds : [0]
Attack rows at :00      : 128,027 (100.0000%)

BENIGN TIMESTAMP PRECISION:
Benign rows             : 97,718
Distinct benign seconds : [0]
Benign rows at :00      : 97,718 (100.0000%)

FILE 2/8 — Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
  chunk 1   rows processed: 150,000
  chunk 2   rows processed

In [24]:
# ==============================================================================================================
# CELL 18 — FINAL CICIDS2017 FORECASTING-DESIGN AUDIT
# ==============================================================================================================
#
# PURPOSE
# -------
# We have already established that:
#
#   - CICIDS2017 CSV attack timestamps are effectively minute-quantized.
#   - 60 seconds is therefore the finest defensible common temporal resolution
#     for CSV-based attack forecasting.
#   - The minute-level temporal dataset contains 10 continuous capture segments.
#
# This cell DOES NOT:
#   - train a model
#   - create train/validation/test splits
#   - scale features
#   - generate synthetic data
#   - modify the saved minute dataset
#
# This cell answers the FINAL experimental-design questions:
#
#   1. How many valid forecasting samples exist at each horizon?
#   2. How many samples are:
#          benign -> benign
#          benign -> attack
#          attack -> benign
#          attack -> attack
#   3. Among future attacks, how many histories are COMPLETELY clean?
#   4. How many independent future attack episodes are represented?
#   5. How sensitive are those counts to episode-gap definitions?
#   6. How many attack types are represented among forecasted episodes?
#
# IMPORTANT TERMINOLOGY
# ---------------------
# "Benign -> Attack" means:
#
#       current minute benign
#              AND
#       future target minute attack
#
# It does NOT automatically mean a new independent attack campaign.
#
# Independent episodes are calculated separately below.
# ==============================================================================================================


import os
import glob
import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------------------
# CONFIGURATION
# --------------------------------------------------------------------------------------------------------------

HISTORY_MINUTES = 15

HORIZONS = [1, 5, 10, 15, 30]

# Number of consecutive benign minutes that may occur INSIDE an attack episode
# before we decide that a new independent episode has begun.
EPISODE_GAP_TOLERANCES = [0, 1, 2, 5]


# --------------------------------------------------------------------------------------------------------------
# DISPLAY SETTINGS
# --------------------------------------------------------------------------------------------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 300)


print("=" * 130)
print("CELL 18 — FINAL CICIDS2017 FORECASTING-DESIGN AUDIT")
print("=" * 130)


# ==============================================================================================================
# STEP 1 — LOCATE / RECOVER THE MINUTE-LEVEL DATASET
# ==============================================================================================================

TEMPORAL_DIR = "/content/drive/MyDrive/Network-Attack-Forecasting/data/temporal"
AUDIT_DIR = "/content/drive/MyDrive/Network-Attack-Forecasting/data/audits"

os.makedirs(AUDIT_DIR, exist_ok=True)


def standardize_minute_dataframe(df):
    """
    Standardize known column-name variants without altering the underlying data.
    """

    df = df.copy()

    rename_map = {}

    # Timestamp variants
    if "Minute" not in df.columns:
        for candidate in [
            "Timestamp",
            "timestamp",
            "Time",
            "time",
            "Datetime",
            "datetime"
        ]:
            if candidate in df.columns:
                rename_map[candidate] = "Minute"
                break

    # Segment variants
    if "Segment_ID" not in df.columns:
        for candidate in [
            "Segment",
            "segment",
            "segment_id",
            "Capture_Segment",
            "capture_segment"
        ]:
            if candidate in df.columns:
                rename_map[candidate] = "Segment_ID"
                break

    # Attack-state variants
    if "Attack_State" not in df.columns:
        for candidate in [
            "Attack_Label",
            "attack_label",
            "Binary_Label",
            "binary_label",
            "Label",
            "label"
        ]:
            if candidate in df.columns:
                rename_map[candidate] = "Attack_State"
                break

    df = df.rename(columns=rename_map)

    return df


# First try RAM
minute_candidates = []

for name, obj in list(globals().items()):

    if isinstance(obj, pd.DataFrame):

        temp = standardize_minute_dataframe(obj)

        required = {"Minute", "Segment_ID", "Attack_State"}

        if required.issubset(temp.columns):

            minute_candidates.append((name, temp))


if minute_candidates:

    # Prefer a dataframe explicitly called minute_df if available
    preferred = [
        item for item in minute_candidates
        if item[0] == "minute_df"
    ]

    if preferred:
        source_name, minute_df = preferred[0]
    else:
        source_name, minute_df = minute_candidates[0]

    print(f"\n✓ Minute dataset recovered from RAM variable: {source_name}")

else:

    # Search likely saved parquet files
    search_patterns = [
        os.path.join(TEMPORAL_DIR, "*minute*.parquet"),
        os.path.join(TEMPORAL_DIR, "*60s*.parquet"),
        os.path.join(TEMPORAL_DIR, "*.parquet"),
    ]

    candidate_paths = []

    for pattern in search_patterns:
        candidate_paths.extend(glob.glob(pattern))

    # Remove duplicates while preserving order
    candidate_paths = list(dict.fromkeys(candidate_paths))

    minute_df = None
    source_path = None

    for path in candidate_paths:

        try:

            test_df = pd.read_parquet(path)
            test_df = standardize_minute_dataframe(test_df)

            required = {"Minute", "Segment_ID", "Attack_State"}

            if required.issubset(test_df.columns):

                # We know the real minute dataset has ~2454 rows.
                # Avoid accidentally selecting tiny audit/metadata tables.
                if len(test_df) >= 2000:

                    minute_df = test_df
                    source_path = path
                    break

        except Exception:
            continue

    if minute_df is None:

        raise RuntimeError(
            "\nCould not automatically locate the saved minute-level dataset.\n"
            "Expected a parquet file in:\n"
            f"{TEMPORAL_DIR}\n\n"
            "containing at least:\n"
            "    Minute\n"
            "    Segment_ID\n"
            "    Attack_State\n"
        )

    print("\n✓ Minute dataset loaded from Drive:")
    print(source_path)


# ==============================================================================================================
# STEP 2 — BASIC VALIDATION
# ==============================================================================================================

minute_df = minute_df.copy()

minute_df["Minute"] = pd.to_datetime(
    minute_df["Minute"],
    errors="coerce"
)

minute_df["Segment_ID"] = pd.to_numeric(
    minute_df["Segment_ID"],
    errors="coerce"
)

minute_df["Attack_State"] = pd.to_numeric(
    minute_df["Attack_State"],
    errors="coerce"
)


if minute_df["Minute"].isna().any():
    raise RuntimeError("Minute dataset contains invalid timestamps.")

if minute_df["Segment_ID"].isna().any():
    raise RuntimeError("Minute dataset contains missing Segment_ID values.")

if minute_df["Attack_State"].isna().any():
    raise RuntimeError("Minute dataset contains missing Attack_State values.")


minute_df["Segment_ID"] = minute_df["Segment_ID"].astype(int)
minute_df["Attack_State"] = minute_df["Attack_State"].astype(int)


invalid_attack_values = set(minute_df["Attack_State"].unique()) - {0, 1}

if invalid_attack_values:
    raise RuntimeError(
        f"Attack_State contains unexpected values: {invalid_attack_values}"
    )


minute_df = (
    minute_df
    .sort_values(["Segment_ID", "Minute"])
    .reset_index(drop=True)
)


duplicate_segment_minutes = minute_df.duplicated(
    subset=["Segment_ID", "Minute"]
).sum()


print("\n" + "=" * 130)
print("BASE DATASET VALIDATION")
print("=" * 130)

print(f"\nMinute rows                  : {len(minute_df):,}")
print(f"Segments                     : {minute_df['Segment_ID'].nunique():,}")
print(f"Attack minutes               : {(minute_df['Attack_State'] == 1).sum():,}")
print(f"Benign minutes               : {(minute_df['Attack_State'] == 0).sum():,}")
print(f"Duplicate segment/minute rows: {duplicate_segment_minutes:,}")

if duplicate_segment_minutes != 0:
    raise RuntimeError(
        "Duplicate minute observations detected. "
        "Do not continue until they are investigated."
    )


# ==============================================================================================================
# STEP 3 — IDENTIFY ATTACK-TYPE INFORMATION IF AVAILABLE
# ==============================================================================================================

attack_type_col = None

for candidate in [
    "Attack_Type",
    "Attack_Types",
    "Attack_Category",
    "Attack_Categories",
    "Attack_Type_Set",
    "Attack_Type_List"
]:

    if candidate in minute_df.columns:
        attack_type_col = candidate
        break


print("\nAttack-type column:", attack_type_col if attack_type_col else "not found")

if attack_type_col is None:
    print(
        "NOTE: Independent binary episode analysis will still run.\n"
        "Attack-type coverage will be reported only if type information is available."
    )


# ==============================================================================================================
# STEP 4 — CREATE EPISODE IDS FOR MULTIPLE GAP DEFINITIONS
# ==============================================================================================================

def build_episode_ids(segment_df, gap_tolerance):

    """
    Assign an episode ID to every attack-active minute.

    gap_tolerance = 0:
        Every benign minute terminates the current attack episode.

    gap_tolerance = 1:
        One benign minute may occur between attack-active minutes
        without splitting the episode.

    etc.

    IMPORTANT:
    Episodes never cross Segment_ID boundaries.
    """

    seg = segment_df.sort_values("Minute").copy()

    states = seg["Attack_State"].to_numpy(dtype=int)

    episode_ids = np.full(len(seg), -1, dtype=int)

    episode_counter = -1
    last_attack_index = None

    for i, state in enumerate(states):

        if state != 1:
            continue

        if last_attack_index is None:

            episode_counter += 1

        else:

            benign_gap = i - last_attack_index - 1

            if benign_gap > gap_tolerance:
                episode_counter += 1

        episode_ids[i] = episode_counter
        last_attack_index = i

    return episode_ids


episode_tables = {}
episode_lookup = {}


for tolerance in EPISODE_GAP_TOLERANCES:

    all_episode_rows = []

    global_episode_counter = 0

    minute_episode_map = {}

    for segment_id, seg in minute_df.groupby("Segment_ID", sort=True):

        seg = seg.sort_values("Minute").copy()

        local_ids = build_episode_ids(
            seg,
            gap_tolerance=tolerance
        )

        # Convert local episode IDs to globally unique IDs
        local_unique = sorted(
            x for x in np.unique(local_ids)
            if x >= 0
        )

        local_to_global = {}

        for local_id in local_unique:

            local_to_global[local_id] = global_episode_counter
            global_episode_counter += 1

        for idx, local_id in zip(seg.index, local_ids):

            if local_id >= 0:
                minute_episode_map[idx] = local_to_global[local_id]


    # Build metadata table
    for episode_id in range(global_episode_counter):

        indices = [
            idx
            for idx, ep_id in minute_episode_map.items()
            if ep_id == episode_id
        ]

        ep_rows = minute_df.loc[indices].sort_values("Minute")

        if len(ep_rows) == 0:
            continue

        segment_id = int(ep_rows["Segment_ID"].iloc[0])

        start_time = ep_rows["Minute"].min()
        end_time = ep_rows["Minute"].max()

        attack_minutes = len(ep_rows)

        # Span includes any tolerated benign gaps between first and last attack minute
        span_minutes = int(
            (end_time - start_time).total_seconds() / 60
        ) + 1

        benign_inside_span = span_minutes - attack_minutes

        row = {
            "Gap_Tolerance_Minutes": tolerance,
            "Episode_ID": episode_id,
            "Segment_ID": segment_id,
            "Episode_Start": start_time,
            "Episode_End": end_time,
            "Attack_Minutes": attack_minutes,
            "Episode_Span_Minutes": span_minutes,
            "Benign_Minutes_Inside_Span": benign_inside_span
        }

        if attack_type_col is not None:

            values = (
                ep_rows[attack_type_col]
                .dropna()
                .astype(str)
                .tolist()
            )

            cleaned_types = set()

            for value in values:

                # Works for ordinary strings and common joined representations
                for separator in ["|", ",", ";"]:

                    if separator in value:

                        pieces = value.split(separator)

                        for piece in pieces:
                            piece = piece.strip()

                            if piece and piece.lower() not in {
                                "benign",
                                "normal",
                                "none",
                                "nan",
                                "[]"
                            }:
                                cleaned_types.add(piece)

                        break

                else:

                    value = value.strip()

                    if value and value.lower() not in {
                        "benign",
                        "normal",
                        "none",
                        "nan",
                        "[]"
                    }:
                        cleaned_types.add(value)

            row["Attack_Types"] = tuple(sorted(cleaned_types))
            row["Attack_Type_Count"] = len(cleaned_types)

        all_episode_rows.append(row)


    episode_df = pd.DataFrame(all_episode_rows)

    episode_tables[tolerance] = episode_df
    episode_lookup[tolerance] = minute_episode_map


# ==============================================================================================================
# STEP 5 — EPISODE DEFINITION SENSITIVITY
# ==============================================================================================================

episode_sensitivity_rows = []

for tolerance, ep_df in episode_tables.items():

    if len(ep_df) == 0:

        episode_sensitivity_rows.append({
            "Gap_Tolerance_Minutes": tolerance,
            "Independent_Episodes": 0,
            "Median_Episode_Span_Minutes": np.nan,
            "Max_Episode_Span_Minutes": np.nan,
            "Median_Attack_Minutes": np.nan,
            "Max_Benign_Minutes_Inside_Episode": np.nan
        })

        continue

    episode_sensitivity_rows.append({

        "Gap_Tolerance_Minutes":
            tolerance,

        "Independent_Episodes":
            len(ep_df),

        "Median_Episode_Span_Minutes":
            ep_df["Episode_Span_Minutes"].median(),

        "Max_Episode_Span_Minutes":
            ep_df["Episode_Span_Minutes"].max(),

        "Median_Attack_Minutes":
            ep_df["Attack_Minutes"].median(),

        "Max_Benign_Minutes_Inside_Episode":
            ep_df["Benign_Minutes_Inside_Span"].max()
    })


episode_sensitivity_df = pd.DataFrame(
    episode_sensitivity_rows
)


print("\n" + "=" * 130)
print("EPISODE-DEFINITION SENSITIVITY")
print("=" * 130)

print(
    episode_sensitivity_df.to_string(
        index=False
    )
)


# ==============================================================================================================
# STEP 6 — BUILD MULTI-HORIZON FORECASTING AUDIT
# ==============================================================================================================

forecast_rows = []

# Store future attack timestamps for later episode matching
forecast_positive_records = []


for segment_id, seg in minute_df.groupby("Segment_ID", sort=True):

    seg = (
        seg
        .sort_values("Minute")
        .reset_index()
        .rename(columns={"index": "Original_Index"})
    )

    times = seg["Minute"].to_numpy()
    states = seg["Attack_State"].to_numpy(dtype=int)

    n = len(seg)

    for horizon in HORIZONS:

        # Exact target at t + horizon.
        #
        # Since the minute representation has complete minute bins inside
        # each validated segment, row offset == minute offset.
        for current_pos in range(HISTORY_MINUTES - 1, n - horizon):

            history_start_pos = current_pos - HISTORY_MINUTES + 1
            history_end_pos = current_pos
            future_pos = current_pos + horizon

            history_states = states[
                history_start_pos:
                history_end_pos + 1
            ]

            current_state = int(states[current_pos])
            future_state = int(states[future_pos])

            history_contains_attack = int(
                np.any(history_states == 1)
            )

            completely_clean_history = int(
                np.all(history_states == 0)
            )

            if current_state == 0 and future_state == 0:
                transition = "Benign_to_Benign"

            elif current_state == 0 and future_state == 1:
                transition = "Benign_to_Attack"

            elif current_state == 1 and future_state == 0:
                transition = "Attack_to_Benign"

            else:
                transition = "Attack_to_Attack"


            row = {

                "Segment_ID":
                    int(segment_id),

                "Horizon_Minutes":
                    horizon,

                "History_Start":
                    seg.loc[history_start_pos, "Minute"],

                "History_End":
                    seg.loc[history_end_pos, "Minute"],

                "Future_Time":
                    seg.loc[future_pos, "Minute"],

                "Current_State":
                    current_state,

                "Future_State":
                    future_state,

                "Transition":
                    transition,

                "History_Contains_Attack":
                    history_contains_attack,

                "Completely_Clean_History":
                    completely_clean_history,

                "Future_Original_Index":
                    int(seg.loc[future_pos, "Original_Index"])
            }

            forecast_rows.append(row)

            if future_state == 1:

                forecast_positive_records.append(row.copy())


forecast_df = pd.DataFrame(forecast_rows)


# ==============================================================================================================
# STEP 7 — TEMPORAL SAFETY CHECK
# ==============================================================================================================

expected_history_delta = HISTORY_MINUTES - 1

forecast_df["Observed_History_Span_Minutes"] = (
    (
        forecast_df["History_End"]
        - forecast_df["History_Start"]
    )
    .dt.total_seconds()
    / 60
)

forecast_df["Observed_Forecast_Delta_Minutes"] = (
    (
        forecast_df["Future_Time"]
        - forecast_df["History_End"]
    )
    .dt.total_seconds()
    / 60
)


invalid_history = (
    forecast_df["Observed_History_Span_Minutes"]
    != expected_history_delta
).sum()

invalid_horizon = (
    forecast_df["Observed_Forecast_Delta_Minutes"]
    != forecast_df["Horizon_Minutes"]
).sum()


print("\n" + "=" * 130)
print("TEMPORAL SAFETY VALIDATION")
print("=" * 130)

print(f"\nInvalid 15-minute histories : {invalid_history:,}")
print(f"Invalid exact horizons      : {invalid_horizon:,}")


if invalid_history != 0 or invalid_horizon != 0:

    raise RuntimeError(
        "Temporal safety validation failed. "
        "A forecasting window contains missing minutes or an invalid horizon."
    )


# ==============================================================================================================
# STEP 8 — STATE TRANSITION SUMMARY
# ==============================================================================================================

transition_summary = (
    forecast_df
    .groupby(
        ["Horizon_Minutes", "Transition"]
    )
    .size()
    .unstack(fill_value=0)
    .reset_index()
)


for column in [
    "Benign_to_Benign",
    "Benign_to_Attack",
    "Attack_to_Benign",
    "Attack_to_Attack"
]:

    if column not in transition_summary.columns:
        transition_summary[column] = 0


transition_summary["Total_Samples"] = (
    transition_summary[
        [
            "Benign_to_Benign",
            "Benign_to_Attack",
            "Attack_to_Benign",
            "Attack_to_Attack"
        ]
    ]
    .sum(axis=1)
)


transition_summary = transition_summary[
    [
        "Horizon_Minutes",
        "Total_Samples",
        "Benign_to_Benign",
        "Benign_to_Attack",
        "Attack_to_Benign",
        "Attack_to_Attack"
    ]
]


print("\n" + "=" * 130)
print("CURRENT STATE -> FUTURE STATE")
print("=" * 130)

print(
    transition_summary.to_string(
        index=False
    )
)


# ==============================================================================================================
# STEP 9 — CLEAN-HISTORY FORECASTING SUMMARY
# ==============================================================================================================

clean_history_rows = []

for horizon in HORIZONS:

    h = forecast_df[
        forecast_df["Horizon_Minutes"] == horizon
    ]

    future_positive = h[
        h["Future_State"] == 1
    ]

    clean_positive = future_positive[
        future_positive["Completely_Clean_History"] == 1
    ]

    attack_history_positive = future_positive[
        future_positive["History_Contains_Attack"] == 1
    ]

    clean_history_all = h[
        h["Completely_Clean_History"] == 1
    ]

    clean_history_negative = clean_history_all[
        clean_history_all["Future_State"] == 0
    ]

    clean_history_rows.append({

        "Horizon_Minutes":
            horizon,

        "Total_Samples":
            len(h),

        "Future_Attack_Positive":
            len(future_positive),

        "Completely_Clean_History_Positive":
            len(clean_positive),

        "Attack_In_History_Positive":
            len(attack_history_positive),

        "All_Clean_History_Samples":
            len(clean_history_all),

        "Clean_History_Future_Benign":
            len(clean_history_negative),

        "Clean_History_Positive_Percent":
            (
                100 * len(clean_positive) / len(clean_history_all)
                if len(clean_history_all) > 0
                else np.nan
            )
    })


clean_history_summary = pd.DataFrame(
    clean_history_rows
)


print("\n" + "=" * 130)
print("COMPLETELY CLEAN 15-MINUTE HISTORY AUDIT")
print("=" * 130)

print(
    clean_history_summary.to_string(
        index=False
    )
)


# ==============================================================================================================
# STEP 10 — MATCH FUTURE ATTACK TARGETS TO INDEPENDENT EPISODES
# ==============================================================================================================

episode_forecast_rows = []


for tolerance in EPISODE_GAP_TOLERANCES:

    ep_df = episode_tables[tolerance]

    if len(ep_df) == 0:
        continue

    # Build lookup:
    # every attack minute -> episode ID
    minute_to_episode = {}

    for _, ep in ep_df.iterrows():

        segment_id = int(ep["Segment_ID"])

        seg_attack_rows = minute_df[
            (minute_df["Segment_ID"] == segment_id)
            &
            (minute_df["Attack_State"] == 1)
            &
            (minute_df["Minute"] >= ep["Episode_Start"])
            &
            (minute_df["Minute"] <= ep["Episode_End"])
        ]

        for _, attack_row in seg_attack_rows.iterrows():

            minute_to_episode[
                (
                    segment_id,
                    attack_row["Minute"]
                )
            ] = int(ep["Episode_ID"])


    for horizon in HORIZONS:

        positives = forecast_df[
            (forecast_df["Horizon_Minutes"] == horizon)
            &
            (forecast_df["Future_State"] == 1)
        ].copy()


        positives["Episode_ID"] = positives.apply(

            lambda row: minute_to_episode.get(
                (
                    int(row["Segment_ID"]),
                    row["Future_Time"]
                ),
                np.nan
            ),

            axis=1
        )


        unmatched = positives["Episode_ID"].isna().sum()

        if unmatched > 0:

            raise RuntimeError(
                f"{unmatched} future attack targets at horizon {horizon} "
                f"could not be matched to an episode for tolerance {tolerance}."
            )


        all_episode_ids = set(
            positives["Episode_ID"].astype(int)
        )

        benign_current_episode_ids = set(
            positives.loc[
                positives["Current_State"] == 0,
                "Episode_ID"
            ].astype(int)
        )

        clean_history_episode_ids = set(
            positives.loc[
                positives["Completely_Clean_History"] == 1,
                "Episode_ID"
            ].astype(int)
        )


        episode_forecast_rows.append({

            "Gap_Tolerance_Minutes":
                tolerance,

            "Horizon_Minutes":
                horizon,

            "Positive_Forecasting_Windows":
                len(positives),

            "Independent_Future_Episodes":
                len(all_episode_ids),

            "Episodes_Reached_From_Current_Benign":
                len(benign_current_episode_ids),

            "Episodes_Reached_From_Clean_15m_History":
                len(clean_history_episode_ids)
        })


episode_forecast_summary = pd.DataFrame(
    episode_forecast_rows
)


print("\n" + "=" * 130)
print("INDEPENDENT FUTURE ATTACK EPISODE COVERAGE")
print("=" * 130)

print(
    episode_forecast_summary.to_string(
        index=False
    )
)


# ==============================================================================================================
# STEP 11 — ATTACK-TYPE COVERAGE AMONG FORECASTED EPISODES
# ==============================================================================================================

attack_type_coverage_rows = []


if attack_type_col is not None:

    for tolerance in EPISODE_GAP_TOLERANCES:

        ep_df = episode_tables[tolerance].copy()

        if (
            len(ep_df) == 0
            or
            "Attack_Types" not in ep_df.columns
        ):
            continue


        for horizon in HORIZONS:

            positives = forecast_df[
                (forecast_df["Horizon_Minutes"] == horizon)
                &
                (forecast_df["Future_State"] == 1)
            ]


            forecasted_episode_ids = set()

            for _, row in positives.iterrows():

                candidates = ep_df[
                    (ep_df["Segment_ID"] == row["Segment_ID"])
                    &
                    (ep_df["Episode_Start"] <= row["Future_Time"])
                    &
                    (ep_df["Episode_End"] >= row["Future_Time"])
                ]

                if len(candidates):

                    forecasted_episode_ids.add(
                        int(candidates.iloc[0]["Episode_ID"])
                    )


            selected_eps = ep_df[
                ep_df["Episode_ID"].isin(
                    forecasted_episode_ids
                )
            ]


            attack_types = set()

            for types in selected_eps["Attack_Types"]:

                for attack_type in types:
                    attack_types.add(attack_type)


            attack_type_coverage_rows.append({

                "Gap_Tolerance_Minutes":
                    tolerance,

                "Horizon_Minutes":
                    horizon,

                "Forecasted_Episodes":
                    len(selected_eps),

                "Distinct_Attack_Types":
                    len(attack_types),

                "Attack_Types":
                    ", ".join(sorted(attack_types))
            })


    attack_type_coverage_df = pd.DataFrame(
        attack_type_coverage_rows
    )


    print("\n" + "=" * 130)
    print("ATTACK-TYPE COVERAGE AMONG FORECASTED EPISODES")
    print("=" * 130)

    print(
        attack_type_coverage_df.to_string(
            index=False
        )
    )

else:

    attack_type_coverage_df = pd.DataFrame()

    print("\n" + "=" * 130)
    print("ATTACK-TYPE COVERAGE")
    print("=" * 130)

    print(
        "\nAttack-type information is not present in the loaded minute dataframe.\n"
        "Binary episode coverage has still been measured correctly."
    )


# ==============================================================================================================
# STEP 12 — HORIZON COMPARISON TABLE
# ==============================================================================================================

# Use STRICT episode definition here only as a neutral reference.
# We are NOT declaring it the final episode definition.

strict_episode_summary = episode_forecast_summary[
    episode_forecast_summary["Gap_Tolerance_Minutes"] == 0
].copy()


final_horizon_table = (
    clean_history_summary
    .merge(
        transition_summary,
        on=[
            "Horizon_Minutes",
            "Total_Samples"
        ],
        how="left"
    )
    .merge(
        strict_episode_summary[
            [
                "Horizon_Minutes",
                "Independent_Future_Episodes",
                "Episodes_Reached_From_Current_Benign",
                "Episodes_Reached_From_Clean_15m_History"
            ]
        ],
        on="Horizon_Minutes",
        how="left"
    )
)


print("\n" + "=" * 130)
print("FINAL HORIZON COMPARISON — STRICT EPISODE REFERENCE")
print("=" * 130)

print(
    final_horizon_table.to_string(
        index=False
    )
)


# ==============================================================================================================
# STEP 13 — SEGMENT COVERAGE BY HORIZON
# ==============================================================================================================

segment_rows = []


for horizon in HORIZONS:

    h = forecast_df[
        forecast_df["Horizon_Minutes"] == horizon
    ]

    positive = h[
        h["Future_State"] == 1
    ]

    clean_positive = positive[
        positive["Completely_Clean_History"] == 1
    ]

    segment_rows.append({

        "Horizon_Minutes":
            horizon,

        "All_Segments":
            h["Segment_ID"].nunique(),

        "Positive_Target_Segments":
            positive["Segment_ID"].nunique(),

        "Clean_History_Positive_Segments":
            clean_positive["Segment_ID"].nunique()
    })


segment_coverage_df = pd.DataFrame(
    segment_rows
)


print("\n" + "=" * 130)
print("SEGMENT COVERAGE")
print("=" * 130)

print(
    segment_coverage_df.to_string(
        index=False
    )
)


# ==============================================================================================================
# STEP 14 — SAVE AUDIT OUTPUTS
# ==============================================================================================================

output_files = {

    "cell18_episode_definition_sensitivity.csv":
        episode_sensitivity_df,

    "cell18_transition_summary.csv":
        transition_summary,

    "cell18_clean_history_summary.csv":
        clean_history_summary,

    "cell18_independent_episode_coverage.csv":
        episode_forecast_summary,

    "cell18_final_horizon_comparison.csv":
        final_horizon_table,

    "cell18_segment_coverage.csv":
        segment_coverage_df
}


if len(attack_type_coverage_df):

    output_files[
        "cell18_attack_type_episode_coverage.csv"
    ] = attack_type_coverage_df


print("\n" + "=" * 130)
print("SAVING AUDIT OUTPUTS")
print("=" * 130)


for filename, dataframe in output_files.items():

    path = os.path.join(
        AUDIT_DIR,
        filename
    )

    dataframe.to_csv(
        path,
        index=False
    )

    print("✓", path)


# ==============================================================================================================
# STEP 15 — FINAL VALIDATION / INTERPRETATION GUARDRAILS
# ==============================================================================================================

print("\n" + "=" * 130)
print("CELL 18 VALIDATION")
print("=" * 130)

print(f"\n✓ Temporal resolution locked for this audit : 60 seconds")
print(f"✓ History length                           : {HISTORY_MINUTES} minutes")
print(f"✓ Horizons audited                         : {HORIZONS}")
print(f"✓ Episode tolerances audited               : {EPISODE_GAP_TOLERANCES}")
print(f"✓ Invalid history windows                  : {invalid_history}")
print(f"✓ Invalid forecast horizons                : {invalid_horizon}")
print(f"✓ Capture segments represented             : {minute_df['Segment_ID'].nunique()}")


print("\n" + "=" * 130)
print("IMPORTANT INTERPRETATION")
print("=" * 130)

print(
    """
1. Sliding forecasting windows are NOT independent attack events.

2. 'Benign_to_Attack' means the CURRENT minute is benign and the
   future target minute is attack-active. It must NOT automatically
   be described as a new independent attack onset.

3. 'Completely_Clean_History_Positive' is stricter:
   the entire preceding 15-minute history contains no attack minute.

4. Independent episode counts are reported separately for multiple
   episode-gap definitions (0, 1, 2 and 5 minutes).

5. No episode-gap tolerance has been declared the final truth.

6. No horizon has been selected solely because it produces the
   largest number of overlapping forecasting windows.

7. No train/validation/test split has been created.

8. No feature scaling has been performed.

9. No model has been trained.
"""
)


print("=" * 130)
print("CELL 18 COMPLETE")
print("=" * 130)

print(
    """
NEXT:

Inspect the Cell 18 tables.

They will determine:

    - which forecasting horizon(s) we retain,
    - whether the primary task is general attack-state forecasting,
    - how much genuine clean-history warning capability exists,
    - which episode definition is defensible for evaluation,
    - and how the later leakage-safe split must be designed.

DO NOT START MODEL TRAINING YET.
"""
)

print("=" * 130)

CELL 18 — FINAL CICIDS2017 FORECASTING-DESIGN AUDIT

✓ Minute dataset recovered from RAM variable: minute_df

BASE DATASET VALIDATION

Minute rows                  : 2,454
Segments                     : 10
Attack minutes               : 520
Benign minutes               : 1,934
Duplicate segment/minute rows: 0

Attack-type column: Attack_Types

EPISODE-DEFINITION SENSITIVITY
 Gap_Tolerance_Minutes  Independent_Episodes  Median_Episode_Span_Minutes  Max_Episode_Span_Minutes  Median_Attack_Minutes  Max_Benign_Minutes_Inside_Episode
                     0                    82                          2.0                        70                    2.0                                  0
                     1                    39                          3.0                       176                    3.0                                 15
                     2                    26                         11.5                       176                    8.0                           

In [25]:
# ==============================================================================================================
# CELL 19 — LEAKAGE-SAFE SPLIT + ATTACK-TYPE FORECASTING FEASIBILITY AUDIT
# ==============================================================================================================

import pandas as pd
import numpy as np
from collections import defaultdict

print("=" * 126)
print("CELL 19 — LEAKAGE-SAFE SPLIT + ATTACK-TYPE FORECASTING FEASIBILITY AUDIT")
print("=" * 126)

# --------------------------------------------------------------------------------------------------------------
# CONFIGURATION
# --------------------------------------------------------------------------------------------------------------

HISTORY_MINUTES = 15
HORIZONS = [1, 5, 10, 15, 30]

# We do NOT assume one episode definition is correct.
# Cell 18 showed strong sensitivity to this choice.
EPISODE_TOLERANCES = [0, 1, 2, 5]

# Minimum support thresholds are descriptive only.
# They are NOT automatic inclusion/exclusion rules.
MIN_EPISODES_FOR_CANDIDATE = 3
MIN_ATTACK_MINUTES_FOR_CANDIDATE = 5


# --------------------------------------------------------------------------------------------------------------
# RECOVER minute_df
# --------------------------------------------------------------------------------------------------------------

if "minute_df" not in globals():
    raise RuntimeError(
        "minute_df was not found in RAM.\n"
        "Please run the cell that constructs the final minute-level CICIDS2017 dataset first."
    )

df = minute_df.copy()

print("\n✓ minute_df recovered")
print(f"Rows     : {len(df):,}")
print(f"Segments : {df['Segment_ID'].nunique()}")


# --------------------------------------------------------------------------------------------------------------
# VALIDATE REQUIRED COLUMNS
# --------------------------------------------------------------------------------------------------------------

required_columns = [
    "Segment_ID",
    "Minute",
    "Attack_State",
    "Attack_Types"
]

missing = [c for c in required_columns if c not in df.columns]

if missing:
    raise RuntimeError(
        "Required columns missing from minute_df:\n" +
        "\n".join(f"  - {x}" for x in missing)
    )

df["Minute"] = pd.to_datetime(df["Minute"])
df = df.sort_values(["Segment_ID", "Minute"]).reset_index(drop=True)

print("\n✓ Required columns validated")


# --------------------------------------------------------------------------------------------------------------
# NORMALIZE ATTACK TYPE REPRESENTATION
# --------------------------------------------------------------------------------------------------------------
#
# Attack_Types may contain:
#
#     "DDoS"
#
# or multiple types:
#
#     "DDoS | PortScan"
#
# depending on how minute_df was constructed.
#
# This function safely extracts individual attack types.
# --------------------------------------------------------------------------------------------------------------

def parse_attack_types(value):

    if pd.isna(value):
        return []

    value = str(value).strip()

    if value == "" or value.lower() in {
        "none", "nan", "benign", "normal", "[]"
    }:
        return []

    # Handle common separators safely
    for separator in [";", ",", "|"]:
        if separator in value:
            return [
                x.strip()
                for x in value.split(separator)
                if x.strip()
            ]

    return [value]


df["_Attack_Type_List"] = df["Attack_Types"].apply(parse_attack_types)


# --------------------------------------------------------------------------------------------------------------
# BASIC TEMPORAL STRUCTURE
# --------------------------------------------------------------------------------------------------------------

print("\n")
print("=" * 126)
print("CAPTURE SEGMENT SUMMARY")
print("=" * 126)

segment_rows = []

for segment_id, seg in df.groupby("Segment_ID", sort=True):

    seg = seg.sort_values("Minute")

    attack_mask = seg["Attack_State"].astype(int).eq(1)

    types = sorted({
        attack_type
        for values in seg.loc[attack_mask, "_Attack_Type_List"]
        for attack_type in values
    })

    segment_rows.append({
        "Segment_ID": segment_id,
        "Start_Time": seg["Minute"].min(),
        "End_Time": seg["Minute"].max(),
        "Date": seg["Minute"].min().date(),
        "Total_Minutes": len(seg),
        "Benign_Minutes": int((~attack_mask).sum()),
        "Attack_Minutes": int(attack_mask.sum()),
        "Attack_Percent": float(attack_mask.mean() * 100),
        "Distinct_Attack_Types": len(types),
        "Attack_Types": ", ".join(types) if types else "NONE"
    })

segment_summary = pd.DataFrame(segment_rows)

print(segment_summary.to_string(index=False))


# --------------------------------------------------------------------------------------------------------------
# ATTACK TYPE → SEGMENT SUPPORT
# --------------------------------------------------------------------------------------------------------------

all_attack_types = sorted({
    attack_type
    for values in df["_Attack_Type_List"]
    for attack_type in values
})

print("\n")
print("=" * 126)
print("ATTACK-TYPE TEMPORAL SUPPORT")
print("=" * 126)

type_support_rows = []

for attack_type in all_attack_types:

    type_mask = df["_Attack_Type_List"].apply(
        lambda values: attack_type in values
    )

    type_df = df[type_mask]

    segments = sorted(type_df["Segment_ID"].unique().tolist())
    dates = sorted(type_df["Minute"].dt.date.unique().tolist())

    type_support_rows.append({
        "Attack_Type": attack_type,
        "Attack_Minutes": len(type_df),
        "Segments_Present": len(segments),
        "Segment_IDs": ", ".join(map(str, segments)),
        "Dates_Present": len(dates),
        "Dates": ", ".join(map(str, dates))
    })

type_support = pd.DataFrame(type_support_rows)

type_support = type_support.sort_values(
    ["Segments_Present", "Attack_Minutes"],
    ascending=[False, False]
).reset_index(drop=True)

print(type_support.to_string(index=False))


# --------------------------------------------------------------------------------------------------------------
# BUILD ATTACK EPISODES
# --------------------------------------------------------------------------------------------------------------
#
# An episode is defined separately FOR EACH ATTACK TYPE.
#
# Example:
#
#     DDoS minute
#     DDoS minute
#     benign minute
#     DDoS minute
#
# Depending on tolerance:
#
# tolerance = 0  -> potentially separate episodes
# tolerance = 1  -> potentially one episode
#
# Episodes NEVER cross Segment_ID boundaries.
# --------------------------------------------------------------------------------------------------------------

def build_type_episodes(dataframe, attack_type, tolerance_minutes):

    episodes = []

    for segment_id, seg in dataframe.groupby("Segment_ID", sort=True):

        seg = seg.sort_values("Minute").reset_index(drop=True)

        presence = seg["_Attack_Type_List"].apply(
            lambda values: attack_type in values
        ).to_numpy()

        attack_positions = np.where(presence)[0]

        if len(attack_positions) == 0:
            continue

        current_positions = [attack_positions[0]]

        for position in attack_positions[1:]:

            previous_position = current_positions[-1]

            # Number of non-attack minute rows separating attack minutes
            benign_gap = position - previous_position - 1

            if benign_gap <= tolerance_minutes:
                current_positions.append(position)

            else:

                ep_rows = seg.iloc[current_positions]

                episodes.append({
                    "Attack_Type": attack_type,
                    "Segment_ID": segment_id,
                    "Start_Time": ep_rows["Minute"].min(),
                    "End_Time": ep_rows["Minute"].max(),
                    "Attack_Minutes": len(ep_rows),
                    "Span_Minutes": (
                        ep_rows["Minute"].max()
                        - ep_rows["Minute"].min()
                    ).total_seconds() / 60 + 1
                })

                current_positions = [position]

        ep_rows = seg.iloc[current_positions]

        episodes.append({
            "Attack_Type": attack_type,
            "Segment_ID": segment_id,
            "Start_Time": ep_rows["Minute"].min(),
            "End_Time": ep_rows["Minute"].max(),
            "Attack_Minutes": len(ep_rows),
            "Span_Minutes": (
                ep_rows["Minute"].max()
                - ep_rows["Minute"].min()
            ).total_seconds() / 60 + 1
        })

    return pd.DataFrame(episodes)


# --------------------------------------------------------------------------------------------------------------
# EPISODE SUPPORT UNDER MULTIPLE DEFINITIONS
# --------------------------------------------------------------------------------------------------------------

print("\n")
print("=" * 126)
print("ATTACK-TYPE EPISODE SUPPORT")
print("=" * 126)

episode_support_rows = []
episode_tables = {}

for tolerance in EPISODE_TOLERANCES:

    tolerance_episode_frames = []

    for attack_type in all_attack_types:

        ep = build_type_episodes(
            df,
            attack_type,
            tolerance
        )

        if len(ep) > 0:

            ep["Gap_Tolerance_Minutes"] = tolerance
            tolerance_episode_frames.append(ep)

            episode_support_rows.append({
                "Gap_Tolerance_Minutes": tolerance,
                "Attack_Type": attack_type,
                "Independent_Episodes": len(ep),
                "Segments_With_Episodes": ep["Segment_ID"].nunique(),
                "Median_Attack_Minutes_Per_Episode": ep["Attack_Minutes"].median(),
                "Max_Attack_Minutes_Per_Episode": ep["Attack_Minutes"].max(),
                "Median_Episode_Span_Minutes": ep["Span_Minutes"].median(),
                "Max_Episode_Span_Minutes": ep["Span_Minutes"].max()
            })

    if tolerance_episode_frames:
        episode_tables[tolerance] = pd.concat(
            tolerance_episode_frames,
            ignore_index=True
        )
    else:
        episode_tables[tolerance] = pd.DataFrame()


episode_support = pd.DataFrame(episode_support_rows)

for tolerance in EPISODE_TOLERANCES:

    print(f"\nGAP TOLERANCE = {tolerance} MINUTE(S)")
    print("-" * 126)

    subset = episode_support[
        episode_support["Gap_Tolerance_Minutes"] == tolerance
    ].copy()

    subset = subset.sort_values(
        ["Independent_Episodes", "Segments_With_Episodes"],
        ascending=False
    )

    print(subset.to_string(index=False))


# --------------------------------------------------------------------------------------------------------------
# FORECASTABLE SUPPORT BY ATTACK TYPE
# --------------------------------------------------------------------------------------------------------------
#
# This measures whether a particular attack type is actually reachable
# by our forecasting setup:
#
#       previous 15 minutes  --->  target at t + H
#
# We count:
#
# 1. total positive target windows
# 2. current-benign -> future attack-type windows
# 3. completely-clean-history -> future attack-type windows
#
# IMPORTANT:
# These are overlapping forecasting windows, NOT independent events.
# --------------------------------------------------------------------------------------------------------------

print("\n")
print("=" * 126)
print("ATTACK-TYPE FORECASTING SUPPORT")
print("=" * 126)

forecast_rows = []

for horizon in HORIZONS:

    for attack_type in all_attack_types:

        total_valid = 0
        positive_targets = 0
        current_benign_positive = 0
        clean_history_positive = 0

        positive_segments = set()
        clean_positive_segments = set()

        for segment_id, seg in df.groupby("Segment_ID", sort=True):

            seg = seg.sort_values("Minute").reset_index(drop=True)

            n = len(seg)

            for target_idx in range(HISTORY_MINUTES + horizon - 1, n):

                current_idx = target_idx - horizon

                history_start = current_idx - HISTORY_MINUTES + 1

                if history_start < 0:
                    continue

                history = seg.iloc[
                    history_start:current_idx + 1
                ]

                target = seg.iloc[target_idx]

                # Exact temporal safety check
                expected_history_start = (
                    seg.iloc[current_idx]["Minute"]
                    - pd.Timedelta(minutes=HISTORY_MINUTES - 1)
                )

                expected_target_time = (
                    seg.iloc[current_idx]["Minute"]
                    + pd.Timedelta(minutes=horizon)
                )

                if history.iloc[0]["Minute"] != expected_history_start:
                    continue

                if target["Minute"] != expected_target_time:
                    continue

                total_valid += 1

                target_types = target["_Attack_Type_List"]

                if attack_type not in target_types:
                    continue

                positive_targets += 1
                positive_segments.add(segment_id)

                current_attack = int(
                    seg.iloc[current_idx]["Attack_State"]
                )

                history_attack = int(
                    history["Attack_State"].max()
                )

                if current_attack == 0:
                    current_benign_positive += 1

                if history_attack == 0:
                    clean_history_positive += 1
                    clean_positive_segments.add(segment_id)

        forecast_rows.append({
            "Horizon_Minutes": horizon,
            "Attack_Type": attack_type,
            "Valid_Forecasting_Windows": total_valid,
            "Positive_Target_Windows": positive_targets,
            "Current_Benign_to_Attack_Type": current_benign_positive,
            "Clean_15m_History_to_Attack_Type": clean_history_positive,
            "Positive_Target_Segments": len(positive_segments),
            "Clean_History_Positive_Segments": len(clean_positive_segments)
        })


forecast_support = pd.DataFrame(forecast_rows)

for horizon in HORIZONS:

    print(f"\nHORIZON = {horizon} MINUTES")
    print("-" * 126)

    subset = forecast_support[
        forecast_support["Horizon_Minutes"] == horizon
    ].copy()

    subset = subset.sort_values(
        [
            "Clean_15m_History_to_Attack_Type",
            "Positive_Target_Windows"
        ],
        ascending=False
    )

    print(subset.to_string(index=False))


# --------------------------------------------------------------------------------------------------------------
# TEMPORAL ORDER OF SEGMENTS
# --------------------------------------------------------------------------------------------------------------

segment_order = (
    segment_summary
    .sort_values("Start_Time")
    ["Segment_ID"]
    .tolist()
)

print("\n")
print("=" * 126)
print("CHRONOLOGICAL SEGMENT ORDER")
print("=" * 126)

for i, segment_id in enumerate(segment_order, start=1):

    row = segment_summary[
        segment_summary["Segment_ID"] == segment_id
    ].iloc[0]

    print(
        f"{i:2d}. Segment {segment_id} | "
        f"{row['Start_Time']} -> {row['End_Time']} | "
        f"attack minutes={row['Attack_Minutes']:,} | "
        f"types={row['Attack_Types']}"
    )


# --------------------------------------------------------------------------------------------------------------
# CANDIDATE CHRONOLOGICAL SPLITS
# --------------------------------------------------------------------------------------------------------------
#
# We evaluate possible boundaries rather than arbitrarily selecting 70/15/15.
#
# Since there are only 10 capture segments, candidate boundaries are
# constructed at segment boundaries.
#
# Require:
#
#     at least 1 segment in train
#     at least 1 segment in validation
#     at least 1 segment in test
#
# --------------------------------------------------------------------------------------------------------------

print("\n")
print("=" * 126)
print("CANDIDATE LEAKAGE-SAFE TEMPORAL SPLITS")
print("=" * 126)

split_rows = []

N = len(segment_order)

for train_end in range(1, N - 1):

    for val_end in range(train_end + 1, N):

        train_segments = segment_order[:train_end]
        val_segments = segment_order[train_end:val_end]
        test_segments = segment_order[val_end:]

        train_df = df[df["Segment_ID"].isin(train_segments)]
        val_df = df[df["Segment_ID"].isin(val_segments)]
        test_df = df[df["Segment_ID"].isin(test_segments)]

        train_attack = int(train_df["Attack_State"].sum())
        val_attack = int(val_df["Attack_State"].sum())
        test_attack = int(test_df["Attack_State"].sum())

        train_types = {
            x
            for values in train_df["_Attack_Type_List"]
            for x in values
        }

        val_types = {
            x
            for values in val_df["_Attack_Type_List"]
            for x in values
        }

        test_types = {
            x
            for values in test_df["_Attack_Type_List"]
            for x in values
        }

        shared_all = train_types & val_types & test_types

        split_rows.append({
            "Train_End_Index": train_end,
            "Validation_End_Index": val_end,

            "Train_Segments": ",".join(map(str, train_segments)),
            "Validation_Segments": ",".join(map(str, val_segments)),
            "Test_Segments": ",".join(map(str, test_segments)),

            "Train_Minutes": len(train_df),
            "Validation_Minutes": len(val_df),
            "Test_Minutes": len(test_df),

            "Train_Attack_Minutes": train_attack,
            "Validation_Attack_Minutes": val_attack,
            "Test_Attack_Minutes": test_attack,

            "Train_Attack_Types": len(train_types),
            "Validation_Attack_Types": len(val_types),
            "Test_Attack_Types": len(test_types),

            "Attack_Types_Shared_All_Splits": len(shared_all),
            "Shared_Attack_Type_Names": ", ".join(sorted(shared_all))
        })


split_candidates = pd.DataFrame(split_rows)

# Prefer splits with attack activity in all partitions,
# then maximize attack-type overlap.

valid_split_candidates = split_candidates[
    (split_candidates["Train_Attack_Minutes"] > 0) &
    (split_candidates["Validation_Attack_Minutes"] > 0) &
    (split_candidates["Test_Attack_Minutes"] > 0)
].copy()

valid_split_candidates = valid_split_candidates.sort_values(
    [
        "Attack_Types_Shared_All_Splits",
        "Validation_Attack_Minutes",
        "Test_Attack_Minutes"
    ],
    ascending=False
)

print("\nTop candidate temporal splits:")
print(
    valid_split_candidates.head(15).to_string(index=False)
)


# --------------------------------------------------------------------------------------------------------------
# ATTACK-TYPE CANDIDATE SCREEN
# --------------------------------------------------------------------------------------------------------------
#
# IMPORTANT:
#
# This does NOT automatically choose the final five attacks.
#
# It identifies attacks that have:
#
#   - more than trivial attack-minute support
#   - multiple independent episodes
#   - presence across multiple temporal segments
#
# We use tolerance=1 only as a SCREENING VIEW because Cell 18 showed
# that tolerance strongly changes episode counts.
#
# Final evaluation can still report multiple tolerances.
# --------------------------------------------------------------------------------------------------------------

SCREEN_TOLERANCE = 1

screen_episode = episode_support[
    episode_support["Gap_Tolerance_Minutes"] == SCREEN_TOLERANCE
].copy()

candidate_table = type_support.merge(
    screen_episode[
        [
            "Attack_Type",
            "Independent_Episodes",
            "Segments_With_Episodes",
            "Median_Attack_Minutes_Per_Episode"
        ]
    ],
    on="Attack_Type",
    how="left"
)

candidate_table["Independent_Episodes"] = (
    candidate_table["Independent_Episodes"]
    .fillna(0)
    .astype(int)
)

candidate_table["Segments_With_Episodes"] = (
    candidate_table["Segments_With_Episodes"]
    .fillna(0)
    .astype(int)
)

candidate_table["Meets_Initial_Support_Screen"] = (
    (candidate_table["Attack_Minutes"] >= MIN_ATTACK_MINUTES_FOR_CANDIDATE)
    &
    (candidate_table["Independent_Episodes"] >= MIN_EPISODES_FOR_CANDIDATE)
)

candidate_table = candidate_table.sort_values(
    [
        "Meets_Initial_Support_Screen",
        "Segments_With_Episodes",
        "Independent_Episodes",
        "Attack_Minutes"
    ],
    ascending=[False, False, False, False]
)

print("\n")
print("=" * 126)
print("INITIAL ATTACK-TYPE CANDIDATE SCREEN")
print("=" * 126)

print(
    "\nNOTE: This is NOT a final top-5 ranking.\n"
    "It is a feasibility screen based on temporal support.\n"
)

print(candidate_table.to_string(index=False))


# --------------------------------------------------------------------------------------------------------------
# SAVE OUTPUTS
# --------------------------------------------------------------------------------------------------------------

print("\n")
print("=" * 126)
print("SAVING CELL 19 OUTPUTS")
print("=" * 126)

OUTPUT_DIR = (
    "/content/drive/MyDrive/"
    "Network-Attack-Forecasting/data/audits"
)

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

outputs = {
    "cell19_segment_summary.csv":
        segment_summary,

    "cell19_attack_type_temporal_support.csv":
        type_support,

    "cell19_attack_type_episode_support.csv":
        episode_support,

    "cell19_attack_type_forecasting_support.csv":
        forecast_support,

    "cell19_candidate_temporal_splits.csv":
        split_candidates,

    "cell19_attack_type_candidate_screen.csv":
        candidate_table
}

for filename, table in outputs.items():

    path = os.path.join(OUTPUT_DIR, filename)
    table.to_csv(path, index=False)

    print(f"✓ {path}")


# Save detailed episode tables too

for tolerance, table in episode_tables.items():

    path = os.path.join(
        OUTPUT_DIR,
        f"cell19_attack_episodes_tolerance_{tolerance}m.csv"
    )

    table.to_csv(path, index=False)

    print(f"✓ {path}")


# --------------------------------------------------------------------------------------------------------------
# FINAL VALIDATION
# --------------------------------------------------------------------------------------------------------------

print("\n")
print("=" * 126)
print("CELL 19 VALIDATION")
print("=" * 126)

print(f"✓ Capture segments                 : {df['Segment_ID'].nunique()}")
print(f"✓ Attack types                     : {len(all_attack_types)}")
print(f"✓ Horizons inspected               : {HORIZONS}")
print(f"✓ Episode tolerances inspected     : {EPISODE_TOLERANCES}")
print(f"✓ Candidate temporal splits tested : {len(split_candidates)}")

print("\n")
print("=" * 126)
print("IMPORTANT INTERPRETATION")
print("=" * 126)

print("""
1. DO NOT choose attacks solely because they contain many raw flows.

2. For attack-specific forecasting, independent temporal episodes and
   cross-segment support matter more than raw flow count.

3. An attack appearing in only one temporal region may be useful for
   binary attack-state forecasting but cannot support a convincing
   independent train/validation/test attack-specific experiment.

4. The 'candidate screen' is descriptive. It does NOT automatically
   declare which attacks must be retained.

5. Candidate train/validation/test splits are chronological and occur
   only at capture-segment boundaries.

6. No model has been trained.

7. No scaler has been fitted.

8. No final split has been selected.

9. No final subset of attack types has been selected.
""")

print("=" * 126)
print("CELL 19 COMPLETE")
print("=" * 126)

print("""
NEXT:

Inspect:

    A. ATTACK-TYPE TEMPORAL SUPPORT
    B. ATTACK-TYPE EPISODE SUPPORT
    C. ATTACK-TYPE FORECASTING SUPPORT
    D. TOP CANDIDATE TEMPORAL SPLITS
    E. INITIAL ATTACK-TYPE CANDIDATE SCREEN

Then we can make TWO final decisions:

    1. Which leakage-safe train / validation / test split to use.
    2. Which attack types have enough independent temporal support
       for attack-specific forecasting.

DO NOT TRAIN YET.
""")

CELL 19 — LEAKAGE-SAFE SPLIT + ATTACK-TYPE FORECASTING FEASIBILITY AUDIT

✓ minute_df recovered
Rows     : 2,454
Segments : 10

✓ Required columns validated


CAPTURE SEGMENT SUMMARY
 Segment_ID          Start_Time            End_Time       Date  Total_Minutes  Benign_Minutes  Attack_Minutes  Attack_Percent  Distinct_Attack_Types                                                           Attack_Types
          0 2017-07-03 01:00:00 2017-07-03 05:01:00 2017-07-03            242             242               0        0.000000                      0                                                                   NONE
          1 2017-07-03 08:55:00 2017-07-03 12:59:00 2017-07-03            245             245               0        0.000000                      0                                                                   NONE
          2 2017-07-04 01:00:00 2017-07-04 05:00:00 2017-07-04            241             178              63       26.141079                      1         

In [26]:
# ==============================================================================================================
# CELL 20 — EVALUATION PROTOCOL FEASIBILITY AUDIT
# ==============================================================================================================
#
# PURPOSE
# -------
# Before training any forecasting model, determine whether CICIDS2017 supports
# a defensible temporal evaluation protocol.
#
# This cell does NOT:
#   - train a model
#   - normalize features
#   - oversample / undersample
#   - randomly split temporal samples
#   - permanently choose the final split
#
# It audits:
#
#   1. chronological train / validation / test feasibility
#   2. attack-state coverage across each split
#   3. attack-type coverage across each split
#   4. independent episode coverage
#   5. forecasting support for attack types
#   6. which attack types are realistic candidates for type-level forecasting
#
# IMPORTANT
# ---------
# Segments are kept intact.
# No forecasting window may cross a segment boundary.
#
# ==============================================================================================================

import pandas as pd
import numpy as np

print("=" * 126)
print("CELL 20 — EVALUATION PROTOCOL FEASIBILITY AUDIT")
print("=" * 126)


# ==============================================================================================================
# 1. VERIFY REQUIRED OBJECTS
# ==============================================================================================================

required_objects = [
    "minute_df",
    "segment_summary",
    "type_support",
    "episode_support",
    "forecast_support",
    "valid_split_candidates",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Missing required objects:\n"
        + "\n".join(f"  - {name}" for name in missing_objects)
        + "\n\nRun the earlier cells that create these objects before Cell 20."
    )

print("\n✓ Required structures found.")

print(f"\nminute_df rows              : {len(minute_df):,}")
print(f"segments                    : {minute_df['Segment_ID'].nunique():,}")
print(f"attack types in type_support: {len(type_support):,}")
print(f"valid temporal splits       : {len(valid_split_candidates):,}")


# ==============================================================================================================
# 2. BASIC COLUMN VALIDATION
# ==============================================================================================================

required_minute_columns = [
    "Segment_ID",
    "Minute",
    "Attack_State",
    "Attack_Types",
]

missing_cols = [
    c for c in required_minute_columns
    if c not in minute_df.columns
]

if missing_cols:
    raise RuntimeError(
        "minute_df is missing required columns:\n"
        + "\n".join(f"  - {c}" for c in missing_cols)
    )


# ==============================================================================================================
# 3. NORMALIZE ATTACK-TYPE REPRESENTATION
# ==============================================================================================================
#
# Attack_Types may contain:
#
#   NaN
#   ""
#   BENIGN
#   "DoS Hulk"
#   multiple types separated by delimiters
#
# We create a safe parser WITHOUT modifying minute_df.
#

def parse_attack_types(value):

    if pd.isna(value):
        return set()

    # Already stored as a collection
    if isinstance(value, (list, tuple, set, np.ndarray)):
        values = value

    else:
        text = str(value).strip()

        if text == "":
            return set()

        # Remove common collection characters
        text = (
            text.replace("[", "")
                .replace("]", "")
                .replace("{", "")
                .replace("}", "")
                .replace("'", "")
                .replace('"', "")
        )

        # Support common delimiters
        text = text.replace("|", ",").replace(";", ",")

        values = text.split(",")

    result = set()

    for x in values:

        x = str(x).strip()

        if (
            x
            and x.upper() not in {
                "BENIGN",
                "NORMAL",
                "NONE",
                "NAN"
            }
        ):
            result.add(x)

    return result


# ==============================================================================================================
# 4. BUILD SEGMENT-LEVEL ATTACK COVERAGE
# ==============================================================================================================

segment_ids = sorted(
    minute_df["Segment_ID"].dropna().unique().tolist()
)

segment_attack_records = []

for seg_id in segment_ids:

    seg = minute_df[
        minute_df["Segment_ID"] == seg_id
    ].copy()

    attack_types = set()

    for value in seg["Attack_Types"]:
        attack_types.update(parse_attack_types(value))

    segment_attack_records.append({
        "Segment_ID": seg_id,
        "Start_Minute": seg["Minute"].min(),
        "End_Minute": seg["Minute"].max(),
        "Total_Minutes": len(seg),
        "Attack_Minutes": int(seg["Attack_State"].sum()),
        "Benign_Minutes": int((seg["Attack_State"] == 0).sum()),
        "Attack_Types": sorted(attack_types),
        "Attack_Type_Count": len(attack_types),
    })

cell20_segment_attack_coverage = pd.DataFrame(
    segment_attack_records
)

print("\n")
print("=" * 126)
print("SEGMENT-LEVEL ATTACK COVERAGE")
print("=" * 126)

print(
    cell20_segment_attack_coverage.to_string(
        index=False
    )
)


# ==============================================================================================================
# 5. STANDARDIZE VALID SPLIT CANDIDATES
# ==============================================================================================================

split_df = valid_split_candidates.copy()

required_split_columns = [
    "Train_Segments",
    "Validation_Segments",
    "Test_Segments",
]

missing_split_cols = [
    c for c in required_split_columns
    if c not in split_df.columns
]

if missing_split_cols:
    raise RuntimeError(
        "valid_split_candidates is missing required columns:\n"
        + "\n".join(f"  - {c}" for c in missing_split_cols)
    )


def normalize_segment_collection(value):

    if isinstance(value, (list, tuple, set, np.ndarray)):
        return sorted([
            int(x) for x in value
        ])

    if pd.isna(value):
        return []

    text = str(value).strip()

    text = (
        text.replace("[", "")
            .replace("]", "")
            .replace("(", "")
            .replace(")", "")
            .replace("{", "")
            .replace("}", "")
    )

    if text == "":
        return []

    parts = text.replace(";", ",").split(",")

    result = []

    for p in parts:

        p = p.strip()

        if p == "":
            continue

        try:
            result.append(int(float(p)))
        except Exception:
            pass

    return sorted(result)


# ==============================================================================================================
# 6. ATTACK TYPES PRESENT IN A SET OF SEGMENTS
# ==============================================================================================================

def attack_types_for_segments(segment_list):

    subset = cell20_segment_attack_coverage[
        cell20_segment_attack_coverage["Segment_ID"].isin(
            segment_list
        )
    ]

    result = set()

    for values in subset["Attack_Types"]:
        result.update(values)

    return result


def attack_minutes_for_segments(segment_list):

    subset = minute_df[
        minute_df["Segment_ID"].isin(segment_list)
    ]

    return int(subset["Attack_State"].sum())


def total_minutes_for_segments(segment_list):

    return int(
        minute_df[
            minute_df["Segment_ID"].isin(segment_list)
        ].shape[0]
    )


# ==============================================================================================================
# 7. AUDIT EVERY VALID TEMPORAL SPLIT
# ==============================================================================================================

split_records = []

for split_idx, row in split_df.iterrows():

    train_segments = normalize_segment_collection(
        row["Train_Segments"]
    )

    val_segments = normalize_segment_collection(
        row["Validation_Segments"]
    )

    test_segments = normalize_segment_collection(
        row["Test_Segments"]
    )

    train_types = attack_types_for_segments(
        train_segments
    )

    val_types = attack_types_for_segments(
        val_segments
    )

    test_types = attack_types_for_segments(
        test_segments
    )

    # Types evaluable in all three partitions
    common_types = (
        train_types
        & val_types
        & test_types
    )

    # Test types never observed during training
    unseen_test_types = (
        test_types - train_types
    )

    train_minutes = total_minutes_for_segments(
        train_segments
    )

    val_minutes = total_minutes_for_segments(
        val_segments
    )

    test_minutes = total_minutes_for_segments(
        test_segments
    )

    train_attack_minutes = attack_minutes_for_segments(
        train_segments
    )

    val_attack_minutes = attack_minutes_for_segments(
        val_segments
    )

    test_attack_minutes = attack_minutes_for_segments(
        test_segments
    )

    split_records.append({

        "Original_Split_Index": split_idx,

        "Train_Segments": train_segments,
        "Validation_Segments": val_segments,
        "Test_Segments": test_segments,

        "Train_Minutes": train_minutes,
        "Validation_Minutes": val_minutes,
        "Test_Minutes": test_minutes,

        "Train_Attack_Minutes": train_attack_minutes,
        "Validation_Attack_Minutes": val_attack_minutes,
        "Test_Attack_Minutes": test_attack_minutes,

        "Train_Attack_Types": len(train_types),
        "Validation_Attack_Types": len(val_types),
        "Test_Attack_Types": len(test_types),

        "Common_Attack_Types": len(common_types),

        "Common_Type_Names":
            sorted(common_types),

        "Unseen_Test_Attack_Types":
            len(unseen_test_types),

        "Unseen_Test_Type_Names":
            sorted(unseen_test_types),

        "Binary_State_Usable":
            (
                train_attack_minutes > 0
                and val_attack_minutes > 0
                and test_attack_minutes > 0
            ),

        "Type_Level_Usable":
            len(common_types) > 0,
    })


cell20_split_audit = pd.DataFrame(
    split_records
)


# ==============================================================================================================
# 8. DISPLAY SPLIT AUDIT
# ==============================================================================================================

print("\n")
print("=" * 126)
print("VALID TEMPORAL SPLIT AUDIT")
print("=" * 126)

display_columns = [

    "Original_Split_Index",

    "Train_Segments",
    "Validation_Segments",
    "Test_Segments",

    "Train_Minutes",
    "Validation_Minutes",
    "Test_Minutes",

    "Train_Attack_Minutes",
    "Validation_Attack_Minutes",
    "Test_Attack_Minutes",

    "Train_Attack_Types",
    "Validation_Attack_Types",
    "Test_Attack_Types",

    "Common_Attack_Types",
    "Unseen_Test_Attack_Types",

    "Binary_State_Usable",
    "Type_Level_Usable",
]

print(
    cell20_split_audit[
        display_columns
    ].to_string(index=False)
)


# ==============================================================================================================
# 9. ATTACK-TYPE SUPPORT TABLE
# ==============================================================================================================
#
# Merge the three important support perspectives:
#
#   type_support      -> temporal presence
#   episode_support   -> independent episodes
#   forecast_support  -> forecasting-window availability
#
# We DO NOT automatically declare a final set of attack types.
# Instead, we expose their support so the next decision is evidence-based.
#

print("\n")
print("=" * 126)
print("ATTACK-TYPE FORECASTING SUPPORT")
print("=" * 126)


# --------------------------------------------------------------------------------------------------------------
# Select episode definition
# --------------------------------------------------------------------------------------------------------------
#
# Earlier cells evaluated multiple benign-gap tolerances.
#
# Prefer 2-minute tolerance when available because this preserves short
# interruptions without merging very distant attack activity.
#

episode_gap_values = sorted(
    episode_support[
        "Gap_Tolerance_Minutes"
    ].dropna().unique().tolist()
)

if 2 in episode_gap_values:
    selected_episode_gap = 2

elif len(episode_gap_values) > 0:
    selected_episode_gap = episode_gap_values[0]

else:
    selected_episode_gap = None


if selected_episode_gap is not None:

    episode_support_selected = episode_support[
        episode_support["Gap_Tolerance_Minutes"]
        == selected_episode_gap
    ].copy()

else:
    episode_support_selected = episode_support.copy()


print(
    f"\nEpisode gap tolerance used for this audit: "
    f"{selected_episode_gap} minute(s)"
)


# --------------------------------------------------------------------------------------------------------------
# Aggregate forecasting support across horizons
# --------------------------------------------------------------------------------------------------------------

forecast_agg = (
    forecast_support
    .groupby("Attack_Type", as_index=False)
    .agg(
        Total_Valid_Forecasting_Windows=(
            "Valid_Forecasting_Windows",
            "sum"
        ),
        Total_Positive_Target_Windows=(
            "Positive_Target_Windows",
            "sum"
        ),
        Total_Benign_to_Attack_Windows=(
            "Current_Benign_to_Attack_Type",
            "sum"
        ),
        Total_Clean_History_Positive_Windows=(
            "Clean_15m_History_to_Attack_Type",
            "sum"
        ),
        Positive_Target_Segments=(
            "Positive_Target_Segments",
            "max"
        ),
        Clean_History_Positive_Segments=(
            "Clean_History_Positive_Segments",
            "max"
        ),
    )
)


# --------------------------------------------------------------------------------------------------------------
# Merge support tables
# --------------------------------------------------------------------------------------------------------------

episode_cols = [
    "Attack_Type",
    "Independent_Episodes",
    "Segments_With_Episodes",
    "Median_Attack_Minutes_Per_Episode",
    "Max_Attack_Minutes_Per_Episode",
]

episode_cols = [
    c for c in episode_cols
    if c in episode_support_selected.columns
]


type_cols = [
    "Attack_Type",
    "Attack_Minutes",
    "Segments_Present",
    "Dates_Present",
]

type_cols = [
    c for c in type_cols
    if c in type_support.columns
]


cell20_attack_type_support = (
    type_support[type_cols]
    .merge(
        episode_support_selected[episode_cols],
        on="Attack_Type",
        how="left"
    )
    .merge(
        forecast_agg,
        on="Attack_Type",
        how="left"
    )
)


# ==============================================================================================================
# 10. CREATE SUPPORT FLAGS
# ==============================================================================================================
#
# These are NOT claims that an attack type is "good" or "bad".
#
# They are transparent screening conditions for whether there is enough
# repeated temporal evidence to attempt independent type-level evaluation.
#
# Conditions:
#
#   >= 3 independent episodes
#   >= 2 segments
#   >= 2 dates
#   >= 20 attack minutes
#   >= 10 benign -> attack forecasting windows
#
# These thresholds can later be changed explicitly.
#

for col in [
    "Independent_Episodes",
    "Segments_With_Episodes",
    "Attack_Minutes",
    "Segments_Present",
    "Dates_Present",
    "Total_Benign_to_Attack_Windows",
    "Total_Clean_History_Positive_Windows",
]:

    if col in cell20_attack_type_support.columns:

        cell20_attack_type_support[col] = (
            pd.to_numeric(
                cell20_attack_type_support[col],
                errors="coerce"
            ).fillna(0)
        )


cell20_attack_type_support[
    "Repeated_Episode_Support"
] = (
    cell20_attack_type_support[
        "Independent_Episodes"
    ] >= 3
)


cell20_attack_type_support[
    "Multi_Segment_Support"
] = (
    cell20_attack_type_support[
        "Segments_Present"
    ] >= 2
)


if "Dates_Present" in cell20_attack_type_support.columns:

    cell20_attack_type_support[
        "Multi_Date_Support"
    ] = (
        cell20_attack_type_support[
            "Dates_Present"
        ] >= 2
    )

else:

    cell20_attack_type_support[
        "Multi_Date_Support"
    ] = False


cell20_attack_type_support[
    "Attack_Minute_Support"
] = (
    cell20_attack_type_support[
        "Attack_Minutes"
    ] >= 20
)


cell20_attack_type_support[
    "Transition_Forecast_Support"
] = (
    cell20_attack_type_support[
        "Total_Benign_to_Attack_Windows"
    ] >= 10
)


cell20_attack_type_support[
    "Candidate_For_Type_Forecasting"
] = (

    cell20_attack_type_support[
        "Repeated_Episode_Support"
    ]

    &

    cell20_attack_type_support[
        "Multi_Segment_Support"
    ]

    &

    cell20_attack_type_support[
        "Multi_Date_Support"
    ]

    &

    cell20_attack_type_support[
        "Attack_Minute_Support"
    ]

    &

    cell20_attack_type_support[
        "Transition_Forecast_Support"
    ]
)


# ==============================================================================================================
# 11. SORT ATTACK TYPES BY SUPPORT
# ==============================================================================================================

cell20_attack_type_support = (
    cell20_attack_type_support
    .sort_values(
        by=[
            "Candidate_For_Type_Forecasting",
            "Independent_Episodes",
            "Attack_Minutes",
            "Total_Benign_to_Attack_Windows",
        ],
        ascending=[
            False,
            False,
            False,
            False,
        ]
    )
    .reset_index(drop=True)
)


support_display_columns = [

    "Attack_Type",

    "Attack_Minutes",
    "Segments_Present",
    "Dates_Present",

    "Independent_Episodes",
    "Segments_With_Episodes",

    "Total_Positive_Target_Windows",
    "Total_Benign_to_Attack_Windows",
    "Total_Clean_History_Positive_Windows",

    "Repeated_Episode_Support",
    "Multi_Segment_Support",
    "Multi_Date_Support",
    "Attack_Minute_Support",
    "Transition_Forecast_Support",

    "Candidate_For_Type_Forecasting",
]

support_display_columns = [
    c for c in support_display_columns
    if c in cell20_attack_type_support.columns
]

print("\n")
print(
    cell20_attack_type_support[
        support_display_columns
    ].to_string(index=False)
)


# ==============================================================================================================
# 12. CANDIDATE ATTACK TYPES
# ==============================================================================================================

candidate_attack_types = (
    cell20_attack_type_support.loc[
        cell20_attack_type_support[
            "Candidate_For_Type_Forecasting"
        ],
        "Attack_Type"
    ]
    .tolist()
)


print("\n")
print("=" * 126)
print("ATTACK TYPES PASSING THE CURRENT SUPPORT SCREEN")
print("=" * 126)

if candidate_attack_types:

    for i, attack_type in enumerate(
        candidate_attack_types,
        start=1
    ):
        print(
            f"{i:>2}. {attack_type}"
        )

else:

    print(
        "\nNo attack type passes every current support condition."
    )

    print(
        "This does NOT invalidate binary attack-state forecasting."
    )


# ==============================================================================================================
# 13. CHECK WHICH SPLITS SUPPORT THE CANDIDATE ATTACK TYPES
# ==============================================================================================================

candidate_set = set(candidate_attack_types)

candidate_split_records = []

for _, row in cell20_split_audit.iterrows():

    common_types = set(
        row["Common_Type_Names"]
    )

    supported_candidates = (
        candidate_set & common_types
    )

    candidate_split_records.append({

        "Original_Split_Index":
            row["Original_Split_Index"],

        "Train_Segments":
            row["Train_Segments"],

        "Validation_Segments":
            row["Validation_Segments"],

        "Test_Segments":
            row["Test_Segments"],

        "Binary_State_Usable":
            row["Binary_State_Usable"],

        "Candidate_Types_In_All_Splits":
            len(supported_candidates),

        "Candidate_Type_Names":
            sorted(supported_candidates),

        "All_Candidate_Types_In_All_Splits":
            (
                len(candidate_set) > 0
                and candidate_set.issubset(
                    common_types
                )
            ),
    })


cell20_candidate_split_support = pd.DataFrame(
    candidate_split_records
)


print("\n")
print("=" * 126)
print("CANDIDATE ATTACK-TYPE COVERAGE ACROSS TEMPORAL SPLITS")
print("=" * 126)

print(
    cell20_candidate_split_support.to_string(
        index=False
    )
)


# ==============================================================================================================
# 14. SUMMARY STATISTICS
# ==============================================================================================================

binary_usable_count = int(
    cell20_split_audit[
        "Binary_State_Usable"
    ].sum()
)

type_usable_count = int(
    cell20_split_audit[
        "Type_Level_Usable"
    ].sum()
)

if len(candidate_attack_types) > 0:

    full_candidate_split_count = int(
        cell20_candidate_split_support[
            "All_Candidate_Types_In_All_Splits"
        ].sum()
    )

else:

    full_candidate_split_count = 0


print("\n")
print("=" * 126)
print("CELL 20 SUMMARY")
print("=" * 126)

print(
    f"\nValid chronological split candidates examined : "
    f"{len(cell20_split_audit):,}"
)

print(
    f"Binary-state usable splits                    : "
    f"{binary_usable_count:,}"
)

print(
    f"Splits with >=1 common attack type            : "
    f"{type_usable_count:,}"
)

print(
    f"Attack types passing support screen            : "
    f"{len(candidate_attack_types):,}"
)

print(
    f"Splits containing ALL screened attack types    : "
    f"{full_candidate_split_count:,}"
)


# ==============================================================================================================
# 15. FINAL AUDIT INTERPRETATION
# ==============================================================================================================

print("\n")
print("=" * 126)
print("INTERPRETATION")
print("=" * 126)

if binary_usable_count > 0:

    print(
        "\n✓ At least one chronological segment-level split "
        "supports binary future attack-state evaluation."
    )

else:

    print(
        "\n✗ No current chronological split contains attack "
        "activity in train, validation, and test."
    )


if len(candidate_attack_types) > 0:

    print(
        "\n✓ Some attack types have sufficient repeated temporal "
        "support to justify further type-level forecasting analysis."
    )

else:

    print(
        "\n⚠ No attack type currently passes all conservative "
        "type-level support conditions."
    )


if full_candidate_split_count > 0:

    print(
        "\n✓ At least one chronological split contains every "
        "screened candidate attack type in train, validation, and test."
    )

else:

    print(
        "\n⚠ No single split currently contains every screened "
        "candidate attack type across all three partitions."
    )


print(
    "\nIMPORTANT:"
)

print(
    "Binary future attack-state forecasting and attack-type forecasting "
    "should be treated as TWO related but distinct evaluation tasks."
)

print(
    "\nBinary task:"
)

print(
    "    past network sequence -> future benign / attack state"
)

print(
    "\nType-level task:"
)

print(
    "    past network sequence -> future attack category"
)

print(
    "\nThe type-level task should only use attack categories with "
    "sufficient temporal, episode, and split support."
)


print("\n")
print("=" * 126)
print("CELL 20 COMPLETE")
print("=" * 126)

print(
    "\nNO MODEL HAS BEEN TRAINED."
)

print(
    "NO RANDOM SPLIT HAS BEEN CREATED."
)

print(
    "NO FINAL TEMPORAL SPLIT HAS BEEN LOCKED."
)

print(
    "\nNEXT:"
)

print(
    "Inspect the Cell 20 results before choosing:"
)

print(
    "  1. final forecasting horizon(s)"
)

print(
    "  2. final attack types for type-level forecasting"
)

print(
    "  3. final chronological train / validation / test split"
)

print(
    "  4. sequence construction protocol"
)

print("=" * 126)

CELL 20 — EVALUATION PROTOCOL FEASIBILITY AUDIT

✓ Required structures found.

minute_df rows              : 2,454
segments                    : 10
attack types in type_support: 14
valid temporal splits       : 21


SEGMENT-LEVEL ATTACK COVERAGE
 Segment_ID        Start_Minute          End_Minute  Total_Minutes  Attack_Minutes  Benign_Minutes                                                             Attack_Types  Attack_Type_Count
          0 2017-07-03 01:00:00 2017-07-03 05:01:00            242               0             242                                                                       []                  0
          1 2017-07-03 08:55:00 2017-07-03 12:59:00            245               0             245                                                                       []                  0
          2 2017-07-04 01:00:00 2017-07-04 05:00:00            241              63             178                                                            [SSH-Patator]          

In [27]:
# ==============================================================================================================
# CELL 21 — FINAL CHRONOLOGICAL SPLIT SELECTION + BASELINE AUDIT
# ==============================================================================================================
#
# PURPOSE
# -------
# Select a defensible chronological train / validation / test split for the
# MAIN CICIDS2017 forecasting task:
#
#       past 15-minute network sequence
#                    ↓
#       future attack state at t + H
#
# Horizons:
#       H = 1, 5, 10, 15, 30 minutes
#
#
# THIS CELL WILL:
#
#   1. Reconstruct valid forecasting samples for every candidate split.
#   2. Preserve segment boundaries.
#   3. Measure train / validation / test target support at every horizon.
#   4. Measure:
#          - positive future attack targets
#          - negative future benign targets
#          - benign -> attack transitions
#          - attack -> attack continuation
#          - clean-history -> attack targets
#   5. Evaluate simple NON-LEARNED baselines:
#          - persistence baseline
#          - majority-class baseline
#   6. Measure attack-prevalence shift.
#   7. Measure unseen test attack types.
#   8. Screen candidate splits for forecasting feasibility.
#
#
# THIS CELL DOES NOT:
#
#   - train LSTM
#   - train NODE
#   - train LSTM-NODE
#   - scale features
#   - oversample
#   - undersample
#   - randomly shuffle samples
#   - permanently modify minute_df
#
# ==============================================================================================================


import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)


print("=" * 132)
print("CELL 21 — FINAL CHRONOLOGICAL SPLIT SELECTION + BASELINE AUDIT")
print("=" * 132)


# ==============================================================================================================
# 1. CONFIGURATION
# ==============================================================================================================

HISTORY_LENGTH = 15

HORIZONS = [
    1,
    5,
    10,
    15,
    30
]


# Minimum support conditions.
#
# These are feasibility screens, NOT statistical laws.
# We will print the actual counts as well so that the thresholds remain transparent.

MIN_POSITIVE_TARGETS_TRAIN = 20
MIN_POSITIVE_TARGETS_VAL   = 5
MIN_POSITIVE_TARGETS_TEST  = 5

MIN_ONSET_TARGETS_TRAIN = 5
MIN_ONSET_TARGETS_VAL   = 1
MIN_ONSET_TARGETS_TEST  = 1


print(f"\nHistory length : {HISTORY_LENGTH} minutes")
print(f"Horizons       : {HORIZONS}")

print("\nFeasibility screening thresholds:")

print(
    f"  Future-positive targets -> "
    f"train >= {MIN_POSITIVE_TARGETS_TRAIN}, "
    f"val >= {MIN_POSITIVE_TARGETS_VAL}, "
    f"test >= {MIN_POSITIVE_TARGETS_TEST}"
)

print(
    f"  Benign→attack targets   -> "
    f"train >= {MIN_ONSET_TARGETS_TRAIN}, "
    f"val >= {MIN_ONSET_TARGETS_VAL}, "
    f"test >= {MIN_ONSET_TARGETS_TEST}"
)


# ==============================================================================================================
# 2. VERIFY REQUIRED OBJECTS
# ==============================================================================================================

required_objects = [
    "minute_df",
    "valid_split_candidates",
    "cell20_segment_attack_coverage",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:

    raise RuntimeError(
        "Missing required objects:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_objects
        )
        + "\n\nRun Cell 20 before Cell 21."
    )


required_columns = [
    "Segment_ID",
    "Minute",
    "Attack_State",
]

missing_columns = [
    c
    for c in required_columns
    if c not in minute_df.columns
]

if missing_columns:

    raise RuntimeError(
        "minute_df is missing required columns:\n"
        + "\n".join(
            f"  - {c}"
            for c in missing_columns
        )
    )


print("\n✓ Required Cell 20 structures found.")


# ==============================================================================================================
# 3. SAFE SEGMENT COLLECTION PARSER
# ==============================================================================================================

def normalize_segment_collection(value):

    if isinstance(
        value,
        (
            list,
            tuple,
            set,
            np.ndarray
        )
    ):

        return sorted(
            [
                int(x)
                for x in value
            ]
        )

    if pd.isna(value):
        return []

    text = str(value).strip()

    text = (
        text
        .replace("[", "")
        .replace("]", "")
        .replace("(", "")
        .replace(")", "")
        .replace("{", "")
        .replace("}", "")
    )

    if text == "":
        return []

    result = []

    for item in text.replace(";", ",").split(","):

        item = item.strip()

        if item == "":
            continue

        try:

            result.append(
                int(
                    float(item)
                )
            )

        except Exception:
            pass

    return sorted(result)


# ==============================================================================================================
# 4. SAFE ATTACK-TYPE PARSER
# ==============================================================================================================

def parse_attack_types(value):

    if pd.isna(value):
        return set()

    if isinstance(
        value,
        (
            list,
            tuple,
            set,
            np.ndarray
        )
    ):

        values = value

    else:

        text = str(value).strip()

        if text == "":
            return set()

        text = (
            text
            .replace("[", "")
            .replace("]", "")
            .replace("{", "")
            .replace("}", "")
            .replace("'", "")
            .replace('"', "")
            .replace("|", ",")
            .replace(";", ",")
        )

        values = text.split(",")

    result = set()

    for value in values:

        value = str(value).strip()

        if (
            value
            and value.upper()
            not in {
                "BENIGN",
                "NORMAL",
                "NONE",
                "NAN"
            }
        ):

            result.add(value)

    return result


# ==============================================================================================================
# 5. PREPARE A TEMPORALLY SORTED WORKING COPY
# ==============================================================================================================

timeline = (
    minute_df
    .copy()
    .sort_values(
        [
            "Segment_ID",
            "Minute"
        ]
    )
    .reset_index(drop=True)
)


timeline["Attack_State"] = (
    pd.to_numeric(
        timeline["Attack_State"],
        errors="coerce"
    )
    .fillna(0)
    .astype(int)
)


print(
    f"\nTimeline rows : {len(timeline):,}"
)

print(
    f"Segments      : {timeline['Segment_ID'].nunique():,}"
)


# ==============================================================================================================
# 6. CONSTRUCT FORECASTING SAMPLE METADATA
# ==============================================================================================================
#
# IMPORTANT:
#
# We construct samples independently inside each segment.
#
# Therefore:
#
#     history never crosses a segment boundary
#     target never crosses a segment boundary
#
#
# For a row at local index i:
#
#     history = [i-HISTORY_LENGTH+1 ... i]
#
#     target = i + horizon
#
#
# Current_State:
#     attack state at time t
#
# Future_State:
#     attack state at t + H
#
# Clean_History:
#     all HISTORY_LENGTH minutes are benign
#
# New_Onset:
#     current state = benign
#     future state  = attack
#
# Continuation:
#     current state = attack
#     future state  = attack
#
# ==============================================================================================================

sample_records = []


for segment_id, segment in timeline.groupby(
    "Segment_ID",
    sort=True
):

    segment = (
        segment
        .sort_values("Minute")
        .reset_index(drop=True)
    )

    states = (
        segment["Attack_State"]
        .astype(int)
        .to_numpy()
    )

    minutes = (
        segment["Minute"]
        .to_numpy()
    )

    n = len(segment)


    for horizon in HORIZONS:

        last_valid_current_index = (
            n - horizon - 1
        )


        for current_idx in range(
            HISTORY_LENGTH - 1,
            last_valid_current_index + 1
        ):

            history_start_idx = (
                current_idx
                - HISTORY_LENGTH
                + 1
            )

            target_idx = (
                current_idx
                + horizon
            )


            history_states = states[
                history_start_idx:
                current_idx + 1
            ]


            current_state = int(
                states[current_idx]
            )

            future_state = int(
                states[target_idx]
            )


            clean_history = int(
                np.sum(history_states) == 0
            )


            new_onset = int(
                current_state == 0
                and future_state == 1
            )


            continuation = int(
                current_state == 1
                and future_state == 1
            )


            attack_to_benign = int(
                current_state == 1
                and future_state == 0
            )


            benign_to_benign = int(
                current_state == 0
                and future_state == 0
            )


            sample_records.append({

                "Segment_ID":
                    int(segment_id),

                "Horizon":
                    int(horizon),

                "History_Start":
                    pd.Timestamp(
                        minutes[
                            history_start_idx
                        ]
                    ),

                "Current_Time":
                    pd.Timestamp(
                        minutes[
                            current_idx
                        ]
                    ),

                "Target_Time":
                    pd.Timestamp(
                        minutes[
                            target_idx
                        ]
                    ),

                "Current_State":
                    current_state,

                "Future_State":
                    future_state,

                "Clean_History":
                    clean_history,

                "New_Onset":
                    new_onset,

                "Continuation":
                    continuation,

                "Attack_to_Benign":
                    attack_to_benign,

                "Benign_to_Benign":
                    benign_to_benign,
            })


cell21_samples = pd.DataFrame(
    sample_records
)


print("\n")
print("=" * 132)
print("FORECASTING SAMPLE RECONSTRUCTION")
print("=" * 132)

print(
    f"\nTotal horizon-specific samples : "
    f"{len(cell21_samples):,}"
)


sample_summary = (
    cell21_samples
    .groupby(
        "Horizon",
        as_index=False
    )
    .agg(

        Samples=(
            "Future_State",
            "size"
        ),

        Future_Attack=(
            "Future_State",
            "sum"
        ),

        New_Onset=(
            "New_Onset",
            "sum"
        ),

        Continuation=(
            "Continuation",
            "sum"
        ),

        Clean_History=(
            "Clean_History",
            "sum"
        )
    )
)


print("\n")
print(
    sample_summary.to_string(
        index=False
    )
)


# ==============================================================================================================
# 7. VERIFY AGAINST CELL 12 LOGIC
# ==============================================================================================================
#
# These totals should be consistent with the earlier general forecasting audit.
#
# This is an important sanity check before split selection.
#

expected_total_samples = {
    1: 2304,
    5: 2264,
    10: 2214,
    15: 2164,
    30: 2014,
}

expected_positive_targets = {
    1: 518,
    5: 516,
    10: 511,
    15: 502,
    30: 470,
}


validation_records = []


for horizon in HORIZONS:

    subset = cell21_samples[
        cell21_samples["Horizon"]
        == horizon
    ]

    actual_samples = len(
        subset
    )

    actual_positive = int(
        subset["Future_State"].sum()
    )


    expected_samples = (
        expected_total_samples.get(
            horizon
        )
    )

    expected_positive = (
        expected_positive_targets.get(
            horizon
        )
    )


    validation_records.append({

        "Horizon":
            horizon,

        "Actual_Samples":
            actual_samples,

        "Expected_Cell12_Samples":
            expected_samples,

        "Sample_Count_Match":
            actual_samples
            == expected_samples,

        "Actual_Positive":
            actual_positive,

        "Expected_Cell12_Positive":
            expected_positive,

        "Positive_Count_Match":
            actual_positive
            == expected_positive,
    })


cell21_reconstruction_validation = pd.DataFrame(
    validation_records
)


print("\n")
print("=" * 132)
print("CELL 12 RECONSTRUCTION SANITY CHECK")
print("=" * 132)

print(
    cell21_reconstruction_validation
    .to_string(
        index=False
    )
)


all_reconstruction_valid = bool(

    cell21_reconstruction_validation[
        "Sample_Count_Match"
    ].all()

    and

    cell21_reconstruction_validation[
        "Positive_Count_Match"
    ].all()
)


print(
    "\nExact reconstruction match:",
    all_reconstruction_valid
)


if not all_reconstruction_valid:

    raise RuntimeError(
        "\nCell 21 forecasting reconstruction does not match Cell 12.\n"
        "STOP HERE. Do not select a split until the discrepancy is investigated."
    )


# ==============================================================================================================
# 8. BASELINE METRICS FUNCTION
# ==============================================================================================================

def calculate_binary_metrics(
    y_true,
    y_pred
):

    y_true = np.asarray(
        y_true,
        dtype=int
    )

    y_pred = np.asarray(
        y_pred,
        dtype=int
    )


    if len(y_true) == 0:

        return {
            "Accuracy": np.nan,
            "Balanced_Accuracy": np.nan,
            "Precision": np.nan,
            "Recall": np.nan,
            "F1": np.nan,
            "TN": 0,
            "FP": 0,
            "FN": 0,
            "TP": 0,
        }


    tn, fp, fn, tp = (
        confusion_matrix(
            y_true,
            y_pred,
            labels=[0, 1]
        )
        .ravel()
    )


    return {

        "Accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "Balanced_Accuracy":
            balanced_accuracy_score(
                y_true,
                y_pred
            ),

        "Precision":
            precision_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "Recall":
            recall_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "F1":
            f1_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "TN":
            int(tn),

        "FP":
            int(fp),

        "FN":
            int(fn),

        "TP":
            int(tp),
    }


# ==============================================================================================================
# 9. GET ATTACK TYPES FOR SEGMENT COLLECTION
# ==============================================================================================================

def attack_types_for_segments(
    segment_list
):

    subset = (
        cell20_segment_attack_coverage[
            cell20_segment_attack_coverage[
                "Segment_ID"
            ].isin(
                segment_list
            )
        ]
    )

    result = set()

    for values in subset["Attack_Types"]:

        if isinstance(
            values,
            (
                list,
                tuple,
                set,
                np.ndarray
            )
        ):

            result.update(
                values
            )

        else:

            result.update(
                parse_attack_types(
                    values
                )
            )

    return result


# ==============================================================================================================
# 10. AUDIT EVERY SPLIT AT EVERY HORIZON
# ==============================================================================================================

split_horizon_records = []


for split_index, split_row in (
    valid_split_candidates
    .iterrows()
):


    train_segments = (
        normalize_segment_collection(
            split_row[
                "Train_Segments"
            ]
        )
    )


    val_segments = (
        normalize_segment_collection(
            split_row[
                "Validation_Segments"
            ]
        )
    )


    test_segments = (
        normalize_segment_collection(
            split_row[
                "Test_Segments"
            ]
        )
    )


    train_types = (
        attack_types_for_segments(
            train_segments
        )
    )

    val_types = (
        attack_types_for_segments(
            val_segments
        )
    )

    test_types = (
        attack_types_for_segments(
            test_segments
        )
    )


    unseen_test_types = (
        test_types
        - train_types
    )


    for horizon in HORIZONS:


        horizon_samples = (
            cell21_samples[
                cell21_samples[
                    "Horizon"
                ]
                == horizon
            ]
        )


        partition_data = {}


        for partition_name, segments in [

            (
                "Train",
                train_segments
            ),

            (
                "Validation",
                val_segments
            ),

            (
                "Test",
                test_segments
            ),

        ]:


            subset = (
                horizon_samples[
                    horizon_samples[
                        "Segment_ID"
                    ].isin(
                        segments
                    )
                ]
                .copy()
            )


            total = len(
                subset
            )


            positive = int(
                subset[
                    "Future_State"
                ].sum()
            )


            negative = int(
                total
                - positive
            )


            onset = int(
                subset[
                    "New_Onset"
                ].sum()
            )


            continuation = int(
                subset[
                    "Continuation"
                ].sum()
            )


            clean_history_positive = int(

                (
                    (
                        subset[
                            "Clean_History"
                        ]
                        == 1
                    )

                    &

                    (
                        subset[
                            "Future_State"
                        ]
                        == 1
                    )
                )
                .sum()
            )


            prevalence = (

                positive
                / total

                if total > 0

                else np.nan
            )


            partition_data[
                partition_name
            ] = {

                "subset":
                    subset,

                "total":
                    total,

                "positive":
                    positive,

                "negative":
                    negative,

                "onset":
                    onset,

                "continuation":
                    continuation,

                "clean_history_positive":
                    clean_history_positive,

                "prevalence":
                    prevalence,
            }


        # ------------------------------------------------------------------------------------------------------
        # Baselines are evaluated on validation/test.
        #
        # Persistence:
        #
        #       prediction(t + H) = state(t)
        #
        #
        # Majority:
        #
        #       predict the majority TARGET class observed in TRAINING
        #
        # This avoids peeking at validation/test labels.
        # ------------------------------------------------------------------------------------------------------

        train_subset = (
            partition_data[
                "Train"
            ]["subset"]
        )


        if len(train_subset) > 0:

            train_majority_class = int(

                train_subset[
                    "Future_State"
                ]
                .mean()
                >= 0.5
            )

        else:

            train_majority_class = 0


        baseline_results = {}


        for eval_partition in [
            "Validation",
            "Test"
        ]:


            eval_subset = (
                partition_data[
                    eval_partition
                ]["subset"]
            )


            y_true = (
                eval_subset[
                    "Future_State"
                ]
                .astype(int)
                .to_numpy()
            )


            persistence_pred = (
                eval_subset[
                    "Current_State"
                ]
                .astype(int)
                .to_numpy()
            )


            majority_pred = (
                np.full(
                    len(eval_subset),
                    train_majority_class,
                    dtype=int
                )
            )


            persistence_metrics = (
                calculate_binary_metrics(
                    y_true,
                    persistence_pred
                )
            )


            majority_metrics = (
                calculate_binary_metrics(
                    y_true,
                    majority_pred
                )
            )


            baseline_results[
                eval_partition
            ] = {

                "Persistence":
                    persistence_metrics,

                "Majority":
                    majority_metrics,
            }


        # ------------------------------------------------------------------------------------------------------
        # Feasibility screen
        # ------------------------------------------------------------------------------------------------------

        train_positive_ok = (
            partition_data[
                "Train"
            ]["positive"]
            >= MIN_POSITIVE_TARGETS_TRAIN
        )


        val_positive_ok = (
            partition_data[
                "Validation"
            ]["positive"]
            >= MIN_POSITIVE_TARGETS_VAL
        )


        test_positive_ok = (
            partition_data[
                "Test"
            ]["positive"]
            >= MIN_POSITIVE_TARGETS_TEST
        )


        train_onset_ok = (
            partition_data[
                "Train"
            ]["onset"]
            >= MIN_ONSET_TARGETS_TRAIN
        )


        val_onset_ok = (
            partition_data[
                "Validation"
            ]["onset"]
            >= MIN_ONSET_TARGETS_VAL
        )


        test_onset_ok = (
            partition_data[
                "Test"
            ]["onset"]
            >= MIN_ONSET_TARGETS_TEST
        )


        horizon_feasible = (

            train_positive_ok
            and val_positive_ok
            and test_positive_ok

            and

            train_onset_ok
            and val_onset_ok
            and test_onset_ok
        )


        train_prev = (
            partition_data[
                "Train"
            ]["prevalence"]
        )


        val_prev = (
            partition_data[
                "Validation"
            ]["prevalence"]
        )


        test_prev = (
            partition_data[
                "Test"
            ]["prevalence"]
        )


        train_test_shift = (

            abs(
                train_prev
                - test_prev
            )

            if (
                not np.isnan(
                    train_prev
                )

                and

                not np.isnan(
                    test_prev
                )
            )

            else np.nan
        )


        split_horizon_records.append({

            "Original_Split_Index":
                split_index,

            "Horizon":
                horizon,

            "Train_Segments":
                train_segments,

            "Validation_Segments":
                val_segments,

            "Test_Segments":
                test_segments,


            # TRAIN
            "Train_Samples":
                partition_data[
                    "Train"
                ]["total"],

            "Train_Positive":
                partition_data[
                    "Train"
                ]["positive"],

            "Train_Negative":
                partition_data[
                    "Train"
                ]["negative"],

            "Train_Onset":
                partition_data[
                    "Train"
                ]["onset"],

            "Train_Continuation":
                partition_data[
                    "Train"
                ]["continuation"],

            "Train_CleanHistory_Positive":
                partition_data[
                    "Train"
                ][
                    "clean_history_positive"
                ],

            "Train_Prevalence":
                train_prev,


            # VALIDATION
            "Validation_Samples":
                partition_data[
                    "Validation"
                ]["total"],

            "Validation_Positive":
                partition_data[
                    "Validation"
                ]["positive"],

            "Validation_Negative":
                partition_data[
                    "Validation"
                ]["negative"],

            "Validation_Onset":
                partition_data[
                    "Validation"
                ]["onset"],

            "Validation_Continuation":
                partition_data[
                    "Validation"
                ]["continuation"],

            "Validation_CleanHistory_Positive":
                partition_data[
                    "Validation"
                ][
                    "clean_history_positive"
                ],

            "Validation_Prevalence":
                val_prev,


            # TEST
            "Test_Samples":
                partition_data[
                    "Test"
                ]["total"],

            "Test_Positive":
                partition_data[
                    "Test"
                ]["positive"],

            "Test_Negative":
                partition_data[
                    "Test"
                ]["negative"],

            "Test_Onset":
                partition_data[
                    "Test"
                ]["onset"],

            "Test_Continuation":
                partition_data[
                    "Test"
                ]["continuation"],

            "Test_CleanHistory_Positive":
                partition_data[
                    "Test"
                ][
                    "clean_history_positive"
                ],

            "Test_Prevalence":
                test_prev,


            # SHIFT
            "Train_Test_Prevalence_Shift":
                train_test_shift,


            # ATTACK TYPES
            "Train_Attack_Types":
                sorted(
                    train_types
                ),

            "Validation_Attack_Types":
                sorted(
                    val_types
                ),

            "Test_Attack_Types":
                sorted(
                    test_types
                ),

            "Unseen_Test_Attack_Types":
                sorted(
                    unseen_test_types
                ),

            "Unseen_Test_Attack_Type_Count":
                len(
                    unseen_test_types
                ),


            # BASELINES — VALIDATION
            "Val_Persistence_Accuracy":
                baseline_results[
                    "Validation"
                ][
                    "Persistence"
                ][
                    "Accuracy"
                ],

            "Val_Persistence_Balanced_Accuracy":
                baseline_results[
                    "Validation"
                ][
                    "Persistence"
                ][
                    "Balanced_Accuracy"
                ],

            "Val_Persistence_F1":
                baseline_results[
                    "Validation"
                ][
                    "Persistence"
                ][
                    "F1"
                ],

            "Val_Persistence_Recall":
                baseline_results[
                    "Validation"
                ][
                    "Persistence"
                ][
                    "Recall"
                ],

            "Val_Majority_Accuracy":
                baseline_results[
                    "Validation"
                ][
                    "Majority"
                ][
                    "Accuracy"
                ],

            "Val_Majority_Balanced_Accuracy":
                baseline_results[
                    "Validation"
                ][
                    "Majority"
                ][
                    "Balanced_Accuracy"
                ],


            # BASELINES — TEST
            "Test_Persistence_Accuracy":
                baseline_results[
                    "Test"
                ][
                    "Persistence"
                ][
                    "Accuracy"
                ],

            "Test_Persistence_Balanced_Accuracy":
                baseline_results[
                    "Test"
                ][
                    "Persistence"
                ][
                    "Balanced_Accuracy"
                ],

            "Test_Persistence_F1":
                baseline_results[
                    "Test"
                ][
                    "Persistence"
                ][
                    "F1"
                ],

            "Test_Persistence_Recall":
                baseline_results[
                    "Test"
                ][
                    "Persistence"
                ][
                    "Recall"
                ],

            "Test_Majority_Accuracy":
                baseline_results[
                    "Test"
                ][
                    "Majority"
                ][
                    "Accuracy"
                ],

            "Test_Majority_Balanced_Accuracy":
                baseline_results[
                    "Test"
                ][
                    "Majority"
                ][
                    "Balanced_Accuracy"
                ],


            # SCREEN
            "Horizon_Feasible":
                horizon_feasible,
        })


cell21_split_horizon_audit = pd.DataFrame(
    split_horizon_records
)


# ==============================================================================================================
# 11. DISPLAY HORIZON-SPECIFIC SUPPORT
# ==============================================================================================================

print("\n")
print("=" * 132)
print("HORIZON-SPECIFIC SPLIT SUPPORT")
print("=" * 132)


display_columns = [

    "Original_Split_Index",
    "Horizon",

    "Train_Samples",
    "Train_Positive",
    "Train_Onset",

    "Validation_Samples",
    "Validation_Positive",
    "Validation_Onset",

    "Test_Samples",
    "Test_Positive",
    "Test_Onset",

    "Train_Test_Prevalence_Shift",

    "Test_Persistence_Balanced_Accuracy",
    "Test_Majority_Balanced_Accuracy",

    "Horizon_Feasible",
]


print(
    cell21_split_horizon_audit[
        display_columns
    ]
    .to_string(
        index=False
    )
)


# ==============================================================================================================
# 12. SUMMARIZE EACH SPLIT ACROSS ALL FIVE HORIZONS
# ==============================================================================================================

split_summary_records = []


for split_index, group in (
    cell21_split_horizon_audit
    .groupby(
        "Original_Split_Index",
        sort=False
    )
):


    first = (
        group.iloc[0]
    )


    horizons_feasible = int(
        group[
            "Horizon_Feasible"
        ].sum()
    )


    all_horizons_feasible = bool(
        group[
            "Horizon_Feasible"
        ].all()
    )


    minimum_train_positive = int(
        group[
            "Train_Positive"
        ].min()
    )


    minimum_val_positive = int(
        group[
            "Validation_Positive"
        ].min()
    )


    minimum_test_positive = int(
        group[
            "Test_Positive"
        ].min()
    )


    minimum_train_onset = int(
        group[
            "Train_Onset"
        ].min()
    )


    minimum_val_onset = int(
        group[
            "Validation_Onset"
        ].min()
    )


    minimum_test_onset = int(
        group[
            "Test_Onset"
        ].min()
    )


    mean_prevalence_shift = float(
        group[
            "Train_Test_Prevalence_Shift"
        ].mean()
    )


    max_prevalence_shift = float(
        group[
            "Train_Test_Prevalence_Shift"
        ].max()
    )


    mean_test_persistence_bal_acc = float(
        group[
            "Test_Persistence_Balanced_Accuracy"
        ].mean()
    )


    mean_test_persistence_f1 = float(
        group[
            "Test_Persistence_F1"
        ].mean()
    )


    total_train_samples = int(
        group[
            "Train_Samples"
        ].sum()
    )


    total_validation_samples = int(
        group[
            "Validation_Samples"
        ].sum()
    )


    total_test_samples = int(
        group[
            "Test_Samples"
        ].sum()
    )


    split_summary_records.append({

        "Original_Split_Index":
            split_index,

        "Train_Segments":
            first[
                "Train_Segments"
            ],

        "Validation_Segments":
            first[
                "Validation_Segments"
            ],

        "Test_Segments":
            first[
                "Test_Segments"
            ],

        "Horizons_Feasible":
            horizons_feasible,

        "All_5_Horizons_Feasible":
            all_horizons_feasible,

        "Min_Train_Positive":
            minimum_train_positive,

        "Min_Validation_Positive":
            minimum_val_positive,

        "Min_Test_Positive":
            minimum_test_positive,

        "Min_Train_Onset":
            minimum_train_onset,

        "Min_Validation_Onset":
            minimum_val_onset,

        "Min_Test_Onset":
            minimum_test_onset,

        "Mean_Train_Test_Prevalence_Shift":
            mean_prevalence_shift,

        "Max_Train_Test_Prevalence_Shift":
            max_prevalence_shift,

        "Mean_Test_Persistence_Balanced_Accuracy":
            mean_test_persistence_bal_acc,

        "Mean_Test_Persistence_F1":
            mean_test_persistence_f1,

        "Total_Train_Horizon_Samples":
            total_train_samples,

        "Total_Validation_Horizon_Samples":
            total_validation_samples,

        "Total_Test_Horizon_Samples":
            total_test_samples,

        "Unseen_Test_Attack_Type_Count":
            first[
                "Unseen_Test_Attack_Type_Count"
            ],

        "Unseen_Test_Attack_Types":
            first[
                "Unseen_Test_Attack_Types"
            ],
    })


cell21_split_summary = pd.DataFrame(
    split_summary_records
)


# ==============================================================================================================
# 13. SCREEN FULLY FEASIBLE SPLITS
# ==============================================================================================================

cell21_fully_feasible_splits = (
    cell21_split_summary[
        cell21_split_summary[
            "All_5_Horizons_Feasible"
        ]
    ]
    .copy()
)


print("\n")
print("=" * 132)
print("SPLIT SUMMARY ACROSS ALL FIVE HORIZONS")
print("=" * 132)


summary_display_columns = [

    "Original_Split_Index",

    "Train_Segments",
    "Validation_Segments",
    "Test_Segments",

    "Horizons_Feasible",
    "All_5_Horizons_Feasible",

    "Min_Train_Positive",
    "Min_Validation_Positive",
    "Min_Test_Positive",

    "Min_Train_Onset",
    "Min_Validation_Onset",
    "Min_Test_Onset",

    "Mean_Train_Test_Prevalence_Shift",

    "Mean_Test_Persistence_Balanced_Accuracy",

    "Unseen_Test_Attack_Type_Count",
]


print(
    cell21_split_summary[
        summary_display_columns
    ]
    .sort_values(
        by=[
            "All_5_Horizons_Feasible",
            "Min_Test_Onset",
            "Min_Validation_Onset",
            "Min_Train_Onset",
        ],
        ascending=[
            False,
            False,
            False,
            False,
        ]
    )
    .to_string(
        index=False
    )
)


# ==============================================================================================================
# 14. SHOW FULLY FEASIBLE CANDIDATES
# ==============================================================================================================

print("\n")
print("=" * 132)
print("FULLY FEASIBLE SPLITS")
print("=" * 132)


if len(
    cell21_fully_feasible_splits
) == 0:

    print(
        "\nNo split passes the minimum support conditions "
        "for all five horizons."
    )


else:

    print(
        f"\nFully feasible splits: "
        f"{len(cell21_fully_feasible_splits):,}"
    )


    feasible_display = [

        "Original_Split_Index",

        "Train_Segments",
        "Validation_Segments",
        "Test_Segments",

        "Min_Train_Positive",
        "Min_Validation_Positive",
        "Min_Test_Positive",

        "Min_Train_Onset",
        "Min_Validation_Onset",
        "Min_Test_Onset",

        "Mean_Train_Test_Prevalence_Shift",

        "Mean_Test_Persistence_Balanced_Accuracy",

        "Unseen_Test_Attack_Type_Count",
    ]


    print("\n")

    print(
        cell21_fully_feasible_splits[
            feasible_display
        ]
        .sort_values(
            by=[
                "Min_Test_Onset",
                "Min_Validation_Onset",
                "Min_Train_Onset",
            ],
            ascending=[
                False,
                False,
                False,
            ]
        )
        .to_string(
            index=False
        )
    )


# ==============================================================================================================
# 15. CREATE A TRANSPARENT SUPPORT SCORE
# ==============================================================================================================
#
# IMPORTANT:
#
# This score is ONLY a screening aid.
#
# It is NOT automatically treated as the scientifically "best" split.
#
#
# We reward:
#
#   - feasibility across horizons
#   - train onset support
#   - validation onset support
#   - test onset support
#
# We lightly penalize:
#
#   - very large train/test prevalence differences
#
#
# The purpose is to surface strong candidates for manual inspection.
# ==============================================================================================================


ranking_df = (
    cell21_split_summary
    .copy()
)


def minmax(series):

    series = (
        pd.to_numeric(
            series,
            errors="coerce"
        )
    )

    minimum = (
        series.min()
    )

    maximum = (
        series.max()
    )


    if (
        pd.isna(minimum)
        or pd.isna(maximum)
        or maximum == minimum
    ):

        return pd.Series(
            np.ones(
                len(series)
            ),
            index=series.index
        )


    return (
        (series - minimum)
        /
        (maximum - minimum)
    )


ranking_df[
    "Norm_Train_Onset"
] = minmax(
    ranking_df[
        "Min_Train_Onset"
    ]
)


ranking_df[
    "Norm_Val_Onset"
] = minmax(
    ranking_df[
        "Min_Validation_Onset"
    ]
)


ranking_df[
    "Norm_Test_Onset"
] = minmax(
    ranking_df[
        "Min_Test_Onset"
    ]
)


ranking_df[
    "Norm_Prevalence_Shift"
] = minmax(
    ranking_df[
        "Mean_Train_Test_Prevalence_Shift"
    ]
)


ranking_df[
    "Support_Score"
] = (

    0.20
    * ranking_df[
        "Norm_Train_Onset"
    ]

    +

    0.30
    * ranking_df[
        "Norm_Val_Onset"
    ]

    +

    0.40
    * ranking_df[
        "Norm_Test_Onset"
    ]

    +

    0.10
    * (
        1
        - ranking_df[
            "Norm_Prevalence_Shift"
        ]
    )
)


# Non-fully-feasible splits should never outrank a fully feasible split.

ranking_df.loc[
    ~ranking_df[
        "All_5_Horizons_Feasible"
    ],
    "Support_Score"
] = -1


cell21_ranked_candidates = (
    ranking_df
    .sort_values(
        by=[
            "Support_Score",
            "Min_Test_Onset",
            "Min_Validation_Onset",
        ],
        ascending=[
            False,
            False,
            False,
        ]
    )
    .reset_index(drop=True)
)


print("\n")
print("=" * 132)
print("TOP SPLIT CANDIDATES — SCREENING ONLY")
print("=" * 132)


ranking_display = [

    "Original_Split_Index",

    "Train_Segments",
    "Validation_Segments",
    "Test_Segments",

    "Min_Train_Onset",
    "Min_Validation_Onset",
    "Min_Test_Onset",

    "Min_Test_Positive",

    "Mean_Train_Test_Prevalence_Shift",

    "Mean_Test_Persistence_Balanced_Accuracy",

    "Unseen_Test_Attack_Type_Count",

    "Support_Score",
]


print(
    cell21_ranked_candidates[
        ranking_display
    ]
    .head(10)
    .to_string(
        index=False
    )
)


# ==============================================================================================================
# 16. IDENTIFY LEADING CANDIDATE — DO NOT LOCK YET
# ==============================================================================================================

if (
    len(
        cell21_ranked_candidates
    ) > 0

    and

    cell21_ranked_candidates.iloc[0][
        "Support_Score"
    ] >= 0
):


    leading_candidate = (
        cell21_ranked_candidates
        .iloc[0]
    )


    cell21_leading_split_index = (
        leading_candidate[
            "Original_Split_Index"
        ]
    )


    cell21_leading_train_segments = (
        leading_candidate[
            "Train_Segments"
        ]
    )


    cell21_leading_validation_segments = (
        leading_candidate[
            "Validation_Segments"
        ]
    )


    cell21_leading_test_segments = (
        leading_candidate[
            "Test_Segments"
        ]
    )


    print("\n")
    print("=" * 132)
    print("LEADING CANDIDATE FOR MANUAL REVIEW")
    print("=" * 132)


    print(
        f"\nOriginal split index : "
        f"{cell21_leading_split_index}"
    )


    print(
        f"Train segments        : "
        f"{cell21_leading_train_segments}"
    )


    print(
        f"Validation segments   : "
        f"{cell21_leading_validation_segments}"
    )


    print(
        f"Test segments         : "
        f"{cell21_leading_test_segments}"
    )


    print(
        "\nIMPORTANT: this split has NOT been permanently locked."
    )


else:

    cell21_leading_split_index = None

    cell21_leading_train_segments = None

    cell21_leading_validation_segments = None

    cell21_leading_test_segments = None


    print("\n")
    print("=" * 132)
    print("NO FULLY FEASIBLE LEADING SPLIT")
    print("=" * 132)


# ==============================================================================================================
# 17. DETAILED LEADING-CANDIDATE HORIZON TABLE
# ==============================================================================================================

if (
    cell21_leading_split_index
    is not None
):


    leading_horizon_details = (
        cell21_split_horizon_audit[
            cell21_split_horizon_audit[
                "Original_Split_Index"
            ]
            ==
            cell21_leading_split_index
        ]
        .copy()
    )


    print("\n")
    print("=" * 132)
    print("LEADING CANDIDATE — HORIZON-BY-HORIZON DETAILS")
    print("=" * 132)


    detail_columns = [

        "Horizon",

        "Train_Samples",
        "Train_Positive",
        "Train_Onset",
        "Train_Continuation",

        "Validation_Samples",
        "Validation_Positive",
        "Validation_Onset",
        "Validation_Continuation",

        "Test_Samples",
        "Test_Positive",
        "Test_Onset",
        "Test_Continuation",

        "Test_Persistence_Accuracy",
        "Test_Persistence_Balanced_Accuracy",
        "Test_Persistence_F1",
        "Test_Persistence_Recall",

        "Test_Majority_Accuracy",
        "Test_Majority_Balanced_Accuracy",

        "Horizon_Feasible",
    ]


    print(
        leading_horizon_details[
            detail_columns
        ]
        .to_string(
            index=False
        )
    )


# ==============================================================================================================
# 18. BASELINE INTERPRETATION
# ==============================================================================================================

print("\n")
print("=" * 132)
print("BASELINE INTERPRETATION")
print("=" * 132)


print(
    """
Persistence baseline:

    predicted future state = current state

This is extremely important for our task.

If attacks last for several consecutive minutes, persistence can obtain
apparently strong results without actually learning early-warning behaviour.

Therefore the learned models must NOT be evaluated using accuracy alone.


Primary model evaluation should later include:

    • ROC-AUC
    • PR-AUC
    • balanced accuracy
    • precision
    • recall
    • F1
    • confusion matrix


And, critically, we should separately report:

    • benign -> attack transition recall
    • attack -> attack continuation performance
    • clean-history -> future attack performance


This lets us distinguish:

    "the model knows an attack is already happening"

from:

    "the model anticipates a future attack state."
"""
)


# ==============================================================================================================
# 19. FINAL STATUS
# ==============================================================================================================

print("\n")
print("=" * 132)
print("CELL 21 COMPLETE")
print("=" * 132)


print(
    f"\nCandidate chronological splits examined : "
    f"{cell21_split_summary.shape[0]:,}"
)


print(
    f"Fully feasible across all 5 horizons    : "
    f"{cell21_fully_feasible_splits.shape[0]:,}"
)


if (
    cell21_leading_split_index
    is not None
):

    print(
        f"Leading candidate split                 : "
        f"{cell21_leading_split_index}"
    )

else:

    print(
        "Leading candidate split                 : NONE"
    )


print(
    "\nNO MODEL HAS BEEN TRAINED."
)

print(
    "NO FEATURE SCALER HAS BEEN FIT."
)

print(
    "NO OVERSAMPLING HAS BEEN PERFORMED."
)

print(
    "NO FINAL SPLIT HAS BEEN LOCKED."
)


print(
    "\nNEXT STEP:"
)

print(
    "Inspect the Cell 21 output."
)

print(
    "If the leading split is defensible, the next cell will LOCK the split "
    "and construct leakage-safe train / validation / test sequences."
)

print("=" * 132)

CELL 21 — FINAL CHRONOLOGICAL SPLIT SELECTION + BASELINE AUDIT

History length : 15 minutes
Horizons       : [1, 5, 10, 15, 30]

Feasibility screening thresholds:
  Future-positive targets -> train >= 20, val >= 5, test >= 5
  Benign→attack targets   -> train >= 5, val >= 1, test >= 1

✓ Required Cell 20 structures found.

Timeline rows : 2,454
Segments      : 10


FORECASTING SAMPLE RECONSTRUCTION

Total horizon-specific samples : 10,960


 Horizon  Samples  Future_Attack  New_Onset  Continuation  Clean_History
       1     2304            518         81           437           1452
       5     2264            516        107           409           1416
      10     2214            511        131           380           1371
      15     2164            502        170           332           1326
      30     2014            470        207           263           1191


CELL 12 RECONSTRUCTION SANITY CHECK
 Horizon  Actual_Samples  Expected_Cell12_Samples  Sample_Count_Match  Actual_P

In [28]:
# ==============================================================================================================
# CELL 22 — FINAL CHRONOLOGICAL SPLIT LOCK + INTEGRITY VALIDATION
# ==============================================================================================================

import numpy as np
import pandas as pd

print("=" * 126)
print("CELL 22 — FINAL CHRONOLOGICAL SPLIT LOCK + INTEGRITY VALIDATION")
print("=" * 126)

# --------------------------------------------------------------------------------------------------------------
# CONFIGURATION
# --------------------------------------------------------------------------------------------------------------

HISTORY_LENGTH = 15
HORIZONS = [1, 5, 10, 15, 30]

# Final split selected after Cell 21 screening
FINAL_SPLIT_INDEX = 26

TRAIN_SEGMENTS = [0, 1, 2, 3, 4]
VAL_SEGMENTS   = [5]
TEST_SEGMENTS  = [6, 7, 8, 9]

print("\nFINAL SPLIT")
print("-" * 110)
print("Cell 21 candidate index :", FINAL_SPLIT_INDEX)
print("Train segments          :", TRAIN_SEGMENTS)
print("Validation segments     :", VAL_SEGMENTS)
print("Test segments           :", TEST_SEGMENTS)

# --------------------------------------------------------------------------------------------------------------
# FIND TIMELINE DATAFRAME
# --------------------------------------------------------------------------------------------------------------

candidate_names = [
    "minute_df",
    "timeline",
    "minute_timeline",
    "temporal_df",
]

timeline_df = None
timeline_name = None

for name in candidate_names:
    if name in globals() and isinstance(globals()[name], pd.DataFrame):
        obj = globals()[name]

        if (
            "Segment_ID" in obj.columns
            and "Attack_State" in obj.columns
        ):
            timeline_df = obj.copy()
            timeline_name = name
            break

if timeline_df is None:
    raise RuntimeError(
        "Could not locate the minute-level forecasting dataframe.\n"
        "Expected a DataFrame such as minute_df containing:\n"
        "  - Segment_ID\n"
        "  - Attack_State\n\n"
        "Run the previous preprocessing cells first."
    )

print("\n✓ Timeline dataframe found:", timeline_name)
print("Timeline rows             :", len(timeline_df))

# --------------------------------------------------------------------------------------------------------------
# IDENTIFY TIMESTAMP COLUMN
# --------------------------------------------------------------------------------------------------------------

timestamp_candidates = [
    "Timestamp",
    "timestamp",
    "Minute",
    "minute",
    "Datetime",
    "datetime",
]

TIME_COL = None

for col in timestamp_candidates:
    if col in timeline_df.columns:
        TIME_COL = col
        break

if TIME_COL is None:
    datetime_cols = [
        col for col in timeline_df.columns
        if pd.api.types.is_datetime64_any_dtype(timeline_df[col])
    ]

    if len(datetime_cols) > 0:
        TIME_COL = datetime_cols[0]

if TIME_COL is None:
    raise RuntimeError(
        "Could not identify the minute timestamp column."
    )

timeline_df[TIME_COL] = pd.to_datetime(timeline_df[TIME_COL])

print("Timestamp column          :", TIME_COL)

# --------------------------------------------------------------------------------------------------------------
# BASIC SPLIT VALIDATION
# --------------------------------------------------------------------------------------------------------------

print("\n")
print("=" * 126)
print("SPLIT DISJOINTNESS VALIDATION")
print("=" * 126)

train_set = set(TRAIN_SEGMENTS)
val_set   = set(VAL_SEGMENTS)
test_set  = set(TEST_SEGMENTS)

train_val_overlap  = train_set.intersection(val_set)
train_test_overlap = train_set.intersection(test_set)
val_test_overlap   = val_set.intersection(test_set)

print("Train ∩ Validation :", train_val_overlap)
print("Train ∩ Test       :", train_test_overlap)
print("Validation ∩ Test  :", val_test_overlap)

if train_val_overlap or train_test_overlap or val_test_overlap:
    raise RuntimeError("Split segment sets overlap.")

print("\n✓ Train / validation / test segment sets are disjoint.")

# --------------------------------------------------------------------------------------------------------------
# VERIFY SEGMENT COVERAGE
# --------------------------------------------------------------------------------------------------------------

available_segments = sorted(
    timeline_df["Segment_ID"].dropna().unique().tolist()
)

selected_segments = sorted(
    TRAIN_SEGMENTS + VAL_SEGMENTS + TEST_SEGMENTS
)

print("\nAvailable segments :", available_segments)
print("Selected segments  :", selected_segments)

missing_selected_segments = sorted(
    set(selected_segments) - set(available_segments)
)

if missing_selected_segments:
    raise RuntimeError(
        f"Selected segments missing from timeline: {missing_selected_segments}"
    )

print("✓ Every selected segment exists in the timeline.")

# --------------------------------------------------------------------------------------------------------------
# SEGMENT SUMMARY
# --------------------------------------------------------------------------------------------------------------

segment_summary = (
    timeline_df
    .groupby("Segment_ID")
    .agg(
        Start_Time=(TIME_COL, "min"),
        End_Time=(TIME_COL, "max"),
        Minute_Rows=(TIME_COL, "size"),
        Attack_Minutes=("Attack_State", "sum")
    )
    .reset_index()
)

segment_summary["Benign_Minutes"] = (
    segment_summary["Minute_Rows"]
    - segment_summary["Attack_Minutes"]
)

def assign_split(segment_id):
    if segment_id in TRAIN_SEGMENTS:
        return "TRAIN"
    elif segment_id in VAL_SEGMENTS:
        return "VALIDATION"
    elif segment_id in TEST_SEGMENTS:
        return "TEST"
    return "UNUSED"

segment_summary["Final_Split"] = (
    segment_summary["Segment_ID"].apply(assign_split)
)

print("\n")
print("=" * 126)
print("FINAL SEGMENT ASSIGNMENT")
print("=" * 126)

print(
    segment_summary[
        [
            "Segment_ID",
            "Final_Split",
            "Start_Time",
            "End_Time",
            "Minute_Rows",
            "Attack_Minutes",
            "Benign_Minutes",
        ]
    ].to_string(index=False)
)

# --------------------------------------------------------------------------------------------------------------
# CHRONOLOGICAL ORDER VALIDATION
# --------------------------------------------------------------------------------------------------------------

train_df = timeline_df[
    timeline_df["Segment_ID"].isin(TRAIN_SEGMENTS)
].copy()

val_df = timeline_df[
    timeline_df["Segment_ID"].isin(VAL_SEGMENTS)
].copy()

test_df = timeline_df[
    timeline_df["Segment_ID"].isin(TEST_SEGMENTS)
].copy()

train_start = train_df[TIME_COL].min()
train_end   = train_df[TIME_COL].max()

val_start = val_df[TIME_COL].min()
val_end   = val_df[TIME_COL].max()

test_start = test_df[TIME_COL].min()
test_end   = test_df[TIME_COL].max()

print("\n")
print("=" * 126)
print("CHRONOLOGICAL ORDER VALIDATION")
print("=" * 126)

print("TRAIN")
print("  Start :", train_start)
print("  End   :", train_end)

print("\nVALIDATION")
print("  Start :", val_start)
print("  End   :", val_end)

print("\nTEST")
print("  Start :", test_start)
print("  End   :", test_end)

chronological_order_valid = (
    train_end < val_start
    and val_end < test_start
)

print("\nStrict chronological ordering:", chronological_order_valid)

if not chronological_order_valid:
    raise RuntimeError(
        "Chronological split ordering failed."
    )

print("✓ TRAIN occurs before VALIDATION.")
print("✓ VALIDATION occurs before TEST.")

# --------------------------------------------------------------------------------------------------------------
# CREATE PERMANENT SPLIT LABEL
# --------------------------------------------------------------------------------------------------------------

timeline_df["Final_Split"] = "UNUSED"

timeline_df.loc[
    timeline_df["Segment_ID"].isin(TRAIN_SEGMENTS),
    "Final_Split"
] = "TRAIN"

timeline_df.loc[
    timeline_df["Segment_ID"].isin(VAL_SEGMENTS),
    "Final_Split"
] = "VALIDATION"

timeline_df.loc[
    timeline_df["Segment_ID"].isin(TEST_SEGMENTS),
    "Final_Split"
] = "TEST"

# --------------------------------------------------------------------------------------------------------------
# SPLIT-LEVEL SUMMARY
# --------------------------------------------------------------------------------------------------------------

split_summary = (
    timeline_df[
        timeline_df["Final_Split"] != "UNUSED"
    ]
    .groupby("Final_Split")
    .agg(
        Minute_Rows=(TIME_COL, "size"),
        Attack_Minutes=("Attack_State", "sum"),
        Start_Time=(TIME_COL, "min"),
        End_Time=(TIME_COL, "max"),
        Segment_Count=("Segment_ID", "nunique")
    )
    .reset_index()
)

split_summary["Benign_Minutes"] = (
    split_summary["Minute_Rows"]
    - split_summary["Attack_Minutes"]
)

split_summary["Attack_Percent"] = (
    100
    * split_summary["Attack_Minutes"]
    / split_summary["Minute_Rows"]
)

split_order = {
    "TRAIN": 0,
    "VALIDATION": 1,
    "TEST": 2
}

split_summary["_order"] = (
    split_summary["Final_Split"].map(split_order)
)

split_summary = (
    split_summary
    .sort_values("_order")
    .drop(columns="_order")
    .reset_index(drop=True)
)

print("\n")
print("=" * 126)
print("FINAL SPLIT SUMMARY")
print("=" * 126)

print(split_summary.to_string(index=False))

# --------------------------------------------------------------------------------------------------------------
# FORECAST SAMPLE SUPPORT RECHECK
# --------------------------------------------------------------------------------------------------------------

print("\n")
print("=" * 126)
print("FINAL HORIZON SUPPORT RECHECK")
print("=" * 126)

support_rows = []

for horizon in HORIZONS:

    for split_name, segment_ids in [
        ("TRAIN", TRAIN_SEGMENTS),
        ("VALIDATION", VAL_SEGMENTS),
        ("TEST", TEST_SEGMENTS),
    ]:

        sample_count = 0
        positive_count = 0
        onset_count = 0
        continuation_count = 0
        clean_history_count = 0

        for segment_id in segment_ids:

            seg = (
                timeline_df[
                    timeline_df["Segment_ID"] == segment_id
                ]
                .sort_values(TIME_COL)
                .reset_index(drop=True)
            )

            states = seg["Attack_State"].astype(int).to_numpy()

            n = len(seg)

            # Current minute = t
            # History = [t-14, ..., t]
            # Target = t + horizon
            #
            # Therefore:
            # t must be >= HISTORY_LENGTH - 1
            # and t + horizon must exist.

            for current_idx in range(
                HISTORY_LENGTH - 1,
                n - horizon
            ):

                history_start = current_idx - HISTORY_LENGTH + 1
                history_end   = current_idx + 1
                target_idx    = current_idx + horizon

                history_states = states[
                    history_start:history_end
                ]

                current_state = states[current_idx]
                future_state  = states[target_idx]

                sample_count += 1
                positive_count += int(future_state == 1)

                if future_state == 1 and current_state == 0:
                    onset_count += 1

                if future_state == 1 and current_state == 1:
                    continuation_count += 1

                if np.sum(history_states) == 0:
                    clean_history_count += 1

        support_rows.append(
            {
                "Horizon": horizon,
                "Split": split_name,
                "Samples": sample_count,
                "Future_Attack": positive_count,
                "New_Onset": onset_count,
                "Continuation": continuation_count,
                "Clean_History": clean_history_count,
            }
        )

final_split_support = pd.DataFrame(support_rows)

print(final_split_support.to_string(index=False))

# --------------------------------------------------------------------------------------------------------------
# SUPPORT THRESHOLD VALIDATION
# --------------------------------------------------------------------------------------------------------------

MIN_POSITIVE = {
    "TRAIN": 20,
    "VALIDATION": 5,
    "TEST": 5,
}

MIN_ONSET = {
    "TRAIN": 5,
    "VALIDATION": 1,
    "TEST": 1,
}

support_failures = []

for _, row in final_split_support.iterrows():

    split_name = row["Split"]

    if row["Future_Attack"] < MIN_POSITIVE[split_name]:
        support_failures.append(
            f"H={row['Horizon']} {split_name}: insufficient positive targets"
        )

    if row["New_Onset"] < MIN_ONSET[split_name]:
        support_failures.append(
            f"H={row['Horizon']} {split_name}: insufficient onset targets"
        )

if support_failures:

    print("\nSUPPORT FAILURES:")
    for x in support_failures:
        print("  -", x)

    raise RuntimeError(
        "Final split failed forecasting support validation."
    )

print("\n✓ Final split satisfies forecasting support requirements at all horizons.")

# --------------------------------------------------------------------------------------------------------------
# LOCK SPLIT
# --------------------------------------------------------------------------------------------------------------

FINAL_SPLIT_LOCKED = True

FINAL_SPLIT_CONFIG = {
    "cell21_candidate_index": FINAL_SPLIT_INDEX,
    "history_length": HISTORY_LENGTH,
    "horizons": HORIZONS.copy(),
    "train_segments": TRAIN_SEGMENTS.copy(),
    "validation_segments": VAL_SEGMENTS.copy(),
    "test_segments": TEST_SEGMENTS.copy(),
    "timestamp_column": TIME_COL,
}

# Keep the canonical dataframe name for future cells.
minute_df = timeline_df.copy()

print("\n")
print("=" * 126)
print("FINAL SPLIT LOCK")
print("=" * 126)

print("FINAL_SPLIT_LOCKED :", FINAL_SPLIT_LOCKED)
print()
print("Train segments      :", TRAIN_SEGMENTS)
print("Validation segments :", VAL_SEGMENTS)
print("Test segments       :", TEST_SEGMENTS)

print("\nIMPORTANT:")
print("The chronological split is now treated as FIXED for subsequent experiments.")
print("It must NOT be changed after inspecting model test performance.")

print("\n")
print("=" * 126)
print("CELL 22 COMPLETE")
print("=" * 126)

print("""
✓ Split 26 locked.
✓ Segment sets are disjoint.
✓ Chronological ordering validated.
✓ Forecasting support revalidated at all five horizons.
✓ No model has been trained.
✓ No scaler has been fitted.
✓ No oversampling has been performed.

NEXT:
Cell 23 will identify model-safe temporal features and construct
leakage-safe 15-minute sequences separately inside TRAIN,
VALIDATION and TEST.
""")

print("=" * 126)

CELL 22 — FINAL CHRONOLOGICAL SPLIT LOCK + INTEGRITY VALIDATION

FINAL SPLIT
--------------------------------------------------------------------------------------------------------------
Cell 21 candidate index : 26
Train segments          : [0, 1, 2, 3, 4]
Validation segments     : [5]
Test segments           : [6, 7, 8, 9]

✓ Timeline dataframe found: minute_df
Timeline rows             : 2454
Timestamp column          : Minute


SPLIT DISJOINTNESS VALIDATION
Train ∩ Validation : set()
Train ∩ Test       : set()
Validation ∩ Test  : set()

✓ Train / validation / test segment sets are disjoint.

Available segments : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Selected segments  : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
✓ Every selected segment exists in the timeline.


FINAL SEGMENT ASSIGNMENT
 Segment_ID Final_Split          Start_Time            End_Time  Minute_Rows  Attack_Minutes  Benign_Minutes
          0       TRAIN 2017-07-03 01:00:00 2017-07-03 05:01:00          242               0             2

In [29]:
# ==============================================================================================================
# CELL 23 — LEAKAGE-SAFE FEATURE SELECTION + RAW SEQUENCE CONSTRUCTION
# CORRECTED VERSION
# ==============================================================================================================

import numpy as np
import pandas as pd


print("=" * 126)
print("CELL 23 — LEAKAGE-SAFE FEATURE SELECTION + RAW SEQUENCE CONSTRUCTION")
print("=" * 126)


# ==============================================================================================================
# 1. VERIFY REQUIRED OBJECTS
# ==============================================================================================================

required_objects = [
    "minute_df",
    "FINAL_SPLIT_LOCKED",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Missing required objects:\n"
        + "\n".join(f"  - {name}" for name in missing_objects)
        + "\n\nRun the preceding cells first."
    )

if FINAL_SPLIT_LOCKED is not True:
    raise RuntimeError(
        "FINAL_SPLIT_LOCKED is not True.\n"
        "Do not construct model sequences until Cell 22 has locked the split."
    )


# ==============================================================================================================
# 2. LOCK EXPERIMENT SETTINGS
# ==============================================================================================================

HISTORY_LENGTH = 15
FORECAST_HORIZONS = [1, 5, 10, 15, 30]

TIMESTAMP_COL = "Minute"
SEGMENT_COL = "Segment_ID"
TARGET_COL = "Attack_State"

TRAIN_SEGMENTS = [0, 1, 2, 3, 4]
VALIDATION_SEGMENTS = [5]
TEST_SEGMENTS = [6, 7, 8, 9]


print("\n✓ Final split lock detected.")
print(f"History length : {HISTORY_LENGTH}")
print(f"Horizons       : {FORECAST_HORIZONS}")
print(f"Timestamp col  : {TIMESTAMP_COL}")
print(f"Target col     : {TARGET_COL}")

print("\nFinal chronological split:")
print(f"  Train      : {TRAIN_SEGMENTS}")
print(f"  Validation : {VALIDATION_SEGMENTS}")
print(f"  Test       : {TEST_SEGMENTS}")


# ==============================================================================================================
# 3. BASIC TIMELINE VALIDATION
# ==============================================================================================================

required_columns = {
    TIMESTAMP_COL,
    SEGMENT_COL,
    TARGET_COL
}

missing_columns = required_columns - set(minute_df.columns)

if missing_columns:
    raise RuntimeError(
        f"minute_df is missing required columns: {sorted(missing_columns)}"
    )


timeline = minute_df.copy()

timeline[TIMESTAMP_COL] = pd.to_datetime(
    timeline[TIMESTAMP_COL],
    errors="coerce"
)

if timeline[TIMESTAMP_COL].isna().any():
    raise RuntimeError(
        "Invalid timestamps detected in minute_df."
    )

if timeline[SEGMENT_COL].isna().any():
    raise RuntimeError(
        "Missing Segment_ID values detected."
    )

if timeline[TARGET_COL].isna().any():
    raise RuntimeError(
        "Missing Attack_State values detected."
    )


timeline = (
    timeline
    .sort_values(
        [SEGMENT_COL, TIMESTAMP_COL]
    )
    .reset_index(drop=True)
)


# ==============================================================================================================
# 4. FEATURE SELECTION
# ==============================================================================================================
#
# IMPORTANT:
#
# Attack_State:
#     Direct forecasting target.
#
# Attack_Flow_Count:
#     Constructed using CICIDS2017 ground-truth flow labels.
#
# Attack_Ratio:
#     Also constructed using ground-truth flow labels.
#
# Therefore Attack_Flow_Count and Attack_Ratio MUST NOT be model inputs.
#
# They may remain in minute_df for descriptive analysis, but they are excluded
# from X so the forecasting model receives only observable network features.
#
# ==============================================================================================================

LEAKAGE_EXCLUDED_COLUMNS = {
    SEGMENT_COL,
    TARGET_COL,
    "Attack_Flow_Count",
    "Attack_Ratio",
}


numeric_columns = timeline.select_dtypes(
    include=[np.number]
).columns.tolist()


candidate_features = [
    col
    for col in numeric_columns
    if col not in LEAKAGE_EXCLUDED_COLUMNS
]


print("\n")
print("=" * 126)
print("INITIAL NUMERIC FEATURE SCREEN")
print("=" * 126)

print(f"Numeric columns found       : {len(numeric_columns)}")
print(
    "Excluded target/meta cols   : "
    f"{sorted(LEAKAGE_EXCLUDED_COLUMNS)}"
)
print(f"Initial candidate features  : {len(candidate_features)}")


# ==============================================================================================================
# 5. REMOVE CONSTANT / ALL-INVALID FEATURES IF ANY
# ==============================================================================================================

usable_features = []
removed_features = []

for feature in candidate_features:

    numeric_series = pd.to_numeric(
        timeline[feature],
        errors="coerce"
    )

    finite_series = numeric_series.replace(
        [np.inf, -np.inf],
        np.nan
    )

    valid_values = finite_series.dropna()

    if len(valid_values) == 0:

        removed_features.append(
            (feature, "no finite values")
        )

        continue

    if valid_values.nunique() <= 1:

        removed_features.append(
            (feature, "constant")
        )

        continue

    usable_features.append(feature)


MODEL_FEATURES = usable_features.copy()
N_MODEL_FEATURES = len(MODEL_FEATURES)


if N_MODEL_FEATURES == 0:
    raise RuntimeError(
        "No usable model features remain after leakage-safe screening."
    )


print("\n")
print("=" * 126)
print("MODEL FEATURE SET")
print("=" * 126)

print(f"Usable model features : {N_MODEL_FEATURES}")

for i, feature in enumerate(MODEL_FEATURES, start=1):
    print(f"{i:3d}. {feature}")


if removed_features:

    print("\nRemoved unusable features:")

    for feature, reason in removed_features:
        print(f"  - {feature}: {reason}")


# ==============================================================================================================
# 6. HARD LEAKAGE ASSERTIONS
# ==============================================================================================================

forbidden_model_features = {
    SEGMENT_COL,
    TARGET_COL,
    "Attack_Flow_Count",
    "Attack_Ratio",
    "Label",
    "label",
}

leaked_features = sorted(
    set(MODEL_FEATURES)
    & forbidden_model_features
)

if leaked_features:
    raise RuntimeError(
        "LEAKAGE DETECTED. Forbidden features remain in MODEL_FEATURES:\n"
        + "\n".join(
            f"  - {feature}"
            for feature in leaked_features
        )
    )


# Additional defensive name screen.
#
# We do NOT automatically delete every feature containing the word
# "attack", because that could incorrectly remove a legitimate feature.
# Instead, suspicious names are surfaced for manual inspection.

suspicious_names = [
    feature
    for feature in MODEL_FEATURES
    if any(
        token in feature.lower()
        for token in [
            "attack",
            "label",
            "target",
            "future"
        ]
    )
]


if suspicious_names:

    print("\n⚠ Suspicious feature names detected:")

    for feature in suspicious_names:
        print(f"  - {feature}")

    raise RuntimeError(
        "Potential target-derived feature names remain. "
        "Inspect them before proceeding."
    )


print(
    "\n✓ Direct target and known target-derived columns "
    "are absent from MODEL_FEATURES."
)


# ==============================================================================================================
# 7. FEATURE QUALITY SUMMARY
# ==============================================================================================================

quality_rows = []

for feature in MODEL_FEATURES:

    series = pd.to_numeric(
        timeline[feature],
        errors="coerce"
    )

    finite_series = series.replace(
        [np.inf, -np.inf],
        np.nan
    )

    quality_rows.append({
        "Feature": feature,
        "Missing_Count": int(
            finite_series.isna().sum()
        ),
        "Missing_Percent": float(
            finite_series.isna().mean() * 100
        ),
        "Unique_Values": int(
            finite_series.nunique(dropna=True)
        ),
    })


cell23_feature_quality = pd.DataFrame(
    quality_rows
)


print("\n")
print("=" * 126)
print("FEATURE QUALITY SUMMARY")
print("=" * 126)

print(
    cell23_feature_quality.to_string(
        index=False
    )
)


# ==============================================================================================================
# 8. RAW NON-FINITE VALUE AUDIT
# ==============================================================================================================

feature_matrix = (
    timeline[MODEL_FEATURES]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
    .to_numpy(dtype=np.float64)
)

nan_count = int(
    np.isnan(feature_matrix).sum()
)

positive_inf_count = int(
    np.isposinf(feature_matrix).sum()
)

negative_inf_count = int(
    np.isneginf(feature_matrix).sum()
)

total_nonfinite = int(
    (~np.isfinite(feature_matrix)).sum()
)


print("\n")
print("=" * 126)
print("RAW NON-FINITE VALUE AUDIT")
print("=" * 126)

print(f"NaN values       : {nan_count:,}")
print(f"+Infinity values : {positive_inf_count:,}")
print(f"-Infinity values : {negative_inf_count:,}")
print(f"Total non-finite : {total_nonfinite:,}")


if total_nonfinite == 0:
    print(
        "\n✓ All selected raw model features are finite."
    )
else:
    print(
        "\n⚠ Non-finite values exist."
    )
    print(
        "They have NOT been filled here."
    )
    print(
        "Any imputation must be fitted using TRAIN data only."
    )


# ==============================================================================================================
# 9. ASSIGN FINAL SPLIT TO EACH MINUTE
# ==============================================================================================================

segment_to_split = {}

for seg in TRAIN_SEGMENTS:
    segment_to_split[seg] = "TRAIN"

for seg in VALIDATION_SEGMENTS:
    segment_to_split[seg] = "VALIDATION"

for seg in TEST_SEGMENTS:
    segment_to_split[seg] = "TEST"


timeline["Final_Split"] = (
    timeline[SEGMENT_COL]
    .map(segment_to_split)
)


if timeline["Final_Split"].isna().any():

    bad_segments = sorted(
        timeline.loc[
            timeline["Final_Split"].isna(),
            SEGMENT_COL
        ]
        .unique()
        .tolist()
    )

    raise RuntimeError(
        f"Unassigned segments detected: {bad_segments}"
    )


# ==============================================================================================================
# 10. BUILD LEAKAGE-SAFE RAW SEQUENCES
# ==============================================================================================================
#
# Definition:
#
# History:
#     t-14, ..., t
#
# Target:
#     Attack_State at t + H
#
# Requirements:
#
# 1. History and target belong to SAME Segment_ID.
# 2. No sequence crosses capture discontinuities.
# 3. History contains exactly 15 consecutive minute rows.
# 4. Target occurs exactly H minutes after the final history minute.
# 5. No scaling/imputation occurs here.
#
# ==============================================================================================================


def build_sequences_for_segments(
    dataframe,
    segment_ids,
    horizon,
    feature_columns,
    history_length=15
):

    X_list = []
    y_list = []
    metadata_rows = []

    for segment_id in segment_ids:

        segment_df = (
            dataframe[
                dataframe[SEGMENT_COL] == segment_id
            ]
            .sort_values(TIMESTAMP_COL)
            .reset_index(drop=True)
        )

        if len(segment_df) == 0:
            continue

        timestamps = pd.to_datetime(
            segment_df[TIMESTAMP_COL]
        )

        timestamp_to_position = {
            timestamp: position
            for position, timestamp
            in enumerate(timestamps)
        }

        feature_values = (
            segment_df[feature_columns]
            .apply(
                pd.to_numeric,
                errors="coerce"
            )
            .to_numpy(dtype=np.float64)
        )

        target_values = (
            pd.to_numeric(
                segment_df[TARGET_COL],
                errors="coerce"
            )
            .to_numpy()
        )

        # --------------------------------------------------------------
        # final_history_position = position corresponding to time t
        # --------------------------------------------------------------

        for final_history_position in range(
            history_length - 1,
            len(segment_df)
        ):

            history_start_position = (
                final_history_position
                - history_length
                + 1
            )

            history_end_position = (
                final_history_position
            )

            history_start_time = timestamps.iloc[
                history_start_position
            ]

            history_end_time = timestamps.iloc[
                history_end_position
            ]

            # ----------------------------------------------------------
            # Require truly consecutive minute history.
            # ----------------------------------------------------------

            expected_history_times = pd.date_range(
                start=history_start_time,
                periods=history_length,
                freq="1min"
            )

            actual_history_times = pd.DatetimeIndex(
                timestamps.iloc[
                    history_start_position:
                    history_end_position + 1
                ]
            )

            if not actual_history_times.equals(
                expected_history_times
            ):
                continue

            # ----------------------------------------------------------
            # Exact future target timestamp
            # ----------------------------------------------------------

            target_time = (
                history_end_time
                + pd.Timedelta(minutes=horizon)
            )

            if target_time not in timestamp_to_position:
                continue

            target_position = (
                timestamp_to_position[target_time]
            )

            # Defensive segment check
            if (
                segment_df.loc[
                    target_position,
                    SEGMENT_COL
                ]
                != segment_id
            ):
                continue

            history_matrix = feature_values[
                history_start_position:
                history_end_position + 1
            ]

            if history_matrix.shape != (
                history_length,
                len(feature_columns)
            ):
                continue

            target_value = target_values[
                target_position
            ]

            if pd.isna(target_value):
                continue

            target_value = int(target_value)

            if target_value not in [0, 1]:
                raise RuntimeError(
                    f"Unexpected target value "
                    f"{target_value} in segment "
                    f"{segment_id}."
                )

            history_attack_states = (
                pd.to_numeric(
                    segment_df[TARGET_COL].iloc[
                        history_start_position:
                        history_end_position + 1
                    ],
                    errors="coerce"
                )
                .fillna(0)
                .astype(int)
                .to_numpy()
            )

            current_state = int(
                history_attack_states[-1]
            )

            attack_in_history = int(
                np.any(
                    history_attack_states == 1
                )
            )

            clean_history = int(
                attack_in_history == 0
            )

            new_onset = int(
                current_state == 0
                and target_value == 1
            )

            continuation = int(
                current_state == 1
                and target_value == 1
            )

            X_list.append(
                history_matrix
            )

            y_list.append(
                target_value
            )

            metadata_rows.append({
                "Segment_ID": int(segment_id),
                "History_Start": history_start_time,
                "History_End": history_end_time,
                "Target_Time": target_time,
                "Horizon_Minutes": int(horizon),
                "Current_State": current_state,
                "Target_State": target_value,
                "Attack_In_History": attack_in_history,
                "Clean_History": clean_history,
                "New_Onset": new_onset,
                "Continuation": continuation,
            })


    if len(X_list) == 0:

        X = np.empty(
            (
                0,
                history_length,
                len(feature_columns)
            ),
            dtype=np.float64
        )

        y = np.empty(
            (0,),
            dtype=np.int8
        )

    else:

        X = np.stack(
            X_list
        ).astype(np.float64)

        y = np.asarray(
            y_list,
            dtype=np.int8
        )


    metadata = pd.DataFrame(
        metadata_rows
    )

    return X, y, metadata


# ==============================================================================================================
# 11. CONSTRUCT ALL HORIZONS
# ==============================================================================================================

cell23_raw_sequences = {}
cell23_metadata = {}


print("\n")
print("=" * 126)
print("BUILDING RAW FORECASTING SEQUENCES")
print("=" * 126)


for horizon in FORECAST_HORIZONS:

    X_train, y_train, meta_train = (
        build_sequences_for_segments(
            dataframe=timeline,
            segment_ids=TRAIN_SEGMENTS,
            horizon=horizon,
            feature_columns=MODEL_FEATURES,
            history_length=HISTORY_LENGTH
        )
    )

    X_val, y_val, meta_val = (
        build_sequences_for_segments(
            dataframe=timeline,
            segment_ids=VALIDATION_SEGMENTS,
            horizon=horizon,
            feature_columns=MODEL_FEATURES,
            history_length=HISTORY_LENGTH
        )
    )

    X_test, y_test, meta_test = (
        build_sequences_for_segments(
            dataframe=timeline,
            segment_ids=TEST_SEGMENTS,
            horizon=horizon,
            feature_columns=MODEL_FEATURES,
            history_length=HISTORY_LENGTH
        )
    )


    cell23_raw_sequences[horizon] = {

        "TRAIN": {
            "X": X_train,
            "y": y_train
        },

        "VALIDATION": {
            "X": X_val,
            "y": y_val
        },

        "TEST": {
            "X": X_test,
            "y": y_test
        },
    }


    cell23_metadata[horizon] = {

        "TRAIN": meta_train,

        "VALIDATION": meta_val,

        "TEST": meta_test,
    }


    print(
        f"\nHorizon = {horizon} minutes"
    )

    print(
        f"  Train      : {X_train.shape}"
    )

    print(
        f"  Validation : {X_val.shape}"
    )

    print(
        f"  Test       : {X_test.shape}"
    )


# ==============================================================================================================
# 12. SEQUENCE SHAPE VALIDATION
# ==============================================================================================================

expected_tail_shape = (
    HISTORY_LENGTH,
    N_MODEL_FEATURES
)

shape_errors = []


for horizon in FORECAST_HORIZONS:

    for split_name in [
        "TRAIN",
        "VALIDATION",
        "TEST"
    ]:

        X = (
            cell23_raw_sequences[horizon]
            [split_name]["X"]
        )

        y = (
            cell23_raw_sequences[horizon]
            [split_name]["y"]
        )

        if X.ndim != 3:

            shape_errors.append(
                f"H={horizon}, "
                f"{split_name}: "
                f"X.ndim={X.ndim}"
            )

            continue

        if X.shape[1:] != expected_tail_shape:

            shape_errors.append(
                f"H={horizon}, "
                f"{split_name}: "
                f"X.shape={X.shape}"
            )

        if len(X) != len(y):

            shape_errors.append(
                f"H={horizon}, "
                f"{split_name}: "
                f"len(X)={len(X)}, "
                f"len(y)={len(y)}"
            )


print("\n")
print("=" * 126)
print("SEQUENCE SHAPE VALIDATION")
print("=" * 126)


if shape_errors:

    raise RuntimeError(
        "Sequence shape validation failed:\n"
        + "\n".join(shape_errors)
    )


print(
    f"✓ Every X tensor has shape: "
    f"(samples, {HISTORY_LENGTH}, "
    f"{N_MODEL_FEATURES})"
)


# ==============================================================================================================
# 13. RECONSTRUCT SUPPORT DIRECTLY FROM CELL 23
# ==============================================================================================================

support_rows = []


for horizon in FORECAST_HORIZONS:

    for split_name in [
        "TRAIN",
        "VALIDATION",
        "TEST"
    ]:

        y = (
            cell23_raw_sequences[horizon]
            [split_name]["y"]
        )

        metadata = (
            cell23_metadata[horizon]
            [split_name]
        )

        if len(metadata) == 0:

            future_attack = 0
            new_onset = 0
            continuation = 0
            clean_history = 0

        else:

            future_attack = int(
                (y == 1).sum()
            )

            new_onset = int(
                metadata["New_Onset"].sum()
            )

            continuation = int(
                metadata["Continuation"].sum()
            )

            clean_history = int(
                metadata["Clean_History"].sum()
            )


        support_rows.append({

            "Horizon": int(horizon),

            "Split": split_name,

            "Samples_Cell23": int(
                len(y)
            ),

            "Future_Attack_Cell23": (
                future_attack
            ),

            "New_Onset_Cell23": (
                new_onset
            ),

            "Continuation_Cell23": (
                continuation
            ),

            "Clean_History_Cell23": (
                clean_history
            ),
        })


cell23_support = pd.DataFrame(
    support_rows
)


print("\n")
print("=" * 126)
print("CELL 23 RECONSTRUCTED SUPPORT")
print("=" * 126)

print(
    cell23_support.to_string(
        index=False
    )
)


# ==============================================================================================================
# 14. CELL 22 SUPPORT RECONSTRUCTION CHECK
# ==============================================================================================================

print("\n")
print("=" * 126)
print("CELL 22 SUPPORT RECONSTRUCTION CHECK")
print("=" * 126)


# Expected values printed and locked by Cell 22.
#
# These are NOT used for training.
# They are only an integrity checksum proving that Cell 23 reconstructed
# the exact same forecasting windows as Cell 22.

CELL22_EXPECTED_SUPPORT = {

    1: {
        "TRAIN":      [1151, 140, 16, 124, 935],
        "VALIDATION": [243,   73,  9,  64, 122],
        "TEST":       [910,  305, 56, 249, 395],
    },

    5: {
        "TRAIN":      [1131, 140, 24, 116, 915],
        "VALIDATION": [239,   73, 22,  51, 118],
        "TEST":       [894,  303, 61, 242, 383],
    },

    10: {
        "TRAIN":      [1106, 140, 27, 113, 890],
        "VALIDATION": [234,   72, 23,  49, 113],
        "TEST":       [874,  299, 81, 218, 368],
    },

    15: {
        "TRAIN":      [1081, 136, 39,  97, 865],
        "VALIDATION": [229,   72, 30,  42, 108],
        "TEST":       [854,  294,101, 193, 353],
    },

    30: {
        "TRAIN":      [1006, 121, 54,  67, 790],
        "VALIDATION": [214,   72, 36,  36,  93],
        "TEST":       [794,  277,117, 160, 308],
    },
}


comparison_rows = []


for horizon in FORECAST_HORIZONS:

    for split_name in [
        "TRAIN",
        "VALIDATION",
        "TEST"
    ]:

        actual_row = (
            cell23_support[
                (cell23_support["Horizon"] == horizon)
                &
                (cell23_support["Split"] == split_name)
            ]
            .iloc[0]
        )

        expected = (
            CELL22_EXPECTED_SUPPORT[horizon]
            [split_name]
        )

        expected_samples = expected[0]
        expected_future_attack = expected[1]
        expected_new_onset = expected[2]
        expected_continuation = expected[3]
        expected_clean_history = expected[4]


        sample_match = (
            int(
                actual_row[
                    "Samples_Cell23"
                ]
            )
            == expected_samples
        )

        attack_match = (
            int(
                actual_row[
                    "Future_Attack_Cell23"
                ]
            )
            == expected_future_attack
        )

        onset_match = (
            int(
                actual_row[
                    "New_Onset_Cell23"
                ]
            )
            == expected_new_onset
        )

        continuation_match = (
            int(
                actual_row[
                    "Continuation_Cell23"
                ]
            )
            == expected_continuation
        )

        clean_match = (
            int(
                actual_row[
                    "Clean_History_Cell23"
                ]
            )
            == expected_clean_history
        )


        comparison_rows.append({

            "Horizon": horizon,

            "Split": split_name,

            "Samples_Cell23": int(
                actual_row[
                    "Samples_Cell23"
                ]
            ),

            "Samples_Cell22": (
                expected_samples
            ),

            "Sample_Match": (
                sample_match
            ),

            "Future_Attack_Match": (
                attack_match
            ),

            "New_Onset_Match": (
                onset_match
            ),

            "Continuation_Match": (
                continuation_match
            ),

            "Clean_History_Match": (
                clean_match
            ),

            "All_Match": all([
                sample_match,
                attack_match,
                onset_match,
                continuation_match,
                clean_match
            ]),
        })


cell23_cell22_comparison = pd.DataFrame(
    comparison_rows
)


print(
    cell23_cell22_comparison.to_string(
        index=False
    )
)


EXACT_CELL22_SUPPORT_MATCH = bool(
    cell23_cell22_comparison[
        "All_Match"
    ].all()
)


print(
    "\nExact Cell 22 support match:",
    EXACT_CELL22_SUPPORT_MATCH
)


if not EXACT_CELL22_SUPPORT_MATCH:

    raise RuntimeError(
        "Cell 23 does NOT reproduce Cell 22 "
        "forecasting support exactly.\n"
        "Do not proceed to preprocessing."
    )


print(
    "\n✓ Cell 23 reproduces the locked "
    "Cell 22 forecasting windows exactly."
)


# ==============================================================================================================
# 15. SPLIT / SEGMENT INTEGRITY CHECK
# ==============================================================================================================

integrity_errors = []


expected_segments_by_split = {

    "TRAIN": set(TRAIN_SEGMENTS),

    "VALIDATION": set(
        VALIDATION_SEGMENTS
    ),

    "TEST": set(TEST_SEGMENTS),
}


for horizon in FORECAST_HORIZONS:

    for split_name in [
        "TRAIN",
        "VALIDATION",
        "TEST"
    ]:

        metadata = (
            cell23_metadata[horizon]
            [split_name]
        )

        if len(metadata) == 0:
            continue

        observed_segments = set(
            metadata[
                "Segment_ID"
            ].unique().tolist()
        )

        illegal_segments = (
            observed_segments
            -
            expected_segments_by_split[
                split_name
            ]
        )

        if illegal_segments:

            integrity_errors.append(
                f"H={horizon}, "
                f"{split_name}: "
                f"illegal segments "
                f"{sorted(illegal_segments)}"
            )


print("\n")
print("=" * 126)
print("SPLIT / SEGMENT INTEGRITY")
print("=" * 126)


if integrity_errors:

    raise RuntimeError(
        "Split integrity failed:\n"
        + "\n".join(
            integrity_errors
        )
    )


print(
    "✓ No forecasting sequence crosses "
    "TRAIN / VALIDATION / TEST boundaries."
)

print(
    "✓ Every sequence remains inside its "
    "assigned capture segment."
)


# ==============================================================================================================
# 16. TARGET DISTRIBUTION SUMMARY
# ==============================================================================================================

distribution_rows = []


for horizon in FORECAST_HORIZONS:

    for split_name in [
        "TRAIN",
        "VALIDATION",
        "TEST"
    ]:

        y = (
            cell23_raw_sequences[horizon]
            [split_name]["y"]
        )

        benign = int(
            (y == 0).sum()
        )

        attack = int(
            (y == 1).sum()
        )

        total = int(
            len(y)
        )

        attack_percent = (
            100 * attack / total
            if total > 0
            else np.nan
        )

        distribution_rows.append({

            "Horizon": horizon,

            "Split": split_name,

            "Samples": total,

            "Benign": benign,

            "Attack": attack,

            "Attack_Percent": (
                attack_percent
            ),
        })


cell23_target_distribution = pd.DataFrame(
    distribution_rows
)


print("\n")
print("=" * 126)
print("TARGET DISTRIBUTION")
print("=" * 126)

print(
    cell23_target_distribution.to_string(
        index=False
    )
)


# ==============================================================================================================
# 17. FINAL STATE
# ==============================================================================================================

CELL23_COMPLETE = True

CELL23_LEAKAGE_SAFE = True

CELL23_RAW_SEQUENCE_READY = True


print("\n")
print("=" * 126)
print("CELL 23 COMPLETE")
print("=" * 126)

print(
    f"\n✓ Model features retained : "
    f"{N_MODEL_FEATURES}"
)

print(
    "✓ Attack_State excluded from X."
)

print(
    "✓ Attack_Flow_Count excluded from X."
)

print(
    "✓ Attack_Ratio excluded from X."
)

print(
    "✓ No scaling has been performed."
)

print(
    "✓ No imputation has been performed."
)

print(
    "✓ No oversampling has been performed."
)

print(
    "✓ No model has been trained."
)

print(
    "✓ Raw sequences are separated by "
    "TRAIN / VALIDATION / TEST."
)

print(
    "✓ Cell 22 forecasting support "
    "reproduced exactly."
)

print("\nObjects created:")
print("  MODEL_FEATURES")
print("  N_MODEL_FEATURES")
print("  cell23_feature_quality")
print("  cell23_raw_sequences")
print("  cell23_metadata")
print("  cell23_support")
print("  cell23_cell22_comparison")
print("  cell23_target_distribution")

print("\nNEXT:")
print(
    "Cell 24 should perform TRAIN-ONLY preprocessing:"
)
print(
    "  1. inspect non-finite values"
)
print(
    "  2. fit any required imputer on TRAIN only"
)
print(
    "  3. fit feature scaler on TRAIN only"
)
print(
    "  4. transform validation/test using TRAIN-fitted parameters"
)
print(
    "  5. preserve the 15 × feature temporal structure"
)

print("=" * 126)

CELL 23 — LEAKAGE-SAFE FEATURE SELECTION + RAW SEQUENCE CONSTRUCTION

✓ Final split lock detected.
History length : 15
Horizons       : [1, 5, 10, 15, 30]
Timestamp col  : Minute
Target col     : Attack_State

Final chronological split:
  Train      : [0, 1, 2, 3, 4]
  Validation : [5]
  Test       : [6, 7, 8, 9]


INITIAL NUMERIC FEATURE SCREEN
Numeric columns found       : 129
Excluded target/meta cols   : ['Attack_Flow_Count', 'Attack_Ratio', 'Attack_State', 'Segment_ID']
Initial candidate features  : 125


MODEL FEATURE SET
Usable model features : 125
  1. Flow_Count
  2. Unique_Source_IPs
  3. Unique_Destination_IPs
  4. Unique_Source_Ports
  5. Unique_Destination_Ports
  6. Unique_Protocols
  7. TCP_Flow_Count
  8. UDP_Flow_Count
  9. Other_Protocol_Flow_Count
 10. Total Fwd Packets_sum
 11. Total Backward Packets_sum
 12. Total Length of Fwd Packets_sum
 13. Total Length of Bwd Packets_sum
 14. FIN Flag Count_sum
 15. SYN Flag Count_sum
 16. RST Flag Count_sum
 17. PSH Flag Coun

In [30]:
# ==============================================================================================================
# CELL 24 — TRAIN-ONLY FEATURE SCALING + MODEL-READY SEQUENCE CONSTRUCTION
# ==============================================================================================================

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler


print("=" * 126)
print("CELL 24 — TRAIN-ONLY FEATURE SCALING + MODEL-READY SEQUENCE CONSTRUCTION")
print("=" * 126)


# ==============================================================================================================
# 1. VERIFY REQUIRED CELL 23 OBJECTS
# ==============================================================================================================

required_objects = [
    "CELL23_COMPLETE",
    "CELL23_LEAKAGE_SAFE",
    "CELL23_RAW_SEQUENCE_READY",
    "cell23_raw_sequences",
    "cell23_metadata",
    "MODEL_FEATURES",
    "N_MODEL_FEATURES",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Missing required Cell 23 objects:\n"
        + "\n".join(f"  - {name}" for name in missing_objects)
        + "\n\nRun the corrected Cell 23 first."
    )


if CELL23_COMPLETE is not True:
    raise RuntimeError(
        "CELL23_COMPLETE is not True."
    )

if CELL23_LEAKAGE_SAFE is not True:
    raise RuntimeError(
        "CELL23_LEAKAGE_SAFE is not True."
    )

if CELL23_RAW_SEQUENCE_READY is not True:
    raise RuntimeError(
        "CELL23_RAW_SEQUENCE_READY is not True."
    )


FORECAST_HORIZONS = [1, 5, 10, 15, 30]
HISTORY_LENGTH = 15


print("\n✓ Cell 23 objects found.")
print("✓ Leakage-safe feature set confirmed.")
print(f"Model features : {N_MODEL_FEATURES}")
print(f"History length : {HISTORY_LENGTH}")
print(f"Horizons       : {FORECAST_HORIZONS}")


# ==============================================================================================================
# 2. VERIFY RAW SEQUENCES BEFORE PREPROCESSING
# ==============================================================================================================

print("\n")
print("=" * 126)
print("RAW INPUT VALIDATION")
print("=" * 126)


raw_validation_rows = []


for horizon in FORECAST_HORIZONS:

    if horizon not in cell23_raw_sequences:
        raise RuntimeError(
            f"Horizon {horizon} missing from cell23_raw_sequences."
        )

    for split_name in ["TRAIN", "VALIDATION", "TEST"]:

        if split_name not in cell23_raw_sequences[horizon]:
            raise RuntimeError(
                f"{split_name} missing for horizon {horizon}."
            )

        X = cell23_raw_sequences[horizon][split_name]["X"]
        y = cell23_raw_sequences[horizon][split_name]["y"]

        if not isinstance(X, np.ndarray):
            raise RuntimeError(
                f"H={horizon}, {split_name}: X is not a numpy array."
            )

        if not isinstance(y, np.ndarray):
            raise RuntimeError(
                f"H={horizon}, {split_name}: y is not a numpy array."
            )

        if X.ndim != 3:
            raise RuntimeError(
                f"H={horizon}, {split_name}: expected 3D X, got {X.shape}."
            )

        if X.shape[1] != HISTORY_LENGTH:
            raise RuntimeError(
                f"H={horizon}, {split_name}: "
                f"expected history length {HISTORY_LENGTH}, got {X.shape[1]}."
            )

        if X.shape[2] != N_MODEL_FEATURES:
            raise RuntimeError(
                f"H={horizon}, {split_name}: "
                f"expected {N_MODEL_FEATURES} features, got {X.shape[2]}."
            )

        if len(X) != len(y):
            raise RuntimeError(
                f"H={horizon}, {split_name}: X/y length mismatch."
            )

        finite_count = int(np.isfinite(X).sum())
        total_values = int(X.size)

        nonfinite_count = total_values - finite_count

        raw_validation_rows.append({
            "Horizon": horizon,
            "Split": split_name,
            "Samples": len(X),
            "Shape": str(X.shape),
            "Nonfinite_Values": nonfinite_count,
        })


cell24_raw_validation = pd.DataFrame(
    raw_validation_rows
)


print(
    cell24_raw_validation.to_string(
        index=False
    )
)


if cell24_raw_validation["Nonfinite_Values"].sum() != 0:
    raise RuntimeError(
        "Non-finite values detected in Cell 23 sequences.\n"
        "Do NOT continue to scaling."
    )


print("\n✓ All raw sequence values are finite.")
print("✓ No imputation is required.")


# ==============================================================================================================
# 3. IMPORTANT PREPROCESSING POLICY
# ==============================================================================================================

print("\n")
print("=" * 126)
print("PREPROCESSING POLICY")
print("=" * 126)

print("""
Scaling method:
    StandardScaler

Leakage rule:
    scaler.fit(...)       -> TRAIN ONLY

    scaler.transform(...) -> TRAIN
    scaler.transform(...) -> VALIDATION
    scaler.transform(...) -> TEST

Important:
    Validation and test values NEVER participate in fitting
    the scaler mean or variance.

Sequence structure:
    Scaling is feature-wise across all TRAIN samples and
    all 15 historical time steps.

    Temporal order is preserved.

No imputation:
    Cell 23 found zero NaN / +Inf / -Inf values.

No balancing:
    No SMOTE, oversampling, undersampling or class weighting
    is performed in this cell.

No model:
    No forecasting model is trained in this cell.
""")


# ==============================================================================================================
# 4. STORAGE OBJECTS
# ==============================================================================================================

cell24_scalers = {}

cell24_scaled_sequences = {}

cell24_metadata = {}

cell24_scaler_statistics = {}

cell24_scaling_summary_rows = []


# ==============================================================================================================
# 5. SCALE EACH HORIZON INDEPENDENTLY
# ==============================================================================================================
#
# Why one scaler per horizon?
#
# Cell 23 produces slightly different TRAIN sample sets for each
# forecasting horizon because the valid future target positions differ.
#
# Therefore each horizon gets a scaler fitted ONLY on the training
# histories actually used by that horizon.
#
# ==============================================================================================================


for horizon in FORECAST_HORIZONS:

    print("\n")
    print("-" * 126)
    print(f"HORIZON = {horizon} MINUTES")
    print("-" * 126)


    # ----------------------------------------------------------------------------------------------------------
    # Raw arrays
    # ----------------------------------------------------------------------------------------------------------

    X_train_raw = (
        cell23_raw_sequences[horizon]["TRAIN"]["X"]
        .astype(np.float64, copy=True)
    )

    y_train = (
        cell23_raw_sequences[horizon]["TRAIN"]["y"]
        .astype(np.int8, copy=True)
    )


    X_val_raw = (
        cell23_raw_sequences[horizon]["VALIDATION"]["X"]
        .astype(np.float64, copy=True)
    )

    y_val = (
        cell23_raw_sequences[horizon]["VALIDATION"]["y"]
        .astype(np.int8, copy=True)
    )


    X_test_raw = (
        cell23_raw_sequences[horizon]["TEST"]["X"]
        .astype(np.float64, copy=True)
    )

    y_test = (
        cell23_raw_sequences[horizon]["TEST"]["y"]
        .astype(np.int8, copy=True)
    )


    # ----------------------------------------------------------------------------------------------------------
    # Flatten TRAIN only for scaler fitting
    #
    # (samples, time, features)
    #            ↓
    # (samples × time, features)
    #
    # Each feature column is therefore standardized independently.
    # ----------------------------------------------------------------------------------------------------------

    train_shape = X_train_raw.shape
    val_shape = X_val_raw.shape
    test_shape = X_test_raw.shape


    X_train_flat = X_train_raw.reshape(
        -1,
        N_MODEL_FEATURES
    )

    X_val_flat = X_val_raw.reshape(
        -1,
        N_MODEL_FEATURES
    )

    X_test_flat = X_test_raw.reshape(
        -1,
        N_MODEL_FEATURES
    )


    # ----------------------------------------------------------------------------------------------------------
    # Fit scaler ONLY on TRAIN
    # ----------------------------------------------------------------------------------------------------------

    scaler = StandardScaler()

    scaler.fit(
        X_train_flat
    )


    # ----------------------------------------------------------------------------------------------------------
    # Transform all three splits using TRAIN-fitted scaler
    # ----------------------------------------------------------------------------------------------------------

    X_train_scaled_flat = scaler.transform(
        X_train_flat
    )

    X_val_scaled_flat = scaler.transform(
        X_val_flat
    )

    X_test_scaled_flat = scaler.transform(
        X_test_flat
    )


    # ----------------------------------------------------------------------------------------------------------
    # Restore temporal structure
    # ----------------------------------------------------------------------------------------------------------

    X_train_scaled = (
        X_train_scaled_flat
        .reshape(train_shape)
        .astype(np.float32)
    )

    X_val_scaled = (
        X_val_scaled_flat
        .reshape(val_shape)
        .astype(np.float32)
    )

    X_test_scaled = (
        X_test_scaled_flat
        .reshape(test_shape)
        .astype(np.float32)
    )


    # ----------------------------------------------------------------------------------------------------------
    # Store scaler
    # ----------------------------------------------------------------------------------------------------------

    cell24_scalers[horizon] = scaler


    # ----------------------------------------------------------------------------------------------------------
    # Store model-ready arrays
    # ----------------------------------------------------------------------------------------------------------

    cell24_scaled_sequences[horizon] = {

        "TRAIN": {
            "X": X_train_scaled,
            "y": y_train,
        },

        "VALIDATION": {
            "X": X_val_scaled,
            "y": y_val,
        },

        "TEST": {
            "X": X_test_scaled,
            "y": y_test,
        },
    }


    # Metadata is NOT scaled or modified.

    cell24_metadata[horizon] = {

        "TRAIN":
            cell23_metadata[horizon]["TRAIN"].copy(),

        "VALIDATION":
            cell23_metadata[horizon]["VALIDATION"].copy(),

        "TEST":
            cell23_metadata[horizon]["TEST"].copy(),
    }


    # ----------------------------------------------------------------------------------------------------------
    # Store scaler parameters for later reproducibility
    # ----------------------------------------------------------------------------------------------------------

    cell24_scaler_statistics[horizon] = {

        "mean": scaler.mean_.copy(),

        "scale": scaler.scale_.copy(),

        "var": scaler.var_.copy(),

        "n_samples_seen": scaler.n_samples_seen_,
    }


    # ----------------------------------------------------------------------------------------------------------
    # Validation
    # ----------------------------------------------------------------------------------------------------------

    train_nonfinite = int(
        (~np.isfinite(X_train_scaled)).sum()
    )

    val_nonfinite = int(
        (~np.isfinite(X_val_scaled)).sum()
    )

    test_nonfinite = int(
        (~np.isfinite(X_test_scaled)).sum()
    )


    if train_nonfinite != 0:
        raise RuntimeError(
            f"H={horizon}: non-finite TRAIN values after scaling."
        )

    if val_nonfinite != 0:
        raise RuntimeError(
            f"H={horizon}: non-finite VALIDATION values after scaling."
        )

    if test_nonfinite != 0:
        raise RuntimeError(
            f"H={horizon}: non-finite TEST values after scaling."
        )


    if X_train_scaled.shape != X_train_raw.shape:
        raise RuntimeError(
            f"H={horizon}: TRAIN shape changed during scaling."
        )

    if X_val_scaled.shape != X_val_raw.shape:
        raise RuntimeError(
            f"H={horizon}: VALIDATION shape changed during scaling."
        )

    if X_test_scaled.shape != X_test_raw.shape:
        raise RuntimeError(
            f"H={horizon}: TEST shape changed during scaling."
        )


    # ----------------------------------------------------------------------------------------------------------
    # Check TRAIN normalization
    #
    # Since scaler was fitted on flattened TRAIN histories,
    # flattened scaled TRAIN should have approximately:
    #
    # mean = 0
    # std  = 1
    #
    # Features with zero variance would be special, but Cell 23
    # already removed globally constant features.
    # ----------------------------------------------------------------------------------------------------------

    train_scaled_flat_check = (
        X_train_scaled
        .reshape(
            -1,
            N_MODEL_FEATURES
        )
        .astype(np.float64)
    )


    train_feature_means = (
        train_scaled_flat_check.mean(axis=0)
    )

    train_feature_stds = (
        train_scaled_flat_check.std(axis=0)
    )


    max_abs_train_mean = float(
        np.max(
            np.abs(
                train_feature_means
            )
        )
    )


    # Some features can still be constant within a particular horizon's
    # TRAIN subset even though they are nonconstant globally.
    #
    # StandardScaler safely maps those features to zero.
    # Therefore std≈1 is checked only where scaler.var_ > 0.

    nonconstant_train_mask = (
        scaler.var_ > 0
    )


    if np.any(nonconstant_train_mask):

        max_abs_train_std_error = float(
            np.max(
                np.abs(
                    train_feature_stds[
                        nonconstant_train_mask
                    ]
                    - 1.0
                )
            )
        )

    else:

        max_abs_train_std_error = 0.0


    constant_train_features = int(
        (~nonconstant_train_mask).sum()
    )


    cell24_scaling_summary_rows.append({

        "Horizon": horizon,

        "Train_Samples":
            len(X_train_scaled),

        "Validation_Samples":
            len(X_val_scaled),

        "Test_Samples":
            len(X_test_scaled),

        "Features":
            N_MODEL_FEATURES,

        "History_Length":
            HISTORY_LENGTH,

        "Train_Flat_Rows_Used_For_Fit":
            len(X_train_flat),

        "Constant_In_Train":
            constant_train_features,

        "Max_Abs_Train_Mean":
            max_abs_train_mean,

        "Max_Abs_Train_Std_Error":
            max_abs_train_std_error,

        "Train_Nonfinite":
            train_nonfinite,

        "Validation_Nonfinite":
            val_nonfinite,

        "Test_Nonfinite":
            test_nonfinite,
    })


    print(
        f"Scaler fitted on TRAIN only : "
        f"{X_train_flat.shape}"
    )

    print(
        f"TRAIN      : {X_train_scaled.shape}"
    )

    print(
        f"VALIDATION : {X_val_scaled.shape}"
    )

    print(
        f"TEST       : {X_test_scaled.shape}"
    )

    print(
        f"Constant features inside this TRAIN subset : "
        f"{constant_train_features}"
    )

    print(
        f"Max |TRAIN scaled mean|     : "
        f"{max_abs_train_mean:.8f}"
    )

    print(
        f"Max |TRAIN scaled std - 1|  : "
        f"{max_abs_train_std_error:.8f}"
    )


# ==============================================================================================================
# 6. SCALING SUMMARY
# ==============================================================================================================

cell24_scaling_summary = pd.DataFrame(
    cell24_scaling_summary_rows
)


print("\n")
print("=" * 126)
print("SCALING SUMMARY")
print("=" * 126)

print(
    cell24_scaling_summary.to_string(
        index=False
    )
)


# ==============================================================================================================
# 7. VERIFY LABELS WERE NOT MODIFIED
# ==============================================================================================================

print("\n")
print("=" * 126)
print("TARGET PRESERVATION CHECK")
print("=" * 126)


target_errors = []


for horizon in FORECAST_HORIZONS:

    for split_name in [
        "TRAIN",
        "VALIDATION",
        "TEST"
    ]:

        y_before = (
            cell23_raw_sequences[horizon]
            [split_name]["y"]
        )

        y_after = (
            cell24_scaled_sequences[horizon]
            [split_name]["y"]
        )

        if not np.array_equal(
            y_before,
            y_after
        ):

            target_errors.append(
                f"H={horizon}, {split_name}"
            )


if target_errors:

    raise RuntimeError(
        "Targets changed during preprocessing:\n"
        + "\n".join(target_errors)
    )


print(
    "✓ Every y array is identical to Cell 23."
)


# ==============================================================================================================
# 8. VERIFY SAMPLE ORDER / METADATA PRESERVATION
# ==============================================================================================================

metadata_errors = []


for horizon in FORECAST_HORIZONS:

    for split_name in [
        "TRAIN",
        "VALIDATION",
        "TEST"
    ]:

        original_meta = (
            cell23_metadata[horizon]
            [split_name]
        )

        copied_meta = (
            cell24_metadata[horizon]
            [split_name]
        )

        if not original_meta.equals(
            copied_meta
        ):

            metadata_errors.append(
                f"H={horizon}, {split_name}"
            )


print("\n")
print("=" * 126)
print("SAMPLE ORDER / METADATA CHECK")
print("=" * 126)


if metadata_errors:

    raise RuntimeError(
        "Metadata changed during preprocessing:\n"
        + "\n".join(metadata_errors)
    )


print(
    "✓ Metadata preserved exactly."
)

print(
    "✓ Sample ordering preserved exactly."
)


# ==============================================================================================================
# 9. EXPLICIT LEAKAGE CHECK
# ==============================================================================================================

print("\n")
print("=" * 126)
print("PREPROCESSING LEAKAGE CHECK")
print("=" * 126)


for horizon in FORECAST_HORIZONS:

    scaler = cell24_scalers[horizon]

    expected_train_rows = (
        cell23_raw_sequences[horizon]
        ["TRAIN"]["X"].shape[0]
        * HISTORY_LENGTH
    )

    actual_seen = scaler.n_samples_seen_

    if np.ndim(actual_seen) == 0:

        actual_seen_value = int(
            actual_seen
        )

        if (
            actual_seen_value
            != expected_train_rows
        ):

            raise RuntimeError(
                f"H={horizon}: scaler saw "
                f"{actual_seen_value} rows, "
                f"expected TRAIN-only "
                f"{expected_train_rows}."
            )

    else:

        actual_seen_array = np.asarray(
            actual_seen
        )

        if not np.all(
            actual_seen_array
            == expected_train_rows
        ):

            raise RuntimeError(
                f"H={horizon}: scaler sample count "
                f"does not match TRAIN-only rows."
            )


    print(
        f"H={horizon:2d} min : "
        f"scaler fitted on exactly "
        f"{expected_train_rows:,} TRAIN time-step rows"
    )


print(
    "\n✓ No validation rows participated in scaler fitting."
)

print(
    "✓ No test rows participated in scaler fitting."
)


# ==============================================================================================================
# 10. MODEL-READY ARRAY ALIASES
# ==============================================================================================================
#
# These aliases make later model cells easier to write.
#
# Example:
#
#     X_train_5
#     y_train_5
#     X_val_5
#     y_val_5
#     X_test_5
#     y_test_5
#
# ==============================================================================================================


for horizon in FORECAST_HORIZONS:

    globals()[f"X_train_{horizon}"] = (
        cell24_scaled_sequences[horizon]
        ["TRAIN"]["X"]
    )

    globals()[f"y_train_{horizon}"] = (
        cell24_scaled_sequences[horizon]
        ["TRAIN"]["y"]
    )


    globals()[f"X_val_{horizon}"] = (
        cell24_scaled_sequences[horizon]
        ["VALIDATION"]["X"]
    )

    globals()[f"y_val_{horizon}"] = (
        cell24_scaled_sequences[horizon]
        ["VALIDATION"]["y"]
    )


    globals()[f"X_test_{horizon}"] = (
        cell24_scaled_sequences[horizon]
        ["TEST"]["X"]
    )

    globals()[f"y_test_{horizon}"] = (
        cell24_scaled_sequences[horizon]
        ["TEST"]["y"]
    )


# ==============================================================================================================
# 11. DISPLAY FINAL MODEL-READY SHAPES
# ==============================================================================================================

print("\n")
print("=" * 126)
print("FINAL MODEL-READY ARRAYS")
print("=" * 126)


for horizon in FORECAST_HORIZONS:

    print(
        f"\nHorizon = {horizon} minutes"
    )

    print(
        f"  X_train_{horizon:<2} : "
        f"{globals()[f'X_train_{horizon}'].shape}"
    )

    print(
        f"  y_train_{horizon:<2} : "
        f"{globals()[f'y_train_{horizon}'].shape}"
    )

    print(
        f"  X_val_{horizon:<2}   : "
        f"{globals()[f'X_val_{horizon}'].shape}"
    )

    print(
        f"  y_val_{horizon:<2}   : "
        f"{globals()[f'y_val_{horizon}'].shape}"
    )

    print(
        f"  X_test_{horizon:<2}  : "
        f"{globals()[f'X_test_{horizon}'].shape}"
    )

    print(
        f"  y_test_{horizon:<2}  : "
        f"{globals()[f'y_test_{horizon}'].shape}"
    )


# ==============================================================================================================
# 12. FINAL LOCK
# ==============================================================================================================

CELL24_COMPLETE = True

CELL24_TRAIN_ONLY_SCALING = True

CELL24_MODEL_READY = True


print("\n")
print("=" * 126)
print("CELL 24 COMPLETE")
print("=" * 126)

print("""
✓ StandardScaler fitted independently for each horizon.
✓ Every scaler was fitted on TRAIN histories only.
✓ Validation was transform-only.
✓ Test was transform-only.
✓ Temporal sequence structure was preserved.
✓ Targets were not modified.
✓ Metadata and sample ordering were not modified.
✓ No imputation was necessary.
✓ No balancing was performed.
✓ No model was trained.

MODEL INPUT SHAPE:
    (samples, 15, 125)

Objects created:
    cell24_scalers
    cell24_scaler_statistics
    cell24_scaled_sequences
    cell24_metadata
    cell24_scaling_summary

Convenience arrays:
    X_train_1,  X_val_1,  X_test_1
    X_train_5,  X_val_5,  X_test_5
    X_train_10, X_val_10, X_test_10
    X_train_15, X_val_15, X_test_15
    X_train_30, X_val_30, X_test_30

and corresponding y_* arrays.

NEXT:
    Do NOT touch TEST for model selection.

    Cell 25 should establish the forecasting baselines and evaluation
    metrics before training the main LSTM / temporal model.

    In particular we should compare against simple baselines such as:

        - majority / always-benign baseline
        - current-state persistence baseline
        - simple non-temporal ML baseline

    Then the temporal models have to demonstrate improvement over those
    baselines rather than merely reporting accuracy.
""")

print("=" * 126)

CELL 24 — TRAIN-ONLY FEATURE SCALING + MODEL-READY SEQUENCE CONSTRUCTION

✓ Cell 23 objects found.
✓ Leakage-safe feature set confirmed.
Model features : 125
History length : 15
Horizons       : [1, 5, 10, 15, 30]


RAW INPUT VALIDATION
 Horizon      Split  Samples           Shape  Nonfinite_Values
       1      TRAIN     1151 (1151, 15, 125)                 0
       1 VALIDATION      243  (243, 15, 125)                 0
       1       TEST      910  (910, 15, 125)                 0
       5      TRAIN     1131 (1131, 15, 125)                 0
       5 VALIDATION      239  (239, 15, 125)                 0
       5       TEST      894  (894, 15, 125)                 0
      10      TRAIN     1106 (1106, 15, 125)                 0
      10 VALIDATION      234  (234, 15, 125)                 0
      10       TEST      874  (874, 15, 125)                 0
      15      TRAIN     1081 (1081, 15, 125)                 0
      15 VALIDATION      229  (229, 15, 125)                 0
      1

In [31]:
# ==============================================================================================================
# CELL 25 — FINAL TARGET FEASIBILITY AUDIT
# Attack Type + Victim + Multi-Horizon Support
# ==============================================================================================================

import numpy as np
import pandas as pd
from collections import Counter

print("=" * 130)
print("CELL 25 — FINAL TARGET FEASIBILITY AUDIT")
print("=" * 130)


# ==============================================================================================================
# 1. CONFIGURATION
# ==============================================================================================================

FORECAST_HORIZONS = [1, 5, 10, 15, 30]
HISTORY_LENGTH = 15

TRAIN_SEGMENTS = [0, 1, 2, 3, 4]
VALIDATION_SEGMENTS = [5]
TEST_SEGMENTS = [6, 7, 8, 9]

SPLIT_SEGMENTS = {
    "TRAIN": TRAIN_SEGMENTS,
    "VALIDATION": VALIDATION_SEGMENTS,
    "TEST": TEST_SEGMENTS,
}

TOP_K_ATTACKS_TO_INSPECT = 10

BENIGN_NAMES = {
    "BENIGN",
    "NORMAL",
    "0",
    "NONE",
    "NO ATTACK",
    "NO_ATTACK",
    ""
}


# ==============================================================================================================
# 2. VERIFY CORE OBJECTS
# ==============================================================================================================

required_objects = [
    "minute_df",
    "MODEL_FEATURES",
    "CELL24_COMPLETE",
]

missing = [
    x for x in required_objects
    if x not in globals()
]

if missing:
    raise RuntimeError(
        "Missing required objects:\n"
        + "\n".join(f"  - {x}" for x in missing)
        + "\n\nRun the previous cells first."
    )

if CELL24_COMPLETE is not True:
    raise RuntimeError(
        "Cell 24 has not completed successfully."
    )

timeline = minute_df.copy()

print("\n✓ minute_df found")
print(f"Rows     : {len(timeline):,}")
print(f"Columns  : {len(timeline.columns):,}")


# ==============================================================================================================
# 3. IDENTIFY IMPORTANT TIMELINE COLUMNS
# ==============================================================================================================

def find_column(df, candidates):

    exact = {
        str(c).strip().lower(): c
        for c in df.columns
    }

    for candidate in candidates:
        key = candidate.strip().lower()

        if key in exact:
            return exact[key]

    return None


TIME_COL = find_column(
    timeline,
    [
        "Minute",
        "Timestamp",
        "timestamp",
        "Time",
        "Datetime",
    ]
)

SEGMENT_COL = find_column(
    timeline,
    [
        "Segment_ID",
        "segment_id",
        "Segment",
    ]
)

ATTACK_STATE_COL = find_column(
    timeline,
    [
        "Attack_State",
        "attack_state",
        "Binary_Label",
        "Attack_Label",
    ]
)


if TIME_COL is None:
    raise RuntimeError(
        "Could not identify timeline timestamp column."
    )

if SEGMENT_COL is None:
    raise RuntimeError(
        "Could not identify Segment_ID column."
    )

if ATTACK_STATE_COL is None:
    raise RuntimeError(
        "Could not identify Attack_State column."
    )


timeline[TIME_COL] = pd.to_datetime(
    timeline[TIME_COL]
)

timeline = timeline.sort_values(
    [SEGMENT_COL, TIME_COL]
).reset_index(drop=True)


print("\nCritical timeline columns:")
print(f"Timestamp     : {TIME_COL}")
print(f"Segment       : {SEGMENT_COL}")
print(f"Attack state  : {ATTACK_STATE_COL}")


# ==============================================================================================================
# 4. FIND ATTACK-TYPE INFORMATION
# ==============================================================================================================
#
# Earlier cells may have stored attack information under different names.
# We inspect all likely columns rather than assuming one exact name.
#
# ==============================================================================================================

attack_type_candidates = [
    "Attack_Type",
    "Attack_Type_Mode",
    "Dominant_Attack_Type",
    "Attack_Category",
    "Label_Mode",
    "Dominant_Label",
    "Attack_Label_Name",
]

ATTACK_TYPE_COL = find_column(
    timeline,
    attack_type_candidates
)


print("\n")
print("=" * 130)
print("ATTACK-TYPE SOURCE CHECK")
print("=" * 130)


if ATTACK_TYPE_COL is not None:

    print(
        f"✓ Attack-type information found directly in minute_df: "
        f"{ATTACK_TYPE_COL}"
    )

else:

    print("Attack-type column was NOT found directly in minute_df.")
    print("\nSearching notebook objects for previously-created")
    print("minute-level attack-type structures...")


# ==============================================================================================================
# 5. SEARCH EXISTING NOTEBOOK DATAFRAMES IF NECESSARY
# ==============================================================================================================

attack_source_df = None
attack_source_name = None


if ATTACK_TYPE_COL is None:

    globals_snapshot = list(globals().items())

    likely_type_names = [
        "Attack_Type",
        "Attack_Category",
        "Label",
        "Dominant_Attack_Type",
        "Attack_Type_Mode",
        "Dominant_Label",
    ]

    for obj_name, obj in globals_snapshot:

        if not isinstance(obj, pd.DataFrame):
            continue

        if obj is timeline:
            continue

        possible_type_col = find_column(
            obj,
            likely_type_names
        )

        possible_time_col = find_column(
            obj,
            [
                "Minute",
                "Timestamp",
                "timestamp",
                "Time",
            ]
        )

        possible_segment_col = find_column(
            obj,
            [
                "Segment_ID",
                "segment_id",
                "Segment",
            ]
        )

        if (
            possible_type_col is not None
            and possible_time_col is not None
        ):

            attack_source_df = obj
            attack_source_name = obj_name

            print(
                f"Possible attack-type dataframe found: {obj_name}"
            )

            print(
                f"  time column : {possible_time_col}"
            )

            print(
                f"  type column : {possible_type_col}"
            )

            if possible_segment_col:
                print(
                    f"  segment col : {possible_segment_col}"
                )

            break


# ==============================================================================================================
# 6. TRY TO CONSTRUCT MINUTE-LEVEL ATTACK TYPE IF SOURCE EXISTS
# ==============================================================================================================

if ATTACK_TYPE_COL is None and attack_source_df is not None:

    src = attack_source_df.copy()

    src_time_col = find_column(
        src,
        [
            "Minute",
            "Timestamp",
            "timestamp",
            "Time",
        ]
    )

    src_type_col = find_column(
        src,
        [
            "Attack_Type",
            "Attack_Category",
            "Label",
            "Dominant_Attack_Type",
            "Attack_Type_Mode",
            "Dominant_Label",
        ]
    )

    src_segment_col = find_column(
        src,
        [
            "Segment_ID",
            "segment_id",
            "Segment",
        ]
    )


    src[src_time_col] = pd.to_datetime(
        src[src_time_col],
        errors="coerce"
    )


    # If raw timestamps exist, reduce them to minute level.
    src["_Audit_Minute"] = (
        src[src_time_col]
        .dt.floor("min")
    )


    def clean_attack_name(value):

        if pd.isna(value):
            return None

        value = str(value).strip()

        if value.upper() in BENIGN_NAMES:
            return None

        return value


    src["_Audit_Attack_Type"] = (
        src[src_type_col]
        .map(clean_attack_name)
    )


    attack_only_src = src[
        src["_Audit_Attack_Type"].notna()
    ].copy()


    if len(attack_only_src) > 0:

        group_cols = ["_Audit_Minute"]

        if src_segment_col is not None:
            group_cols.append(src_segment_col)


        # Dominant attack type per minute.
        def dominant_value(series):

            vc = series.value_counts()

            if len(vc) == 0:
                return None

            return vc.index[0]


        minute_attack_types = (
            attack_only_src
            .groupby(group_cols)["_Audit_Attack_Type"]
            .agg(dominant_value)
            .reset_index()
        )


        merge_left = [TIME_COL]
        merge_right = ["_Audit_Minute"]


        if src_segment_col is not None:

            merge_left.append(SEGMENT_COL)
            merge_right.append(src_segment_col)


        timeline = timeline.merge(
            minute_attack_types,
            how="left",
            left_on=merge_left,
            right_on=merge_right
        )


        timeline["Attack_Type_Audit"] = (
            timeline["_Audit_Attack_Type"]
        )

        ATTACK_TYPE_COL = "Attack_Type_Audit"

        drop_cols = [
            "_Audit_Minute",
            "_Audit_Attack_Type",
        ]

        if (
            src_segment_col is not None
            and src_segment_col != SEGMENT_COL
            and src_segment_col in timeline.columns
        ):
            drop_cols.append(src_segment_col)


        timeline.drop(
            columns=[
                c for c in drop_cols
                if c in timeline.columns
            ],
            inplace=True
        )


        print(
            f"\n✓ Constructed minute-level attack types "
            f"from '{attack_source_name}'."
        )


# ==============================================================================================================
# 7. ATTACK-TYPE AVAILABILITY VERDICT
# ==============================================================================================================

print("\n")
print("=" * 130)
print("ATTACK-TYPE AVAILABILITY")
print("=" * 130)


ATTACK_TYPE_AUDIT_AVAILABLE = (
    ATTACK_TYPE_COL is not None
)


if ATTACK_TYPE_AUDIT_AVAILABLE:

    print(
        f"✓ Attack type column available: {ATTACK_TYPE_COL}"
    )

else:

    print("✗ No trustworthy minute-level attack-type source was found.")
    print()
    print("IMPORTANT:")
    print("Cell 25 will continue with victim feasibility.")
    print()
    print("Attack-type selection will NOT be invented.")
    print("If necessary, the next cell will reconstruct attack-type")
    print("labels directly from the original CICIDS2017 flow labels.")


# ==============================================================================================================
# 8. ASSIGN FINAL SPLITS TO TIMELINE
# ==============================================================================================================

timeline["_Audit_Split"] = None

for split_name, segments in SPLIT_SEGMENTS.items():

    timeline.loc[
        timeline[SEGMENT_COL].isin(segments),
        "_Audit_Split"
    ] = split_name


if timeline["_Audit_Split"].isna().any():

    bad_segments = sorted(
        timeline.loc[
            timeline["_Audit_Split"].isna(),
            SEGMENT_COL
        ].unique()
    )

    raise RuntimeError(
        f"Timeline contains unassigned segments: {bad_segments}"
    )


# ==============================================================================================================
# 9. CLEAN ATTACK-TYPE LABELS
# ==============================================================================================================

if ATTACK_TYPE_AUDIT_AVAILABLE:

    def normalize_attack_type(value, attack_state):

        if int(attack_state) == 0:
            return "BENIGN"

        if pd.isna(value):
            return "UNKNOWN_ATTACK"

        value = str(value).strip()

        if value.upper() in BENIGN_NAMES:
            return "UNKNOWN_ATTACK"

        return value


    timeline["_Audit_Attack_Type"] = [
        normalize_attack_type(t, s)
        for t, s in zip(
            timeline[ATTACK_TYPE_COL],
            timeline[ATTACK_STATE_COL]
        )
    ]


# ==============================================================================================================
# 10. ATTACK-TYPE MINUTE SUPPORT BY SPLIT
# ==============================================================================================================

if ATTACK_TYPE_AUDIT_AVAILABLE:

    print("\n")
    print("=" * 130)
    print("ATTACK-TYPE MINUTE SUPPORT BY SPLIT")
    print("=" * 130)


    attack_minute_support = (
        timeline[
            timeline[ATTACK_STATE_COL] == 1
        ]
        .groupby(
            ["_Audit_Attack_Type", "_Audit_Split"]
        )
        .size()
        .unstack(fill_value=0)
    )


    for split_name in [
        "TRAIN",
        "VALIDATION",
        "TEST"
    ]:

        if split_name not in attack_minute_support.columns:
            attack_minute_support[split_name] = 0


    attack_minute_support = (
        attack_minute_support[
            ["TRAIN", "VALIDATION", "TEST"]
        ]
        .reset_index()
    )


    attack_minute_support[
        "Total_Attack_Minutes"
    ] = (
        attack_minute_support[
            ["TRAIN", "VALIDATION", "TEST"]
        ].sum(axis=1)
    )


    attack_minute_support = (
        attack_minute_support
        .sort_values(
            ["TRAIN", "Total_Attack_Minutes"],
            ascending=False
        )
        .reset_index(drop=True)
    )


    print(
        attack_minute_support.to_string(
            index=False
        )
    )


# ==============================================================================================================
# 11. ATTACK EPISODE DETECTION
# ==============================================================================================================

if ATTACK_TYPE_AUDIT_AVAILABLE:

    print("\n")
    print("=" * 130)
    print("ATTACK-TYPE EPISODE SUPPORT")
    print("=" * 130)


    episode_rows = []


    for segment_id, seg in timeline.groupby(
        SEGMENT_COL,
        sort=True
    ):

        seg = (
            seg.sort_values(TIME_COL)
            .reset_index(drop=True)
        )


        previous_type = "BENIGN"
        previous_time = None

        episode_id = -1


        for i, row in seg.iterrows():

            current_type = row[
                "_Audit_Attack_Type"
            ]

            current_time = row[
                TIME_COL
            ]

            current_attack = int(
                row[ATTACK_STATE_COL]
            )


            new_episode = False


            if current_attack == 1:

                if previous_type == "BENIGN":
                    new_episode = True

                elif current_type != previous_type:
                    new_episode = True

                elif previous_time is not None:

                    gap = (
                        current_time
                        - previous_time
                    ).total_seconds()

                    if gap > 60:
                        new_episode = True


            if new_episode:
                episode_id += 1


            if current_attack == 1:

                episode_rows.append({
                    "Segment_ID":
                        segment_id,

                    "Split":
                        row["_Audit_Split"],

                    "Episode_ID":
                        episode_id,

                    "Attack_Type":
                        current_type,

                    "Minute":
                        current_time,
                })


            previous_type = current_type
            previous_time = current_time


    episode_df = pd.DataFrame(
        episode_rows
    )


    if len(episode_df) > 0:

        episode_summary = (
            episode_df
            .groupby(
                [
                    "Attack_Type",
                    "Split",
                    "Segment_ID",
                    "Episode_ID"
                ]
            )
            .agg(
                Start_Time=("Minute", "min"),
                End_Time=("Minute", "max"),
                Duration_Minutes=("Minute", "size"),
            )
            .reset_index()
        )


        attack_episode_support = (
            episode_summary
            .groupby(
                ["Attack_Type", "Split"]
            )
            .size()
            .unstack(fill_value=0)
        )


        for split_name in [
            "TRAIN",
            "VALIDATION",
            "TEST"
        ]:

            if split_name not in attack_episode_support.columns:
                attack_episode_support[split_name] = 0


        attack_episode_support = (
            attack_episode_support[
                ["TRAIN", "VALIDATION", "TEST"]
            ]
            .reset_index()
        )


        attack_episode_support[
            "Total_Episodes"
        ] = (
            attack_episode_support[
                ["TRAIN", "VALIDATION", "TEST"]
            ].sum(axis=1)
        )


        attack_episode_support = (
            attack_episode_support
            .sort_values(
                ["TRAIN", "Total_Episodes"],
                ascending=False
            )
            .reset_index(drop=True)
        )


        print(
            attack_episode_support.to_string(
                index=False
            )
        )

    else:

        episode_summary = pd.DataFrame()
        attack_episode_support = pd.DataFrame()

        print("No attack episodes found.")


# ==============================================================================================================
# 12. FUTURE ATTACK-TYPE FORECASTING SUPPORT
# ==============================================================================================================

if ATTACK_TYPE_AUDIT_AVAILABLE:

    print("\n")
    print("=" * 130)
    print("FUTURE ATTACK-TYPE TARGET SUPPORT")
    print("=" * 130)


    future_type_rows = []


    for horizon in FORECAST_HORIZONS:

        for split_name, segments in SPLIT_SEGMENTS.items():

            split_df = timeline[
                timeline[SEGMENT_COL].isin(segments)
            ]


            for segment_id, seg in split_df.groupby(
                SEGMENT_COL,
                sort=True
            ):

                seg = (
                    seg.sort_values(TIME_COL)
                    .reset_index(drop=True)
                )


                if len(seg) < (
                    HISTORY_LENGTH + horizon
                ):
                    continue


                for target_idx in range(
                    HISTORY_LENGTH - 1 + horizon,
                    len(seg)
                ):

                    history_end_idx = (
                        target_idx - horizon
                    )

                    history_start_idx = (
                        history_end_idx
                        - HISTORY_LENGTH
                        + 1
                    )


                    if history_start_idx < 0:
                        continue


                    history = seg.iloc[
                        history_start_idx:
                        history_end_idx + 1
                    ]


                    target = seg.iloc[
                        target_idx
                    ]


                    target_attack = int(
                        target[ATTACK_STATE_COL]
                    )


                    if target_attack != 1:
                        continue


                    target_type = target[
                        "_Audit_Attack_Type"
                    ]


                    history_attack_present = bool(
                        (
                            history[ATTACK_STATE_COL]
                            == 1
                        ).any()
                    )


                    current_state = int(
                        history.iloc[-1][
                            ATTACK_STATE_COL
                        ]
                    )


                    future_type_rows.append({

                        "Horizon":
                            horizon,

                        "Split":
                            split_name,

                        "Segment_ID":
                            segment_id,

                        "Target_Time":
                            target[TIME_COL],

                        "Attack_Type":
                            target_type,

                        "Clean_History":
                            not history_attack_present,

                        "Current_Benign":
                            current_state == 0,
                    })


    future_type_support_df = pd.DataFrame(
        future_type_rows
    )


    if len(future_type_support_df) == 0:

        print(
            "No future attack-type targets could be constructed."
        )

    else:

        type_forecasting_support = (
            future_type_support_df
            .groupby(
                [
                    "Attack_Type",
                    "Horizon",
                    "Split"
                ]
            )
            .agg(
                Future_Positive=(
                    "Attack_Type",
                    "size"
                ),

                Clean_History=(
                    "Clean_History",
                    "sum"
                ),

                Current_Benign=(
                    "Current_Benign",
                    "sum"
                ),
            )
            .reset_index()
        )


        print(
            type_forecasting_support.to_string(
                index=False
            )
        )


# ==============================================================================================================
# 13. TRAIN-ONLY ATTACK-TYPE RANKING
# ==============================================================================================================

if ATTACK_TYPE_AUDIT_AVAILABLE:

    print("\n")
    print("=" * 130)
    print("TRAIN-ONLY ATTACK-TYPE CANDIDATE RANKING")
    print("=" * 130)


    train_minutes = (
        attack_minute_support[
            [
                "_Audit_Attack_Type",
                "TRAIN"
            ]
        ]
        .rename(
            columns={
                "_Audit_Attack_Type":
                    "Attack_Type",

                "TRAIN":
                    "Train_Attack_Minutes"
            }
        )
    )


    if (
        "attack_episode_support" in globals()
        and len(attack_episode_support) > 0
    ):

        train_episodes = (
            attack_episode_support[
                [
                    "Attack_Type",
                    "TRAIN"
                ]
            ]
            .rename(
                columns={
                    "TRAIN":
                        "Train_Episodes"
                }
            )
        )

    else:

        train_episodes = pd.DataFrame(
            columns=[
                "Attack_Type",
                "Train_Episodes"
            ]
        )


    train_future = (
        future_type_support_df[
            future_type_support_df[
                "Split"
            ] == "TRAIN"
        ]
        .groupby("Attack_Type")
        .agg(
            Train_Future_Targets=(
                "Attack_Type",
                "size"
            ),

            Train_Clean_History_Targets=(
                "Clean_History",
                "sum"
            ),

            Train_Current_Benign_Targets=(
                "Current_Benign",
                "sum"
            ),
        )
        .reset_index()
    )


    candidate_attack_ranking = (
        train_minutes
        .merge(
            train_episodes,
            on="Attack_Type",
            how="left"
        )
        .merge(
            train_future,
            on="Attack_Type",
            how="left"
        )
        .fillna(0)
    )


    numeric_cols = [
        "Train_Attack_Minutes",
        "Train_Episodes",
        "Train_Future_Targets",
        "Train_Clean_History_Targets",
        "Train_Current_Benign_Targets",
    ]


    for col in numeric_cols:

        candidate_attack_ranking[col] = (
            candidate_attack_ranking[col]
            .astype(int)
        )


    # Ranking is based ONLY on TRAIN support.
    #
    # This is NOT yet final class selection.
    # Validation/test are deliberately excluded from ranking.

    candidate_attack_ranking = (
        candidate_attack_ranking
        .sort_values(
            [
                "Train_Current_Benign_Targets",
                "Train_Clean_History_Targets",
                "Train_Episodes",
                "Train_Attack_Minutes",
            ],
            ascending=False
        )
        .reset_index(drop=True)
    )


    candidate_attack_ranking[
        "Train_Rank"
    ] = (
        np.arange(
            1,
            len(candidate_attack_ranking) + 1
        )
    )


    candidate_attack_ranking = (
        candidate_attack_ranking[
            [
                "Train_Rank",
                "Attack_Type",
                "Train_Attack_Minutes",
                "Train_Episodes",
                "Train_Future_Targets",
                "Train_Clean_History_Targets",
                "Train_Current_Benign_Targets",
            ]
        ]
    )


    print(
        candidate_attack_ranking
        .head(TOP_K_ATTACKS_TO_INSPECT)
        .to_string(index=False)
    )


    print()
    print(
        "IMPORTANT: This is a TRAIN-ONLY ranking."
    )

    print(
        "The top five have NOT been automatically locked."
    )


# ==============================================================================================================
# 14. VALIDATION / TEST PRESENCE FOR TRAIN-RANKED TYPES
# ==============================================================================================================

if ATTACK_TYPE_AUDIT_AVAILABLE:

    print("\n")
    print("=" * 130)
    print("GENERALIZATION SUPPORT FOR TRAIN-RANKED ATTACK TYPES")
    print("=" * 130)


    generalization_table = (
        candidate_attack_ranking
        .merge(
            attack_minute_support[
                [
                    "_Audit_Attack_Type",
                    "VALIDATION",
                    "TEST"
                ]
            ].rename(
                columns={
                    "_Audit_Attack_Type":
                        "Attack_Type",

                    "VALIDATION":
                        "Validation_Attack_Minutes",

                    "TEST":
                        "Test_Attack_Minutes",
                }
            ),
            on="Attack_Type",
            how="left"
        )
        .fillna(0)
    )


    if len(attack_episode_support) > 0:

        generalization_table = (
            generalization_table
            .merge(
                attack_episode_support[
                    [
                        "Attack_Type",
                        "VALIDATION",
                        "TEST"
                    ]
                ].rename(
                    columns={
                        "VALIDATION":
                            "Validation_Episodes",

                        "TEST":
                            "Test_Episodes",
                    }
                ),
                on="Attack_Type",
                how="left"
            )
            .fillna(0)
        )


    for col in generalization_table.columns:

        if (
            col != "Attack_Type"
            and col != "Train_Rank"
        ):

            try:
                generalization_table[col] = (
                    generalization_table[col]
                    .astype(int)
                )
            except:
                pass


    print(
        generalization_table
        .head(TOP_K_ATTACKS_TO_INSPECT)
        .to_string(index=False)
    )


# ==============================================================================================================
# 15. SEARCH FOR VICTIM / DESTINATION INFORMATION
# ==============================================================================================================

print("\n")
print("=" * 130)
print("VICTIM TARGET SOURCE CHECK")
print("=" * 130)


victim_candidates = [
    "Destination IP",
    "Destination_IP",
    "dst_ip",
    "Dst IP",
    "Dst_IP",
    "DestinationIP",
]

VICTIM_COL = find_column(
    timeline,
    victim_candidates
)

victim_source_df = None
victim_source_name = None


if VICTIM_COL is not None:

    print(
        f"✓ Victim candidate found directly in minute_df: "
        f"{VICTIM_COL}"
    )

else:

    globals_snapshot = list(globals().items())

    for obj_name, obj in globals_snapshot:

        if not isinstance(obj, pd.DataFrame):
            continue

        possible_victim_col = find_column(
            obj,
            victim_candidates
        )

        possible_time_col = find_column(
            obj,
            [
                "Minute",
                "Timestamp",
                "timestamp",
                "Time",
            ]
        )

        if (
            possible_victim_col is not None
            and possible_time_col is not None
        ):

            victim_source_df = obj
            victim_source_name = obj_name

            print(
                f"Possible victim source dataframe: "
                f"{obj_name}"
            )

            print(
                f"  victim column : "
                f"{possible_victim_col}"
            )

            print(
                f"  time column   : "
                f"{possible_time_col}"
            )

            break


# ==============================================================================================================
# 16. VICTIM SUPPORT AUDIT
# ==============================================================================================================

victim_support = None
victim_overlap_summary = None


if VICTIM_COL is not None:

    victim_audit_df = timeline[
        timeline[ATTACK_STATE_COL] == 1
    ].copy()

    victim_audit_df["_Victim"] = (
        victim_audit_df[VICTIM_COL]
        .astype(str)
        .str.strip()
    )


elif victim_source_df is not None:

    src = victim_source_df.copy()

    src_victim_col = find_column(
        src,
        victim_candidates
    )

    src_time_col = find_column(
        src,
        [
            "Minute",
            "Timestamp",
            "timestamp",
            "Time",
        ]
    )

    src_attack_col = find_column(
        src,
        [
            "Attack_State",
            "Attack_Label",
            "Binary_Label",
        ]
    )

    src_label_col = find_column(
        src,
        [
            "Label",
            "Attack_Type",
            "Attack_Category",
        ]
    )


    src[src_time_col] = pd.to_datetime(
        src[src_time_col],
        errors="coerce"
    )


    # Determine attack rows if possible.

    if src_attack_col is not None:

        attack_mask = (
            pd.to_numeric(
                src[src_attack_col],
                errors="coerce"
            )
            .fillna(0)
            .astype(int)
            == 1
        )

    elif src_label_col is not None:

        attack_mask = ~(
            src[src_label_col]
            .astype(str)
            .str.strip()
            .str.upper()
            .isin(BENIGN_NAMES)
        )

    else:

        attack_mask = pd.Series(
            True,
            index=src.index
        )


    victim_audit_df = src[
        attack_mask
    ].copy()


    victim_audit_df["_Victim"] = (
        victim_audit_df[src_victim_col]
        .astype(str)
        .str.strip()
    )


    victim_audit_df["_Minute"] = (
        victim_audit_df[src_time_col]
        .dt.floor("min")
    )


    # Attach split from the validated timeline by minute.
    minute_split_map = (
        timeline[
            [TIME_COL, "_Audit_Split"]
        ]
        .drop_duplicates()
        .rename(
            columns={
                TIME_COL: "_Minute"
            }
        )
    )


    victim_audit_df = (
        victim_audit_df
        .merge(
            minute_split_map,
            on="_Minute",
            how="left"
        )
    )


else:

    victim_audit_df = None


if victim_audit_df is None:

    print()
    print(
        "✗ No trustworthy victim/destination source found."
    )

    print(
        "Victim forecasting will NOT be assumed feasible."
    )


else:

    split_col = (
        "_Audit_Split"
        if "_Audit_Split" in victim_audit_df.columns
        else None
    )


    if split_col is None:

        print()
        print(
            "Victim data exists, but could not be assigned "
            "reliably to the locked chronological splits."
        )

    else:

        victim_audit_df = victim_audit_df[
            victim_audit_df[split_col].notna()
        ].copy()


        victim_support = (
            victim_audit_df
            .groupby(
                ["_Victim", split_col]
            )
            .size()
            .unstack(fill_value=0)
        )


        for split_name in [
            "TRAIN",
            "VALIDATION",
            "TEST"
        ]:

            if split_name not in victim_support.columns:
                victim_support[split_name] = 0


        victim_support = (
            victim_support[
                ["TRAIN", "VALIDATION", "TEST"]
            ]
            .reset_index()
        )


        victim_support[
            "Total"
        ] = (
            victim_support[
                ["TRAIN", "VALIDATION", "TEST"]
            ].sum(axis=1)
        )


        victim_support = (
            victim_support
            .sort_values(
                ["TRAIN", "Total"],
                ascending=False
            )
            .reset_index(drop=True)
        )


        print("\nTop victim candidates by TRAIN support:")
        print()

        print(
            victim_support
            .head(20)
            .to_string(index=False)
        )


        train_victims = set(
            victim_support.loc[
                victim_support["TRAIN"] > 0,
                "_Victim"
            ]
        )

        val_victims = set(
            victim_support.loc[
                victim_support["VALIDATION"] > 0,
                "_Victim"
            ]
        )

        test_victims = set(
            victim_support.loc[
                victim_support["TEST"] > 0,
                "_Victim"
            ]
        )


        val_seen = (
            val_victims & train_victims
        )

        test_seen = (
            test_victims & train_victims
        )


        victim_overlap_summary = pd.DataFrame([
            {
                "Metric":
                    "Unique TRAIN victims",

                "Value":
                    len(train_victims),
            },

            {
                "Metric":
                    "Unique VALIDATION victims",

                "Value":
                    len(val_victims),
            },

            {
                "Metric":
                    "Validation victims seen in TRAIN",

                "Value":
                    len(val_seen),
            },

            {
                "Metric":
                    "Validation overlap percent",

                "Value":
                    (
                        100 * len(val_seen)
                        / max(len(val_victims), 1)
                    ),
            },

            {
                "Metric":
                    "Unique TEST victims",

                "Value":
                    len(test_victims),
            },

            {
                "Metric":
                    "Test victims seen in TRAIN",

                "Value":
                    len(test_seen),
            },

            {
                "Metric":
                    "Test overlap percent",

                "Value":
                    (
                        100 * len(test_seen)
                        / max(len(test_victims), 1)
                    ),
            },
        ])


        print("\n")
        print("Victim cross-split overlap:")
        print()

        print(
            victim_overlap_summary.to_string(
                index=False
            )
        )


# ==============================================================================================================
# 17. LEAD-TIME TARGET DESIGN
# ==============================================================================================================

print("\n")
print("=" * 130)
print("LEAD-TIME DESIGN")
print("=" * 130)

print("""
Lead time will NOT be treated as an arbitrary continuous regression
target at this stage.

The forecasting system already has five explicit future horizons:

    +1 minute
    +5 minutes
    +10 minutes
    +15 minutes
    +30 minutes

Each horizon asks:

    "What is the future network state at t + H?"

During final inference, the earliest horizon producing a sufficiently
confident future-attack prediction can be used as the system's
forecast-warning horizon.

Example:

    +1 min   -> benign
    +5 min   -> benign
    +10 min  -> attack
    +15 min  -> attack
    +30 min  -> attack

Earliest positive forecast horizon = 10 minutes.

Thresholds must later be chosen using VALIDATION data only.
TEST must remain untouched during threshold/model selection.
""")


# ==============================================================================================================
# 18. FINAL AUDIT SUMMARY
# ==============================================================================================================

print("\n")
print("=" * 130)
print("FINAL TARGET FEASIBILITY SUMMARY")
print("=" * 130)


print("\n1. ATTACK PRESENCE")
print("-" * 70)

print(
    "✓ Already fully prepared by Cells 1–24."
)

print(
    "Target = future binary Attack_State."
)


print("\n2. ATTACK TYPE")
print("-" * 70)

if ATTACK_TYPE_AUDIT_AVAILABLE:

    print(
        "✓ Minute-level attack-type information is available."
    )

    print(
        "✓ TRAIN-only candidate ranking has been constructed."
    )

    print(
        "✓ Validation/test support has been reported separately."
    )

    print(
        "⚠ No five attack types have been locked yet."
    )

else:

    print(
        "⚠ Attack-type reconstruction is still required."
    )

    print(
        "No classes were guessed or fabricated."
    )


print("\n3. VICTIM")
print("-" * 70)

if victim_audit_df is not None:

    print(
        "✓ Destination/victim information was found."
    )

    if victim_support is not None:

        print(
            "✓ Cross-split victim support has been measured."
        )

        print(
            "Victim target should be locked only after inspecting "
            "the overlap results above."
        )

else:

    print(
        "⚠ Victim information was not available in a form "
        "safe to use yet."
    )


print("\n4. LEAD TIME")
print("-" * 70)

print(
    "✓ Lead-time framework = [1, 5, 10, 15, 30] minute horizons."
)


# ==============================================================================================================
# 19. SAVE AUDIT OBJECTS
# ==============================================================================================================

cell25_timeline = timeline.copy()

cell25_attack_type_available = (
    ATTACK_TYPE_AUDIT_AVAILABLE
)

cell25_attack_type_column = (
    ATTACK_TYPE_COL
)

cell25_victim_column = (
    VICTIM_COL
)


if ATTACK_TYPE_AUDIT_AVAILABLE:

    cell25_attack_minute_support = (
        attack_minute_support.copy()
    )

    cell25_attack_episode_support = (
        attack_episode_support.copy()
    )

    cell25_future_type_support = (
        future_type_support_df.copy()
    )

    cell25_candidate_attack_ranking = (
        candidate_attack_ranking.copy()
    )

    cell25_attack_generalization = (
        generalization_table.copy()
    )


if victim_support is not None:

    cell25_victim_support = (
        victim_support.copy()
    )


if victim_overlap_summary is not None:

    cell25_victim_overlap_summary = (
        victim_overlap_summary.copy()
    )


CELL25_COMPLETE = True


# ==============================================================================================================
# 20. FINAL MESSAGE
# ==============================================================================================================

print("\n")
print("=" * 130)
print("CELL 25 COMPLETE")
print("=" * 130)

print("""
NO MODEL HAS BEEN TRAINED.

NO ATTACK TYPE HAS BEEN SELECTED USING TEST PERFORMANCE.

NO VICTIM CLASS HAS BEEN LOCKED.

NO TEST-SET THRESHOLD HAS BEEN CHOSEN.

Cell 25 has audited the remaining outputs needed for the final system:

    past 15-minute network history
                    ↓
        future attack probability
                    ↓
             attack type
                    ↓
             likely victim
                    ↓
              lead time
                    ↓
             final alert

NEXT:

Inspect the Cell 25 output before writing Cell 26.

Cell 26 should lock:

    1. which attack types are actually defensible,
    2. whether exact victim-IP forecasting is feasible,
    3. the final target encodings,

before Logistic Regression / Random Forest / LSTM / NODE /
LSTM-NODE training begins.
""")

print("=" * 130)

CELL 25 — FINAL TARGET FEASIBILITY AUDIT

✓ minute_df found
Rows     : 2,454
Columns  : 133

Critical timeline columns:
Timestamp     : Minute
Segment       : Segment_ID
Attack state  : Attack_State


ATTACK-TYPE SOURCE CHECK
Attack-type column was NOT found directly in minute_df.

Searching notebook objects for previously-created
minute-level attack-type structures...
Possible attack-type dataframe found: header
  time column :  Timestamp
  type column :  Label


ATTACK-TYPE AVAILABILITY
✗ No trustworthy minute-level attack-type source was found.

IMPORTANT:
Cell 25 will continue with victim feasibility.

Attack-type selection will NOT be invented.
If necessary, the next cell will reconstruct attack-type
labels directly from the original CICIDS2017 flow labels.


VICTIM TARGET SOURCE CHECK
Possible victim source dataframe: header
  victim column :  Destination IP
  time column   :  Timestamp

Top victim candidates by TRAIN support:

Empty DataFrame
Columns: [_Victim, TRAIN, VALIDATION

In [32]:
# ==============================================================================================================
# 4. LOAD THE CORRECT TRAFFICLABELLING CICIDS2017 FILES
# ==============================================================================================================

print("\n")
print("=" * 140)
print("LOADING ORIGINAL CICIDS2017 TRAFFICLABELLING FILES")
print("=" * 140)

TRAFFIC_FOLDER = "/content/drive/MyDrive/TrafficLabelling "

expected_names = [
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "Monday-WorkingHours.pcap_ISCX.csv",
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "Tuesday-WorkingHours.pcap_ISCX.csv",
    "Wednesday-workingHours.pcap_ISCX.csv",
]

if not os.path.isdir(TRAFFIC_FOLDER):
    raise RuntimeError(
        f"TrafficLabelling folder not found:\n{TRAFFIC_FOLDER}"
    )

# Match filenames case-insensitively, but ONLY inside TrafficLabelling.
folder_files = {
    filename.lower(): os.path.join(TRAFFIC_FOLDER, filename)
    for filename in os.listdir(TRAFFIC_FOLDER)
    if filename.lower().endswith(".csv")
}

raw_files = []

for expected in expected_names:

    key = expected.lower()

    if key not in folder_files:
        raise RuntimeError(
            f"Required TrafficLabelling file not found:\n"
            f"{expected}\n\n"
            f"Folder searched:\n{TRAFFIC_FOLDER}"
        )

    raw_files.append(folder_files[key])


print(f"\n✓ All {len(raw_files)} TrafficLabelling files found.\n")

for i, path in enumerate(raw_files, start=1):

    # Validate header BEFORE any large file is processed.
    header_check = pd.read_csv(
        path,
        nrows=0,
        low_memory=False
    )

    normalized_columns = [
        str(c).replace("\ufeff", "").strip()
        for c in header_check.columns
    ]

    critical = [
        "Flow ID",
        "Source IP",
        "Source Port",
        "Destination IP",
        "Destination Port",
        "Protocol",
        "Timestamp",
        "Label"
    ]

    missing = [
        c for c in critical
        if c not in normalized_columns
    ]

    size_mb = os.path.getsize(path) / (1024 ** 2)

    print(f"{i}. {os.path.basename(path)}")
    print(f"   Size       : {size_mb:.2f} MB")
    print(f"   Columns    : {len(header_check.columns)}")
    print(f"   Timestamp  : {'✓' if 'Timestamp' in normalized_columns else '✗'}")
    print(f"   Source IP  : {'✓' if 'Source IP' in normalized_columns else '✗'}")
    print(f"   Dest. IP   : {'✓' if 'Destination IP' in normalized_columns else '✗'}")
    print(f"   Label      : {'✓' if 'Label' in normalized_columns else '✗'}")
    print(f"   Path       : {path}")

    if missing:
        raise RuntimeError(
            "\nWRONG CICIDS2017 VERSION DETECTED.\n"
            f"File: {path}\n"
            f"Missing critical columns: {missing}\n\n"
            "Cell 25B requires the original TrafficLabelling files, "
            "not the 79-column MachineLearningCVE files."
        )

print("\n✓ Correct timestamped TrafficLabelling dataset verified.")



LOADING ORIGINAL CICIDS2017 TRAFFICLABELLING FILES

✓ All 8 TrafficLabelling files found.

1. Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
   Size       : 91.65 MB
   Columns    : 85
   Timestamp  : ✓
   Source IP  : ✓
   Dest. IP   : ✓
   Label      : ✓
   Path       : /content/drive/MyDrive/TrafficLabelling /Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
2. Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
   Size       : 97.16 MB
   Columns    : 85
   Timestamp  : ✓
   Source IP  : ✓
   Dest. IP   : ✓
   Label      : ✓
   Path       : /content/drive/MyDrive/TrafficLabelling /Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
3. Friday-WorkingHours-Morning.pcap_ISCX.csv
   Size       : 71.89 MB
   Columns    : 85
   Timestamp  : ✓
   Source IP  : ✓
   Dest. IP   : ✓
   Label      : ✓
   Path       : /content/drive/MyDrive/TrafficLabelling /Friday-WorkingHours-Morning.pcap_ISCX.csv
4. Monday-WorkingHours.pcap_ISCX.csv
   Size       : 256.20 MB
   Columns    : 85
   Timest

In [33]:
# ==============================================================================================================
# CELL 25C — ATTACK-TYPE + VICTIM METADATA RECONSTRUCTION — CORRECTED
# ==============================================================================================================

import os
import gc
import numpy as np
import pandas as pd
from collections import Counter, defaultdict

print("=" * 140)
print("CELL 25C — ATTACK-TYPE + VICTIM METADATA RECONSTRUCTION")
print("=" * 140)


# ==============================================================================================================
# 1. VALIDATE REQUIRED OBJECTS
# ==============================================================================================================

if "minute_df" not in globals():
    raise RuntimeError("minute_df not found. Do not continue.")

if "raw_files" not in globals():
    raise RuntimeError("raw_files not found. Run Cell 25B first.")

required_cols = {"Minute", "Segment_ID", "Attack_State"}

missing = required_cols - set(minute_df.columns)

if missing:
    raise RuntimeError(
        f"minute_df missing required columns: {sorted(missing)}"
    )

print("\n✓ Required objects found.")
print(f"minute_df rows : {len(minute_df):,}")
print(f"raw files      : {len(raw_files)}")


# ==============================================================================================================
# 2. CLEAN TIMELINE COPY
# ==============================================================================================================

timeline = minute_df[
    ["Minute", "Segment_ID", "Attack_State"]
].copy()

timeline["Minute"] = pd.to_datetime(
    timeline["Minute"],
    errors="raise"
)

timeline["Segment_ID"] = (
    timeline["Segment_ID"].astype(int)
)

timeline["Attack_State"] = (
    timeline["Attack_State"].astype(int)
)

if timeline.duplicated(
    ["Segment_ID", "Minute"]
).any():
    raise RuntimeError(
        "Duplicate Segment_ID + Minute rows in minute_df."
    )


# ==============================================================================================================
# 3. LOCKED SPLIT FROM CELL 22
# ==============================================================================================================

TRAIN_SEGMENTS = {0, 1, 2, 3, 4}
VALIDATION_SEGMENTS = {5}
TEST_SEGMENTS = {6, 7, 8, 9}


def split_name(segment_id):

    segment_id = int(segment_id)

    if segment_id in TRAIN_SEGMENTS:
        return "TRAIN"

    if segment_id in VALIDATION_SEGMENTS:
        return "VALIDATION"

    if segment_id in TEST_SEGMENTS:
        return "TEST"

    raise ValueError(
        f"Unexpected Segment_ID: {segment_id}"
    )


expected_segments = (
    TRAIN_SEGMENTS
    | VALIDATION_SEGMENTS
    | TEST_SEGMENTS
)

actual_segments = set(
    timeline["Segment_ID"].unique()
)

if actual_segments != expected_segments:
    raise RuntimeError(
        f"Segment mismatch.\n"
        f"Expected: {sorted(expected_segments)}\n"
        f"Found   : {sorted(actual_segments)}"
    )


# ==============================================================================================================
# 4. EXACT SEGMENT BOUNDARIES
# ==============================================================================================================

segment_bounds = (
    timeline
    .groupby("Segment_ID")
    ["Minute"]
    .agg(["min", "max"])
    .reset_index()
    .rename(
        columns={
            "min": "Start_Minute",
            "max": "End_Minute"
        }
    )
)

valid_keys = set(
    zip(
        timeline["Segment_ID"],
        timeline["Minute"]
    )
)

print("\n✓ Locked chronological split preserved.")
print("TRAIN      :", sorted(TRAIN_SEGMENTS))
print("VALIDATION :", sorted(VALIDATION_SEGMENTS))
print("TEST       :", sorted(TEST_SEGMENTS))


# ==============================================================================================================
# 5. FRESH STORAGE
#
# IMPORTANT:
# These are deliberately recreated here.
# Nothing from the failed previous Cell 25C run is reused.
# ==============================================================================================================

minute_attack_counts = defaultdict(Counter)
minute_victim_counts = defaultdict(Counter)

global_attack_flow_counts = Counter()
global_victim_flow_counts = Counter()

total_rows = 0
mapped_rows = 0
attack_rows = 0
invalid_timestamps = 0


# ==============================================================================================================
# 6. PROCESS ALL 8 TRAFFICLABELLING FILES
# ==============================================================================================================

print("\n")
print("=" * 140)
print("PROCESSING TRAFFICLABELLING FILES")
print("=" * 140)

CHUNK_SIZE = 100_000

for file_no, path in enumerate(raw_files, start=1):

    filename = os.path.basename(path)

    print(
        f"\n[{file_no}/8] {filename}"
    )

    # ------------------------------------------------------------------------------------------
    # Latin-1 is intentional.
    #
    # CICIDS2017 TrafficLabelling contains non-UTF8 bytes in at least
    # the WebAttacks file. Latin-1 maps every byte 0–255 and therefore
    # prevents UnicodeDecodeError.
    #
    # Timestamp / Label / Destination IP remain ASCII-compatible.
    # ------------------------------------------------------------------------------------------

    header = pd.read_csv(
        path,
        nrows=0,
        encoding="latin-1"
    )

    normalized = {
        str(c).replace("\ufeff", "").strip(): c
        for c in header.columns
    }

    needed = [
        "Timestamp",
        "Label",
        "Destination IP"
    ]

    missing = [
        x for x in needed
        if x not in normalized
    ]

    if missing:
        raise RuntimeError(
            f"\n{filename}\n"
            f"Missing required columns: {missing}"
        )

    ts_col = normalized["Timestamp"]
    label_col = normalized["Label"]
    dst_col = normalized["Destination IP"]

    file_rows = 0
    file_mapped = 0
    file_attacks = 0

    reader = pd.read_csv(
        path,
        usecols=[
            ts_col,
            label_col,
            dst_col
        ],
        chunksize=CHUNK_SIZE,
        encoding="latin-1",
        low_memory=False
    )

    for chunk_no, chunk in enumerate(
        reader,
        start=1
    ):

        chunk.columns = [
            str(c).replace("\ufeff", "").strip()
            for c in chunk.columns
        ]

        chunk = chunk.rename(
            columns={
                "Destination IP":
                    "Destination_IP"
            }
        )

        n = len(chunk)

        total_rows += n
        file_rows += n


        # ==================================================================
        # TIMESTAMP PARSING
        # ==================================================================

        raw_ts = (
            chunk["Timestamp"]
            .astype(str)
            .str.strip()
        )

        # CICIDS2017 is day/month/year.
        parsed_ts = pd.to_datetime(
            raw_ts,
            dayfirst=True,
            errors="coerce"
        )

        bad_ts = int(
            parsed_ts.isna().sum()
        )

        invalid_timestamps += bad_ts

        chunk["Timestamp"] = parsed_ts

        chunk = chunk.loc[
            chunk["Timestamp"].notna()
        ].copy()

        if chunk.empty:
            continue

        chunk["Minute"] = (
            chunk["Timestamp"]
            .dt.floor("min")
        )


        # ==================================================================
        # MAP TO THE EXISTING VALIDATED SEGMENTS
        # ==================================================================

        chunk["Segment_ID"] = -1

        for b in segment_bounds.itertuples(
            index=False
        ):

            mask = (
                (chunk["Minute"] >= b.Start_Minute)
                &
                (chunk["Minute"] <= b.End_Minute)
            )

            chunk.loc[
                mask,
                "Segment_ID"
            ] = int(b.Segment_ID)


        # ==================================================================
        # EXACT VALIDATED MINUTE FILTER
        # ==================================================================

        keys = zip(
            chunk["Segment_ID"].astype(int),
            chunk["Minute"]
        )

        keep = np.fromiter(
            (
                key in valid_keys
                for key in keys
            ),
            dtype=bool,
            count=len(chunk)
        )

        chunk = chunk.loc[
            keep
        ].copy()

        this_mapped = len(chunk)

        mapped_rows += this_mapped
        file_mapped += this_mapped

        if chunk.empty:
            continue


        # ==================================================================
        # NORMALIZE LABEL
        # ==================================================================

        chunk["Label"] = (
            chunk["Label"]
            .astype(str)
            .str.strip()
        )

        benign = (
            chunk["Label"]
            .str.casefold()
            .eq("benign")
        )

        attacks = chunk.loc[
            ~benign,
            [
                "Segment_ID",
                "Minute",
                "Label",
                "Destination_IP"
            ]
        ].copy()

        this_attacks = len(attacks)

        attack_rows += this_attacks
        file_attacks += this_attacks

        if attacks.empty:
            continue


        attacks["Attack_Type"] = (
            attacks["Label"]
            .astype(str)
            .str.strip()
        )

        attacks["Destination_IP"] = (
            attacks["Destination_IP"]
            .astype(str)
            .str.strip()
        )


        # ==================================================================
        # ATTACK TYPE COUNTS PER MINUTE
        # ==================================================================

        grouped_types = (
            attacks
            .groupby(
                [
                    "Segment_ID",
                    "Minute",
                    "Attack_Type"
                ],
                observed=True
            )
            .size()
        )

        for (
            segment_id,
            minute,
            attack_type
        ), count in grouped_types.items():

            key = (
                int(segment_id),
                minute
            )

            count = int(count)

            minute_attack_counts[
                key
            ][attack_type] += count

            global_attack_flow_counts[
                attack_type
            ] += count


        # ==================================================================
        # VICTIM COUNTS PER MINUTE
        # ==================================================================

        grouped_victims = (
            attacks
            .groupby(
                [
                    "Segment_ID",
                    "Minute",
                    "Destination_IP"
                ],
                observed=True
            )
            .size()
        )

        for (
            segment_id,
            minute,
            victim
        ), count in grouped_victims.items():

            key = (
                int(segment_id),
                minute
            )

            count = int(count)

            minute_victim_counts[
                key
            ][victim] += count

            global_victim_flow_counts[
                victim
            ] += count


        del attacks
        del chunk

        gc.collect()


    print(
        f"    rows        : {file_rows:,}"
    )

    print(
        f"    mapped      : {file_mapped:,}"
    )

    print(
        f"    attack rows : {file_attacks:,}"
    )


# ==============================================================================================================
# 7. INGESTION SUMMARY
# ==============================================================================================================

print("\n")
print("=" * 140)
print("INGESTION SUMMARY")
print("=" * 140)

print(
    f"Total raw rows read        : {total_rows:,}"
)

print(
    f"Mapped to validated minutes: {mapped_rows:,}"
)

print(
    f"Attack flow rows           : {attack_rows:,}"
)

print(
    f"Invalid timestamps         : {invalid_timestamps:,}"
)


# ==============================================================================================================
# 8. BUILD MINUTE-LEVEL METADATA
# ==============================================================================================================

records = []

for r in timeline.itertuples(
    index=False
):

    key = (
        int(r.Segment_ID),
        r.Minute
    )

    type_counter = (
        minute_attack_counts.get(
            key,
            Counter()
        )
    )

    victim_counter = (
        minute_victim_counts.get(
            key,
            Counter()
        )
    )

    attack_types = tuple(
        sorted(type_counter.keys())
    )

    victims = tuple(
        sorted(victim_counter.keys())
    )

    attack_flow_count = int(
        sum(type_counter.values())
    )

    reconstructed_state = int(
        attack_flow_count > 0
    )

    if type_counter:

        dominant_type, dominant_type_count = (
            type_counter.most_common(1)[0]
        )

    else:

        dominant_type = "BENIGN"
        dominant_type_count = 0


    if victim_counter:

        dominant_victim, dominant_victim_count = (
            victim_counter.most_common(1)[0]
        )

    else:

        dominant_victim = None
        dominant_victim_count = 0


    records.append(
        {
            "Segment_ID":
                int(r.Segment_ID),

            "Minute":
                r.Minute,

            "Final_Split":
                split_name(r.Segment_ID),

            "Reconstructed_Attack_State":
                reconstructed_state,

            "Attack_Flow_Count":
                attack_flow_count,

            "Attack_Type_Count":
                len(attack_types),

            "Attack_Types_Present":
                attack_types,

            "Dominant_Attack_Type":
                dominant_type,

            "Dominant_Attack_Flow_Count":
                int(dominant_type_count),

            "Victim_Count":
                len(victims),

            "Victims_Present":
                victims,

            "Dominant_Victim_IP":
                dominant_victim,

            "Dominant_Victim_Attack_Flow_Count":
                int(dominant_victim_count)
        }
    )


cell25c_minute_metadata = (
    pd.DataFrame(records)
)


# ==============================================================================================================
# 9. BINARY CONSISTENCY CHECK
# ==============================================================================================================

check = timeline.merge(
    cell25c_minute_metadata[
        [
            "Segment_ID",
            "Minute",
            "Reconstructed_Attack_State"
        ]
    ],
    on=[
        "Segment_ID",
        "Minute"
    ],
    how="left",
    validate="one_to_one"
)

check["Match"] = (
    check["Attack_State"]
    ==
    check["Reconstructed_Attack_State"]
)

mismatch_count = int(
    (~check["Match"]).sum()
)

cell25c_binary_mismatches = (
    check.loc[
        ~check["Match"]
    ].copy()
)


print("\n")
print("=" * 140)
print("BINARY TARGET CONSISTENCY CHECK")
print("=" * 140)

print(
    f"Existing Attack_State positives : "
    f"{int(check['Attack_State'].sum()):,}"
)

print(
    f"Reconstructed positives         : "
    f"{int(check['Reconstructed_Attack_State'].sum()):,}"
)

print(
    f"Matches                         : "
    f"{int(check['Match'].sum()):,} / {len(check):,}"
)

print(
    f"Mismatches                      : "
    f"{mismatch_count:,}"
)

print(
    f"Agreement                       : "
    f"{100 * check['Match'].mean():.4f}%"
)

if mismatch_count == 0:

    print(
        "\n✓ Perfect binary agreement."
    )

else:

    print(
        "\n⚠ Mismatches found. Existing Attack_State has NOT been changed."
    )

    print(
        cell25c_binary_mismatches
        .head(20)
        .to_string(index=False)
    )


# ==============================================================================================================
# 10. ATTACK FLOW DISTRIBUTION
# ==============================================================================================================

cell25c_attack_flow_distribution = (
    pd.DataFrame(
        [
            {
                "Attack_Type": k,
                "Attack_Flows": v
            }
            for k, v
            in global_attack_flow_counts.items()
        ]
    )
)

if not cell25c_attack_flow_distribution.empty:

    cell25c_attack_flow_distribution = (
        cell25c_attack_flow_distribution
        .sort_values(
            "Attack_Flows",
            ascending=False
        )
        .reset_index(drop=True)
    )

    total_attack_flows = (
        cell25c_attack_flow_distribution[
            "Attack_Flows"
        ].sum()
    )

    cell25c_attack_flow_distribution[
        "Percent"
    ] = (
        100
        * cell25c_attack_flow_distribution[
            "Attack_Flows"
        ]
        / total_attack_flows
    )


print("\n")
print("=" * 140)
print("ATTACK-TYPE FLOW DISTRIBUTION")
print("=" * 140)

print(
    cell25c_attack_flow_distribution
    .to_string(
        index=False,
        formatters={
            "Percent":
                "{:.4f}".format
        }
    )
)


# ==============================================================================================================
# 11. ATTACK TYPE MINUTE SUPPORT BY SPLIT
# ==============================================================================================================

type_records = []

for r in cell25c_minute_metadata.itertuples(
    index=False
):

    for attack_type in r.Attack_Types_Present:

        type_records.append(
            {
                "Attack_Type":
                    attack_type,

                "Final_Split":
                    r.Final_Split,

                "Segment_ID":
                    r.Segment_ID,

                "Minute":
                    r.Minute
            }
        )


cell25c_attack_type_minutes = (
    pd.DataFrame(type_records)
)

if not cell25c_attack_type_minutes.empty:

    support = (
        cell25c_attack_type_minutes
        .groupby(
            [
                "Attack_Type",
                "Final_Split"
            ]
        )
        .size()
        .unstack(
            fill_value=0
        )
    )

    for s in [
        "TRAIN",
        "VALIDATION",
        "TEST"
    ]:

        if s not in support.columns:
            support[s] = 0

    support = (
        support[
            [
                "TRAIN",
                "VALIDATION",
                "TEST"
            ]
        ]
        .reset_index()
    )

    support[
        "Total_Attack_Minutes"
    ] = (
        support[
            [
                "TRAIN",
                "VALIDATION",
                "TEST"
            ]
        ]
        .sum(axis=1)
    )

    cell25c_attack_type_support = (
        support
        .sort_values(
            "Total_Attack_Minutes",
            ascending=False
        )
        .reset_index(drop=True)
    )

else:

    cell25c_attack_type_support = (
        pd.DataFrame()
    )


print("\n")
print("=" * 140)
print("ATTACK-TYPE MINUTE SUPPORT BY LOCKED SPLIT")
print("=" * 140)

print(
    cell25c_attack_type_support
    .to_string(index=False)
)


# ==============================================================================================================
# 12. DOMINANT ATTACK SUPPORT
# ==============================================================================================================

dominant = (
    cell25c_minute_metadata.loc[
        cell25c_minute_metadata[
            "Dominant_Attack_Type"
        ] != "BENIGN"
    ]
)

if not dominant.empty:

    dom_support = (
        dominant
        .groupby(
            [
                "Dominant_Attack_Type",
                "Final_Split"
            ]
        )
        .size()
        .unstack(
            fill_value=0
        )
    )

    for s in [
        "TRAIN",
        "VALIDATION",
        "TEST"
    ]:

        if s not in dom_support.columns:
            dom_support[s] = 0

    dom_support = (
        dom_support[
            [
                "TRAIN",
                "VALIDATION",
                "TEST"
            ]
        ]
        .reset_index()
    )

    dom_support[
        "Total_Dominant_Minutes"
    ] = (
        dom_support[
            [
                "TRAIN",
                "VALIDATION",
                "TEST"
            ]
        ].sum(axis=1)
    )

    cell25c_dominant_attack_support = (
        dom_support
        .sort_values(
            "Total_Dominant_Minutes",
            ascending=False
        )
        .reset_index(drop=True)
    )

else:

    cell25c_dominant_attack_support = (
        pd.DataFrame()
    )


print("\n")
print("=" * 140)
print("DOMINANT ATTACK-TYPE SUPPORT BY LOCKED SPLIT")
print("=" * 140)

print(
    cell25c_dominant_attack_support
    .to_string(index=False)
)


# ==============================================================================================================
# 13. MULTI-TYPE AUDIT
# ==============================================================================================================

attack_minutes = (
    cell25c_minute_metadata.loc[
        cell25c_minute_metadata[
            "Reconstructed_Attack_State"
        ] == 1
    ]
)

single_type = int(
    (
        attack_minutes[
            "Attack_Type_Count"
        ] == 1
    ).sum()
)

multi_type = int(
    (
        attack_minutes[
            "Attack_Type_Count"
        ] > 1
    ).sum()
)


print("\n")
print("=" * 140)
print("MULTI-ATTACK MINUTE AUDIT")
print("=" * 140)

print(
    f"Attack-positive minutes : {len(attack_minutes):,}"
)

print(
    f"Single-type minutes     : {single_type:,}"
)

print(
    f"Multi-type minutes      : {multi_type:,}"
)

if len(attack_minutes):

    print(
        f"Multi-type percent      : "
        f"{100 * multi_type / len(attack_minutes):.4f}%"
    )


# ==============================================================================================================
# 14. VICTIM SUPPORT BY SPLIT
# ==============================================================================================================

victim_records = []

for r in cell25c_minute_metadata.itertuples(
    index=False
):

    for victim in r.Victims_Present:

        victim_records.append(
            {
                "Victim_IP":
                    victim,

                "Final_Split":
                    r.Final_Split,

                "Segment_ID":
                    r.Segment_ID,

                "Minute":
                    r.Minute
            }
        )


cell25c_victim_minutes = (
    pd.DataFrame(victim_records)
)

if not cell25c_victim_minutes.empty:

    victim_support = (
        cell25c_victim_minutes
        .groupby(
            [
                "Victim_IP",
                "Final_Split"
            ]
        )
        .size()
        .unstack(
            fill_value=0
        )
    )

    for s in [
        "TRAIN",
        "VALIDATION",
        "TEST"
    ]:

        if s not in victim_support.columns:
            victim_support[s] = 0

    victim_support = (
        victim_support[
            [
                "TRAIN",
                "VALIDATION",
                "TEST"
            ]
        ]
        .reset_index()
    )

    victim_support[
        "Total_Attack_Minutes"
    ] = (
        victim_support[
            [
                "TRAIN",
                "VALIDATION",
                "TEST"
            ]
        ].sum(axis=1)
    )

    cell25c_victim_support = (
        victim_support
        .sort_values(
            [
                "TRAIN",
                "Total_Attack_Minutes"
            ],
            ascending=False
        )
        .reset_index(drop=True)
    )

else:

    cell25c_victim_support = (
        pd.DataFrame()
    )


print("\n")
print("=" * 140)
print("TOP VICTIM IPs BY TRAIN SUPPORT")
print("=" * 140)

print(
    cell25c_victim_support
    .head(30)
    .to_string(index=False)
)


# ==============================================================================================================
# 15. VICTIM CROSS-SPLIT OVERLAP
# ==============================================================================================================

def victims_for(split):

    if cell25c_victim_minutes.empty:
        return set()

    return set(
        cell25c_victim_minutes.loc[
            cell25c_victim_minutes[
                "Final_Split"
            ] == split,
            "Victim_IP"
        ]
    )


train_victims = victims_for("TRAIN")
val_victims = victims_for("VALIDATION")
test_victims = victims_for("TEST")

val_overlap = (
    train_victims
    & val_victims
)

test_overlap = (
    train_victims
    & test_victims
)


cell25c_victim_overlap_summary = (
    pd.DataFrame(
        [
            [
                "Unique TRAIN victims",
                len(train_victims)
            ],

            [
                "Unique VALIDATION victims",
                len(val_victims)
            ],

            [
                "VALIDATION victims seen in TRAIN",
                len(val_overlap)
            ],

            [
                "VALIDATION overlap percent",
                (
                    100
                    * len(val_overlap)
                    / len(val_victims)
                    if val_victims
                    else 0
                )
            ],

            [
                "Unique TEST victims",
                len(test_victims)
            ],

            [
                "TEST victims seen in TRAIN",
                len(test_overlap)
            ],

            [
                "TEST overlap percent",
                (
                    100
                    * len(test_overlap)
                    / len(test_victims)
                    if test_victims
                    else 0
                )
            ]
        ],
        columns=[
            "Metric",
            "Value"
        ]
    )
)


print("\n")
print("=" * 140)
print("VICTIM CROSS-SPLIT OVERLAP")
print("=" * 140)

print(
    cell25c_victim_overlap_summary
    .to_string(index=False)
)


# ==============================================================================================================
# 16. ENRICHED DATAFRAME — DO NOT OVERWRITE minute_df
# ==============================================================================================================

cell25c_minute_df_enriched = (
    minute_df.merge(
        cell25c_minute_metadata,
        on=[
            "Segment_ID",
            "Minute"
        ],
        how="left",
        validate="one_to_one"
    )
)

if (
    len(cell25c_minute_df_enriched)
    !=
    len(minute_df)
):
    raise RuntimeError(
        "Metadata merge changed timeline row count."
    )


# ==============================================================================================================
# 17. COMPLETE
# ==============================================================================================================

CELL25C_COMPLETE = True

print("\n")
print("=" * 140)
print("CELL 25C — FINAL RECONSTRUCTION SUMMARY")
print("=" * 140)

print(
    f"Timeline minutes               : {len(minute_df):,}"
)

print(
    f"Attack-positive minutes        : "
    f"{int(cell25c_minute_metadata['Reconstructed_Attack_State'].sum()):,}"
)

print(
    f"Distinct attack labels         : "
    f"{len(global_attack_flow_counts):,}"
)

print(
    f"Distinct attacked destinations : "
    f"{len(global_victim_flow_counts):,}"
)

print(
    f"Binary mismatches              : {mismatch_count:,}"
)

print(
    f"Multi-type attack minutes      : {multi_type:,}"
)

print("\n✓ Metadata reconstruction complete.")
print("✓ Original minute_df NOT overwritten.")
print("✓ Locked chronological split NOT changed.")
print("✓ No attack types selected yet.")
print("✓ No victims selected yet.")
print("✓ No model trained.")

print("\nNEXT: Cell 26 — final target definition.")

print("=" * 140)

CELL 25C — ATTACK-TYPE + VICTIM METADATA RECONSTRUCTION

✓ Required objects found.
minute_df rows : 2,454
raw files      : 8

✓ Locked chronological split preserved.
TRAIN      : [0, 1, 2, 3, 4]
VALIDATION : [5]
TEST       : [6, 7, 8, 9]


PROCESSING TRAFFICLABELLING FILES

[1/8] Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
    rows        : 225,745
    mapped      : 225,745
    attack rows : 128,027

[2/8] Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
    rows        : 286,467
    mapped      : 286,467
    attack rows : 158,930

[3/8] Friday-WorkingHours-Morning.pcap_ISCX.csv
    rows        : 191,033
    mapped      : 191,033
    attack rows : 1,966

[4/8] Monday-WorkingHours.pcap_ISCX.csv
    rows        : 529,918
    mapped      : 529,918
    attack rows : 0

[5/8] Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
    rows        : 288,602
    mapped      : 288,602
    attack rows : 36

[6/8] Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
    rows      

In [34]:
# ==============================================================================================================
# CELL 26 — FINAL MULTI-OUTPUT TARGET CONSTRUCTION + ATTACK-TYPE TEMPORAL FEASIBILITY AUDIT
# ==============================================================================================================

import numpy as np
import pandas as pd
from collections import defaultdict

print("=" * 150)
print("CELL 26 — FINAL MULTI-OUTPUT TARGET CONSTRUCTION + ATTACK-TYPE TEMPORAL FEASIBILITY AUDIT")
print("=" * 150)


# ==============================================================================================================
# 1. REQUIRED OBJECTS
# ==============================================================================================================

required_objects = [
    "minute_df",
    "cell25c_minute_metadata",
    "cell25c_attack_type_support",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Missing required objects:\n"
        + "\n".join(f"  - {x}" for x in missing_objects)
        + "\nRun Cell 25C first."
    )


required_minute_cols = {
    "Minute",
    "Segment_ID",
    "Attack_State"
}

missing_cols = (
    required_minute_cols
    - set(minute_df.columns)
)

if missing_cols:
    raise RuntimeError(
        f"minute_df missing columns: {sorted(missing_cols)}"
    )


print("\n✓ Required Cell 25C objects found.")
print(f"Timeline rows : {len(minute_df):,}")
print(
    f"Segments      : "
    f"{minute_df['Segment_ID'].nunique():,}"
)


# ==============================================================================================================
# 2. LOCK EXPERIMENT CONSTANTS
# ==============================================================================================================

HISTORY_LENGTH = 15

FORECAST_HORIZONS = [
    1,
    5,
    10,
    15,
    30
]

TRAIN_SEGMENTS = {
    0, 1, 2, 3, 4
}

VALIDATION_SEGMENTS = {
    5
}

TEST_SEGMENTS = {
    6, 7, 8, 9
}


def split_from_segment(segment_id):

    segment_id = int(segment_id)

    if segment_id in TRAIN_SEGMENTS:
        return "TRAIN"

    if segment_id in VALIDATION_SEGMENTS:
        return "VALIDATION"

    if segment_id in TEST_SEGMENTS:
        return "TEST"

    raise ValueError(
        f"Unknown Segment_ID: {segment_id}"
    )


print("\nForecasting design:")
print(
    f"History length : {HISTORY_LENGTH} minutes"
)
print(
    f"Horizons       : {FORECAST_HORIZONS}"
)

print("\nPrimary chronological split:")
print(
    f"TRAIN      : {sorted(TRAIN_SEGMENTS)}"
)
print(
    f"VALIDATION : {sorted(VALIDATION_SEGMENTS)}"
)
print(
    f"TEST       : {sorted(TEST_SEGMENTS)}"
)


# ==============================================================================================================
# 3. CLEAN ATTACK-TYPE NAMES
#
# IMPORTANT:
# Latin-1 decoding caused the en-dash in Web Attack labels to appear as
# the byte-compatible character "–/" depending on display.
#
# We normalize only representation.
# We do NOT merge different attack classes.
# ==============================================================================================================

def normalize_attack_label(label):

    if label is None:
        return None

    label = str(label).strip()

    # Normalize the known CICIDS2017 Web Attack separator.
    label = label.replace("\x96", "-")
    label = label.replace("–", "-")
    label = label.replace("—", "-")
    label = label.replace("", "-")

    # Normalize whitespace.
    label = " ".join(
        label.split()
    )

    return label


metadata = (
    cell25c_minute_metadata.copy()
)

metadata["Minute"] = pd.to_datetime(
    metadata["Minute"],
    errors="raise"
)

metadata["Dominant_Attack_Type"] = (
    metadata["Dominant_Attack_Type"]
    .map(normalize_attack_label)
)

metadata["Attack_Types_Present"] = (
    metadata["Attack_Types_Present"]
    .apply(
        lambda values: tuple(
            normalize_attack_label(v)
            for v in values
        )
    )
)


# ==============================================================================================================
# 4. LOCK ALL OBSERVED ATTACK TYPES
#
# NO TOP-5 FILTER.
# NO FREQUENCY FILTER.
# NO TEST-PERFORMANCE FILTER.
# ==============================================================================================================

observed_attack_types = sorted(
    {
        attack_type

        for values
        in metadata[
            "Attack_Types_Present"
        ]

        for attack_type
        in values

        if attack_type
        and attack_type.upper() != "BENIGN"
    }
)

ALL_ATTACK_TYPES = (
    observed_attack_types.copy()
)

ATTACK_TYPE_TO_ID = {
    attack_type: idx
    for idx, attack_type
    in enumerate(ALL_ATTACK_TYPES)
}

ATTACK_ID_TO_TYPE = {
    idx: attack_type
    for attack_type, idx
    in ATTACK_TYPE_TO_ID.items()
}


print("\n")
print("=" * 150)
print("LOCKED ATTACK-TYPE VOCABULARY")
print("=" * 150)

print(
    f"Observed attack classes : "
    f"{len(ALL_ATTACK_TYPES)}"
)

for idx, attack_type in enumerate(
    ALL_ATTACK_TYPES
):

    print(
        f"{idx:2d} -> {attack_type}"
    )


if len(ALL_ATTACK_TYPES) != 14:

    print(
        "\n⚠ Expected 14 CICIDS2017 attack labels "
        f"from Cell 25C, but found "
        f"{len(ALL_ATTACK_TYPES)}."
    )

else:

    print(
        "\n✓ All 14 observed attack types retained."
    )


# ==============================================================================================================
# 5. CREATE ONE EXACT TEMPORAL TABLE
# ==============================================================================================================

timeline = (
    minute_df[
        [
            "Minute",
            "Segment_ID",
            "Attack_State"
        ]
    ]
    .copy()
)

timeline["Minute"] = pd.to_datetime(
    timeline["Minute"],
    errors="raise"
)

timeline = timeline.merge(
    metadata[
        [
            "Minute",
            "Segment_ID",
            "Dominant_Attack_Type",
            "Attack_Types_Present",
            "Dominant_Victim_IP",
            "Victims_Present"
        ]
    ],
    on=[
        "Minute",
        "Segment_ID"
    ],
    how="left",
    validate="one_to_one"
)


if len(timeline) != len(minute_df):

    raise RuntimeError(
        "Metadata merge changed timeline size."
    )


timeline["Final_Split"] = (
    timeline["Segment_ID"]
    .map(split_from_segment)
)


# ==============================================================================================================
# 6. VALIDATE ONE ATTACK TYPE PER ATTACK-POSITIVE MINUTE
#
# Cell 25C reported zero multi-type attack minutes.
# We verify that again before using a single multiclass target.
# ==============================================================================================================

timeline[
    "Attack_Type_Count_Check"
] = (
    timeline[
        "Attack_Types_Present"
    ]
    .apply(
        lambda x:
        len(x)
        if isinstance(x, tuple)
        else 0
    )
)


attack_rows = timeline.loc[
    timeline["Attack_State"] == 1
]

multi_type_rows = attack_rows.loc[
    attack_rows[
        "Attack_Type_Count_Check"
    ] > 1
]


print("\n")
print("=" * 150)
print("MULTICLASS TARGET VALIDITY")
print("=" * 150)

print(
    f"Attack-positive minutes : "
    f"{len(attack_rows):,}"
)

print(
    f"Multi-type minutes      : "
    f"{len(multi_type_rows):,}"
)


if len(multi_type_rows) > 0:

    raise RuntimeError(
        "Multiple attack types occur in the same minute. "
        "A single multiclass target would not be valid."
    )

print(
    "✓ Exactly one attack class can represent "
    "each attack-positive minute."
)


# ==============================================================================================================
# 7. CURRENT-MINUTE CLASS TARGET
#
# Class 0 = BENIGN
# Classes 1..14 = observed attack types
#
# This is NOT yet a model target.
# It is the canonical state representation used to construct FUTURE targets.
# ==============================================================================================================

STATE_CLASSES = [
    "BENIGN"
] + ALL_ATTACK_TYPES

STATE_TO_ID = {
    state: idx
    for idx, state
    in enumerate(STATE_CLASSES)
}

ID_TO_STATE = {
    idx: state
    for state, idx
    in STATE_TO_ID.items()
}


def current_state_label(row):

    if int(row["Attack_State"]) == 0:
        return "BENIGN"

    return normalize_attack_label(
        row["Dominant_Attack_Type"]
    )


timeline["Current_State_Label"] = (
    timeline.apply(
        current_state_label,
        axis=1
    )
)

timeline["Current_State_ID"] = (
    timeline[
        "Current_State_Label"
    ]
    .map(STATE_TO_ID)
)


if timeline[
    "Current_State_ID"
].isna().any():

    bad = timeline.loc[
        timeline[
            "Current_State_ID"
        ].isna()
    ]

    raise RuntimeError(
        "Unknown current-state labels detected:\n"
        + str(
            bad[
                [
                    "Minute",
                    "Segment_ID",
                    "Current_State_Label"
                ]
            ].head(20)
        )
    )


# ==============================================================================================================
# 8. BUILD FUTURE TARGETS WITHOUT CROSSING SEGMENTS
#
# IMPORTANT:
# shift(-H) is performed WITHIN each Segment_ID.
# Therefore no target crosses a capture discontinuity.
# ==============================================================================================================

timeline = (
    timeline
    .sort_values(
        [
            "Segment_ID",
            "Minute"
        ]
    )
    .reset_index(drop=True)
)


for horizon in FORECAST_HORIZONS:

    grouped = timeline.groupby(
        "Segment_ID",
        sort=False
    )

    timeline[
        f"Future_Attack_H{horizon}"
    ] = (
        grouped[
            "Attack_State"
        ]
        .shift(-horizon)
    )

    timeline[
        f"Future_State_Label_H{horizon}"
    ] = (
        grouped[
            "Current_State_Label"
        ]
        .shift(-horizon)
    )

    timeline[
        f"Future_State_ID_H{horizon}"
    ] = (
        timeline[
            f"Future_State_Label_H{horizon}"
        ]
        .map(STATE_TO_ID)
    )

    timeline[
        f"Future_Victim_H{horizon}"
    ] = (
        grouped[
            "Dominant_Victim_IP"
        ]
        .shift(-horizon)
    )


# ==============================================================================================================
# 9. VALIDATE TARGET ALIGNMENT
#
# Because minute_df contains one row per minute inside each validated segment,
# row shift H should equal exactly H minutes.
# ==============================================================================================================

alignment_records = []


for horizon in FORECAST_HORIZONS:

    future_time = (
        timeline
        .groupby(
            "Segment_ID",
            sort=False
        )["Minute"]
        .shift(-horizon)
    )

    valid = future_time.notna()

    observed_delta = (
        future_time[valid]
        -
        timeline.loc[
            valid,
            "Minute"
        ]
    ).dt.total_seconds() / 60.0

    invalid_count = int(
        (
            observed_delta
            != horizon
        ).sum()
    )

    alignment_records.append(
        {
            "Horizon":
                horizon,

            "Valid_Target_Rows":
                int(valid.sum()),

            "Invalid_Time_Alignment":
                invalid_count
        }
    )


cell26_target_alignment = (
    pd.DataFrame(
        alignment_records
    )
)


print("\n")
print("=" * 150)
print("FUTURE TARGET ALIGNMENT VALIDATION")
print("=" * 150)

print(
    cell26_target_alignment
    .to_string(index=False)
)


if (
    cell26_target_alignment[
        "Invalid_Time_Alignment"
    ].sum()
    != 0
):

    raise RuntimeError(
        "Future target alignment error detected."
    )

print(
    "\n✓ Every future target is exactly H minutes ahead."
)


# ==============================================================================================================
# 10. CONSTRUCT VALID 15-MINUTE FORECAST ORIGINS
#
# We only retain t where:
#
#     t-14 ... t
#
# exists continuously inside the SAME segment.
#
# This is the input sequence ending at forecast origin t.
# ==============================================================================================================

valid_origin_records = []


for segment_id, seg in timeline.groupby(
    "Segment_ID",
    sort=True
):

    seg = (
        seg
        .sort_values("Minute")
        .reset_index()
    )

    for local_i in range(
        HISTORY_LENGTH - 1,
        len(seg)
    ):

        history_start_i = (
            local_i
            - HISTORY_LENGTH
            + 1
        )

        history_start_time = (
            seg.loc[
                history_start_i,
                "Minute"
            ]
        )

        origin_time = (
            seg.loc[
                local_i,
                "Minute"
            ]
        )

        expected_duration = (
            HISTORY_LENGTH - 1
        )

        actual_duration = (
            origin_time
            - history_start_time
        ).total_seconds() / 60.0

        if actual_duration != expected_duration:
            continue

        original_index = int(
            seg.loc[
                local_i,
                "index"
            ]
        )

        valid_origin_records.append(
            {
                "Timeline_Index":
                    original_index,

                "Segment_ID":
                    int(segment_id),

                "Final_Split":
                    split_from_segment(
                        segment_id
                    ),

                "History_Start":
                    history_start_time,

                "Forecast_Origin":
                    origin_time
            }
        )


cell26_valid_origins = (
    pd.DataFrame(
        valid_origin_records
    )
)


print("\n")
print("=" * 150)
print("VALID 15-MINUTE FORECAST ORIGINS")
print("=" * 150)

print(
    cell26_valid_origins
    .groupby("Final_Split")
    .size()
    .rename("Valid_Origins")
    .to_string()
)

print(
    f"\nTotal valid origins : "
    f"{len(cell26_valid_origins):,}"
)


# ==============================================================================================================
# 11. FUTURE BINARY + MULTICLASS SUPPORT AT EACH HORIZON
# ==============================================================================================================

future_support_records = []


for horizon in FORECAST_HORIZONS:

    target_col = (
        f"Future_State_Label_H{horizon}"
    )

    attack_col = (
        f"Future_Attack_H{horizon}"
    )

    origin_subset = (
        timeline.loc[
            cell26_valid_origins[
                "Timeline_Index"
            ].values
        ]
        .copy()
    )

    origin_subset = (
        origin_subset.loc[
            origin_subset[
                target_col
            ].notna()
        ]
    )

    for split in [
        "TRAIN",
        "VALIDATION",
        "TEST"
    ]:

        part = (
            origin_subset.loc[
                origin_subset[
                    "Final_Split"
                ] == split
            ]
        )

        future_support_records.append(
            {
                "Horizon":
                    horizon,

                "Split":
                    split,

                "Samples":
                    len(part),

                "Future_Benign":
                    int(
                        (
                            part[
                                target_col
                            ]
                            == "BENIGN"
                        ).sum()
                    ),

                "Future_Attack":
                    int(
                        (
                            part[
                                target_col
                            ]
                            != "BENIGN"
                        ).sum()
                    ),

                "Distinct_Attack_Types":
                    int(
                        part.loc[
                            part[
                                target_col
                            ]
                            != "BENIGN",
                            target_col
                        ].nunique()
                    )
            }
        )


cell26_future_support = (
    pd.DataFrame(
        future_support_records
    )
)


print("\n")
print("=" * 150)
print("GENERAL FORECAST TARGET SUPPORT")
print("=" * 150)

print(
    cell26_future_support
    .to_string(index=False)
)


# ==============================================================================================================
# 12. ATTACK-TYPE SUPPORT AT EACH HORIZON AND SPLIT
#
# This shows exactly why the primary chronological split cannot be blindly
# used as an ordinary closed-set 14-class attack classifier.
# ==============================================================================================================

attack_type_horizon_records = []


for horizon in FORECAST_HORIZONS:

    target_col = (
        f"Future_State_Label_H{horizon}"
    )

    origins = (
        timeline.loc[
            cell26_valid_origins[
                "Timeline_Index"
            ].values
        ]
        .copy()
    )

    origins = origins.loc[
        origins[target_col].notna()
    ]

    origins = origins.loc[
        origins[target_col] != "BENIGN"
    ]

    for attack_type in ALL_ATTACK_TYPES:

        record = {
            "Horizon":
                horizon,

            "Attack_Type":
                attack_type
        }

        for split in [
            "TRAIN",
            "VALIDATION",
            "TEST"
        ]:

            record[split] = int(
                (
                    (
                        origins[
                            "Final_Split"
                        ] == split
                    )
                    &
                    (
                        origins[
                            target_col
                        ] == attack_type
                    )
                ).sum()
            )

        record[
            "Total"
        ] = (
            record["TRAIN"]
            +
            record["VALIDATION"]
            +
            record["TEST"]
        )

        attack_type_horizon_records.append(
            record
        )


cell26_attack_type_horizon_support = (
    pd.DataFrame(
        attack_type_horizon_records
    )
)


print("\n")
print("=" * 150)
print("ATTACK-TYPE FORECAST SUPPORT — PRIMARY CHRONOLOGICAL SPLIT")
print("=" * 150)


for horizon in FORECAST_HORIZONS:

    print(
        f"\nHORIZON = +{horizon} MINUTES"
    )

    print("-" * 100)

    display_df = (
        cell26_attack_type_horizon_support.loc[
            cell26_attack_type_horizon_support[
                "Horizon"
            ] == horizon,
            [
                "Attack_Type",
                "TRAIN",
                "VALIDATION",
                "TEST",
                "Total"
            ]
        ]
        .sort_values(
            "Total",
            ascending=False
        )
    )

    print(
        display_df.to_string(
            index=False
        )
    )


# ==============================================================================================================
# 13. PRIMARY-SPLIT CLOSED-SET FEASIBILITY
# ==============================================================================================================

closed_set_records = []


for horizon in FORECAST_HORIZONS:

    hdf = (
        cell26_attack_type_horizon_support.loc[
            cell26_attack_type_horizon_support[
                "Horizon"
            ] == horizon
        ]
    )

    train_types = set(
        hdf.loc[
            hdf["TRAIN"] > 0,
            "Attack_Type"
        ]
    )

    val_types = set(
        hdf.loc[
            hdf["VALIDATION"] > 0,
            "Attack_Type"
        ]
    )

    test_types = set(
        hdf.loc[
            hdf["TEST"] > 0,
            "Attack_Type"
        ]
    )

    unseen_val = (
        val_types
        - train_types
    )

    unseen_test = (
        test_types
        - train_types
    )

    closed_set_records.append(
        {
            "Horizon":
                horizon,

            "Train_Attack_Types":
                len(train_types),

            "Validation_Attack_Types":
                len(val_types),

            "Test_Attack_Types":
                len(test_types),

            "Validation_Types_Unseen_In_Train":
                len(unseen_val),

            "Test_Types_Unseen_In_Train":
                len(unseen_test),

            "Closed_Set_14_Class_Feasible":
                (
                    len(train_types)
                    == len(ALL_ATTACK_TYPES)
                    and
                    len(unseen_val) == 0
                    and
                    len(unseen_test) == 0
                )
        }
    )


cell26_primary_multiclass_feasibility = (
    pd.DataFrame(
        closed_set_records
    )
)


print("\n")
print("=" * 150)
print("PRIMARY CHRONOLOGICAL SPLIT — 14-CLASS FEASIBILITY")
print("=" * 150)

print(
    cell26_primary_multiclass_feasibility
    .to_string(index=False)
)


# ==============================================================================================================
# 14. RECONSTRUCT ATTACK EPISODES
#
# An episode is a maximal consecutive run of the SAME attack type
# inside the SAME validated capture segment.
#
# This is critical because adjacent attack minutes are NOT independent.
# ==============================================================================================================

episode_records = []

episode_id = 0


for segment_id, seg in timeline.groupby(
    "Segment_ID",
    sort=True
):

    seg = (
        seg
        .sort_values("Minute")
        .reset_index(drop=True)
    )

    current_type = None
    current_start = None
    current_end = None
    current_minutes = 0


    def close_episode():

        nonlocal_dummy = None


    for i, row in seg.iterrows():

        state = row[
            "Current_State_Label"
        ]

        minute = row[
            "Minute"
        ]

        if state == "BENIGN":

            if current_type is not None:

                episode_records.append(
                    {
                        "Episode_ID":
                            episode_id,

                        "Segment_ID":
                            int(segment_id),

                        "Attack_Type":
                            current_type,

                        "Start_Minute":
                            current_start,

                        "End_Minute":
                            current_end,

                        "Attack_Minutes":
                            current_minutes,

                        "Original_Split":
                            split_from_segment(
                                segment_id
                            )
                    }
                )

                episode_id += 1

                current_type = None
                current_start = None
                current_end = None
                current_minutes = 0

            continue


        if current_type is None:

            current_type = state
            current_start = minute
            current_end = minute
            current_minutes = 1

            continue


        expected_next = (
            current_end
            +
            pd.Timedelta(minutes=1)
        )


        if (
            state == current_type
            and
            minute == expected_next
        ):

            current_end = minute
            current_minutes += 1

        else:

            episode_records.append(
                {
                    "Episode_ID":
                        episode_id,

                    "Segment_ID":
                        int(segment_id),

                    "Attack_Type":
                        current_type,

                    "Start_Minute":
                        current_start,

                    "End_Minute":
                        current_end,

                    "Attack_Minutes":
                        current_minutes,

                    "Original_Split":
                        split_from_segment(
                            segment_id
                        )
                }
            )

            episode_id += 1

            current_type = state
            current_start = minute
            current_end = minute
            current_minutes = 1


    if current_type is not None:

        episode_records.append(
            {
                "Episode_ID":
                    episode_id,

                "Segment_ID":
                    int(segment_id),

                "Attack_Type":
                    current_type,

                "Start_Minute":
                    current_start,

                "End_Minute":
                    current_end,

                "Attack_Minutes":
                    current_minutes,

                "Original_Split":
                    split_from_segment(
                        segment_id
                    )
            }
        )

        episode_id += 1


cell26_attack_episodes = (
    pd.DataFrame(
        episode_records
    )
)


# ==============================================================================================================
# 15. ATTACK EPISODE SUPPORT
# ==============================================================================================================

episode_support = (
    cell26_attack_episodes
    .groupby("Attack_Type")
    .agg(
        Episode_Count=(
            "Episode_ID",
            "nunique"
        ),

        Segment_Count=(
            "Segment_ID",
            "nunique"
        ),

        Attack_Minutes=(
            "Attack_Minutes",
            "sum"
        ),

        Median_Episode_Minutes=(
            "Attack_Minutes",
            "median"
        ),

        Max_Episode_Minutes=(
            "Attack_Minutes",
            "max"
        )
    )
    .reset_index()
)


# Make sure all 14 classes appear.
episode_support = (
    pd.DataFrame(
        {
            "Attack_Type":
                ALL_ATTACK_TYPES
        }
    )
    .merge(
        episode_support,
        on="Attack_Type",
        how="left"
    )
    .fillna(0)
)


cell26_attack_episode_support = (
    episode_support
    .sort_values(
        [
            "Episode_Count",
            "Attack_Minutes"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)


print("\n")
print("=" * 150)
print("INDEPENDENT ATTACK-EPISODE SUPPORT")
print("=" * 150)

print(
    cell26_attack_episode_support
    .to_string(index=False)
)


# ==============================================================================================================
# 16. ATTACK-TYPE TEMPORAL EVALUATION FEASIBILITY
#
# IMPORTANT:
#
# We are NOT creating a random window split here.
#
# We ask whether each attack type has enough independent temporal episodes
# to potentially support:
#
#     train episode(s)
#     validation episode(s)
#     test episode(s)
#
# A minimum of 3 episodes is the absolute structural minimum.
#
# This does NOT automatically mean the class is statistically strong.
# ==============================================================================================================

cell26_attack_episode_support[
    "Minimum_3_Episodes"
] = (
    cell26_attack_episode_support[
        "Episode_Count"
    ] >= 3
)

cell26_attack_episode_support[
    "Multiple_Segments"
] = (
    cell26_attack_episode_support[
        "Segment_Count"
    ] >= 2
)


print("\n")
print("=" * 150)
print("EPISODE-BASED TYPE-EVALUATION FEASIBILITY")
print("=" * 150)

print(
    cell26_attack_episode_support[
        [
            "Attack_Type",
            "Episode_Count",
            "Segment_Count",
            "Attack_Minutes",
            "Minimum_3_Episodes",
            "Multiple_Segments"
        ]
    ]
    .to_string(index=False)
)


# ==============================================================================================================
# 17. VICTIM GENERALIZATION AUDIT
#
# We deliberately do NOT create a fixed 11-class target yet.
#
# Instead we classify future victim identities as:
#
#     SEEN_IN_TRAIN
#     UNSEEN_IN_TRAIN
#
# This tells us whether a closed-set victim classifier would be meaningful.
# ==============================================================================================================

train_attack_metadata = (
    timeline.loc[
        (
            timeline["Final_Split"]
            == "TRAIN"
        )
        &
        (
            timeline["Attack_State"]
            == 1
        )
    ]
)

train_victims = set(
    train_attack_metadata[
        "Dominant_Victim_IP"
    ]
    .dropna()
    .astype(str)
)


victim_generalization_records = []


for horizon in FORECAST_HORIZONS:

    victim_col = (
        f"Future_Victim_H{horizon}"
    )

    attack_col = (
        f"Future_Attack_H{horizon}"
    )

    origins = (
        timeline.loc[
            cell26_valid_origins[
                "Timeline_Index"
            ].values
        ]
        .copy()
    )

    origins = origins.loc[
        origins[
            attack_col
        ].notna()
    ]

    future_attacks = origins.loc[
        origins[
            attack_col
        ] == 1
    ]

    for split in [
        "TRAIN",
        "VALIDATION",
        "TEST"
    ]:

        part = (
            future_attacks.loc[
                future_attacks[
                    "Final_Split"
                ] == split
            ]
        )

        victims = (
            part[
                victim_col
            ]
            .dropna()
            .astype(str)
        )

        seen = int(
            victims.isin(
                train_victims
            ).sum()
        )

        unseen = int(
            (~victims.isin(
                train_victims
            )).sum()
        )

        victim_generalization_records.append(
            {
                "Horizon":
                    horizon,

                "Split":
                    split,

                "Future_Attack_Samples":
                    len(part),

                "Victim_Label_Available":
                    len(victims),

                "Seen_Train_Victim":
                    seen,

                "Unseen_Train_Victim":
                    unseen,

                "Seen_Percent":
                    (
                        100
                        * seen
                        / len(victims)
                        if len(victims)
                        else 0
                    )
            }
        )


cell26_victim_generalization = (
    pd.DataFrame(
        victim_generalization_records
    )
)


print("\n")
print("=" * 150)
print("FUTURE VICTIM GENERALIZATION")
print("=" * 150)

print(
    cell26_victim_generalization
    .to_string(
        index=False,
        formatters={
            "Seen_Percent":
                "{:.2f}".format
        }
    )
)


# ==============================================================================================================
# 18. DEFINE LEAD-TIME / WARNING-HORIZON SEMANTICS
# ==============================================================================================================

LEAD_TIME_HORIZONS = (
    FORECAST_HORIZONS.copy()
)

LEAD_TIME_DEFINITION = (
    "The system produces future attack probabilities at "
    "+1, +5, +10, +15 and +30 minutes. "
    "After thresholds are selected using VALIDATION data only, "
    "the earliest horizon satisfying the warning rule is reported "
    "as the earliest positive forecast horizon. "
    "It is not treated as an independently regressed continuous time-to-attack."
)


print("\n")
print("=" * 150)
print("WARNING-HORIZON DEFINITION")
print("=" * 150)

print(
    LEAD_TIME_DEFINITION
)


# ==============================================================================================================
# 19. LOCK TARGET DESIGN
# ==============================================================================================================

CELL26_TARGET_DESIGN = {

    "history_length_minutes":
        HISTORY_LENGTH,

    "forecast_horizons_minutes":
        FORECAST_HORIZONS,

    "primary_binary_task":
        "future Attack_State at each horizon",

    "attack_type_classes":
        ALL_ATTACK_TYPES,

    "number_of_attack_types":
        len(ALL_ATTACK_TYPES),

    "attack_type_target":
        "future attack type at each horizon",

    "attack_type_primary_split_warning":
        (
            "The Cell 22 chronological split must not be "
            "assumed to support ordinary closed-set multiclass "
            "evaluation when future classes are absent from TRAIN."
        ),

    "attack_type_secondary_protocol":
        (
            "Episode/block-aware temporal evaluation must be "
            "designed after inspecting independent episode support. "
            "Random overlapping-window splitting is prohibited."
        ),

    "victim_target":
        (
            "Exact closed-set victim-IP classification is not "
            "locked. Victim/destination-risk formulation remains "
            "to be chosen from generalization evidence."
        ),

    "warning_horizon":
        LEAD_TIME_DEFINITION,

    "primary_train_segments":
        sorted(TRAIN_SEGMENTS),

    "primary_validation_segments":
        sorted(VALIDATION_SEGMENTS),

    "primary_test_segments":
        sorted(TEST_SEGMENTS)
}


# ==============================================================================================================
# 20. SAVE CANONICAL TARGET TABLE
#
# This table contains target metadata only.
# It does NOT scale features.
# It does NOT train models.
# ==============================================================================================================

target_columns = [
    "Minute",
    "Segment_ID",
    "Final_Split",
    "Attack_State",
    "Current_State_Label",
    "Current_State_ID"
]


for horizon in FORECAST_HORIZONS:

    target_columns.extend(
        [
            f"Future_Attack_H{horizon}",
            f"Future_State_Label_H{horizon}",
            f"Future_State_ID_H{horizon}",
            f"Future_Victim_H{horizon}"
        ]
    )


cell26_target_table = (
    timeline[
        target_columns
    ]
    .copy()
)


# ==============================================================================================================
# 21. FINAL SUMMARY
# ==============================================================================================================

CELL26_COMPLETE = True


print("\n")
print("=" * 150)
print("CELL 26 — FINAL TARGET DESIGN SUMMARY")
print("=" * 150)

print(
    f"""
PRIMARY INPUT
-------------
Past {HISTORY_LENGTH}-minute network sequence.

PRIMARY FORECAST HORIZONS
-------------------------
{FORECAST_HORIZONS}

OUTPUT 1 — ATTACK PRESENCE
--------------------------
Binary future Attack_State at every horizon.

This remains evaluated using the LOCKED Cell 22
chronological TRAIN / VALIDATION / TEST split.


OUTPUT 2 — ATTACK TYPE
----------------------
All {len(ALL_ATTACK_TYPES)} observed CICIDS2017 attack types are retained.

No top-5 filtering.
No frequency-based class deletion.
No class has been selected using model performance.

Future attack type has now been constructed at every horizon.

However, the main chronological split is NOT automatically
treated as a valid closed-set 14-class evaluation protocol.

Independent attack episodes have therefore been reconstructed.
Their support must determine the secondary type-evaluation protocol.


OUTPUT 3 — VICTIM / DESTINATION
-------------------------------
Future destination metadata has been reconstructed.

Exact fixed-class IP classification has NOT been locked because
many future victims may be unseen during TRAIN.

The next design stage may use a destination-risk / candidate-ranking
formulation rather than memorizing IP identities.


OUTPUT 4 — WARNING HORIZON
--------------------------
The model will produce predictions at:

    +1
    +5
    +10
    +15
    +30 minutes

Thresholds will later be selected using VALIDATION only.

The earliest horizon satisfying the warning rule will be reported
as the earliest positive forecast horizon.


MODELS PLANNED
--------------
Baseline:
    Logistic Regression
    Random Forest

Temporal:
    LSTM
    Neural ODE
    LSTM + Neural ODE


IMPORTANT
---------
NO model has been trained.
NO scaler has been fitted.
NO oversampling has been performed.
NO TEST threshold has been selected.
NO attack class has been dropped.
NO random overlapping-window split has been created.
The Cell 22 chronological split has NOT been modified.
"""
)

print("=" * 150)
print("CELL 26 COMPLETE")
print("=" * 150)

print(
    "\nNEXT:\n"
    "Inspect the attack-episode support and primary-split feasibility tables.\n"
    "Then Cell 27 will construct the actual model-ready datasets/protocols "
    "without leakage."
)

CELL 26 — FINAL MULTI-OUTPUT TARGET CONSTRUCTION + ATTACK-TYPE TEMPORAL FEASIBILITY AUDIT

✓ Required Cell 25C objects found.
Timeline rows : 2,454
Segments      : 10

Forecasting design:
History length : 15 minutes
Horizons       : [1, 5, 10, 15, 30]

Primary chronological split:
TRAIN      : [0, 1, 2, 3, 4]
VALIDATION : [5]
TEST       : [6, 7, 8, 9]


LOCKED ATTACK-TYPE VOCABULARY
Observed attack classes : 14
 0 -> Bot
 1 -> DDoS
 2 -> DoS GoldenEye
 3 -> DoS Hulk
 4 -> DoS Slowhttptest
 5 -> DoS slowloris
 6 -> FTP-Patator
 7 -> Heartbleed
 8 -> Infiltration
 9 -> PortScan
10 -> SSH-Patator
11 -> Web Attack - Brute Force
12 -> Web Attack - Sql Injection
13 -> Web Attack - XSS

✓ All 14 observed attack types retained.


MULTICLASS TARGET VALIDITY
Attack-positive minutes : 520
Multi-type minutes      : 0
✓ Exactly one attack class can represent each attack-positive minute.


FUTURE TARGET ALIGNMENT VALIDATION
 Horizon  Valid_Target_Rows  Invalid_Time_Alignment
       1               2

In [35]:
# ==============================================================================================================
# CELL 27 — FINAL EXISTING DATASET LOCK
# ==============================================================================================================

import numpy as np

print("=" * 130)
print("CELL 27 — FINAL EXISTING DATASET LOCK")
print("=" * 130)

HORIZONS = [1, 5, 10, 15, 30]

# --------------------------------------------------------------------------------------------------------------
# Expected validated sample counts
# --------------------------------------------------------------------------------------------------------------

EXPECTED = {
    1:  {"TRAIN": 1151, "VALIDATION": 243, "TEST": 910},
    5:  {"TRAIN": 1131, "VALIDATION": 239, "TEST": 894},
    10: {"TRAIN": 1106, "VALIDATION": 234, "TEST": 874},
    15: {"TRAIN": 1081, "VALIDATION": 229, "TEST": 854},
    30: {"TRAIN": 1006, "VALIDATION": 214, "TEST": 794},
}

# --------------------------------------------------------------------------------------------------------------
# Use the ALREADY-CREATED horizon-specific arrays.
#
# IMPORTANT:
# X_train_1, X_train_5, etc. are the existing Cell 24 arrays.
# Nothing is recreated here.
# --------------------------------------------------------------------------------------------------------------

FINAL_MODEL_DATASETS = {}

for h in HORIZONS:

    required_names = [
        f"X_train_{h}",
        f"X_val_{h}",
        f"X_test_{h}",
        f"y_train_{h}",
        f"y_val_{h}",
        f"y_test_{h}",
    ]

    missing = [
        name for name in required_names
        if name not in globals()
    ]

    if missing:
        raise RuntimeError(
            f"H={h}: missing existing objects:\n"
            + "\n".join(f"  - {name}" for name in missing)
        )

    Xtr = globals()[f"X_train_{h}"]
    Xva = globals()[f"X_val_{h}"]
    Xte = globals()[f"X_test_{h}"]

    ytr = globals()[f"y_train_{h}"]
    yva = globals()[f"y_val_{h}"]
    yte = globals()[f"y_test_{h}"]

    # ------------------------------------------------------------------------------------------
    # Integrity checks
    # ------------------------------------------------------------------------------------------

    for split, X, y in [
        ("TRAIN", Xtr, ytr),
        ("VALIDATION", Xva, yva),
        ("TEST", Xte, yte),
    ]:

        if not isinstance(X, np.ndarray):
            raise RuntimeError(
                f"H={h} {split}: X is not a NumPy array."
            )

        if not isinstance(y, np.ndarray):
            raise RuntimeError(
                f"H={h} {split}: y is not a NumPy array."
            )

        if X.ndim != 3:
            raise RuntimeError(
                f"H={h} {split}: expected 3D X, got {X.shape}"
            )

        if X.shape[1:] != (15, 125):
            raise RuntimeError(
                f"H={h} {split}: expected (*, 15, 125), got {X.shape}"
            )

        if len(X) != len(y):
            raise RuntimeError(
                f"H={h} {split}: X/y mismatch: "
                f"{len(X)} vs {len(y)}"
            )

        expected_n = EXPECTED[h][split]

        if len(X) != expected_n:
            raise RuntimeError(
                f"H={h} {split}: expected {expected_n} samples, "
                f"found {len(X)}."
            )

        if np.isnan(X).any():
            raise RuntimeError(
                f"H={h} {split}: NaN detected."
            )

        if np.isinf(X).any():
            raise RuntimeError(
                f"H={h} {split}: Inf detected."
            )

        unique_y = set(np.unique(y).tolist())

        if not unique_y.issubset({0, 1}):
            raise RuntimeError(
                f"H={h} {split}: invalid binary labels {unique_y}"
            )

    # ------------------------------------------------------------------------------------------
    # Lock references
    # ------------------------------------------------------------------------------------------

    FINAL_MODEL_DATASETS[h] = {

        "TRAIN": {
            "X": Xtr,
            "y_attack": ytr,
        },

        "VALIDATION": {
            "X": Xva,
            "y_attack": yva,
        },

        "TEST": {
            "X": Xte,
            "y_attack": yte,
        }
    }


# --------------------------------------------------------------------------------------------------------------
# Summary
# --------------------------------------------------------------------------------------------------------------

print("\nFINAL LOCKED DATASETS")
print("-" * 130)

for h in HORIZONS:

    print(f"\nHORIZON +{h} MIN")

    for split in ["TRAIN", "VALIDATION", "TEST"]:

        X = FINAL_MODEL_DATASETS[h][split]["X"]
        y = FINAL_MODEL_DATASETS[h][split]["y_attack"]

        positives = int(y.sum())
        negatives = len(y) - positives

        print(
            f"  {split:<10} "
            f"X={str(X.shape):<20} "
            f"Attack={positives:<5} "
            f"Benign={negatives:<5}"
        )


# --------------------------------------------------------------------------------------------------------------
# Lock
# --------------------------------------------------------------------------------------------------------------

FINAL_MODEL_DATA_LOCKED = True
CELL27_COMPLETE = True


print("\n" + "=" * 130)
print("CELL 27 COMPLETE")
print("=" * 130)

print("""
✓ All five existing horizon datasets found.
✓ Existing Cell 24 arrays reused.
✓ No dataset rebuilt.
✓ No features regenerated.
✓ No sequences regenerated.
✓ No scaling performed.
✓ No split changed.
✓ No model trained.

CANONICAL MODEL INPUT:

    FINAL_MODEL_DATASETS[horizon][split]["X"]
    FINAL_MODEL_DATASETS[horizon][split]["y_attack"]

Example:

    FINAL_MODEL_DATASETS[10]["TRAIN"]["X"]
    FINAL_MODEL_DATASETS[10]["TRAIN"]["y_attack"]

From this point onward:

    DO NOT rebuild datasets.
    DO NOT recreate sequences.
    DO NOT fit another scaler.

NEXT = BASELINE MODEL TRAINING.
""")

print("=" * 130)

CELL 27 — FINAL EXISTING DATASET LOCK

FINAL LOCKED DATASETS
----------------------------------------------------------------------------------------------------------------------------------

HORIZON +1 MIN
  TRAIN      X=(1151, 15, 125)      Attack=140   Benign=1011 
  VALIDATION X=(243, 15, 125)       Attack=73    Benign=170  
  TEST       X=(910, 15, 125)       Attack=305   Benign=605  

HORIZON +5 MIN
  TRAIN      X=(1131, 15, 125)      Attack=140   Benign=991  
  VALIDATION X=(239, 15, 125)       Attack=73    Benign=166  
  TEST       X=(894, 15, 125)       Attack=303   Benign=591  

HORIZON +10 MIN
  TRAIN      X=(1106, 15, 125)      Attack=140   Benign=966  
  VALIDATION X=(234, 15, 125)       Attack=72    Benign=162  
  TEST       X=(874, 15, 125)       Attack=299   Benign=575  

HORIZON +15 MIN
  TRAIN      X=(1081, 15, 125)      Attack=136   Benign=945  
  VALIDATION X=(229, 15, 125)       Attack=72    Benign=157  
  TEST       X=(854, 15, 125)       Attack=294   Benign=560 